In [1]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 0: setup and frozen observation protocol
#
# Scientific question
# ---------------------------------------------------------------
# Can a SINGLE continuous multivariate time series be sliced
# into multi-resolution endpoint families such that the
# extrapolated persistent generator recovers the instantaneous
# network active at each slice start?
#
# Physical system:
#   EXACTLY the frozen Stage-4 N=8 benchmark.
#
# What changes:
#   observation protocol only.
#
# Old:
#   many independent initial conditions
#
# New:
#   one continuous trajectory
#   -> many temporally indexed slice starts
#   -> four endpoint horizons per start
# ================================================================

from pathlib import Path
from itertools import combinations, combinations_with_replacement
from collections import Counter, defaultdict

import importlib
import numpy as np
import pandas as pd
import sympy as sp


# ================================================================
# Frozen physical-system reproducibility
# ================================================================

SEED = 20260811

weight_rng = np.random.default_rng(
    SEED
)

trajectory_rng = np.random.default_rng(
    SEED + 202
)


# ================================================================
# Output directory
# ================================================================

OUTPUT_DIR = Path(
    "./stage6_n8_timeseries_slicing"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Physical-system dimensions
# ================================================================

N = 8

nodes = tuple(
    range(1, N + 1)
)

x_symbols = sp.symbols(
    f"x1:{N + 1}"
)


# ================================================================
# Frozen nonlinear interaction
#
# phi(x) = x + 0.5 x^2
# ================================================================

LAMBDA = 0.5
LAMBDA_EXACT = sp.Rational(1, 2)


# ================================================================
# ONE-TIME-SERIES observation protocol
#
# Fixed physical snapshot duration:
#
#       tau_snapshot = 0.06
#
# Dense trajectory observation:
#
#       dt_obs = 0.0005
#
# Multi-resolution slice endpoints:
#
#       eps = [0.005, 0.0075, 0.010, 0.015]
#
# All eps values are exact integer multiples of dt_obs.
# ================================================================

TAU_SNAPSHOT = 0.060

DT_OBS = 0.0005

EPS_SLICE = np.array(
    [
        0.0050,
        0.0075,
        0.0100,
        0.0150,
    ],
    dtype=float,
)

EPS_MAX = float(
    EPS_SLICE.max()
)


# ================================================================
# Repeated temporal protocol
#
# One continuous trajectory:
#
#   G1 -> G2 -> ... -> G6
#      -> G1 -> G2 -> ...
#
# Six cycles are generated initially.
#
# Block roles:
#
#   cycles 0-3 : fitting
#   cycle 4    : validation
#   cycle 5    : external test
# ================================================================

N_CYCLES = 6

FIT_CYCLES = (
    0, 1, 2, 3
)

VALIDATION_CYCLES = (
    4,
)

TEST_CYCLES = (
    5,
)


# ================================================================
# Slice-start spacing
#
# This controls how densely starting states are extracted from
# the ONE trajectory.
#
# It does NOT define independent experiments.
# ================================================================

SLICE_STRIDE = 0.0015


# ================================================================
# Exact grid compatibility
# ================================================================

def integer_steps(
    duration,
    dt=DT_OBS,
):
    steps = int(
        round(
            float(duration)
            /
            float(dt)
        )
    )

    if not np.isclose(
        steps * dt,
        duration,
        rtol=0.0,
        atol=1e-14,
    ):
        raise ValueError(
            f"{duration} is not an integer "
            f"multiple of dt={dt}."
        )

    return steps


SNAPSHOT_STEPS = integer_steps(
    TAU_SNAPSHOT
)

EPS_STEPS = np.array(
    [
        integer_steps(epsilon)
        for epsilon
        in EPS_SLICE
    ],
    dtype=int,
)

SLICE_STRIDE_STEPS = integer_steps(
    SLICE_STRIDE
)

CYCLE_STEPS = (
    6
    *
    SNAPSHOT_STEPS
)

TOTAL_STEPS = (
    N_CYCLES
    *
    CYCLE_STEPS
)

TOTAL_TIME = (
    TOTAL_STEPS
    *
    DT_OBS
)


# ================================================================
# Load frozen TSC v3.6 core
# ================================================================

try:
    import TSC_AGLASSO_v3_6 as tsc_core

except ModuleNotFoundError:
    import TSC_AGLASSO as tsc_core


tsc_core = importlib.reload(
    tsc_core
)

assert (
    getattr(
        tsc_core,
        "_IMPLEMENTATION_VERSION",
        None,
    )
    ==
    "v3.6"
)


# ================================================================
# Basic configuration audit
# ================================================================

print("=" * 70)
print("N=8 ONE-TIME-SERIES SLICING EXPERIMENT")
print("=" * 70)

print(
    f"N                           : "
    f"{N}"
)

print(
    f"snapshot duration           : "
    f"{TAU_SNAPSHOT}"
)

print(
    f"dense observation dt        : "
    f"{DT_OBS}"
)

print(
    f"steps / snapshot            : "
    f"{SNAPSHOT_STEPS}"
)

print(
    f"slice epsilon values        : "
    f"{EPS_SLICE}"
)

print(
    f"epsilon steps               : "
    f"{EPS_STEPS}"
)

print(
    f"slice-start stride          : "
    f"{SLICE_STRIDE}"
)

print(
    f"slice stride steps          : "
    f"{SLICE_STRIDE_STEPS}"
)

print(
    f"cycles                      : "
    f"{N_CYCLES}"
)

print(
    f"total trajectory duration   : "
    f"{TOTAL_TIME}"
)

print(
    f"fit cycles                  : "
    f"{FIT_CYCLES}"
)

print(
    f"validation cycles           : "
    f"{VALIDATION_CYCLES}"
)

print(
    f"test cycles                 : "
    f"{TEST_CYCLES}"
)

print(
    f"TSC implementation          : "
    f"{tsc_core._IMPLEMENTATION_VERSION}"
)


# ================================================================
# Expected nominal number of eligible interior starts
#
# No start is allowed if its largest epsilon crosses a snapshot
# boundary.
# ================================================================

LAST_LOCAL_START_STEP = (
    SNAPSHOT_STEPS
    -
    int(EPS_STEPS.max())
)

START_OFFSETS = np.arange(
    0,
    LAST_LOCAL_START_STEP + 1,
    SLICE_STRIDE_STEPS,
    dtype=int,
)

STARTS_PER_SNAPSHOT_PER_CYCLE = len(
    START_OFFSETS
)

print()
print("Nominal slicing capacity")
print("-" * 70)

print(
    f"eligible starts / snapshot / cycle : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE}"
)

print(
    f"eligible starts / snapshot total   : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * N_CYCLES}"
)

print(
    f"fit starts / snapshot              : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(FIT_CYCLES)}"
)

print(
    f"validation starts / snapshot       : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(VALIDATION_CYCLES)}"
)

print(
    f"test starts / snapshot             : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(TEST_CYCLES)}"
)


# ================================================================
# Hard guards
# ================================================================

assert EPS_MAX < TAU_SNAPSHOT

assert (
    set(FIT_CYCLES)
    .isdisjoint(
        VALIDATION_CYCLES
    )
)

assert (
    set(FIT_CYCLES)
    .isdisjoint(
        TEST_CYCLES
    )
)

assert (
    set(VALIDATION_CYCLES)
    .isdisjoint(
        TEST_CYCLES
    )
)

assert (
    set(FIT_CYCLES)
    |
    set(VALIDATION_CYCLES)
    |
    set(TEST_CYCLES)
) == set(
    range(N_CYCLES)
)


print()
print("=" * 70)
print("Cell 0 PASSED.")
print("=" * 70)

N=8 ONE-TIME-SERIES SLICING EXPERIMENT
N                           : 8
snapshot duration           : 0.06
dense observation dt        : 0.0005
steps / snapshot            : 120
slice epsilon values        : [0.005  0.0075 0.01   0.015 ]
epsilon steps               : [10 15 20 30]
slice-start stride          : 0.0015
slice stride steps          : 3
cycles                      : 6
total trajectory duration   : 2.16
fit cycles                  : (0, 1, 2, 3)
validation cycles           : (4,)
test cycles                 : (5,)
TSC implementation          : v3.6

Nominal slicing capacity
----------------------------------------------------------------------
eligible starts / snapshot / cycle : 31
eligible starts / snapshot total   : 186
fit starts / snapshot              : 124
validation starts / snapshot       : 31
test starts / snapshot             : 31

Cell 0 PASSED.


In [2]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 1: exact frozen Stage-4 physical system
#
# IMPORTANT
# ---------------------------------------------------------------
# This cell reproduces the physical system from the original
# Stage-4 N=8 benchmark.
#
# Nothing in the microscopic dynamics is changed:
#
#   - same 12 microscopic edges
#   - same heterogeneous edge weights
#   - same six temporal snapshots
#   - same nonlinear pairwise interaction
#   - same two persistent native triads
#
# Only the observation protocol will change in later cells.
# ================================================================


# ================================================================
# Microscopic pairwise graph
# ================================================================

microscopic_edges = (
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 7),
    (7, 8),
    (8, 1),
    (1, 3),
    (2, 4),
    (3, 5),
    (5, 7),
)

assert len(microscopic_edges) == 12
assert len(set(microscopic_edges)) == 12


# ================================================================
# Frozen heterogeneous edge weights
#
# EXACT Stage-4 construction:
#
#   integer weight ~ Uniform{800,...,1200}
#   physical weight = integer / 1000
#
# weight_rng was initialized in Cell 0 with:
#
#       SEED = 20260811
#
# ================================================================

edge_weight_integer = {
    edge:
        int(
            weight_rng.integers(
                800,
                1201,
            )
        )

    for edge
    in microscopic_edges
}

edge_weight = {
    edge:
        value / 1000.0

    for edge, value
    in edge_weight_integer.items()
}

edge_weight_exact = {
    edge:
        sp.Rational(
            value,
            1000,
        )

    for edge, value
    in edge_weight_integer.items()
}


# ================================================================
# Frozen six-snapshot temporal protocol
#
# Every microscopic edge appears exactly twice over one complete
# six-snapshot cycle.
# ================================================================

snapshots = (

    # G1
    (
        (1, 2),
        (3, 4),
        (5, 6),
        (7, 8),
    ),

    # G2
    (
        (2, 3),
        (4, 5),
        (6, 7),
        (8, 1),
    ),

    # G3
    (
        (1, 3),
        (2, 4),
        (3, 5),
        (5, 7),
    ),

    # G4
    (
        (1, 2),
        (4, 5),
        (7, 8),
        (3, 5),
    ),

    # G5
    (
        (2, 3),
        (5, 6),
        (8, 1),
        (5, 7),
    ),

    # G6
    (
        (3, 4),
        (6, 7),
        (1, 3),
        (2, 4),
    ),
)


# ================================================================
# Snapshot consistency audit
# ================================================================

snapshot_edge_counts = Counter(
    edge

    for snapshot
    in snapshots

    for edge
    in snapshot
)

assert len(snapshots) == 6

assert all(
    len(snapshot) == 4
    for snapshot
    in snapshots
)

assert (
    set(snapshot_edge_counts)
    ==
    set(microscopic_edges)
)

assert all(
    snapshot_edge_counts[edge] == 2
    for edge
    in microscopic_edges
)


# ================================================================
# Nonlinear pairwise interaction
#
#       phi(x) = x + 0.5 x^2
# ================================================================

def phi_symbolic(z):
    return (
        z
        +
        LAMBDA_EXACT * z**2
    )


def phi_numeric(z):
    return (
        z
        +
        LAMBDA * z**2
    )


# ================================================================
# Pairwise microscopic vector field
#
# For edge (i,j):
#
#   F_i = w_ij [phi(x_j) - phi(x_i)]
#   F_j = -F_i
#
# Hence every edge field conserves:
#
#       sum_i x_i
# ================================================================

def edge_field_symbolic(
    edge,
):

    i, j = edge

    if edge not in edge_weight_exact:
        raise KeyError(
            f"Unknown microscopic edge: "
            f"{edge}"
        )

    w = edge_weight_exact[
        edge
    ]

    field = sp.zeros(
        N,
        1,
    )

    interaction = (
        w
        *
        (
            phi_symbolic(
                x_symbols[j - 1]
            )
            -
            phi_symbolic(
                x_symbols[i - 1]
            )
        )
    )

    field[
        i - 1
    ] += interaction

    field[
        j - 1
    ] -= interaction

    return field


def edge_field_numeric(
    X,
    edge,
):

    i, j = edge

    if edge not in edge_weight:
        raise KeyError(
            f"Unknown microscopic edge: "
            f"{edge}"
        )

    w = edge_weight[
        edge
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    interaction = (
        w
        *
        (
            phi_numeric(
                X[..., j - 1]
            )
            -
            phi_numeric(
                X[..., i - 1]
            )
        )
    )

    F[
        ..., i - 1
    ] += interaction

    F[
        ..., j - 1
    ] -= interaction

    return F


# ================================================================
# Persistent native triadic interactions
#
# Frozen Stage-4 definition:
#
# For triad h={i,j,k},
#
#   T_i =
#       g x_j x_k
#       - g/2 x_i x_j
#       - g/2 x_i x_k
#
# with cyclic permutations for j and k.
#
# These interactions are present in EVERY snapshot.
# ================================================================

NATIVE_TRIADS = (
    (1, 2, 3),
    (2, 5, 8),
)

NATIVE_G = 0.02

NATIVE_G_EXACT = sp.Rational(
    1,
    50,
)


def triad_field_symbolic(
    triad,
    g=NATIVE_G_EXACT,
):

    i, j, k = triad

    xi = x_symbols[
        i - 1
    ]

    xj = x_symbols[
        j - 1
    ]

    xk = x_symbols[
        k - 1
    ]

    field = sp.zeros(
        N,
        1,
    )

    field[
        i - 1
    ] += (
        g * xj * xk
        -
        g / 2 * xi * xj
        -
        g / 2 * xi * xk
    )

    field[
        j - 1
    ] += (
        g * xi * xk
        -
        g / 2 * xj * xi
        -
        g / 2 * xj * xk
    )

    field[
        k - 1
    ] += (
        g * xi * xj
        -
        g / 2 * xk * xi
        -
        g / 2 * xk * xj
    )

    return field


def triad_field_numeric(
    X,
    triad,
    g=NATIVE_G,
):

    i, j, k = triad

    xi = X[
        ..., i - 1
    ]

    xj = X[
        ..., j - 1
    ]

    xk = X[
        ..., k - 1
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    F[
        ..., i - 1
    ] += (
        g * xj * xk
        -
        0.5 * g * xi * xj
        -
        0.5 * g * xi * xk
    )

    F[
        ..., j - 1
    ] += (
        g * xi * xk
        -
        0.5 * g * xj * xi
        -
        0.5 * g * xj * xk
    )

    F[
        ..., k - 1
    ] += (
        g * xi * xj
        -
        0.5 * g * xk * xi
        -
        0.5 * g * xk * xj
    )

    return F


# ================================================================
# Precompute symbolic microscopic fields
# ================================================================

edge_fields_symbolic = {
    edge:
        edge_field_symbolic(
            edge
        )

    for edge
    in microscopic_edges
}


native_triad_fields_symbolic = {
    triad:
        triad_field_symbolic(
            triad
        )

    for triad
    in NATIVE_TRIADS
}


# ================================================================
# Persistent native contribution
# ================================================================

native_total_symbolic = sum(
    native_triad_fields_symbolic.values(),
    sp.zeros(
        N,
        1,
    ),
)


def native_total_numeric(
    X,
):

    F = np.zeros_like(
        X,
        dtype=float,
    )

    for triad in NATIVE_TRIADS:

        F += triad_field_numeric(
            X,
            triad,
        )

    return F


# ================================================================
# Full symbolic snapshot generators
#
# Each G_m contains:
#
#   4 active pairwise edges
#   +
#   2 persistent native triads
# ================================================================

snapshot_fields_symbolic = []


for snapshot in snapshots:

    pairwise_part = sum(
        (
            edge_fields_symbolic[
                edge
            ]

            for edge
            in snapshot
        ),
        sp.zeros(
            N,
            1,
        ),
    )

    full_field = (
        pairwise_part
        +
        native_total_symbolic
    )

    snapshot_fields_symbolic.append(
        sp.Matrix(
            [
                sp.expand(
                    component
                )

                for component
                in full_field
            ]
        )
    )


# ================================================================
# Numerical snapshot generator
# ================================================================

def snapshot_field_numeric(
    X,
    snapshot_index,
):
    """
    Evaluate instantaneous generator G_m.

    Parameters
    ----------
    X
        State or batch with shape (..., N).

    snapshot_index
        Python index 0,...,5.
    """

    snapshot = snapshots[
        snapshot_index
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    # Temporally active pairwise sector
    for edge in snapshot:

        F += edge_field_numeric(
            X,
            edge,
        )

    # Persistent native HOI sector
    F += native_total_numeric(
        X
    )

    return F


# ================================================================
# Physical consistency audit
#
# Use a separate RNG so this diagnostic does NOT consume the
# trajectory RNG reserved for Cell 2.
# ================================================================

audit_rng = np.random.default_rng(
    SEED + 303
)

X_check = audit_rng.uniform(
    -0.5,
    0.5,
    size=(
        32,
        N,
    ),
)


# Every microscopic symbolic field conserves total state.
for edge, field in (
    edge_fields_symbolic.items()
):

    assert (
        sp.simplify(
            sum(field)
        )
        ==
        0
    )


for triad, field in (
    native_triad_fields_symbolic.items()
):

    assert (
        sp.simplify(
            sum(field)
        )
        ==
        0
    )


# Every full snapshot generator also conserves total state.
max_conservation_error = 0.0


for snapshot_index in range(
    len(snapshots)
):

    F_check = (
        snapshot_field_numeric(
            X_check,
            snapshot_index,
        )
    )

    error = float(
        np.max(
            np.abs(
                F_check.sum(
                    axis=1
                )
            )
        )
    )

    max_conservation_error = max(
        max_conservation_error,
        error,
    )


assert (
    max_conservation_error
    <
    1e-13
)


# ================================================================
# Exact instantaneous structural-support oracle
#
# Used ONLY later for synthetic scoring.
#
# For a monomial appearing in output i:
#
#       support =
#           {output i}
#           union
#           {variables in monomial}
#
# This is exactly the TSC structural-support convention.
# ================================================================

def symbolic_structural_supports(
    field,
):

    supports = set()

    for output_index, expr in enumerate(
        field
    ):

        poly = sp.Poly(
            sp.expand(
                expr
            ),
            *x_symbols,
        )

        for powers, coefficient in (
            poly.terms()
        ):

            if (
                sp.simplify(
                    coefficient
                )
                ==
                0
            ):
                continue

            variables = {
                j + 1

                for j, power
                in enumerate(
                    powers
                )

                if power > 0
            }

            support = tuple(
                sorted(
                    variables
                    |
                    {
                        output_index
                        + 1
                    }
                )
            )

            supports.add(
                support
            )

    return tuple(
        sorted(
            supports,
            key=lambda s: (
                len(s),
                s,
            ),
        )
    )


SNAPSHOT_ORACLE_SUPPORTS = {
    snapshot_index:
        symbolic_structural_supports(
            snapshot_fields_symbolic[
                snapshot_index
            ]
        )

    for snapshot_index
    in range(
        len(snapshots)
    )
}


SNAPSHOT_ORACLE_COUNTS = {
    snapshot_index:
        Counter(
            len(support)

            for support
            in supports
        )

    for snapshot_index, supports
    in SNAPSHOT_ORACLE_SUPPORTS.items()
}


# ================================================================
# Report
# ================================================================

print("=" * 70)
print("FROZEN STAGE-4 PHYSICAL SYSTEM")
print("=" * 70)

print(
    f"Nodes                       : "
    f"{N}"
)

print(
    f"Microscopic pair edges      : "
    f"{len(microscopic_edges)}"
)

print(
    f"Temporal snapshots          : "
    f"{len(snapshots)}"
)

print(
    f"Active pair edges / snapshot: "
    f"{len(snapshots[0])}"
)

print(
    f"Persistent native triads    : "
    f"{NATIVE_TRIADS}"
)

print(
    f"Native triad strength       : "
    f"{NATIVE_G}"
)

print(
    f"Max conservation error      : "
    f"{max_conservation_error:.3e}"
)


print()
print("Frozen edge weights")
print("-" * 70)

for edge in microscopic_edges:

    print(
        f"{edge}: "
        f"{edge_weight[edge]:.3f}"
    )


print()
print("Temporal protocol")
print("-" * 70)

for m, snapshot in enumerate(
    snapshots,
    start=1,
):

    print(
        f"G{m}: "
        f"{snapshot}"
    )


print()
print("Instantaneous support oracle")
print("-" * 70)

for snapshot_index in range(
    len(snapshots)
):

    supports = (
        SNAPSHOT_ORACLE_SUPPORTS[
            snapshot_index
        ]
    )

    counts = (
        SNAPSHOT_ORACLE_COUNTS[
            snapshot_index
        ]
    )

    print(
        f"G{snapshot_index + 1}: "
        f"{len(supports):2d} supports "
        f"| by size = "
        f"{dict(sorted(counts.items()))}"
    )


# Frozen checks
EXPECTED_SNAPSHOT_COUNTS = (
    {1: 8, 2: 9, 3: 2},
    {1: 8, 2: 9, 3: 2},
    {1: 6, 2: 9, 3: 2},
    {1: 7, 2: 9, 3: 2},
    {1: 7, 2: 9, 3: 2},
    {1: 6, 2: 9, 3: 2},
)


for snapshot_index, expected in enumerate(
    EXPECTED_SNAPSHOT_COUNTS
):

    assert (
        dict(
            SNAPSHOT_ORACLE_COUNTS[
                snapshot_index
            ]
        )
        ==
        expected
    )


print()
print("=" * 70)
print("Cell 1 PASSED.")
print("=" * 70)

FROZEN STAGE-4 PHYSICAL SYSTEM
Nodes                       : 8
Microscopic pair edges      : 12
Temporal snapshots          : 6
Active pair edges / snapshot: 4
Persistent native triads    : ((1, 2, 3), (2, 5, 8))
Native triad strength       : 0.02
Max conservation error      : 2.776e-16

Frozen edge weights
----------------------------------------------------------------------
(1, 2): 1.006
(2, 3): 0.911
(3, 4): 1.105
(4, 5): 0.917
(5, 6): 1.177
(6, 7): 1.119
(7, 8): 0.917
(8, 1): 0.944
(1, 3): 1.026
(2, 4): 1.002
(3, 5): 1.067
(5, 7): 1.031

Temporal protocol
----------------------------------------------------------------------
G1: ((1, 2), (3, 4), (5, 6), (7, 8))
G2: ((2, 3), (4, 5), (6, 7), (8, 1))
G3: ((1, 3), (2, 4), (3, 5), (5, 7))
G4: ((1, 2), (4, 5), (7, 8), (3, 5))
G5: ((2, 3), (5, 6), (8, 1), (5, 7))
G6: ((3, 4), (6, 7), (1, 3), (2, 4))

Instantaneous support oracle
----------------------------------------------------------------------
G1: 19 supports | by size = {1: 8, 2: 9

In [3]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 2: generate ONE continuous repeated-snapshot trajectory
#
# Physical protocol:
#
#   G1 -> G2 -> G3 -> G4 -> G5 -> G6
#      -> G1 -> G2 -> ...  (6 cycles total)
#
# IMPORTANT
# ---------------------------------------------------------------
# There is exactly ONE initial condition.
#
# After this cell, every endpoint pair used for inference must be
# READ from this stored trajectory.
#
# We will NOT restart the simulator from individual slice starts.
# ================================================================

import time


# ================================================================
# Reinitialize trajectory RNG explicitly
#
# Same Stage-4 data seed:
#
#       SEED + 202
#
# This guarantees that previous diagnostic random draws cannot
# affect the formal trajectory initial condition.
# ================================================================

TRAJECTORY_SEED = SEED + 202

trajectory_rng = np.random.default_rng(
    TRAJECTORY_SEED
)


# ================================================================
# Single initial condition
#
# Same state range as the original Stage-4 benchmark.
# ================================================================

X_TRAJ_INITIAL = trajectory_rng.uniform(
    -0.5,
    0.5,
    size=N,
)


# ================================================================
# One fixed RK4 step
#
# dt = DT_OBS = 5e-4
#
# This is finer than the original Stage-4 maximum RK4 step
# of 1e-3.
# ================================================================

def rk4_step_snapshot(
    x,
    snapshot_index,
    dt=DT_OBS,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    k1 = snapshot_field_numeric(
        x,
        snapshot_index,
    )

    k2 = snapshot_field_numeric(
        x + 0.5 * dt * k1,
        snapshot_index,
    )

    k3 = snapshot_field_numeric(
        x + 0.5 * dt * k2,
        snapshot_index,
    )

    k4 = snapshot_field_numeric(
        x + dt * k3,
        snapshot_index,
    )

    return (
        x
        +
        (dt / 6.0)
        * (
            k1
            + 2.0 * k2
            + 2.0 * k3
            + k4
        )
    )


# ================================================================
# Allocate the ONE observed trajectory
#
# TRAJECTORY_STATES[k] = X(t_k)
#
# with
#
#       t_k = k * DT_OBS
#
# Shape:
#
#       (TOTAL_STEPS + 1, N)
# ================================================================

TRAJECTORY_TIMES = (
    np.arange(
        TOTAL_STEPS + 1,
        dtype=float,
    )
    *
    DT_OBS
)

TRAJECTORY_STATES = np.empty(
    (
        TOTAL_STEPS + 1,
        N,
    ),
    dtype=float,
)

TRAJECTORY_STATES[0] = (
    X_TRAJ_INITIAL
)


# ================================================================
# Metadata for every integration interval
#
# STEP_SNAPSHOT[k] tells us which generator acts on:
#
#       [t_k, t_{k+1})
#
# STEP_CYCLE[k] tells us which repeated protocol cycle it belongs to.
# ================================================================

STEP_SNAPSHOT = np.empty(
    TOTAL_STEPS,
    dtype=int,
)

STEP_CYCLE = np.empty(
    TOTAL_STEPS,
    dtype=int,
)


# ================================================================
# Generate the continuous trajectory
# ================================================================

tic = time.perf_counter()

for step in range(
    TOTAL_STEPS
):

    cycle_index = (
        step
        //
        CYCLE_STEPS
    )

    step_inside_cycle = (
        step
        %
        CYCLE_STEPS
    )

    snapshot_index = (
        step_inside_cycle
        //
        SNAPSHOT_STEPS
    )

    assert (
        0
        <= snapshot_index
        < 6
    )

    STEP_CYCLE[
        step
    ] = cycle_index

    STEP_SNAPSHOT[
        step
    ] = snapshot_index

    TRAJECTORY_STATES[
        step + 1
    ] = rk4_step_snapshot(
        TRAJECTORY_STATES[
            step
        ],
        snapshot_index,
        DT_OBS,
    )


TRAJECTORY_RUNTIME = (
    time.perf_counter()
    -
    tic
)


# ================================================================
# Fundamental integrity checks
# ================================================================

assert TRAJECTORY_TIMES.shape == (
    TOTAL_STEPS + 1,
)

assert TRAJECTORY_STATES.shape == (
    TOTAL_STEPS + 1,
    N,
)

assert STEP_SNAPSHOT.shape == (
    TOTAL_STEPS,
)

assert STEP_CYCLE.shape == (
    TOTAL_STEPS,
)

assert np.all(
    np.isfinite(
        TRAJECTORY_STATES
    )
)

assert np.isclose(
    TRAJECTORY_TIMES[-1],
    TOTAL_TIME,
)


# ================================================================
# Exact protocol-count audit
#
# Every cycle must contain exactly:
#
#       120 steps of G1
#       ...
#       120 steps of G6
# ================================================================

for cycle_index in range(
    N_CYCLES
):

    cycle_mask = (
        STEP_CYCLE
        ==
        cycle_index
    )

    counts = Counter(
        STEP_SNAPSHOT[
            cycle_mask
        ]
    )

    assert counts == {
        snapshot_index:
            SNAPSHOT_STEPS

        for snapshot_index
        in range(6)
    }


# ================================================================
# Conservation audit
#
# Every microscopic field conserves sum_i x_i, so the entire
# switched trajectory must conserve the same total mass.
# ================================================================

INITIAL_MASS = float(
    X_TRAJ_INITIAL.sum()
)

TRAJECTORY_MASS = (
    TRAJECTORY_STATES.sum(
        axis=1
    )
)

MAX_TRAJECTORY_MASS_ERROR = float(
    np.max(
        np.abs(
            TRAJECTORY_MASS
            -
            INITIAL_MASS
        )
    )
)

assert (
    MAX_TRAJECTORY_MASS_ERROR
    <
    1e-11
)


# ================================================================
# Dynamical-excitation audit
#
# This matters specifically for one-time-series inference.
#
# Because the network is conservative but mixing/dissipative,
# the trajectory may progressively approach consensus.
#
# Define:
#
#   node_spread(t)
#       = RMS deviation of node states around their instantaneous
#         node mean.
#
# Since mass is conserved, the node mean is constant.
# ================================================================

TRAJECTORY_NODE_MEAN = (
    TRAJECTORY_STATES.mean(
        axis=1
    )
)

TRAJECTORY_NODE_SPREAD = np.sqrt(
    np.mean(
        (
            TRAJECTORY_STATES
            -
            TRAJECTORY_NODE_MEAN[
                :, None
            ]
        )
        ** 2,
        axis=1,
    )
)


# ================================================================
# Per-cycle state-space diagnostics
# ================================================================

cycle_rows = []

for cycle_index in range(
    N_CYCLES
):

    start_step = (
        cycle_index
        *
        CYCLE_STEPS
    )

    end_step = (
        (cycle_index + 1)
        *
        CYCLE_STEPS
    )

    X_cycle = (
        TRAJECTORY_STATES[
            start_step:
            end_step + 1
        ]
    )

    spread_cycle = (
        TRAJECTORY_NODE_SPREAD[
            start_step:
            end_step + 1
        ]
    )

    state_excursion = (
        np.sqrt(
            np.mean(
                (
                    X_cycle
                    -
                    X_cycle[0]
                )
                ** 2
            )
        )
    )

    cycle_rows.append({
        "cycle":
            cycle_index,

        "t_start":
            TRAJECTORY_TIMES[
                start_step
            ],

        "t_end":
            TRAJECTORY_TIMES[
                end_step
            ],

        "spread_start":
            spread_cycle[0],

        "spread_end":
            spread_cycle[-1],

        "spread_mean":
            spread_cycle.mean(),

        "state_excursion_rms":
            state_excursion,

        "state_min":
            X_cycle.min(),

        "state_max":
            X_cycle.max(),
    })


TRAJECTORY_CYCLE_AUDIT = pd.DataFrame(
    cycle_rows
)


# ================================================================
# Snapshot-boundary state table
#
# Useful later when we diagnose inference near switching points.
# ================================================================

boundary_rows = []

for cycle_index in range(
    N_CYCLES
):

    for snapshot_index in range(
        6
    ):

        step = (
            cycle_index
            *
            CYCLE_STEPS

            +
            snapshot_index
            *
            SNAPSHOT_STEPS
        )

        boundary_rows.append({
            "cycle":
                cycle_index,

            "snapshot":
                snapshot_index + 1,

            "step":
                step,

            "time":
                TRAJECTORY_TIMES[
                    step
                ],

            "node_spread":
                TRAJECTORY_NODE_SPREAD[
                    step
                ],
        })


SNAPSHOT_BOUNDARY_TABLE = pd.DataFrame(
    boundary_rows
)


# ================================================================
# Report
# ================================================================

print("=" * 70)
print("ONE CONTINUOUS N=8 TRAJECTORY")
print("=" * 70)

print(
    f"trajectory seed             : "
    f"{TRAJECTORY_SEED}"
)

print(
    f"initial state               : "
    f"{np.array2string(
        X_TRAJ_INITIAL,
        precision=6
    )}"
)

print(
    f"initial mass                : "
    f"{INITIAL_MASS:.12f}"
)

print(
    f"time interval               : "
    f"[0, {TOTAL_TIME}]"
)

print(
    f"observation interval        : "
    f"{DT_OBS}"
)

print(
    f"observed states             : "
    f"{len(TRAJECTORY_STATES)}"
)

print(
    f"integration intervals       : "
    f"{TOTAL_STEPS}"
)

print(
    f"runtime                     : "
    f"{TRAJECTORY_RUNTIME:.3f} s"
)

print(
    f"max conservation error      : "
    f"{MAX_TRAJECTORY_MASS_ERROR:.3e}"
)

print(
    f"global state range          : "
    f"[{TRAJECTORY_STATES.min():.6f}, "
    f"{TRAJECTORY_STATES.max():.6f}]"
)

print(
    f"node spread: start -> end   : "
    f"{TRAJECTORY_NODE_SPREAD[0]:.6e} "
    f"-> "
    f"{TRAJECTORY_NODE_SPREAD[-1]:.6e}"
)

print(
    f"spread retention            : "
    f"{TRAJECTORY_NODE_SPREAD[-1] /
       TRAJECTORY_NODE_SPREAD[0]:.6f}"
)


print()
print("Per-cycle excitation audit")
print("-" * 70)

display(
    TRAJECTORY_CYCLE_AUDIT.style.format({
        "t_start":
            "{:.3f}",

        "t_end":
            "{:.3f}",

        "spread_start":
            "{:.6e}",

        "spread_end":
            "{:.6e}",

        "spread_mean":
            "{:.6e}",

        "state_excursion_rms":
            "{:.6e}",

        "state_min":
            "{:.6f}",

        "state_max":
            "{:.6f}",
    })
)


# ================================================================
# Save the ONE observed time series
#
# Ground-truth snapshot labels are saved separately as synthetic
# oracle metadata. They will not be supplied to the learner in
# the later unlabelled experiment.
# ================================================================

TRAJECTORY_FILE = (
    OUTPUT_DIR
    /
    "n8_single_continuous_trajectory.npz"
)

np.savez_compressed(
    TRAJECTORY_FILE,

    times=
        TRAJECTORY_TIMES,

    states=
        TRAJECTORY_STATES,

    dt_obs=
        np.array(
            DT_OBS
        ),

    initial_state=
        X_TRAJ_INITIAL,

    trajectory_seed=
        np.array(
            TRAJECTORY_SEED
        ),

    # Synthetic oracle metadata
    step_snapshot=
        STEP_SNAPSHOT,

    step_cycle=
        STEP_CYCLE,
)


TRAJECTORY_CYCLE_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "n8_single_trajectory_cycle_audit.csv",
    index=False,
)


print()
print(
    f"Saved trajectory            : "
    f"{TRAJECTORY_FILE}"
)

print()
print("=" * 70)
print("Cell 2 PASSED.")
print("=" * 70)

ONE CONTINUOUS N=8 TRAJECTORY
trajectory seed             : 20261013
initial state               : [-0.140814 -0.264819  0.181982 -0.390419  0.272227 -0.116231 -0.228057
 -0.418635]
initial mass                : -1.104766581081
time interval               : [0, 2.16]
observation interval        : 0.0005
observed states             : 4321
integration intervals       : 4320
runtime                     : 7.072 s
max conservation error      : 1.110e-15
global state range          : [-0.418635, 0.272227]
node spread: start -> end   : 2.339393e-01 -> 5.001122e-02
spread retention            : 0.213779

Per-cycle excitation audit
----------------------------------------------------------------------


,cycle,t_start,t_end,spread_start,spread_end,spread_mean,state_excursion_rms,state_min,state_max
0,0,0.000,0.360,2.339393e-01,1.585989e-01,1.923930e-01,5.179997e-02,-0.418635,0.272227
1,1,0.360,0.720,1.585989e-01,1.152496e-01,1.347842e-01,3.084650e-02,-0.384521,0.120355
2,2,0.720,1.080,1.152496e-01,8.881113e-02,1.007675e-01,1.937449e-02,-0.351540,0.033930
3,3,1.080,1.440,8.881113e-02,7.149441e-02,7.934365e-02,1.283483e-02,-0.321530,-0.017926
4,4,1.440,1.800,7.149441e-02,5.925699e-02,6.480398e-02,8.993320e-03,-0.295087,-0.050432
5,5,1.800,2.160,5.925699e-02,5.001122e-02,5.419280e-02,6.659965e-03,-0.272202,-0.071664



Saved trajectory            : stage6_n8_timeseries_slicing\n8_single_continuous_trajectory.npz

Cell 2 PASSED.


In [4]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 3: construct multi-resolution endpoint slices
#
# IMPORTANT
# ---------------------------------------------------------------
# Every X0 and XF in this cell is READ from the single trajectory
# generated in Cell 2.
#
# There is NO restart, NO reintegration, and NO new initial state.
#
# We first construct only snapshot-interior slices:
#
#   t_s --------> t_s + eps_l
#
# such that the entire interval lies inside the same microscopic
# snapshot G_m.
#
# This is the clean test of:
#
#   epsilon -> 0 intercept
#       =
#   instantaneous generator at the slice start.
# ================================================================


# ================================================================
# Helper: assign full protocol cycle to statistical role
# ================================================================

def cycle_role(cycle_index):

    if cycle_index in FIT_CYCLES:
        return "fit"

    if cycle_index in VALIDATION_CYCLES:
        return "validation"

    if cycle_index in TEST_CYCLES:
        return "test"

    raise ValueError(
        f"Cycle {cycle_index} has no assigned role."
    )


# ================================================================
# Construct slice records
# ================================================================

slice_X0 = []
slice_XF = []
slice_records = []

sample_index = 0


for cycle_index in range(
    N_CYCLES
):

    cycle_base_step = (
        cycle_index
        *
        CYCLE_STEPS
    )

    role = cycle_role(
        cycle_index
    )

    for snapshot_index in range(
        6
    ):

        snapshot_start_step = (
            cycle_base_step
            +
            snapshot_index
            *
            SNAPSHOT_STEPS
        )

        snapshot_end_step = (
            snapshot_start_step
            +
            SNAPSHOT_STEPS
        )

        for local_offset_step in (
            START_OFFSETS
        ):

            start_step = (
                snapshot_start_step
                +
                int(local_offset_step)
            )

            endpoint_steps = (
                start_step
                +
                EPS_STEPS
            )

            # ------------------------------------------------
            # Hard no-crossing condition
            #
            # Equality is allowed:
            #
            #   endpoint == snapshot boundary
            #
            # because all integration intervals before that
            # endpoint are still generated by G_m.
            # ------------------------------------------------

            assert (
                endpoint_steps.max()
                <=
                snapshot_end_step
            )

            # Every integration interval from start until the
            # largest endpoint must carry the same snapshot label.
            interval_labels = (
                STEP_SNAPSHOT[
                    start_step:
                    int(
                        endpoint_steps.max()
                    )
                ]
            )

            assert len(
                interval_labels
            ) == int(
                EPS_STEPS.max()
            )

            assert np.all(
                interval_labels
                ==
                snapshot_index
            )

            # ------------------------------------------------
            # READ the beginning state
            # ------------------------------------------------

            x0 = (
                TRAJECTORY_STATES[
                    start_step
                ]
                .copy()
            )

            # ------------------------------------------------
            # READ all four endpoint states
            #
            # Shape for this sample:
            #
            #       (L, N) = (4, 8)
            # ------------------------------------------------

            xf_family = (
                TRAJECTORY_STATES[
                    endpoint_steps
                ]
                .copy()
            )

            slice_X0.append(
                x0
            )

            slice_XF.append(
                xf_family
            )

            slice_records.append({
                "sample_index":
                    sample_index,

                "cycle":
                    cycle_index,

                # human-readable G1,...,G6 numbering
                "snapshot":
                    snapshot_index + 1,

                # Python index retained separately
                "snapshot_index":
                    snapshot_index,

                "role":
                    role,

                "start_step":
                    start_step,

                "start_time":
                    TRAJECTORY_TIMES[
                        start_step
                    ],

                "local_offset_step":
                    int(
                        local_offset_step
                    ),

                "local_offset_time":
                    float(
                        local_offset_step
                        *
                        DT_OBS
                    ),

                "time_to_boundary":
                    float(
                        (
                            snapshot_end_step
                            -
                            start_step
                        )
                        *
                        DT_OBS
                    ),

                "largest_endpoint_step":
                    int(
                        endpoint_steps.max()
                    ),

                "largest_endpoint_time":
                    float(
                        TRAJECTORY_TIMES[
                            endpoint_steps.max()
                        ]
                    ),
            })

            sample_index += 1


# ================================================================
# Convert to native TSC orientation
#
# Temporary stacked slice_XF:
#
#       (M, L, N)
#
# Native v3.6:
#
#       (L, M, N)
# ================================================================

X0_SLICES = np.stack(
    slice_X0,
    axis=0,
)

XF_SLICES = np.stack(
    slice_XF,
    axis=0,
)

XF_SLICES = np.transpose(
    XF_SLICES,
    (
        1,
        0,
        2,
    ),
)

SLICE_METADATA = pd.DataFrame(
    slice_records
)


# ================================================================
# Expected dimensions
# ================================================================

EXPECTED_TOTAL_SLICES = (
    N_CYCLES
    *
    6
    *
    STARTS_PER_SNAPSHOT_PER_CYCLE
)

assert EXPECTED_TOTAL_SLICES == 1116

assert X0_SLICES.shape == (
    EXPECTED_TOTAL_SLICES,
    N,
)

assert XF_SLICES.shape == (
    len(EPS_SLICE),
    EXPECTED_TOTAL_SLICES,
    N,
)

assert len(
    SLICE_METADATA
) == EXPECTED_TOTAL_SLICES


# ================================================================
# Strong single-source audit
#
# Randomly inspect stored slice entries and prove that their
# beginning/end states are exact reads from TRAJECTORY_STATES.
#
# No tolerance should be needed: these are array copies.
# ================================================================

audit_slice_rng = np.random.default_rng(
    SEED + 404
)

AUDIT_SAMPLE_INDICES = (
    audit_slice_rng.choice(
        EXPECTED_TOTAL_SLICES,
        size=25,
        replace=False,
    )
)


for sample in AUDIT_SAMPLE_INDICES:

    row = (
        SLICE_METADATA.iloc[
            sample
        ]
    )

    start_step = int(
        row["start_step"]
    )

    assert np.array_equal(
        X0_SLICES[
            sample
        ],
        TRAJECTORY_STATES[
            start_step
        ],
    )

    for ell, eps_step in enumerate(
        EPS_STEPS
    ):

        endpoint_step = (
            start_step
            +
            int(eps_step)
        )

        assert np.array_equal(
            XF_SLICES[
                ell,
                sample,
            ],
            TRAJECTORY_STATES[
                endpoint_step
            ],
        )


# ================================================================
# Role indices
# ================================================================

FIT_SLICE_IDX = (
    SLICE_METADATA.index[
        SLICE_METADATA[
            "role"
        ]
        ==
        "fit"
    ]
    .to_numpy(
        dtype=int
    )
)

VALIDATION_SLICE_IDX = (
    SLICE_METADATA.index[
        SLICE_METADATA[
            "role"
        ]
        ==
        "validation"
    ]
    .to_numpy(
        dtype=int
    )
)

TEST_SLICE_IDX = (
    SLICE_METADATA.index[
        SLICE_METADATA[
            "role"
        ]
        ==
        "test"
    ]
    .to_numpy(
        dtype=int
    )
)


# ================================================================
# Snapshot-specific indices
#
# These labels are SYNTHETIC ORACLE information.
#
# They are used only in the first proof-of-concept experiment:
#
#       "If I know which snapshot a slice belongs to,
#        can I recover that instantaneous network?"
#
# Later we will remove these labels and use sliding windows.
# ================================================================

SLICE_INDICES = {}

for snapshot_index in range(
    6
):

    for role in (
        "fit",
        "validation",
        "test",
    ):

        idx = (
            SLICE_METADATA.index[
                (
                    SLICE_METADATA[
                        "snapshot_index"
                    ]
                    ==
                    snapshot_index
                )
                &
                (
                    SLICE_METADATA[
                        "role"
                    ]
                    ==
                    role
                )
            ]
            .to_numpy(
                dtype=int
            )
        )

        SLICE_INDICES[
            (
                snapshot_index,
                role,
            )
        ] = idx


# ================================================================
# Count audit
# ================================================================

assert len(
    FIT_SLICE_IDX
) == (
    4
    *
    6
    *
    31
)

assert len(
    VALIDATION_SLICE_IDX
) == (
    1
    *
    6
    *
    31
)

assert len(
    TEST_SLICE_IDX
) == (
    1
    *
    6
    *
    31
)


for snapshot_index in range(
    6
):

    assert len(
        SLICE_INDICES[
            (
                snapshot_index,
                "fit",
            )
        ]
    ) == 124

    assert len(
        SLICE_INDICES[
            (
                snapshot_index,
                "validation",
            )
        ]
    ) == 31

    assert len(
        SLICE_INDICES[
            (
                snapshot_index,
                "test",
            )
        ]
    ) == 31


# ================================================================
# Simple state-cloud coverage diagnostic
#
# This is NOT yet the formal feature identifiability audit.
#
# For each snapshot/role, measure how much the slice-start states
# vary around their own sample mean:
#
#       cloud_rms
#           =
#       sqrt(mean((X - mean(X))^2))
#
# Cell 4 will inspect the actual polynomial design rank /
# conditioning.
# ================================================================

coverage_rows = []


for snapshot_index in range(
    6
):

    for role in (
        "fit",
        "validation",
        "test",
    ):

        idx = (
            SLICE_INDICES[
                (
                    snapshot_index,
                    role,
                )
            ]
        )

        X_cloud = (
            X0_SLICES[
                idx
            ]
        )

        cloud_mean = (
            X_cloud.mean(
                axis=0,
                keepdims=True,
            )
        )

        cloud_rms = float(
            np.sqrt(
                np.mean(
                    (
                        X_cloud
                        -
                        cloud_mean
                    )
                    ** 2
                )
            )
        )

        coverage_rows.append({
            "snapshot":
                snapshot_index + 1,

            "role":
                role,

            "n_starts":
                len(idx),

            "cloud_rms":
                cloud_rms,

            "state_min":
                float(
                    X_cloud.min()
                ),

            "state_max":
                float(
                    X_cloud.max()
                ),

            "t_min":
                float(
                    SLICE_METADATA.loc[
                        idx,
                        "start_time",
                    ].min()
                ),

            "t_max":
                float(
                    SLICE_METADATA.loc[
                        idx,
                        "start_time",
                    ].max()
                ),
        })


SLICE_COVERAGE_AUDIT = pd.DataFrame(
    coverage_rows
)


# ================================================================
# Endpoint displacement audit
#
# This confirms that the four horizon families behave locally
# and monotonically, but does NOT use the ground-truth generator.
# ================================================================

displacement_rows = []


for ell, epsilon in enumerate(
    EPS_SLICE
):

    displacement = np.sqrt(
        np.mean(
            (
                XF_SLICES[
                    ell
                ]
                -
                X0_SLICES
            )
            ** 2,
            axis=1,
        )
    )

    displacement_rows.append({
        "epsilon":
            epsilon,

        "mean_endpoint_rms":
            float(
                displacement.mean()
            ),

        "median_endpoint_rms":
            float(
                np.median(
                    displacement
                )
            ),

        "max_endpoint_rms":
            float(
                displacement.max()
            ),
    })


SLICE_DISPLACEMENT_AUDIT = pd.DataFrame(
    displacement_rows
)


# ================================================================
# Report
# ================================================================

print("=" * 70)
print("ONE-TRAJECTORY MULTI-RESOLUTION SLICE DATASET")
print("=" * 70)

print(
    f"X0 shape                    : "
    f"{X0_SLICES.shape}"
)

print(
    f"XF shape                    : "
    f"{XF_SLICES.shape}"
)

print(
    f"epsilon                     : "
    f"{EPS_SLICE}"
)

print(
    f"total slice starts          : "
    f"{EXPECTED_TOTAL_SLICES}"
)

print(
    f"fit slice starts            : "
    f"{len(FIT_SLICE_IDX)}"
)

print(
    f"validation slice starts     : "
    f"{len(VALIDATION_SLICE_IDX)}"
)

print(
    f"test slice starts           : "
    f"{len(TEST_SLICE_IDX)}"
)

print()
print(
    "Each start has 4 endpoint observations, "
    "all read from the same stored trajectory."
)


print()
print("Snapshot / role counts")
print("-" * 70)

COUNT_TABLE = pd.crosstab(
    SLICE_METADATA[
        "snapshot"
    ],
    SLICE_METADATA[
        "role"
    ],
)

display(
    COUNT_TABLE
)


print()
print("Slice-start state-cloud coverage")
print("-" * 70)

display(
    SLICE_COVERAGE_AUDIT.style.format({
        "cloud_rms":
            "{:.6e}",

        "state_min":
            "{:.6f}",

        "state_max":
            "{:.6f}",

        "t_min":
            "{:.4f}",

        "t_max":
            "{:.4f}",
    })
)


print()
print("Endpoint displacement")
print("-" * 70)

display(
    SLICE_DISPLACEMENT_AUDIT.style.format({
        "epsilon":
            "{:.4f}",

        "mean_endpoint_rms":
            "{:.6e}",

        "median_endpoint_rms":
            "{:.6e}",

        "max_endpoint_rms":
            "{:.6e}",
    })
)


# ================================================================
# Save sliced observation dataset
# ================================================================

SLICE_DATA_FILE = (
    OUTPUT_DIR
    /
    "n8_single_trajectory_interior_slices.npz"
)

np.savez_compressed(
    SLICE_DATA_FILE,

    X0=
        X0_SLICES,

    XF=
        XF_SLICES,

    eps=
        EPS_SLICE,

    fit_indices=
        FIT_SLICE_IDX,

    validation_indices=
        VALIDATION_SLICE_IDX,

    test_indices=
        TEST_SLICE_IDX,
)


SLICE_METADATA.to_csv(
    OUTPUT_DIR
    /
    "n8_single_trajectory_slice_metadata.csv",
    index=False,
)


SLICE_COVERAGE_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "n8_single_trajectory_slice_coverage.csv",
    index=False,
)


print()
print(
    f"Saved slice dataset         : "
    f"{SLICE_DATA_FILE}"
)

print()
print("=" * 70)
print("Cell 3 PASSED.")
print("=" * 70)

ONE-TRAJECTORY MULTI-RESOLUTION SLICE DATASET
X0 shape                    : (1116, 8)
XF shape                    : (4, 1116, 8)
epsilon                     : [0.005  0.0075 0.01   0.015 ]
total slice starts          : 1116
fit slice starts            : 744
validation slice starts     : 186
test slice starts           : 186

Each start has 4 endpoint observations, all read from the same stored trajectory.

Snapshot / role counts
----------------------------------------------------------------------


role,fit,test,validation
snapshot,,,
1,124,31,31
2,124,31,31
3,124,31,31
4,124,31,31
5,124,31,31
6,124,31,31



Slice-start state-cloud coverage
----------------------------------------------------------------------


,snapshot,role,n_starts,cloud_rms,state_min,state_max,t_min,t_max
0,1,fit,124,5.941240e-02,-0.418635,0.272227,0.0000,1.1250
1,1,validation,31,8.716546e-04,-0.295087,-0.050432,1.4400,1.4850
2,1,test,31,6.943204e-04,-0.272202,-0.071664,1.8000,1.8450
3,2,fit,124,5.407751e-02,-0.411878,0.245021,0.0600,1.1850
4,2,validation,31,1.002637e-03,-0.288702,-0.052087,1.5000,1.5450
5,2,test,31,7.726081e-04,-0.266529,-0.072314,1.8600,1.9050
6,3,fit,124,4.985847e-02,-0.401602,0.215480,0.1200,1.2450
7,3,validation,31,7.210004e-04,-0.283206,-0.056331,1.5600,1.6050
8,3,test,31,5.620676e-04,-0.262005,-0.075092,1.9200,1.9650
9,4,fit,124,4.691078e-02,-0.401619,0.184670,0.1800,1.3050



Endpoint displacement
----------------------------------------------------------------------


,epsilon,mean_endpoint_rms,median_endpoint_rms,max_endpoint_rms
0,0.0050,6.606474e-04,5.051596e-04,1.901124e-03
1,0.0075,9.882243e-04,7.558167e-04,2.843942e-03
2,0.0100,1.313986e-03,1.005202e-03,3.781635e-03
3,0.0150,1.960114e-03,1.500524e-03,5.641762e-03



Saved slice dataset         : stage6_n8_timeseries_slicing\n8_single_trajectory_interior_slices.npz

Cell 3 PASSED.


In [5]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 4: local-generator recovery + identifiability audit
#
# Questions
# ---------------------------------------------------------------
# A. Does multi-resolution slicing recover the instantaneous
#    generator active at the slice start?
#
#       (X(t), X(t+eps_l))
#              |
#              v
#       eps -> 0 intercept
#              |
#              v
#          G_m(X(t)) ?
#
# B. Does ONE deterministic trajectory contain enough excitation
#    to identify the true polynomial dynamics?
#
# No AGLASSO is run in this cell.
# ================================================================


# ================================================================
# Frozen generic learner class
#
# Same broad Stage-4 hypothesis class:
#
#       polynomial degree <= 3
#       structural support <= 4
#
# The true instantaneous fields occupy only a sparse subset.
# ================================================================

MAX_POLY_DEGREE_STAGE6 = 3
MAX_SUPPORT_SIZE_STAGE6 = 4


# ================================================================
# 1. Frozen v3.6 four-point cubic extrapolation
#
# XF_SLICES already has native orientation:
#
#       (L, M, N)
# ================================================================

(
    A_SLICE_EST,
    G1_SLICE_EST,
    EPS_USED_SLICE,
) = tsc_core._resolution_extrapolation(
    X0_SLICES,
    XF_SLICES,
    EPS_SLICE,
    n_points=4,
    polynomial_order=3,
)


assert A_SLICE_EST.shape == (
    len(X0_SLICES),
    N,
)

assert G1_SLICE_EST.shape == (
    len(X0_SLICES),
    N,
)

assert np.allclose(
    EPS_USED_SLICE,
    EPS_SLICE,
)


# ================================================================
# 2. Exact instantaneous generator at every slice start
#
# SYNTHETIC ORACLE — audit only.
#
# For a slice labelled snapshot m:
#
#       true target = G_m(X_start)
#
# This information is NOT used in the extrapolation.
# ================================================================

SLICE_SNAPSHOT_INDEX = (
    SLICE_METADATA[
        "snapshot_index"
    ]
    .to_numpy(
        dtype=int
    )
)

G_SLICE_TRUE = np.empty_like(
    X0_SLICES,
    dtype=float,
)


for snapshot_index in range(6):

    idx = np.flatnonzero(
        SLICE_SNAPSHOT_INDEX
        ==
        snapshot_index
    )

    G_SLICE_TRUE[
        idx
    ] = snapshot_field_numeric(
        X0_SLICES[
            idx
        ],
        snapshot_index,
    )


# ================================================================
# Helpers
# ================================================================

def relative_fro_error(
    estimate,
    truth,
):
    estimate = np.asarray(
        estimate,
        dtype=float,
    )

    truth = np.asarray(
        truth,
        dtype=float,
    )

    denominator = max(
        float(
            np.linalg.norm(
                truth
            )
        ),
        np.finfo(float).tiny,
    )

    return float(
        np.linalg.norm(
            estimate
            -
            truth
        )
        /
        denominator
    )


def rms_field(
    X,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    return float(
        np.sqrt(
            np.mean(
                X ** 2
            )
        )
    )


def numerical_rank_and_condition(
    X,
):
    """
    Numerical matrix rank and condition number on its nonzero
    singular-value subspace.

    For an underdetermined matrix n < p, nullity is reported
    separately; condition_nonzero does NOT hide that nullspace.
    """

    X = np.asarray(
        X,
        dtype=float,
    )

    singular_values = np.linalg.svd(
        X,
        compute_uv=False,
    )

    if len(singular_values) == 0:

        return {
            "rank": 0,
            "nullity": X.shape[1],
            "condition_nonzero": np.inf,
            "s_min_nonzero": 0.0,
            "s_max": 0.0,
        }

    tol = (
        max(
            X.shape
        )
        *
        np.finfo(float).eps
        *
        singular_values[0]
    )

    nonzero = (
        singular_values[
            singular_values > tol
        ]
    )

    rank = len(
        nonzero
    )

    if rank == 0:

        condition = np.inf
        s_min = 0.0

    else:

        condition = float(
            nonzero[0]
            /
            nonzero[-1]
        )

        s_min = float(
            nonzero[-1]
        )

    return {
        "rank":
            int(rank),

        "nullity":
            int(
                X.shape[1]
                -
                rank
            ),

        "condition_nonzero":
            condition,

        "s_min_nonzero":
            s_min,

        "s_max":
            float(
                singular_values[0]
            ),
    }


# ================================================================
# 3. Generator-reconstruction audit
# ================================================================

generator_rows = []


# Overall
generator_rows.append({
    "snapshot":
        "all",

    "role":
        "all",

    "n":
        len(X0_SLICES),

    "true_field_rms":
        rms_field(
            G_SLICE_TRUE
        ),

    "estimated_field_rms":
        rms_field(
            A_SLICE_EST
        ),

    "abs_rms_error":
        rms_field(
            A_SLICE_EST
            -
            G_SLICE_TRUE
        ),

    "relative_error":
        relative_fro_error(
            A_SLICE_EST,
            G_SLICE_TRUE,
        ),
})


# Snapshot x statistical role
for snapshot_index in range(6):

    for role in (
        "fit",
        "validation",
        "test",
    ):

        idx = (
            SLICE_INDICES[
                (
                    snapshot_index,
                    role,
                )
            ]
        )

        generator_rows.append({
            "snapshot":
                f"G{snapshot_index + 1}",

            "role":
                role,

            "n":
                len(idx),

            "true_field_rms":
                rms_field(
                    G_SLICE_TRUE[
                        idx
                    ]
                ),

            "estimated_field_rms":
                rms_field(
                    A_SLICE_EST[
                        idx
                    ]
                ),

            "abs_rms_error":
                rms_field(
                    A_SLICE_EST[
                        idx
                    ]
                    -
                    G_SLICE_TRUE[
                        idx
                    ]
                ),

            "relative_error":
                relative_fro_error(
                    A_SLICE_EST[
                        idx
                    ],
                    G_SLICE_TRUE[
                        idx
                    ],
                ),
        })


GENERATOR_RECOVERY_AUDIT = pd.DataFrame(
    generator_rows
)


# ================================================================
# 4. Build exact Stage-4 broad polynomial library
#
# This library knows ONLY:
#
#       degree <= 3
#       support <= 4
#
# It does NOT know:
#
#       active edges
#       native triads
#       snapshot supports
# ================================================================

REFERENCE_LIBRARY = (
    tsc_core.StructuralPolynomialLibrary(
        n_nodes=N,
        max_polynomial_degree=
            MAX_POLY_DEGREE_STAGE6,
        max_interaction_order=
            MAX_SUPPORT_SIZE_STAGE6,
    )
)


print("=" * 70)
print("LOCAL GENERATOR RECOVERY")
print("=" * 70)

print(
    f"epsilon used                : "
    f"{EPS_USED_SLICE}"
)

print(
    f"estimated intercept shape   : "
    f"{A_SLICE_EST.shape}"
)

print(
    f"broad polynomial features   : "
    f"{REFERENCE_LIBRARY.n_features}"
)

print(
    f"broad structural groups     : "
    f"{len(REFERENCE_LIBRARY.structural_groups)}"
)


# ================================================================
# 5. Convert symbolic instantaneous generators into the learner's
#    raw polynomial coefficient basis
#
# This is oracle-side bookkeeping ONLY.
# ================================================================

EXPONENT_TO_FEATURE = {
    tuple(
        exponent
    ):
        feature_index

    for feature_index, exponent
    in enumerate(
        REFERENCE_LIBRARY.exponents
    )
}


def symbolic_field_to_library_coefficients(
    field_symbolic,
    library,
):

    B_raw = np.zeros(
        (
            library.n_features,
            N,
        ),
        dtype=float,
    )

    extracted_supports = set()

    for output_index in range(N):

        polynomial = sp.Poly(
            sp.expand(
                field_symbolic[
                    output_index
                ]
            ),
            *x_symbols,
        )

        for exponent, coefficient in (
            polynomial.terms()
        ):

            if (
                sp.simplify(
                    coefficient
                )
                ==
                0
            ):
                continue

            exponent = tuple(
                int(v)
                for v
                in exponent
            )

            if (
                exponent
                not in
                EXPONENT_TO_FEATURE
            ):

                raise RuntimeError(
                    "True instantaneous term lies "
                    "outside learner dictionary: "
                    f"output={output_index + 1}, "
                    f"exponent={exponent}, "
                    f"coefficient={coefficient}"
                )

            feature_index = (
                EXPONENT_TO_FEATURE[
                    exponent
                ]
            )

            B_raw[
                feature_index,
                output_index,
            ] = float(
                coefficient
            )

            variables = {
                j + 1

                for j, power
                in enumerate(
                    exponent
                )

                if power > 0
            }

            support = tuple(
                sorted(
                    variables
                    |
                    {
                        output_index + 1
                    }
                )
            )

            extracted_supports.add(
                support
            )

    return (
        B_raw,
        tuple(
            sorted(
                extracted_supports,
                key=lambda s: (
                    len(s),
                    s,
                ),
            )
        ),
    )


SNAPSHOT_TRUE_B_RAW = {}
SNAPSHOT_TRUE_SUPPORTS_FROM_LIBRARY = {}


for snapshot_index in range(6):

    (
        B_true,
        supports_true,
    ) = (
        symbolic_field_to_library_coefficients(
            snapshot_fields_symbolic[
                snapshot_index
            ],
            REFERENCE_LIBRARY,
        )
    )

    SNAPSHOT_TRUE_B_RAW[
        snapshot_index
    ] = B_true

    SNAPSHOT_TRUE_SUPPORTS_FROM_LIBRARY[
        snapshot_index
    ] = supports_true

    # Exact consistency with Cell-1 symbolic support oracle
    assert (
        set(
            supports_true
        )
        ==
        set(
            SNAPSHOT_ORACLE_SUPPORTS[
                snapshot_index
            ]
        )
    )


# ================================================================
# 6. Snapshot-local identifiability audit
#
# For each G_m:
#
#   - use ONLY cycles 0-3 slice starts
#   - fit feature scaling on those states
#   - inspect full broad dictionary
#   - inspect the TRUE active feature subspace separately
#
# The full dictionary is expected to be underdetermined because:
#
#       n_fit = 124
#       p     = 164
#
# Sparse inference can still succeed.
#
# The crucial requirement is that the TRUE active directions for
# each output are full rank on the observed trajectory states.
# ================================================================

identifiability_rows = []
output_identifiability_rows = []


for snapshot_index in range(6):

    fit_idx = (
        SLICE_INDICES[
            (
                snapshot_index,
                "fit",
            )
        ]
    )

    X_fit = (
        X0_SLICES[
            fit_idx
        ]
    )

    G_fit_true = (
        G_SLICE_TRUE[
            fit_idx
        ]
    )

    # ------------------------------------------------------------
    # Fresh library: scaling learned only from this snapshot's
    # fitting states.
    # ------------------------------------------------------------

    local_library = (
        tsc_core.StructuralPolynomialLibrary(
            n_nodes=N,
            max_polynomial_degree=
                MAX_POLY_DEGREE_STAGE6,
            max_interaction_order=
                MAX_SUPPORT_SIZE_STAGE6,
        )
    )

    Theta_scaled = (
        local_library.fit_transform(
            X_fit
        )
    )

    Theta_raw = (
        local_library.evaluate_raw(
            X_fit
        )
    )

    B_true = (
        SNAPSHOT_TRUE_B_RAW[
            snapshot_index
        ]
    )

    # ------------------------------------------------------------
    # Exact dictionary representability
    # ------------------------------------------------------------

    G_from_dictionary = (
        Theta_raw
        @
        B_true
    )

    dictionary_rel_error = (
        relative_fro_error(
            G_from_dictionary,
            G_fit_true,
        )
    )

    dictionary_max_abs_error = float(
        np.max(
            np.abs(
                G_from_dictionary
                -
                G_fit_true
            )
        )
    )

    assert (
        dictionary_rel_error
        <
        1e-12
    )

    assert (
        dictionary_max_abs_error
        <
        1e-12
    )

    # ------------------------------------------------------------
    # Full broad dictionary
    # ------------------------------------------------------------

    full_diag = (
        numerical_rank_and_condition(
            Theta_scaled
        )
    )

    # Max off-diagonal normalized feature inner product.
    #
    # fit_transform gives RMS-normalized columns:
    #
    #       ||theta_j||_2^2 / n = 1
    # ------------------------------------------------------------

    gram = (
        Theta_scaled.T
        @
        Theta_scaled
        /
        Theta_scaled.shape[0]
    )

    gram_offdiag = (
        gram.copy()
    )

    np.fill_diagonal(
        gram_offdiag,
        0.0,
    )

    max_feature_coherence = float(
        np.max(
            np.abs(
                gram_offdiag
            )
        )
    )

    # ------------------------------------------------------------
    # True active scalar directions, output by output
    # ------------------------------------------------------------

    output_conditions = []
    output_full_rank_flags = []
    active_counts = []


    for output_index in range(N):

        active_features = np.flatnonzero(
            np.abs(
                B_true[
                    :,
                    output_index
                ]
            )
            >
            1e-14
        )

        n_active = len(
            active_features
        )

        active_counts.append(
            n_active
        )

        if n_active == 0:

            rank_active = 0
            condition_active = 1.0
            full_rank_active = True

        else:

            Theta_active = (
                Theta_scaled[
                    :,
                    active_features
                ]
            )

            active_diag = (
                numerical_rank_and_condition(
                    Theta_active
                )
            )

            rank_active = (
                active_diag[
                    "rank"
                ]
            )

            condition_active = (
                active_diag[
                    "condition_nonzero"
                ]
            )

            full_rank_active = (
                rank_active
                ==
                n_active
            )

        output_conditions.append(
            condition_active
        )

        output_full_rank_flags.append(
            full_rank_active
        )

        output_identifiability_rows.append({
            "snapshot":
                snapshot_index + 1,

            "output_node":
                output_index + 1,

            "n_fit":
                len(fit_idx),

            "active_scalar_features":
                n_active,

            "active_rank":
                rank_active,

            "active_full_rank":
                full_rank_active,

            "active_condition":
                condition_active,
        })


    true_supports = (
        SNAPSHOT_ORACLE_SUPPORTS[
            snapshot_index
        ]
    )


    identifiability_rows.append({
        "snapshot":
            snapshot_index + 1,

        "n_fit":
            len(fit_idx),

        "n_features":
            Theta_scaled.shape[1],

        "full_design_rank":
            full_diag[
                "rank"
            ],

        "full_design_nullity":
            full_diag[
                "nullity"
            ],

        "full_nonzero_condition":
            full_diag[
                "condition_nonzero"
            ],

        "max_feature_coherence":
            max_feature_coherence,

        "true_structural_groups":
            len(
                true_supports
            ),

        "true_scalar_directions":
            int(
                np.count_nonzero(
                    np.abs(
                        B_true
                    )
                    >
                    1e-14
                )
            ),

        "max_active_features_per_output":
            int(
                max(
                    active_counts
                )
            ),

        "all_outputs_oracle_full_rank":
            bool(
                all(
                    output_full_rank_flags
                )
            ),

        "max_oracle_condition":
            float(
                max(
                    output_conditions
                )
            ),

        "median_oracle_condition":
            float(
                np.median(
                    output_conditions
                )
            ),

        "dictionary_rel_error":
            dictionary_rel_error,
    })


SNAPSHOT_IDENTIFIABILITY_AUDIT = (
    pd.DataFrame(
        identifiability_rows
    )
)

OUTPUT_IDENTIFIABILITY_AUDIT = (
    pd.DataFrame(
        output_identifiability_rows
    )
)


# ================================================================
# 7. Report
# ================================================================

print()
print("Generator recovery by snapshot / role")
print("-" * 70)

display(
    GENERATOR_RECOVERY_AUDIT.style.format({
        "true_field_rms":
            "{:.6e}",

        "estimated_field_rms":
            "{:.6e}",

        "abs_rms_error":
            "{:.6e}",

        "relative_error":
            "{:.6e}",
    })
)


print()
print("=" * 70)
print("SINGLE-TRAJECTORY IDENTIFIABILITY AUDIT")
print("=" * 70)

display(
    SNAPSHOT_IDENTIFIABILITY_AUDIT.style.format({
        "full_nonzero_condition":
            "{:.6e}",

        "max_feature_coherence":
            "{:.8f}",

        "max_oracle_condition":
            "{:.6e}",

        "median_oracle_condition":
            "{:.6e}",

        "dictionary_rel_error":
            "{:.6e}",
    })
)


print()
print("Per-output oracle active-subspace audit")
print("-" * 70)

display(
    OUTPUT_IDENTIFIABILITY_AUDIT.style.format({
        "active_condition":
            "{:.6e}",
    })
)


# ================================================================
# 8. Hard conclusions that can already be tested
# ================================================================

GLOBAL_GENERATOR_REL_ERROR = (
    relative_fro_error(
        A_SLICE_EST,
        G_SLICE_TRUE,
    )
)

ALL_ORACLE_SUBSPACES_IDENTIFIABLE = bool(
    SNAPSHOT_IDENTIFIABILITY_AUDIT[
        "all_outputs_oracle_full_rank"
    ].all()
)


print()
print("=" * 70)
print("CELL 4 SUMMARY")
print("=" * 70)

print(
    f"Global generator relative error : "
    f"{GLOBAL_GENERATOR_REL_ERROR:.6e}"
)

print(
    f"All true active subspaces "
    f"full rank                     : "
    f"{ALL_ORACLE_SUBSPACES_IDENTIFIABLE}"
)

print(
    "Broad dictionary is "
    "underdetermined by design    : "
    f"{REFERENCE_LIBRARY.n_features > 124}"
)


# ================================================================
# 9. Save audit products
# ================================================================

GENERATOR_AUDIT_FILE = (
    OUTPUT_DIR
    /
    "n8_single_trajectory_generator_audit.csv"
)

IDENTIFIABILITY_AUDIT_FILE = (
    OUTPUT_DIR
    /
    "n8_single_trajectory_identifiability_audit.csv"
)

OUTPUT_IDENTIFIABILITY_FILE = (
    OUTPUT_DIR
    /
    "n8_single_trajectory_output_identifiability.csv"
)

GENERATOR_RECOVERY_AUDIT.to_csv(
    GENERATOR_AUDIT_FILE,
    index=False,
)

SNAPSHOT_IDENTIFIABILITY_AUDIT.to_csv(
    IDENTIFIABILITY_AUDIT_FILE,
    index=False,
)

OUTPUT_IDENTIFIABILITY_AUDIT.to_csv(
    OUTPUT_IDENTIFIABILITY_FILE,
    index=False,
)


np.savez_compressed(
    OUTPUT_DIR
    /
    "n8_single_trajectory_generator_reconstruction.npz",

    A_est=
        A_SLICE_EST,

    G1_est=
        G1_SLICE_EST,

    G_true=
        G_SLICE_TRUE,

    X0=
        X0_SLICES,

    eps=
        EPS_USED_SLICE,

    snapshot_index=
        SLICE_SNAPSHOT_INDEX,
)


print()
print("=" * 70)
print("Cell 4 PASSED.")
print("=" * 70)

LOCAL GENERATOR RECOVERY
epsilon used                : [0.005  0.0075 0.01   0.015 ]
estimated intercept shape   : (1116, 8)
broad polynomial features   : 164
broad structural groups     : 162

Generator recovery by snapshot / role
----------------------------------------------------------------------


,snapshot,role,n,true_field_rms,estimated_field_rms,abs_rms_error,relative_error
0,all,all,1116,1.599567e-01,1.599567e-01,9.462074e-10,5.915399e-09
1,G1,fit,124,2.254865e-01,2.254865e-01,2.657076e-10,1.178375e-09
2,G1,validation,31,6.498800e-02,6.498800e-02,2.949544e-11,4.538597e-10
3,G1,test,31,5.176490e-02,5.176490e-02,1.717176e-11,3.317259e-10
4,G2,fit,124,2.093935e-01,2.093935e-01,8.042765e-11,3.840981e-10
5,G2,validation,31,7.475401e-02,7.475401e-02,3.039367e-11,4.065825e-10
6,G2,test,31,5.760397e-02,5.760397e-02,2.451751e-11,4.256219e-10
7,G3,fit,124,1.693691e-01,1.693691e-01,1.163350e-09,6.868726e-09
8,G3,validation,31,5.376296e-02,5.376296e-02,1.088905e-10,2.025381e-09
9,G3,test,31,4.191103e-02,4.191103e-02,7.330367e-11,1.749030e-09



SINGLE-TRAJECTORY IDENTIFIABILITY AUDIT


,snapshot,n_fit,n_features,full_design_rank,full_design_nullity,full_nonzero_condition,max_feature_coherence,true_structural_groups,true_scalar_directions,max_active_features_per_output,all_outputs_oracle_full_rank,max_oracle_condition,median_oracle_condition,dictionary_rel_error
0,1,124,164,25,139,2.551135e+13,0.99998837,19,50,10,True,2.284051e+06,1.912185e+04,1.502203e-16
1,2,124,164,23,141,4.194965e+12,0.99998066,19,50,10,True,9.286296e+05,7.236349e+03,1.660773e-16
2,3,124,164,25,139,9.502035e+12,0.99996837,17,46,10,True,1.729424e+06,1.010020e+04,1.855870e-16
3,4,124,164,25,139,9.166839e+12,0.99996687,18,48,10,True,2.288101e+06,5.219257e+04,1.661627e-16
4,5,124,164,25,139,2.615076e+12,0.99998195,18,48,10,True,1.168871e+06,9.325974e+03,1.822941e-16
5,6,124,164,26,138,1.878184e+13,0.99997368,17,46,10,True,2.111911e+06,1.853126e+03,1.822925e-16



Per-output oracle active-subspace audit
----------------------------------------------------------------------


,snapshot,output_node,n_fit,active_scalar_features,active_rank,active_full_rank,active_condition
0,1,1,124,7,7,True,5.593533e+04
1,1,2,124,10,10,True,2.284051e+06
2,1,3,124,7,7,True,6.123509e+04
3,1,4,124,4,4,True,1.366525e+03
4,1,5,124,7,7,True,1.194524e+04
5,1,6,124,4,4,True,5.049343e+01
6,1,7,124,4,4,True,7.968757e+02
7,1,8,124,7,7,True,2.629847e+04
8,2,1,124,7,7,True,1.661231e+04
9,2,2,124,10,10,True,9.286296e+05



CELL 4 SUMMARY
Global generator relative error : 5.915399e-09
All true active subspaces full rank                     : True
Broad dictionary is underdetermined by design    : True

Cell 4 PASSED.


In [6]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 5: blind recovery of instantaneous microscopic network G1
#
# Observation hierarchy
# ---------------------------------------------------------------
# LOCAL RESPONSE FAMILY:
#
#   one start X(t_s)
#       ->
#   four endpoints
#
#       eps = [0.005, 0.0075, 0.010, 0.015]
#
#   maximum response span = eps_max = 0.015
#
#
# LOCAL-STATIONARITY REGIME:
#
#   G1 lasts DeltaT = 0.06
#
#   many different starts t_s are taken inside G1.
#
#
# Current synthetic pilot:
#
#   cycles 0-3 G1 segments : fit
#   cycle 4 G1 segment     : validation
#   cycle 5 G1 segment     : external test
#
#
# Scientific question
# ---------------------------------------------------------------
# Given:
#
#       X(t_s)
#       A_est(t_s) ~= G1(X(t_s))
#
# recovered only from one stored time series,
#
# can broad polynomial sparse inference recover:
#
#       1. the dynamical function G1
#       2. its structural supports
#       3. the microscopic network active during G1
#
# Oracle topology / interaction law enters ONLY after fitting.
# ================================================================

import hashlib
import time


# ================================================================
# Target microscopic regime
# ================================================================

SNAPSHOT_PILOT = 0       # Python index
G_LABEL = "G1"


# ================================================================
# Exact temporal-block indices
# ================================================================

IDX_G1_FIT = (
    SLICE_INDICES[
        (
            SNAPSHOT_PILOT,
            "fit",
        )
    ]
)

IDX_G1_VAL = (
    SLICE_INDICES[
        (
            SNAPSHOT_PILOT,
            "validation",
        )
    ]
)

IDX_G1_TEST = (
    SLICE_INDICES[
        (
            SNAPSHOT_PILOT,
            "test",
        )
    ]
)


assert len(IDX_G1_FIT) == 124
assert len(IDX_G1_VAL) == 31
assert len(IDX_G1_TEST) == 31


# ================================================================
# State / reconstructed-generator datasets
#
# These A_est values came from Cell 4:
#
#       four endpoint responses
#            ->
#       epsilon -> 0 extrapolation
#            ->
#       instantaneous generator sample
#
# No ground-truth generator is used here.
# ================================================================

X_G1_FIT = (
    X0_SLICES[
        IDX_G1_FIT
    ]
)

Y_G1_FIT = (
    A_SLICE_EST[
        IDX_G1_FIT
    ]
)


X_G1_VAL = (
    X0_SLICES[
        IDX_G1_VAL
    ]
)

Y_G1_VAL = (
    A_SLICE_EST[
        IDX_G1_VAL
    ]
)


X_G1_TEST = (
    X0_SLICES[
        IDX_G1_TEST
    ]
)

Y_G1_TEST = (
    A_SLICE_EST[
        IDX_G1_TEST
    ]
)


assert X_G1_FIT.shape == (
    124,
    N,
)

assert X_G1_VAL.shape == (
    31,
    N,
)

assert X_G1_TEST.shape == (
    31,
    N,
)


# ================================================================
# Fresh broad structural polynomial library
#
# Learner knows only:
#
#       degree <= 3
#       structural support <= 4
#
# It does NOT know:
#
#       active G1 edges
#       native triads
#       phi(x)
#       true coefficients
# ================================================================

G1_LIBRARY = (
    tsc_core.StructuralPolynomialLibrary(
        n_nodes=N,
        max_polynomial_degree=
            MAX_POLY_DEGREE_STAGE6,
        max_interaction_order=
            MAX_SUPPORT_SIZE_STAGE6,
    )
)


# ================================================================
# STRICT TEMPORAL FEATURE SCALING
#
# IMPORTANT CHANGE FROM PREVIOUS CELL 5:
#
# Learn feature scales ONLY from cycles 0-3.
#
# Validation/test states do not influence:
#
#       feature RMS scaling
#
# at all.
# ================================================================

THETA_G1_FIT = (
    G1_LIBRARY.fit_transform(
        X_G1_FIT
    )
)

THETA_G1_VAL = (
    G1_LIBRARY.transform(
        X_G1_VAL
    )
)

THETA_G1_TEST = (
    G1_LIBRARY.transform(
        X_G1_TEST
    )
)


# ================================================================
# _AdaptiveGroupLasso expects validation to be contained inside
# its supplied training matrix.
#
# Therefore concatenate:
#
#       [fit block | validation block]
#
# but preserve their ordering exactly.
#
# Our custom split below will recover:
#
#       first 124 -> fit
#       last   31 -> validation
#
# The solver's final selected support is allowed to refit on
# fit + validation, as usual after model selection.
#
# External test remains completely untouched.
# ================================================================

THETA_G1_TRAIN = np.vstack(
    [
        THETA_G1_FIT,
        THETA_G1_VAL,
    ]
)

Y_G1_TRAIN = np.vstack(
    [
        Y_G1_FIT,
        Y_G1_VAL,
    ]
)


N_G1_FIXED_FIT = len(
    Y_G1_FIT
)

N_G1_FIXED_VAL = len(
    Y_G1_VAL
)


assert THETA_G1_TRAIN.shape == (
    155,
    G1_LIBRARY.n_features,
)

assert Y_G1_TRAIN.shape == (
    155,
    N,
)


# ================================================================
# Temporal metadata sanity check
# ================================================================

FIT_CYCLES_G1 = set(
    SLICE_METADATA.loc[
        IDX_G1_FIT,
        "cycle",
    ]
)

VAL_CYCLES_G1 = set(
    SLICE_METADATA.loc[
        IDX_G1_VAL,
        "cycle",
    ]
)

TEST_CYCLES_G1 = set(
    SLICE_METADATA.loc[
        IDX_G1_TEST,
        "cycle",
    ]
)


assert FIT_CYCLES_G1 == {
    0, 1, 2, 3
}

assert VAL_CYCLES_G1 == {
    4
}

assert TEST_CYCLES_G1 == {
    5
}


print("=" * 70)
print("G1 MICROSCOPIC NETWORK INFERENCE FROM ONE TIME SERIES")
print("=" * 70)

print(
    f"local regime duration DeltaT: "
    f"{TAU_SNAPSHOT}"
)

print(
    f"response horizons epsilon   : "
    f"{EPS_SLICE}"
)

print(
    f"maximum response span       : "
    f"{EPS_MAX}"
)

print(
    f"slice-start stride          : "
    f"{SLICE_STRIDE}"
)

print()

print(
    f"fit starts                  : "
    f"{len(X_G1_FIT)} "
    f"(cycles 0-3)"
)

print(
    f"validation starts           : "
    f"{len(X_G1_VAL)} "
    f"(cycle 4)"
)

print(
    f"external test starts        : "
    f"{len(X_G1_TEST)} "
    f"(cycle 5)"
)

print()

print(
    f"broad scalar features       : "
    f"{G1_LIBRARY.n_features}"
)

print(
    f"candidate structural groups : "
    f"{len(G1_LIBRARY.structural_groups)}"
)

print(
    f"fit generator RMS           : "
    f"{rms_field(Y_G1_FIT):.6e}"
)

print(
    f"validation generator RMS    : "
    f"{rms_field(Y_G1_VAL):.6e}"
)

print(
    f"test generator RMS          : "
    f"{rms_field(Y_G1_TEST):.6e}"
)


# ================================================================
# Block-aware AGLASSO
#
# ONLY the internal validation split is changed.
#
# Everything else remains frozen v3.6:
#
#   adaptive Ridge pilot
#   adaptive group weights
#   KKT working-set expansion
#   event-driven lambda continuation
#   one-SE validation selection
#   post-selection OLS refit
# ================================================================

class FixedTemporalBlockAGLASSO(
    tsc_core._AdaptiveGroupLasso
):

    def __init__(
        self,
        *args,
        fixed_fit_count,
        fixed_validation_count,
        **kwargs,
    ):

        super().__init__(
            *args,
            **kwargs,
        )

        self.fixed_fit_count = int(
            fixed_fit_count
        )

        self.fixed_validation_count = int(
            fixed_validation_count
        )


    def _fit_validation_split(
        self,
        Theta,
        Y,
    ):

        n = Theta.shape[0]

        expected = (
            self.fixed_fit_count
            +
            self.fixed_validation_count
        )

        if n != expected:

            raise ValueError(
                "Temporal block split expected "
                f"{expected} samples, "
                f"received {n}."
            )


        fit_idx = np.arange(
            0,
            self.fixed_fit_count,
            dtype=int,
        )

        val_idx = np.arange(
            self.fixed_fit_count,
            expected,
            dtype=int,
        )


        return (
            Theta[
                fit_idx
            ],

            Y[
                fit_idx
            ],

            Theta[
                val_idx
            ],

            Y[
                val_idx
            ],

            fit_idx,
            val_idx,
        )


# ================================================================
# Frozen v3.6 F0 hyperparameters
# ================================================================

STAGE6_CFG = (
    tsc_core.TSCInference(
        max_interaction_order=
            MAX_SUPPORT_SIZE_STAGE6,

        max_polynomial_degree=
            MAX_POLY_DEGREE_STAGE6,

        temporal_order=1,

        verbose=True,
    )
)


G1_ESTIMATOR = (
    FixedTemporalBlockAGLASSO(

        library=
            G1_LIBRARY,

        fixed_fit_count=
            N_G1_FIXED_FIT,

        fixed_validation_count=
            N_G1_FIXED_VAL,

        n_lambdas=
            STAGE6_CFG._F0_N_LAMBDAS,

        lambda_min_ratio=
            STAGE6_CFG._F0_LAMBDA_MIN_RATIO,

        coarse_n_lambdas=
            STAGE6_CFG._F0_COARSE_N_LAMBDAS,

        refine_n_lambdas=
            STAGE6_CFG._F0_REFINE_N_LAMBDAS,

        gamma=
            STAGE6_CFG._ADAPTIVE_GAMMA,

        delta_ratio=
            STAGE6_CFG._ADAPTIVE_DELTA_RATIO,

        max_iter=
            STAGE6_CFG._MAX_ITER,

        tol=
            STAGE6_CFG._SOLVER_TOL,

        active_group_tol=
            STAGE6_CFG._ACTIVE_GROUP_TOL,

        ridge_condition_target=
            STAGE6_CFG._RIDGE_CONDITION_TARGET,

        ridge_alpha_floor_ratio=
            STAGE6_CFG._RIDGE_ALPHA_FLOOR_RATIO,

        kkt_tol=
            STAGE6_CFG._KKT_TOL,

        max_kkt_expansions=
            STAGE6_CFG._MAX_KKT_EXPANSIONS,

        # Ignored by custom block splitter,
        # retained for frozen configuration completeness.
        validation_fraction=
            STAGE6_CFG._VALIDATION_FRACTION,

        validation_seed=
            STAGE6_CFG._VALIDATION_SEED,

        early_stop_min_evals=
            STAGE6_CFG._EARLY_STOP_MIN_EVALS,

        early_stop_patience=
            STAGE6_CFG._EARLY_STOP_PATIENCE,

        early_stop_relative_degradation=
            STAGE6_CFG._EARLY_STOP_RELATIVE_DEGRADATION,

        early_stop_support_growth_ratio=
            STAGE6_CFG._EARLY_STOP_SUPPORT_GROWTH_RATIO,

        exact_fit_plateau_patience=
            STAGE6_CFG._EXACT_FIT_PLATEAU_PATIENCE,

        exact_fit_relative_error=
            STAGE6_CFG._EXACT_FIT_RELATIVE_ERROR,

        adaptive_geometric_ratio=
            STAGE6_CFG._ADAPTIVE_GEOMETRIC_RATIO,

        adaptive_entry_fraction=
            STAGE6_CFG._ADAPTIVE_ENTRY_FRACTION,

        adaptive_max_jump_decades=
            STAGE6_CFG._ADAPTIVE_MAX_JUMP_DECADES,

        adaptive_max_evals=
            STAGE6_CFG._F0_ADAPTIVE_MAX_EVALS,

        enable_pruning=False,

        verbose=True,

        label=
            "Stage6 one-series G1 F^(0)",
    )
)


# ================================================================
# Checkpoint fingerprint
# ================================================================

def stage6_g1_fingerprint(
    X_fit,
    Y_fit,
    X_val,
    Y_val,
):

    h = hashlib.sha256()

    h.update(
        b"STAGE6_G1_STRICT_TEMPORAL_BLOCK_v2"
    )

    for name, array in (

        ("X_fit", X_fit),
        ("Y_fit", Y_fit),

        ("X_val", X_val),
        ("Y_val", Y_val),

        ("eps", EPS_SLICE),
    ):

        arr = np.ascontiguousarray(
            np.asarray(
                array
            )
        )

        h.update(
            name.encode(
                "utf-8"
            )
        )

        h.update(
            str(
                arr.dtype
            ).encode(
                "utf-8"
            )
        )

        h.update(
            np.asarray(
                arr.shape,
                dtype=np.int64,
            ).tobytes()
        )

        h.update(
            arr.view(
                np.uint8
            ).tobytes()
        )


    # Feature scaling is part of the actual experiment.
    h.update(
        np.ascontiguousarray(
            G1_LIBRARY.feature_scale
        )
        .view(
            np.uint8
        )
        .tobytes()
    )


    return h.hexdigest()


G1_FINGERPRINT = (
    stage6_g1_fingerprint(

        X_G1_FIT,
        Y_G1_FIT,

        X_G1_VAL,
        Y_G1_VAL,
    )
)


G1_CHECKPOINT = (
    OUTPUT_DIR
    /
    "stage6_G1_strict_block_aglasso_partial.pkl.gz"
)


# ================================================================
# Run blind sparse inference
# ================================================================

print()
print("=" * 70)
print("STARTING BLOCK-AWARE v3.6 AGLASSO")
print("=" * 70)

tic = time.perf_counter()


G1_RESULT = (
    G1_ESTIMATOR.fit(

        THETA_G1_TRAIN,
        Y_G1_TRAIN,

        Theta_test=
            THETA_G1_TEST,

        Y_test=
            Y_G1_TEST,

        checkpoint_path=
            G1_CHECKPOINT,

        checkpoint_fingerprint=
            G1_FINGERPRINT,

        resume_from_checkpoint=True,
    )
)


G1_RUNTIME = (
    time.perf_counter()
    -
    tic
)


# ================================================================
# Synthetic oracle enters ONLY NOW
# ================================================================

TRUE_G1_SUPPORTS = set(
    SNAPSHOT_ORACLE_SUPPORTS[
        SNAPSHOT_PILOT
    ]
)

PRED_G1_SUPPORTS = set(
    G1_RESULT.selected_supports
)


# ================================================================
# Structural scoring helper
# ================================================================

def support_metrics(
    predicted,
    truth,
):

    predicted = set(
        predicted
    )

    truth = set(
        truth
    )


    tp = len(
        predicted
        &
        truth
    )

    fp = len(
        predicted
        -
        truth
    )

    fn = len(
        truth
        -
        predicted
    )


    precision = (
        tp
        /
        (tp + fp)

        if tp + fp > 0

        else 0.0
    )

    recall = (
        tp
        /
        (tp + fn)

        if tp + fn > 0

        else 0.0
    )

    f1 = (
        2.0
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
        )

        if (
            precision
            +
            recall
        ) > 0

        else 0.0
    )


    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }


# ================================================================
# Overall + by-order support metrics
# ================================================================

G1_SUPPORT_METRICS = (
    support_metrics(
        PRED_G1_SUPPORTS,
        TRUE_G1_SUPPORTS,
    )
)


order_rows = []


for order in (
    1, 2, 3, 4
):

    predicted_order = {
        support

        for support
        in PRED_G1_SUPPORTS

        if len(
            support
        ) == order
    }

    true_order = {
        support

        for support
        in TRUE_G1_SUPPORTS

        if len(
            support
        ) == order
    }


    metrics = support_metrics(
        predicted_order,
        true_order,
    )


    order_rows.append({

        "order":
            order,

        "true":
            len(
                true_order
            ),

        "selected":
            len(
                predicted_order
            ),

        **metrics,
    })


G1_ORDER_METRICS = pd.DataFrame(
    order_rows
)


# ================================================================
# Raw coefficient scoring
#
# Synthetic diagnostic only.
# ================================================================

B_G1_TRUE = (
    SNAPSHOT_TRUE_B_RAW[
        SNAPSHOT_PILOT
    ]
)

B_G1_EST = (
    G1_RESULT.coefficients_raw
)


G1_COEFF_REL_ERROR = float(

    np.linalg.norm(
        B_G1_EST
        -
        B_G1_TRUE
    )

    /

    np.linalg.norm(
        B_G1_TRUE
    )
)


# ================================================================
# Field predictions
# ================================================================

Y_G1_FIT_PRED = (
    THETA_G1_FIT
    @
    G1_RESULT.coefficients_scaled
)

Y_G1_VAL_PRED = (
    THETA_G1_VAL
    @
    G1_RESULT.coefficients_scaled
)

Y_G1_TEST_PRED = (
    THETA_G1_TEST
    @
    G1_RESULT.coefficients_scaled
)


# ================================================================
# Exact physical generator values
#
# Oracle diagnostic only.
# ================================================================

G1_TRUE_FIT = (
    G_SLICE_TRUE[
        IDX_G1_FIT
    ]
)

G1_TRUE_VAL = (
    G_SLICE_TRUE[
        IDX_G1_VAL
    ]
)

G1_TRUE_TEST = (
    G_SLICE_TRUE[
        IDX_G1_TEST
    ]
)


# ================================================================
# Field errors
# ================================================================

G1_FIELD_ERROR_RECON_FIT = (
    relative_fro_error(
        Y_G1_FIT_PRED,
        Y_G1_FIT,
    )
)

G1_FIELD_ERROR_TRUE_FIT = (
    relative_fro_error(
        Y_G1_FIT_PRED,
        G1_TRUE_FIT,
    )
)


G1_FIELD_ERROR_RECON_VAL = (
    relative_fro_error(
        Y_G1_VAL_PRED,
        Y_G1_VAL,
    )
)

G1_FIELD_ERROR_TRUE_VAL = (
    relative_fro_error(
        Y_G1_VAL_PRED,
        G1_TRUE_VAL,
    )
)


G1_FIELD_ERROR_RECON_TEST = (
    relative_fro_error(
        Y_G1_TEST_PRED,
        Y_G1_TEST,
    )
)

G1_FIELD_ERROR_TRUE_TEST = (
    relative_fro_error(
        Y_G1_TEST_PRED,
        G1_TRUE_TEST,
    )
)


# ================================================================
# Support mismatches
# ================================================================

G1_FALSE_POSITIVES = tuple(
    sorted(
        PRED_G1_SUPPORTS
        -
        TRUE_G1_SUPPORTS,

        key=lambda s: (
            len(s),
            s,
        ),
    )
)

G1_FALSE_NEGATIVES = tuple(
    sorted(
        TRUE_G1_SUPPORTS
        -
        PRED_G1_SUPPORTS,

        key=lambda s: (
            len(s),
            s,
        ),
    )
)


# ================================================================
# Main report
# ================================================================

print()
print("=" * 70)
print("G1 BLIND MICROSCOPIC-NETWORK RECOVERY")
print("=" * 70)

print(
    f"Runtime                     : "
    f"{G1_RUNTIME:.2f} s "
    f"= {G1_RUNTIME / 60:.2f} min"
)

print()

print(
    f"True supports               : "
    f"{len(TRUE_G1_SUPPORTS)}"
)

print(
    f"Selected supports           : "
    f"{len(PRED_G1_SUPPORTS)}"
)

print(
    f"TP / FP / FN                : "
    f"{G1_SUPPORT_METRICS['TP']} / "
    f"{G1_SUPPORT_METRICS['FP']} / "
    f"{G1_SUPPORT_METRICS['FN']}"
)

print(
    f"Precision                   : "
    f"{G1_SUPPORT_METRICS['precision']:.6f}"
)

print(
    f"Recall                      : "
    f"{G1_SUPPORT_METRICS['recall']:.6f}"
)

print(
    f"F1                          : "
    f"{G1_SUPPORT_METRICS['F1']:.6f}"
)


print()
print("Model-selection diagnostics")
print("-" * 70)

print(
    f"Selected lambda             : "
    f"{G1_RESULT.lambda_selected:.6e}"
)

print(
    f"Selection method            : "
    f"{G1_RESULT.selection_method}"
)

print(
    f"Internal validation error   : "
    f"{G1_RESULT.validation_relative_error:.6e}"
)

print(
    f"External test error         : "
    f"{G1_RESULT.test_relative_error:.6e}"
)

print(
    f"Final global KKT            : "
    f"{G1_RESULT.final_kkt_satisfied}"
)

print(
    f"Selected-lambda KKT cert.   : "
    f"{G1_RESULT.kkt_certified}"
)

print(
    f"Path boundary hit           : "
    f"{G1_RESULT.path_boundary_hit}"
)

print(
    f"Path stop reason            : "
    f"{G1_RESULT.early_stop_reason}"
)


print()
print("Recovery by structural order")
print("-" * 70)

display(
    G1_ORDER_METRICS.style.format({

        "precision":
            "{:.6f}",

        "recall":
            "{:.6f}",

        "F1":
            "{:.6f}",
    })
)


print()
print("Coefficient / field diagnostics")
print("-" * 70)

print(
    f"Coefficient relative error  : "
    f"{G1_COEFF_REL_ERROR:.6e}"
)

print()

print(
    f"Fit error vs A_est          : "
    f"{G1_FIELD_ERROR_RECON_FIT:.6e}"
)

print(
    f"Fit error vs exact G1       : "
    f"{G1_FIELD_ERROR_TRUE_FIT:.6e}"
)

print()

print(
    f"Validation error vs A_est   : "
    f"{G1_FIELD_ERROR_RECON_VAL:.6e}"
)

print(
    f"Validation error vs exact G1: "
    f"{G1_FIELD_ERROR_TRUE_VAL:.6e}"
)

print()

print(
    f"Test error vs A_est         : "
    f"{G1_FIELD_ERROR_RECON_TEST:.6e}"
)

print(
    f"Test error vs exact G1      : "
    f"{G1_FIELD_ERROR_TRUE_TEST:.6e}"
)


print()
print("False positives")
print("-" * 70)

print(
    G1_FALSE_POSITIVES
    if G1_FALSE_POSITIVES
    else "None"
)


print()
print("False negatives")
print("-" * 70)

print(
    G1_FALSE_NEGATIVES
    if G1_FALSE_NEGATIVES
    else "None"
)


# ================================================================
# Save formal pilot result
# ================================================================

G1_ORDER_METRICS.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_strict_block_order_metrics.csv",
    index=False,
)


G1_RESULT.path_ledger.to_pickle(
    OUTPUT_DIR
    /
    "stage6_G1_strict_block_path_ledger.pkl"
)


np.savez_compressed(
    OUTPUT_DIR
    /
    "stage6_G1_strict_block_result.npz",

    coefficients_raw=
        B_G1_EST,

    feature_scale=
        G1_LIBRARY.feature_scale,

    fit_indices=
        IDX_G1_FIT,

    validation_indices=
        IDX_G1_VAL,

    test_indices=
        IDX_G1_TEST,
)


print()
print("=" * 70)
print("Cell 5 completed.")
print("=" * 70)

G1 MICROSCOPIC NETWORK INFERENCE FROM ONE TIME SERIES
local regime duration DeltaT: 0.06
response horizons epsilon   : [0.005  0.0075 0.01   0.015 ]
maximum response span       : 0.015
slice-start stride          : 0.0015

fit starts                  : 124 (cycles 0-3)
validation starts           : 31 (cycle 4)
external test starts        : 31 (cycle 5)

broad scalar features       : 164
candidate structural groups : 162
fit generator RMS           : 2.254865e-01
validation generator RMS    : 6.498800e-02
test generator RMS          : 5.176490e-02

STARTING BLOCK-AWARE v3.6 AGLASSO
Starting KKT working-set Adaptive Group LASSO for Stage6 one-series G1 F^(0) (KKT-event-driven validation path)...
  internal split = 124 fit / 31 validation states
  adaptive path <= 40 lambda evaluations | geometric ratio=0.750 | entry fraction=0.980
  maximum KKT jump = 2.00 decades | emergency floor = 1e-10 * lambda_max
  local refinement <= 4 lambdas around an interior minimum validation-loss adaptive p

,order,true,selected,TP,FP,FN,precision,recall,F1
0,1,8,0,0,0,8,0.000000,0.000000,0.000000
1,2,9,8,1,7,8,0.125000,0.111111,0.117647
2,3,2,3,0,3,2,0.000000,0.000000,0.000000
3,4,0,0,0,0,0,0.000000,0.000000,0.000000



Coefficient / field diagnostics
----------------------------------------------------------------------
Coefficient relative error  : 9.891938e+01

Fit error vs A_est          : 5.113781e-04
Fit error vs exact G1       : 5.113781e-04

Validation error vs A_est   : 1.831666e-03
Validation error vs exact G1: 1.831666e-03

Test error vs A_est         : 1.394335e-02
Test error vs exact G1      : 1.394335e-02

False positives
----------------------------------------------------------------------
((2, 4), (3, 5), (3, 6), (3, 8), (4, 8), (6, 7), (6, 8), (1, 3, 4), (1, 4, 5), (3, 6, 8))

False negatives
----------------------------------------------------------------------
((1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,), (1, 2), (1, 3), (2, 3), (2, 5), (2, 8), (3, 4), (5, 6), (5, 8), (1, 2, 3), (2, 5, 8))

Cell 5 completed.


In [7]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 5b: diagnose structural non-identifiability
#
# NO sparse inference is run here.
#
# Questions
# ---------------------------------------------------------------
# 1. Does the conserved quantity generate exact polynomial
#    identities inside our degree-1/2/3 dictionary?
#
# 2. Can the true G1 field be represented on the observed
#    trajectory after removing:
#
#       A. the true active scalar directions?
#
#       B. ALL directions belonging to true structural supports?
#
# 3. Does the wrong 11-support model remain accurate:
#
#       - on the observed trajectory?
#       - elsewhere on the same invariant hyperplane?
#       - away from the invariant hyperplane?
#
# This separates:
#
#       endpoint/generator reconstruction
#       structural identifiability
#       invariant-manifold ambiguity
#       single-orbit ambiguity
# ================================================================


# ================================================================
# 0. Basic objects
# ================================================================

C_TRAJ = float(
    INITIAL_MASS
)

assert abs(C_TRAJ) > 1e-12


X_G1_ALL = np.vstack(
    [
        X_G1_FIT,
        X_G1_VAL,
        X_G1_TEST,
    ]
)

G1_TRUE_ALL = np.vstack(
    [
        G1_TRUE_FIT,
        G1_TRUE_VAL,
        G1_TRUE_TEST,
    ]
)


print("=" * 70)
print("CELL 5b — STRUCTURAL NON-IDENTIFIABILITY DIAGNOSIS")
print("=" * 70)

print(
    f"Conserved mass C            : "
    f"{C_TRAJ:.12f}"
)

print(
    f"G1 observed states          : "
    f"{len(X_G1_ALL)}"
)


# ================================================================
# 1. Direct conservation-law audit
#
# Entire trajectory obeys:
#
#       sum_j x_j = C
#
# Therefore, on this invariant manifold:
#
#       x_i
#         =
#       (1/C) sum_j x_i x_j
#
# and:
#
#       x_i x_j
#         =
#       (1/C) sum_k x_i x_j x_k
#
# These identities connect different polynomial degrees.
# ================================================================

mass_residual = (
    X_G1_ALL.sum(
        axis=1
    )
    -
    C_TRAJ
)

MAX_MASS_IDENTITY_ERROR = float(
    np.max(
        np.abs(
            mass_residual
        )
    )
)


# ------------------------------------------------
# Degree 1 -> degree 2 lifting
# ------------------------------------------------

linear_lift_errors = []

sum_x = (
    X_G1_ALL.sum(
        axis=1
    )
)


for i in range(N):

    lhs = (
        X_G1_ALL[
            :,
            i
        ]
    )

    rhs = (
        X_G1_ALL[
            :,
            i
        ]
        *
        sum_x
        /
        C_TRAJ
    )

    err = float(
        np.max(
            np.abs(
                lhs
                -
                rhs
            )
        )
    )

    linear_lift_errors.append(
        err
    )


MAX_LINEAR_TO_QUADRATIC_ERROR = float(
    max(
        linear_lift_errors
    )
)


# ------------------------------------------------
# Degree 2 -> degree 3 lifting
# ------------------------------------------------

quadratic_lift_errors = []


for i in range(N):

    for j in range(
        i,
        N
    ):

        lhs = (
            X_G1_ALL[
                :,
                i
            ]
            *
            X_G1_ALL[
                :,
                j
            ]
        )

        rhs = (
            lhs
            *
            sum_x
            /
            C_TRAJ
        )

        err = float(
            np.max(
                np.abs(
                    lhs
                    -
                    rhs
                )
            )
        )

        quadratic_lift_errors.append(
            err
        )


MAX_QUADRATIC_TO_CUBIC_ERROR = float(
    max(
        quadratic_lift_errors
    )
)


print()
print("Exact invariant-manifold identities")
print("-" * 70)

print(
    f"max |sum(x)-C|              : "
    f"{MAX_MASS_IDENTITY_ERROR:.6e}"
)

print(
    f"max degree 1 -> 2 residual  : "
    f"{MAX_LINEAR_TO_QUADRATIC_ERROR:.6e}"
)

print(
    f"max degree 2 -> 3 residual  : "
    f"{MAX_QUADRATIC_TO_CUBIC_ERROR:.6e}"
)


# ================================================================
# 2. Structural support of every scalar direction
#
# A scalar direction is:
#
#       monomial feature q
#       +
#       output node i
#
# Its TSC structural support is:
#
#       variables(monomial)
#           union
#       {output node}
# ================================================================

EXPONENTS_G1 = [
    tuple(
        int(v)
        for v
        in exponent
    )

    for exponent
    in G1_LIBRARY.exponents
]


def scalar_direction_support(
    feature_index,
    output_index,
):

    exponent = (
        EXPONENTS_G1[
            feature_index
        ]
    )

    variables = {
        j + 1

        for j, power
        in enumerate(
            exponent
        )

        if power > 0
    }

    return tuple(
        sorted(
            variables
            |
            {
                output_index + 1
            }
        )
    )


DIRECTION_SUPPORT_G1 = np.empty(
    (
        G1_LIBRARY.n_features,
        N,
    ),
    dtype=object,
)


for feature_index in range(
    G1_LIBRARY.n_features
):

    for output_index in range(N):

        DIRECTION_SUPPORT_G1[
            feature_index,
            output_index,
        ] = scalar_direction_support(
            feature_index,
            output_index,
        )


# ================================================================
# 3. Design matrices using the SAME fit-only feature scaling
#    already frozen in Cell 5
# ================================================================

THETA_RAW_G1_FIT = (
    G1_LIBRARY.evaluate_raw(
        X_G1_FIT
    )
)

THETA_RAW_G1_VAL = (
    G1_LIBRARY.evaluate_raw(
        X_G1_VAL
    )
)

THETA_RAW_G1_TEST = (
    G1_LIBRARY.evaluate_raw(
        X_G1_TEST
    )
)


# Scaled versions already exist from Cell 5:
#
#       THETA_G1_FIT
#       THETA_G1_VAL
#       THETA_G1_TEST
#
# We use the scaled matrices for least-squares stability.


VALID_MASK_G1 = np.asarray(
    G1_LIBRARY.valid_mask,
    dtype=bool,
)

assert VALID_MASK_G1.shape == (
    G1_LIBRARY.n_features,
    N,
)


TRUE_SCALAR_MASK_G1 = (
    np.abs(
        B_G1_TRUE
    )
    >
    1e-14
)


# ================================================================
# Helper: scalar relative error
# ================================================================

def relative_vector_error(
    estimate,
    truth,
):

    estimate = np.asarray(
        estimate,
        dtype=float,
    )

    truth = np.asarray(
        truth,
        dtype=float,
    )

    denominator = max(
        float(
            np.linalg.norm(
                truth
            )
        ),
        np.finfo(float).tiny,
    )

    return float(
        np.linalg.norm(
            estimate
            -
            truth
        )
        /
        denominator
    )


# ================================================================
# Helper: alternative-representation least squares
#
# Fit ONLY on cycles 0-3.
# Evaluate unchanged coefficients on validation/test.
# ================================================================

def alternative_ls_test(
    output_index,
    candidate_mask,
):

    candidate_indices = np.flatnonzero(
        candidate_mask
    )

    if len(
        candidate_indices
    ) == 0:

        return {
            "n_candidates":
                0,

            "rank":
                0,

            "train_error":
                np.inf,

            "validation_error":
                np.inf,

            "test_error":
                np.inf,
        }


    A_fit = (
        THETA_G1_FIT[
            :,
            candidate_indices
        ]
    )

    y_fit = (
        G1_TRUE_FIT[
            :,
            output_index
        ]
    )


    coefficient, _, rank, _ = (
        np.linalg.lstsq(
            A_fit,
            y_fit,
            rcond=None,
        )
    )


    pred_fit = (
        A_fit
        @
        coefficient
    )

    pred_val = (
        THETA_G1_VAL[
            :,
            candidate_indices
        ]
        @
        coefficient
    )

    pred_test = (
        THETA_G1_TEST[
            :,
            candidate_indices
        ]
        @
        coefficient
    )


    return {
        "n_candidates":
            len(
                candidate_indices
            ),

        "rank":
            int(
                rank
            ),

        "train_error":
            relative_vector_error(
                pred_fit,
                G1_TRUE_FIT[
                    :,
                    output_index
                ],
            ),

        "validation_error":
            relative_vector_error(
                pred_val,
                G1_TRUE_VAL[
                    :,
                    output_index
                ],
            ),

        "test_error":
            relative_vector_error(
                pred_test,
                G1_TRUE_TEST[
                    :,
                    output_index
                ],
            ),
    }


# ================================================================
# 4. Alternative representation A
#
# Remove every TRUE ACTIVE SCALAR DIRECTION.
#
# Question:
#
# Can inactive scalar directions reproduce the true field anyway?
# ================================================================

inactive_scalar_rows = []


for output_index in range(N):

    candidate_mask = (
        VALID_MASK_G1[
            :,
            output_index
        ]
        &
        (
            ~TRUE_SCALAR_MASK_G1[
                :,
                output_index
            ]
        )
    )


    result = alternative_ls_test(
        output_index,
        candidate_mask,
    )


    inactive_scalar_rows.append({

        "output_node":
            output_index + 1,

        "true_active_scalar_directions":
            int(
                TRUE_SCALAR_MASK_G1[
                    :,
                    output_index
                ].sum()
            ),

        **result,
    })


INACTIVE_SCALAR_ALIAS_AUDIT = (
    pd.DataFrame(
        inactive_scalar_rows
    )
)


# ================================================================
# 5. Alternative representation B — stronger test
#
# Remove EVERY scalar direction whose structural support belongs
# to the true G1 structural network.
#
# Therefore the remaining dictionary uses ONLY structurally
# FALSE groups.
#
# Question:
#
# Can a completely wrong structural network reproduce G1 on
# this trajectory?
# ================================================================

false_support_rows = []


for output_index in range(N):

    false_structural_mask = np.array(
        [
            (
                DIRECTION_SUPPORT_G1[
                    feature_index,
                    output_index
                ]
                not in
                TRUE_G1_SUPPORTS
            )

            for feature_index
            in range(
                G1_LIBRARY.n_features
            )
        ],
        dtype=bool,
    )


    candidate_mask = (
        VALID_MASK_G1[
            :,
            output_index
        ]
        &
        false_structural_mask
    )


    result = alternative_ls_test(
        output_index,
        candidate_mask,
    )


    false_support_rows.append({

        "output_node":
            output_index + 1,

        **result,
    })


FALSE_SUPPORT_ALIAS_AUDIT = (
    pd.DataFrame(
        false_support_rows
    )
)


# ================================================================
# 6. Aggregate alternative-model errors
#
# Instead of averaging relative errors over nodes, construct
# complete vector fields from each alternative model.
# ================================================================

def build_alternative_field(
    mode,
):

    pred_fit = np.zeros_like(
        G1_TRUE_FIT
    )

    pred_val = np.zeros_like(
        G1_TRUE_VAL
    )

    pred_test = np.zeros_like(
        G1_TRUE_TEST
    )


    for output_index in range(N):

        if mode == "inactive_scalar":

            candidate_mask = (
                VALID_MASK_G1[
                    :,
                    output_index
                ]
                &
                (
                    ~TRUE_SCALAR_MASK_G1[
                        :,
                        output_index
                    ]
                )
            )


        elif mode == "false_support":

            false_structural_mask = np.array(
                [
                    (
                        DIRECTION_SUPPORT_G1[
                            feature_index,
                            output_index
                        ]
                        not in
                        TRUE_G1_SUPPORTS
                    )

                    for feature_index
                    in range(
                        G1_LIBRARY.n_features
                    )
                ],
                dtype=bool,
            )

            candidate_mask = (
                VALID_MASK_G1[
                    :,
                    output_index
                ]
                &
                false_structural_mask
            )


        else:

            raise ValueError(
                mode
            )


        idx = np.flatnonzero(
            candidate_mask
        )


        if len(idx) == 0:
            continue


        coef, _, _, _ = (
            np.linalg.lstsq(
                THETA_G1_FIT[
                    :,
                    idx
                ],
                G1_TRUE_FIT[
                    :,
                    output_index
                ],
                rcond=None,
            )
        )


        pred_fit[
            :,
            output_index
        ] = (
            THETA_G1_FIT[
                :,
                idx
            ]
            @
            coef
        )

        pred_val[
            :,
            output_index
        ] = (
            THETA_G1_VAL[
                :,
                idx
            ]
            @
            coef
        )

        pred_test[
            :,
            output_index
        ] = (
            THETA_G1_TEST[
                :,
                idx
            ]
            @
            coef
        )


    return (
        pred_fit,
        pred_val,
        pred_test,
    )


(
    ALT_INACTIVE_FIT,
    ALT_INACTIVE_VAL,
    ALT_INACTIVE_TEST,
) = build_alternative_field(
    "inactive_scalar"
)


(
    ALT_FALSE_SUPPORT_FIT,
    ALT_FALSE_SUPPORT_VAL,
    ALT_FALSE_SUPPORT_TEST,
) = build_alternative_field(
    "false_support"
)


ALTERNATIVE_FIELD_SUMMARY = pd.DataFrame(
    [
        {
            "model":
                "remove true scalar directions",

            "fit_error":
                relative_fro_error(
                    ALT_INACTIVE_FIT,
                    G1_TRUE_FIT,
                ),

            "validation_error":
                relative_fro_error(
                    ALT_INACTIVE_VAL,
                    G1_TRUE_VAL,
                ),

            "test_error":
                relative_fro_error(
                    ALT_INACTIVE_TEST,
                    G1_TRUE_TEST,
                ),
        },

        {
            "model":
                "use only false structural supports",

            "fit_error":
                relative_fro_error(
                    ALT_FALSE_SUPPORT_FIT,
                    G1_TRUE_FIT,
                ),

            "validation_error":
                relative_fro_error(
                    ALT_FALSE_SUPPORT_VAL,
                    G1_TRUE_VAL,
                ),

            "test_error":
                relative_fro_error(
                    ALT_FALSE_SUPPORT_TEST,
                    G1_TRUE_TEST,
                ),
        },
    ]
)


# ================================================================
# 7. Off-orbit generalisation of the SELECTED Cell-5 model
#
# Three geometries:
#
# A. observed G1 trajectory states
#
# B. random states constrained to the SAME conserved hyperplane
#
#       sum_i x_i = C
#
# C. unrestricted random states
#
# If B remains accurate but C fails:
#
#       conservation manifold is the dominant ambiguity.
#
# If B also fails badly:
#
#       the particular 1-D trajectory/orbit geometry adds another
#       strong source of non-identifiability.
# ================================================================

OFF_ORBIT_SEED = (
    SEED
    +
    505
)

off_orbit_rng = np.random.default_rng(
    OFF_ORBIT_SEED
)

N_OFF_ORBIT = 2000


# ------------------------------------------------
# Unrestricted random cloud
# ------------------------------------------------

X_RANDOM_FREE = (
    off_orbit_rng.uniform(
        -0.5,
        0.5,
        size=(
            N_OFF_ORBIT,
            N,
        ),
    )
)


# ------------------------------------------------
# Same-mass random cloud
#
# Start with random deviations, remove row mean, then add C/N.
#
# Every row satisfies:
#
#       sum_i x_i = C
#
# but these states are NOT points from the observed trajectory.
# ------------------------------------------------

X_RANDOM_SAME_MASS = (
    off_orbit_rng.uniform(
        -0.5,
        0.5,
        size=(
            N_OFF_ORBIT,
            N,
        ),
    )
)

X_RANDOM_SAME_MASS -= (
    X_RANDOM_SAME_MASS.mean(
        axis=1,
        keepdims=True,
    )
)

X_RANDOM_SAME_MASS += (
    C_TRAJ
    /
    N
)


SAME_MASS_RESIDUAL = float(
    np.max(
        np.abs(
            X_RANDOM_SAME_MASS.sum(
                axis=1
            )
            -
            C_TRAJ
        )
    )
)

assert (
    SAME_MASS_RESIDUAL
    <
    1e-12
)


# ------------------------------------------------
# Exact G1 fields
# ------------------------------------------------

G1_RANDOM_SAME_MASS_TRUE = (
    snapshot_field_numeric(
        X_RANDOM_SAME_MASS,
        SNAPSHOT_PILOT,
    )
)

G1_RANDOM_FREE_TRUE = (
    snapshot_field_numeric(
        X_RANDOM_FREE,
        SNAPSHOT_PILOT,
    )
)


# ------------------------------------------------
# Cell-5 selected sparse model predictions
#
# Uses the feature scaling learned strictly from cycles 0-3.
# ------------------------------------------------

THETA_RANDOM_SAME_MASS = (
    G1_LIBRARY.transform(
        X_RANDOM_SAME_MASS
    )
)

THETA_RANDOM_FREE = (
    G1_LIBRARY.transform(
        X_RANDOM_FREE
    )
)


G1_RANDOM_SAME_MASS_PRED = (
    THETA_RANDOM_SAME_MASS
    @
    G1_RESULT.coefficients_scaled
)

G1_RANDOM_FREE_PRED = (
    THETA_RANDOM_FREE
    @
    G1_RESULT.coefficients_scaled
)


# ------------------------------------------------
# Observed trajectory prediction
# ------------------------------------------------

THETA_G1_ALL = np.vstack(
    [
        THETA_G1_FIT,
        THETA_G1_VAL,
        THETA_G1_TEST,
    ]
)

G1_TRAJECTORY_PRED = (
    THETA_G1_ALL
    @
    G1_RESULT.coefficients_scaled
)


OFF_ORBIT_GENERALISATION = pd.DataFrame(
    [
        {
            "state_geometry":
                "observed G1 trajectory",

            "n_states":
                len(
                    X_G1_ALL
                ),

            "field_rms":
                rms_field(
                    G1_TRUE_ALL
                ),

            "relative_error":
                relative_fro_error(
                    G1_TRAJECTORY_PRED,
                    G1_TRUE_ALL,
                ),
        },

        {
            "state_geometry":
                "random same-mass hyperplane",

            "n_states":
                N_OFF_ORBIT,

            "field_rms":
                rms_field(
                    G1_RANDOM_SAME_MASS_TRUE
                ),

            "relative_error":
                relative_fro_error(
                    G1_RANDOM_SAME_MASS_PRED,
                    G1_RANDOM_SAME_MASS_TRUE,
                ),
        },

        {
            "state_geometry":
                "unrestricted random states",

            "n_states":
                N_OFF_ORBIT,

            "field_rms":
                rms_field(
                    G1_RANDOM_FREE_TRUE
                ),

            "relative_error":
                relative_fro_error(
                    G1_RANDOM_FREE_PRED,
                    G1_RANDOM_FREE_TRUE,
                ),
        },
    ]
)


# ================================================================
# 8. Compare feature-space rank on the three geometries
# ================================================================

rank_rows = []


for label, Theta in (

    (
        "G1 fit trajectory",
        THETA_G1_FIT,
    ),

    (
        "random same-mass hyperplane",
        THETA_RANDOM_SAME_MASS,
    ),

    (
        "unrestricted random states",
        THETA_RANDOM_FREE,
    ),
):

    diag = (
        numerical_rank_and_condition(
            Theta
        )
    )

    rank_rows.append({
        "state_geometry":
            label,

        "n_states":
            Theta.shape[0],

        "n_features":
            Theta.shape[1],

        "rank":
            diag[
                "rank"
            ],

        "nullity":
            diag[
                "nullity"
            ],

        "condition_nonzero":
            diag[
                "condition_nonzero"
            ],
    })


GEOMETRY_RANK_AUDIT = pd.DataFrame(
    rank_rows
)


# ================================================================
# 9. Report
# ================================================================

print()
print("=" * 70)
print("A. INVARIANT-MANIFOLD POLYNOMIAL ALIASING")
print("=" * 70)

print(
    f"max conservation residual   : "
    f"{MAX_MASS_IDENTITY_ERROR:.6e}"
)

print(
    f"degree 1 -> 2 identity error: "
    f"{MAX_LINEAR_TO_QUADRATIC_ERROR:.6e}"
)

print(
    f"degree 2 -> 3 identity error: "
    f"{MAX_QUADRATIC_TO_CUBIC_ERROR:.6e}"
)


print()
print("=" * 70)
print("B. REMOVE TRUE SCALAR DIRECTIONS")
print("=" * 70)

display(
    INACTIVE_SCALAR_ALIAS_AUDIT.style.format({
        "train_error":
            "{:.6e}",

        "validation_error":
            "{:.6e}",

        "test_error":
            "{:.6e}",
    })
)


print()
print("=" * 70)
print("C. USE ONLY FALSE STRUCTURAL SUPPORTS")
print("=" * 70)

display(
    FALSE_SUPPORT_ALIAS_AUDIT.style.format({
        "train_error":
            "{:.6e}",

        "validation_error":
            "{:.6e}",

        "test_error":
            "{:.6e}",
    })
)


print()
print("Aggregate alternative-field errors")
print("-" * 70)

display(
    ALTERNATIVE_FIELD_SUMMARY.style.format({
        "fit_error":
            "{:.6e}",

        "validation_error":
            "{:.6e}",

        "test_error":
            "{:.6e}",
    })
)


print()
print("=" * 70)
print("D. CELL-5 MODEL: ON-ORBIT VS OFF-ORBIT")
print("=" * 70)

display(
    OFF_ORBIT_GENERALISATION.style.format({
        "field_rms":
            "{:.6e}",

        "relative_error":
            "{:.6e}",
    })
)


print()
print("=" * 70)
print("E. FEATURE-SPACE RANK BY STATE GEOMETRY")
print("=" * 70)

display(
    GEOMETRY_RANK_AUDIT.style.format({
        "condition_nonzero":
            "{:.6e}",
    })
)


# ================================================================
# 10. Save
# ================================================================

INACTIVE_SCALAR_ALIAS_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_inactive_scalar_alias_audit.csv",
    index=False,
)

FALSE_SUPPORT_ALIAS_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_false_support_alias_audit.csv",
    index=False,
)

ALTERNATIVE_FIELD_SUMMARY.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_alternative_field_summary.csv",
    index=False,
)

OFF_ORBIT_GENERALISATION.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_off_orbit_generalisation.csv",
    index=False,
)

GEOMETRY_RANK_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_geometry_rank_audit.csv",
    index=False,
)


print()
print("=" * 70)
print("Cell 5b PASSED.")
print("=" * 70)

CELL 5b — STRUCTURAL NON-IDENTIFIABILITY DIAGNOSIS
Conserved mass C            : -1.104766581081
G1 observed states          : 186

Exact invariant-manifold identities
----------------------------------------------------------------------
max |sum(x)-C|              : 8.881784e-16
max degree 1 -> 2 residual  : 3.330669e-16
max degree 2 -> 3 residual  : 1.387779e-16

A. INVARIANT-MANIFOLD POLYNOMIAL ALIASING
max conservation residual   : 8.881784e-16
degree 1 -> 2 identity error: 3.330669e-16
degree 2 -> 3 identity error: 1.387779e-16

B. REMOVE TRUE SCALAR DIRECTIONS


,output_node,true_active_scalar_directions,n_candidates,rank,train_error,validation_error,test_error
0,1,7,157,25,6.912864e-15,4.819342e-03,6.513657e-03
1,2,10,154,25,8.249450e-15,5.091919e-03,5.405019e-03
2,3,7,157,25,1.079551e-15,2.090743e-03,1.070286e-02
3,4,4,160,25,1.356513e-15,2.295866e-03,1.161578e-02
4,5,7,157,25,3.888536e-15,7.063339e-03,4.400824e-02
5,6,4,160,25,3.641672e-15,6.545661e-03,3.987357e-02
6,7,4,160,25,5.556437e-15,7.359135e-04,2.220531e-03
7,8,7,157,25,5.677861e-15,7.768396e-04,2.321524e-03



C. USE ONLY FALSE STRUCTURAL SUPPORTS


,output_node,n_candidates,rank,train_error,validation_error,test_error
0,1,145,25,7.090958e-15,2.681368e-02,5.384005e-02
1,2,129,25,8.189647e-15,1.306661e-02,1.567177e-02
2,3,139,25,2.602109e-15,9.687374e-05,3.571985e-04
3,4,155,25,1.516883e-15,1.720865e-03,8.809523e-03
4,5,139,25,3.790928e-15,7.060191e-03,4.288043e-02
5,6,155,24,3.627257e-15,5.897913e-03,3.487960e-02
6,7,155,25,7.094856e-15,7.126722e-04,2.023094e-03
7,8,139,25,6.918215e-15,5.524607e-04,1.390699e-03



Aggregate alternative-field errors
----------------------------------------------------------------------


...


D. CELL-5 MODEL: ON-ORBIT VS OFF-ORBIT


,state_geometry,n_states,field_rms,relative_error
0,observed G1 trajectory,186,1.872074e-01,1.672655e-03
1,random same-mass hyperplane,2000,3.762259e-01,1.574702e+01
2,unrestricted random states,2000,4.341214e-01,1.442249e+01



E. FEATURE-SPACE RANK BY STATE GEOMETRY


,state_geometry,n_states,n_features,rank,nullity,condition_nonzero
0,G1 fit trajectory,124,164,25,139,2.551135e+13
1,random same-mass hyperplane,2000,164,120,44,2.314722e+02
2,unrestricted random states,2000,164,164,0,3.809522e+02



Cell 5b PASSED.


In [8]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 6: prior-ladder structural identifiability audit
#
# No sparse inference is run here.
#
# Compare, on EXACTLY the same G1 fit trajectory states:
#
#   P0 : generic polynomial
#        degree <= 3, support <= 4
#
#   P1 : correct maximum polynomial degree
#        degree <= 2, support <= 3
#
#   P2 : known physical interaction functions
#        all 28 pair candidates
#        all 56 triad candidates
#        topology / coefficients unknown
#
# Questions
# ---------------------------------------------------------------
# 1. How many parameter directions remain?
# 2. What is their numerical rank / nullity?
# 3. How ill-conditioned are they?
# 4. Is the true active subspace identifiable?
# 5. Can FALSE structural candidates alone still reproduce G1?
#
# This tests whether additional prior information removes the
# single-trajectory structural degeneracy found in Cell 5b.
# ================================================================

from itertools import combinations


# ================================================================
# Common data
# ================================================================

X_PRIOR_AUDIT = (
    X_G1_FIT
)

Y_PRIOR_AUDIT = (
    G1_TRUE_FIT
)

TRUE_G1_MINIMAL_SUPPORTS = set(
    SNAPSHOT_ORACLE_SUPPORTS[
        SNAPSHOT_PILOT
    ]
)


assert X_PRIOR_AUDIT.shape == (
    124,
    N,
)

assert Y_PRIOR_AUDIT.shape == (
    124,
    N,
)


# ================================================================
# Numerical helpers
# ================================================================

def prior_svd_diagnostics(
    X,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    if (
        X.ndim != 2
        or X.shape[1] == 0
    ):
        return {
            "rank": 0,
            "nullity": (
                X.shape[1]
                if X.ndim == 2
                else 0
            ),
            "condition_nonzero": np.inf,
            "smax": 0.0,
            "smin_nonzero": 0.0,
        }

    s = np.linalg.svd(
        X,
        compute_uv=False,
    )

    if len(s) == 0:
        return {
            "rank": 0,
            "nullity": X.shape[1],
            "condition_nonzero": np.inf,
            "smax": 0.0,
            "smin_nonzero": 0.0,
        }

    tol = (
        max(
            X.shape
        )
        *
        np.finfo(float).eps
        *
        s[0]
    )

    nz = (
        s[
            s > tol
        ]
    )

    rank = int(
        len(nz)
    )

    if rank == 0:
        condition = np.inf
        smin = 0.0
    else:
        condition = float(
            nz[0]
            /
            nz[-1]
        )

        smin = float(
            nz[-1]
        )

    return {
        "rank":
            rank,

        "nullity":
            int(
                X.shape[1]
                -
                rank
            ),

        "condition_nonzero":
            condition,

        "smax":
            float(
                s[0]
            ),

        "smin_nonzero":
            smin,
    }


def normalized_column_coherence(
    X,
):
    """
    Maximum absolute normalized inner product between
    distinct nonzero columns.
    """

    X = np.asarray(
        X,
        dtype=float,
    )

    norms = np.linalg.norm(
        X,
        axis=0,
    )

    keep = (
        norms
        >
        0.0
    )

    Xn = (
        X[
            :,
            keep
        ]
        /
        norms[
            keep
        ][
            None,
            :
        ]
    )

    if Xn.shape[1] <= 1:
        return 0.0

    gram = (
        Xn.T
        @
        Xn
    )

    np.fill_diagonal(
        gram,
        0.0,
    )

    return float(
        np.max(
            np.abs(
                gram
            )
        )
    )


# ================================================================
# Robust symbolic -> library coefficient conversion
#
# Unlike the old Cell-4 helper, this constructs the exponent map
# separately for EACH library.
# ================================================================

def symbolic_field_to_library_raw(
    field_symbolic,
    library,
):

    exponent_to_feature = {
        tuple(
            int(v)
            for v
            in exponent
        ):
            feature_index

        for feature_index, exponent
        in enumerate(
            library.exponents
        )
    }

    B_raw = np.zeros(
        (
            library.n_features,
            N,
        ),
        dtype=float,
    )

    for output_index in range(N):

        poly = sp.Poly(
            sp.expand(
                field_symbolic[
                    output_index
                ]
            ),
            *x_symbols,
        )

        for exponent, coefficient in (
            poly.terms()
        ):

            if (
                sp.simplify(
                    coefficient
                )
                ==
                0
            ):
                continue

            exponent = tuple(
                int(v)
                for v
                in exponent
            )

            if (
                exponent
                not in
                exponent_to_feature
            ):
                raise RuntimeError(
                    "True G1 field lies outside "
                    "this prior dictionary: "
                    f"output={output_index + 1}, "
                    f"exponent={exponent}, "
                    f"coefficient={coefficient}"
                )

            feature_index = (
                exponent_to_feature[
                    exponent
                ]
            )

            if not (
                library.valid_mask[
                    feature_index,
                    output_index
                ]
            ):
                raise RuntimeError(
                    "True term exists in polynomial basis "
                    "but violates structural support limit."
                )

            B_raw[
                feature_index,
                output_index,
            ] = float(
                coefficient
            )

    return B_raw


# ================================================================
# Minimal structural support for a scalar polynomial direction
# ================================================================

def library_direction_support(
    library,
    feature_index,
    output_index,
):

    exponent = (
        library.exponents[
            feature_index
        ]
    )

    variables = {
        j + 1

        for j, power
        in enumerate(
            exponent
        )

        if power > 0
    }

    return tuple(
        sorted(
            variables
            |
            {
                output_index + 1
            }
        )
    )


# ================================================================
# Broad polynomial-prior audit
#
# The coefficient model is:
#
#       Y = Theta B
#
# so each valid (feature, output) pair is one scalar parameter
# direction.
#
# Because outputs occupy independent row blocks in the equivalent
# vectorized regression, total parameter rank is the SUM of the
# per-output design ranks.
# ================================================================

def audit_broad_polynomial_prior(
    name,
    library,
    Theta_scaled,
    B_true_raw,
):

    valid_mask = np.asarray(
        library.valid_mask,
        dtype=bool,
    )

    # ------------------------------------------------------------
    # Exact representability
    # ------------------------------------------------------------

    Theta_raw = (
        library.evaluate_raw(
            X_PRIOR_AUDIT
        )
    )

    Y_from_true_dictionary = (
        Theta_raw
        @
        B_true_raw
    )

    representation_error = (
        relative_fro_error(
            Y_from_true_dictionary,
            Y_PRIOR_AUDIT,
        )
    )

    # ------------------------------------------------------------
    # Feature-space geometry
    # ------------------------------------------------------------

    feature_diag = (
        prior_svd_diagnostics(
            Theta_scaled
        )
    )

    feature_coherence = (
        normalized_column_coherence(
            Theta_scaled
        )
    )

    # ------------------------------------------------------------
    # Full coefficient-direction geometry
    #
    # Each output has its own polynomial coefficient vector.
    # ------------------------------------------------------------

    parameter_directions = 0
    parameter_rank = 0

    output_conditions = []

    for output_index in range(N):

        valid_features = np.flatnonzero(
            valid_mask[
                :,
                output_index
            ]
        )

        parameter_directions += len(
            valid_features
        )

        diag_i = (
            prior_svd_diagnostics(
                Theta_scaled[
                    :,
                    valid_features
                ]
            )
        )

        parameter_rank += (
            diag_i[
                "rank"
            ]
        )

        output_conditions.append(
            diag_i[
                "condition_nonzero"
            ]
        )

    parameter_nullity = (
        parameter_directions
        -
        parameter_rank
    )

    # ------------------------------------------------------------
    # True active scalar subspace
    # ------------------------------------------------------------

    true_scalar_mask = (
        np.abs(
            B_true_raw
        )
        >
        1e-14
    )

    true_scalar_directions = int(
        true_scalar_mask.sum()
    )

    true_output_full_rank = []
    true_output_conditions = []

    for output_index in range(N):

        active_features = np.flatnonzero(
            true_scalar_mask[
                :,
                output_index
            ]
        )

        if len(active_features) == 0:

            true_output_full_rank.append(
                True
            )

            true_output_conditions.append(
                1.0
            )

            continue

        diag_true = (
            prior_svd_diagnostics(
                Theta_scaled[
                    :,
                    active_features
                ]
            )
        )

        true_output_full_rank.append(
            diag_true[
                "rank"
            ]
            ==
            len(
                active_features
            )
        )

        true_output_conditions.append(
            diag_true[
                "condition_nonzero"
            ]
        )

    # ------------------------------------------------------------
    # Strong alias test:
    #
    # REMOVE EVERY scalar direction whose structural support is
    # one of the true G1 minimal supports.
    #
    # Then ask whether a completely false structural network can
    # still reproduce G1 on the observed trajectory.
    # ------------------------------------------------------------

    Y_false_only = np.zeros_like(
        Y_PRIOR_AUDIT
    )

    false_parameter_count = 0

    for output_index in range(N):

        candidate_features = []

        for feature_index in range(
            library.n_features
        ):

            if not (
                valid_mask[
                    feature_index,
                    output_index
                ]
            ):
                continue

            support = (
                library_direction_support(
                    library,
                    feature_index,
                    output_index,
                )
            )

            if (
                support
                in
                TRUE_G1_MINIMAL_SUPPORTS
            ):
                continue

            candidate_features.append(
                feature_index
            )

        candidate_features = np.asarray(
            candidate_features,
            dtype=int,
        )

        false_parameter_count += len(
            candidate_features
        )

        if len(candidate_features) == 0:
            continue

        coef, _, _, _ = (
            np.linalg.lstsq(
                Theta_scaled[
                    :,
                    candidate_features
                ],
                Y_PRIOR_AUDIT[
                    :,
                    output_index
                ],
                rcond=None,
            )
        )

        Y_false_only[
            :,
            output_index
        ] = (
            Theta_scaled[
                :,
                candidate_features
            ]
            @
            coef
        )

    false_only_error = (
        relative_fro_error(
            Y_false_only,
            Y_PRIOR_AUDIT,
        )
    )

    return {
        "prior":
            name,

        "dictionary_type":
            "generic polynomial",

        "n_features":
            library.n_features,

        "candidate_groups":
            len(
                library.structural_groups
            ),

        "parameter_directions":
            parameter_directions,

        "parameter_rank":
            parameter_rank,

        "parameter_nullity":
            parameter_nullity,

        "parameter_rank_fraction":
            (
                parameter_rank
                /
                parameter_directions
            ),

        "feature_rank":
            feature_diag[
                "rank"
            ],

        "feature_nullity":
            feature_diag[
                "nullity"
            ],

        "max_coherence":
            feature_coherence,

        "max_nonzero_condition":
            float(
                max(
                    output_conditions
                )
            ),

        "true_groups":
            len(
                TRUE_G1_MINIMAL_SUPPORTS
            ),

        "true_scalar_directions":
            true_scalar_directions,

        "true_active_full_rank":
            bool(
                all(
                    true_output_full_rank
                )
            ),

        "max_true_active_condition":
            float(
                max(
                    true_output_conditions
                )
            ),

        "true_representation_error":
            representation_error,

        "false_only_parameters":
            false_parameter_count,

        "false_only_fit_error":
            false_only_error,

        "full_parameter_identifiability":
            bool(
                parameter_rank
                ==
                parameter_directions
            ),
    }


# ================================================================
# P0 — original broad TSC prior
#
# degree <= 3
# support <= 4
#
# Reuse the STRICT fit-only scaling from Cell 5.
# ================================================================

P0_LIBRARY = (
    G1_LIBRARY
)

P0_THETA = (
    THETA_G1_FIT
)

P0_B_TRUE = (
    B_G1_TRUE
)


P0_AUDIT = (
    audit_broad_polynomial_prior(
        "P0: generic degree<=3",
        P0_LIBRARY,
        P0_THETA,
        P0_B_TRUE,
    )
)


# ================================================================
# P1 — correct maximum polynomial degree
#
# degree <= 2
# support <= 3
#
# This removes ALL cubic directions but does NOT provide the
# actual interaction function.
# ================================================================

P1_LIBRARY = (
    tsc_core.StructuralPolynomialLibrary(
        n_nodes=N,
        max_polynomial_degree=2,
        max_interaction_order=3,
    )
)


P1_THETA = (
    P1_LIBRARY.fit_transform(
        X_PRIOR_AUDIT
    )
)


P1_B_TRUE = (
    symbolic_field_to_library_raw(
        snapshot_fields_symbolic[
            SNAPSHOT_PILOT
        ],
        P1_LIBRARY,
    )
)


P1_AUDIT = (
    audit_broad_polynomial_prior(
        "P1: generic degree<=2",
        P1_LIBRARY,
        P1_THETA,
        P1_B_TRUE,
    )
)


# ================================================================
# P2 — known physical interaction functions
#
# UNKNOWN:
#
#       topology
#       edge coefficients
#       triad coefficients
#
# KNOWN:
#
#       pair interaction shape
#       triad interaction shape
#
# Candidate universe:
#
#       all C(8,2) = 28 pairs
#       all C(8,3) = 56 triads
#
# One candidate physical interaction = one coefficient.
#
# This is qualitatively similar to the strong functional-form
# prior used in conventional topology-inference approaches.
# ================================================================

P2_PAIR_CANDIDATES = tuple(
    combinations(
        nodes,
        2,
    )
)

P2_TRIAD_CANDIDATES = tuple(
    combinations(
        nodes,
        3,
    )
)


assert len(
    P2_PAIR_CANDIDATES
) == 28

assert len(
    P2_TRIAD_CANDIDATES
) == 56


P2_CANDIDATES = (
    tuple(
        (
            "pair",
            edge,
        )

        for edge
        in P2_PAIR_CANDIDATES
    )
    +
    tuple(
        (
            "triad",
            triad,
        )

        for triad
        in P2_TRIAD_CANDIDATES
    )
)


assert len(
    P2_CANDIDATES
) == 84


# ================================================================
# Unit-strength physical interaction fields
#
# Pair coefficient is the unknown w_ij.
#
# Triad coefficient is the unknown g_ijk.
# ================================================================

def unit_pair_interaction_field(
    X,
    edge,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    i, j = edge

    F = np.zeros_like(
        X,
        dtype=float,
    )

    interaction = (
        phi_numeric(
            X[
                ...,
                j - 1
            ]
        )
        -
        phi_numeric(
            X[
                ...,
                i - 1
            ]
        )
    )

    F[
        ...,
        i - 1
    ] += interaction

    F[
        ...,
        j - 1
    ] -= interaction

    return F


def unit_triad_interaction_field(
    X,
    triad,
):

    return triad_field_numeric(
        X,
        triad,
        g=1.0,
    )


# ================================================================
# Build vector-valued physical design
#
# Rows:
#
#       sample x output
#
# Columns:
#
#       candidate physical interaction
#
# Thus:
#
#       Psi shape = (124 * 8, 84)
# ================================================================

P2_PSI_RAW_COLUMNS = []


for interaction_type, support in (
    P2_CANDIDATES
):

    if interaction_type == "pair":

        field = (
            unit_pair_interaction_field(
                X_PRIOR_AUDIT,
                support,
            )
        )

    elif interaction_type == "triad":

        field = (
            unit_triad_interaction_field(
                X_PRIOR_AUDIT,
                support,
            )
        )

    else:

        raise RuntimeError(
            interaction_type
        )

    P2_PSI_RAW_COLUMNS.append(
        field.reshape(
            -1
        )
    )


P2_PSI_RAW = np.column_stack(
    P2_PSI_RAW_COLUMNS
)


assert P2_PSI_RAW.shape == (
    len(
        X_PRIOR_AUDIT
    )
    *
    N,
    84,
)


# ================================================================
# Fit-only RMS scaling
# ================================================================

P2_SCALE = np.sqrt(
    np.mean(
        P2_PSI_RAW ** 2,
        axis=0,
    )
)

P2_SCALE = np.where(
    P2_SCALE > 0.0,
    P2_SCALE,
    1.0,
)


P2_PSI = (
    P2_PSI_RAW
    /
    P2_SCALE[
        None,
        :
    ]
)


# ================================================================
# True physical coefficients
#
# Pair:
#
#       beta_(i,j) = actual heterogeneous weight
#
# if edge is active in G1, else 0.
#
# Triad:
#
#       beta_(i,j,k) = 0.02
#
# for the two persistent native triads.
# ================================================================

EDGE_WEIGHT_CANONICAL = {
    tuple(
        sorted(
            edge
        )
    ):
        weight

    for edge, weight
    in edge_weight.items()
}


TRUE_G1_ACTIVE_PAIRS = {
    tuple(
        sorted(
            edge
        )
    )

    for edge
    in snapshots[
        SNAPSHOT_PILOT
    ]
}


TRUE_G1_ACTIVE_TRIADS = {
    tuple(
        sorted(
            triad
        )
    )

    for triad
    in NATIVE_TRIADS
}


P2_BETA_TRUE_RAW = np.zeros(
    len(
        P2_CANDIDATES
    ),
    dtype=float,
)


for candidate_index, (
    interaction_type,
    support,
) in enumerate(
    P2_CANDIDATES
):

    if (
        interaction_type
        ==
        "pair"
    ):

        if (
            support
            in
            TRUE_G1_ACTIVE_PAIRS
        ):

            P2_BETA_TRUE_RAW[
                candidate_index
            ] = (
                EDGE_WEIGHT_CANONICAL[
                    support
                ]
            )

    else:

        if (
            support
            in
            TRUE_G1_ACTIVE_TRIADS
        ):

            P2_BETA_TRUE_RAW[
                candidate_index
            ] = (
                NATIVE_G
            )


P2_TRUE_ACTIVE_MASK = (
    np.abs(
        P2_BETA_TRUE_RAW
    )
    >
    1e-14
)


assert int(
    P2_TRUE_ACTIVE_MASK.sum()
) == 6


# ================================================================
# Exact representability under P2
# ================================================================

Y_P2_TRUE_VECTOR = (
    Y_PRIOR_AUDIT.reshape(
        -1
    )
)


Y_P2_FROM_TRUE = (
    P2_PSI_RAW
    @
    P2_BETA_TRUE_RAW
)


P2_REPRESENTATION_ERROR = (
    relative_vector_error(
        Y_P2_FROM_TRUE,
        Y_P2_TRUE_VECTOR,
    )
)


assert (
    P2_REPRESENTATION_ERROR
    <
    1e-12
)


# ================================================================
# P2 full physical-candidate geometry
# ================================================================

P2_FULL_DIAG = (
    prior_svd_diagnostics(
        P2_PSI
    )
)


P2_COHERENCE = (
    normalized_column_coherence(
        P2_PSI
    )
)


# ================================================================
# P2 true-active subspace
# ================================================================

P2_TRUE_INDICES = np.flatnonzero(
    P2_TRUE_ACTIVE_MASK
)


P2_TRUE_DIAG = (
    prior_svd_diagnostics(
        P2_PSI[
            :,
            P2_TRUE_INDICES
        ]
    )
)


# ================================================================
# P2 strong alias test
#
# Remove ALL 6 true physical interactions and fit using only the
# remaining 78 physically valid but topologically false
# candidates.
# ================================================================

P2_FALSE_INDICES = np.flatnonzero(
    ~P2_TRUE_ACTIVE_MASK
)


P2_FALSE_COEF, _, _, _ = (
    np.linalg.lstsq(
        P2_PSI[
            :,
            P2_FALSE_INDICES
        ],
        Y_P2_TRUE_VECTOR,
        rcond=None,
    )
)


P2_FALSE_PRED = (
    P2_PSI[
        :,
        P2_FALSE_INDICES
    ]
    @
    P2_FALSE_COEF
)


P2_FALSE_ONLY_ERROR = (
    relative_vector_error(
        P2_FALSE_PRED,
        Y_P2_TRUE_VECTOR,
    )
)


P2_AUDIT = {
    "prior":
        "P2: known interaction functions",

    "dictionary_type":
        "physical interaction candidates",

    # No independent monomial feature count here.
    "n_features":
        np.nan,

    "candidate_groups":
        len(
            P2_CANDIDATES
        ),

    # One coefficient per physical interaction candidate.
    "parameter_directions":
        len(
            P2_CANDIDATES
        ),

    "parameter_rank":
        P2_FULL_DIAG[
            "rank"
        ],

    "parameter_nullity":
        P2_FULL_DIAG[
            "nullity"
        ],

    "parameter_rank_fraction":
        (
            P2_FULL_DIAG[
                "rank"
            ]
            /
            len(
                P2_CANDIDATES
            )
        ),

    # For P2, feature space == parameter space.
    "feature_rank":
        P2_FULL_DIAG[
            "rank"
        ],

    "feature_nullity":
        P2_FULL_DIAG[
            "nullity"
        ],

    "max_coherence":
        P2_COHERENCE,

    "max_nonzero_condition":
        P2_FULL_DIAG[
            "condition_nonzero"
        ],

    # Physical grouping:
    #
    # 4 active pair interactions
    # +
    # 2 active triadic interactions
    "true_groups":
        int(
            P2_TRUE_ACTIVE_MASK.sum()
        ),

    "true_scalar_directions":
        int(
            P2_TRUE_ACTIVE_MASK.sum()
        ),

    "true_active_full_rank":
        bool(
            P2_TRUE_DIAG[
                "rank"
            ]
            ==
            len(
                P2_TRUE_INDICES
            )
        ),

    "max_true_active_condition":
        P2_TRUE_DIAG[
            "condition_nonzero"
        ],

    "true_representation_error":
        P2_REPRESENTATION_ERROR,

    "false_only_parameters":
        len(
            P2_FALSE_INDICES
        ),

    "false_only_fit_error":
        P2_FALSE_ONLY_ERROR,

    "full_parameter_identifiability":
        bool(
            P2_FULL_DIAG[
                "rank"
            ]
            ==
            len(
                P2_CANDIDATES
            )
        ),
}


# ================================================================
# Combined prior ladder
# ================================================================

PRIOR_LADDER_AUDIT = pd.DataFrame(
    [
        P0_AUDIT,
        P1_AUDIT,
        P2_AUDIT,
    ]
)


# ================================================================
# Compact interpretation table
# ================================================================

PRIOR_LADDER_COMPACT = (
    PRIOR_LADDER_AUDIT[
        [
            "prior",
            "candidate_groups",
            "parameter_directions",
            "parameter_rank",
            "parameter_nullity",
            "parameter_rank_fraction",
            "max_coherence",
            "max_nonzero_condition",
            "true_groups",
            "true_active_full_rank",
            "max_true_active_condition",
            "false_only_fit_error",
            "full_parameter_identifiability",
        ]
    ]
    .copy()
)


# ================================================================
# Report
# ================================================================

print("=" * 78)
print("CELL 6 — PRIOR LADDER IDENTIFIABILITY AUDIT")
print("=" * 78)

print(
    f"Same G1 trajectory fit states : "
    f"{len(X_PRIOR_AUDIT)}"
)

print(
    f"State dimension                : "
    f"{N}"
)

print()
print(
    "P0 = generic degree<=3 / support<=4"
)

print(
    "P1 = generic degree<=2 / support<=3"
)

print(
    "P2 = known pair+triad functional forms / unknown topology"
)


print()
print("=" * 78)
print("FULL PRIOR-LADDER TABLE")
print("=" * 78)

display(
    PRIOR_LADDER_AUDIT.style.format({
        "n_features":
            lambda x:
                "—"
                if pd.isna(x)
                else f"{int(x)}",

        "parameter_rank_fraction":
            "{:.6f}",

        "max_coherence":
            "{:.8f}",

        "max_nonzero_condition":
            "{:.6e}",

        "max_true_active_condition":
            "{:.6e}",

        "true_representation_error":
            "{:.6e}",

        "false_only_fit_error":
            "{:.6e}",
    })
)


print()
print("=" * 78)
print("COMPACT IDENTIFIABILITY COMPARISON")
print("=" * 78)

display(
    PRIOR_LADDER_COMPACT.style.format({
        "parameter_rank_fraction":
            "{:.6f}",

        "max_coherence":
            "{:.8f}",

        "max_nonzero_condition":
            "{:.6e}",

        "max_true_active_condition":
            "{:.6e}",

        "false_only_fit_error":
            "{:.6e}",
    })
)


# ================================================================
# Explicit P2 physical candidate summary
# ================================================================

print()
print("=" * 78)
print("P2 PHYSICAL PRIOR")
print("=" * 78)

print(
    f"all pair candidates          : "
    f"{len(P2_PAIR_CANDIDATES)}"
)

print(
    f"all triad candidates         : "
    f"{len(P2_TRIAD_CANDIDATES)}"
)

print(
    f"all physical candidates      : "
    f"{len(P2_CANDIDATES)}"
)

print()

print(
    f"true active G1 pairs         : "
    f"{sorted(TRUE_G1_ACTIVE_PAIRS)}"
)

print(
    f"true active native triads    : "
    f"{sorted(TRUE_G1_ACTIVE_TRIADS)}"
)

print()

print(
    f"P2 full rank                 : "
    f"{P2_FULL_DIAG['rank']} / "
    f"{len(P2_CANDIDATES)}"
)

print(
    f"P2 nullity                   : "
    f"{P2_FULL_DIAG['nullity']}"
)

print(
    f"P2 condition                 : "
    f"{P2_FULL_DIAG['condition_nonzero']:.6e}"
)

print(
    f"P2 coherence                 : "
    f"{P2_COHERENCE:.8f}"
)

print()

print(
    f"P2 true-active rank          : "
    f"{P2_TRUE_DIAG['rank']} / "
    f"{len(P2_TRUE_INDICES)}"
)

print(
    f"P2 true-active condition     : "
    f"{P2_TRUE_DIAG['condition_nonzero']:.6e}"
)

print()

print(
    f"P2 exact representation err : "
    f"{P2_REPRESENTATION_ERROR:.6e}"
)

print(
    f"P2 false-only fit error      : "
    f"{P2_FALSE_ONLY_ERROR:.6e}"
)


# ================================================================
# Decision flags
# ================================================================

print()
print("=" * 78)
print("DECISION FLAGS")
print("=" * 78)

for _, row in (
    PRIOR_LADDER_AUDIT.iterrows()
):

    print(
        f"{row['prior']}:"
    )

    print(
        f"  full parameter identifiability = "
        f"{row['full_parameter_identifiability']}"
    )

    print(
        f"  true active subspace full rank = "
        f"{row['true_active_full_rank']}"
    )

    print(
        f"  false-only fit error           = "
        f"{row['false_only_fit_error']:.6e}"
    )


# ================================================================
# Save
# ================================================================

PRIOR_LADDER_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_prior_ladder_identifiability.csv",
    index=False,
)


np.savez_compressed(
    OUTPUT_DIR
    /
    "stage6_G1_P2_physical_design.npz",

    Psi_raw=
        P2_PSI_RAW,

    Psi_scaled=
        P2_PSI,

    scale=
        P2_SCALE,

    beta_true=
        P2_BETA_TRUE_RAW,

    true_active_mask=
        P2_TRUE_ACTIVE_MASK,
)


print()
print("=" * 78)
print("Cell 6 PASSED.")
print("=" * 78)

CELL 6 — PRIOR LADDER IDENTIFIABILITY AUDIT
Same G1 trajectory fit states : 124
State dimension                : 8

P0 = generic degree<=3 / support<=4
P1 = generic degree<=2 / support<=3
P2 = known pair+triad functional forms / unknown topology

FULL PRIOR-LADDER TABLE


,prior,dictionary_type,n_features,candidate_groups,parameter_directions,parameter_rank,parameter_nullity,parameter_rank_fraction,feature_rank,feature_nullity,max_coherence,max_nonzero_condition,true_groups,true_scalar_directions,true_active_full_rank,max_true_active_condition,true_representation_error,false_only_parameters,false_only_fit_error,full_parameter_identifiability
0,P0: generic degree<=3,generic polynomial,164,162,1312,200,1112,0.152439,25,139,0.99998837,2.551135e+13,19,50,True,2.284051e+06,1.502203e-16,1156,3.475060e-15,False
1,P1: generic degree<=2,generic polynomial,44,92,352,176,176,0.500000,22,22,0.99994614,1.285070e+13,19,50,True,2.284051e+06,1.462223e-16,276,5.383891e-14,False
2,P2: known interaction functions,physical interaction candidates,—,84,84,84,0,1.000000,84,0,0.99310419,9.450075e+07,6,6,True,2.741347e+00,9.659893e-18,78,1.004805e-06,True



COMPACT IDENTIFIABILITY COMPARISON


,prior,candidate_groups,parameter_directions,parameter_rank,parameter_nullity,parameter_rank_fraction,max_coherence,max_nonzero_condition,true_groups,true_active_full_rank,max_true_active_condition,false_only_fit_error,full_parameter_identifiability
0,P0: generic degree<=3,162,1312,200,1112,0.152439,0.99998837,2.551135e+13,19,True,2.284051e+06,3.475060e-15,False
1,P1: generic degree<=2,92,352,176,176,0.500000,0.99994614,1.285070e+13,19,True,2.284051e+06,5.383891e-14,False
2,P2: known interaction functions,84,84,84,0,1.000000,0.99310419,9.450075e+07,6,True,2.741347e+00,1.004805e-06,True



P2 PHYSICAL PRIOR
all pair candidates          : 28
all triad candidates         : 56
all physical candidates      : 84

true active G1 pairs         : [(1, 2), (3, 4), (5, 6), (7, 8)]
true active native triads    : [(1, 2, 3), (2, 5, 8)]

P2 full rank                 : 84 / 84
P2 nullity                   : 0
P2 condition                 : 9.450075e+07
P2 coherence                 : 0.99310419

P2 true-active rank          : 6 / 6
P2 true-active condition     : 2.741347e+00

P2 exact representation err : 9.659893e-18
P2 false-only fit error      : 1.004805e-06

DECISION FLAGS
P0: generic degree<=3:
  full parameter identifiability = False
  true active subspace full rank = True
  false-only fit error           = 3.475060e-15
P1: generic degree<=2:
  full parameter identifiability = False
  true active subspace full rank = True
  false-only fit error           = 5.383891e-14
P2: known interaction functions:
  full parameter identifiability = True
  true active subspace full rank = Tru

In [9]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 7: complete P2 blind microscopic-topology inference
#
# PRIOR:
# ---------------------------------------------------------------
# KNOWN:
#   - pair interaction functional form
#   - triad interaction functional form
#   - these microscopic functional forms are time-invariant
#
# UNKNOWN TO LEARNER:
#   - which pair interactions are active
#   - which triad interactions are active
#   - all interaction coefficients
#
# DATA:
#   - one continuous time series only
#   - G1 local slices
#   - A_SLICE_EST recovered from multi-epsilon endpoint slicing
#
# TEMPORAL SPLIT:
#   cycles 0-3 : fit
#   cycle 4    : validation
#   cycle 5    : external test
#
# CANDIDATE PHYSICAL INTERACTIONS:
#   28 pairs + 56 triads = 84
#
# Scientific question:
#
#   Does known interaction-function prior make one-trajectory
#   microscopic topology recovery possible?
#
# NO oracle topology enters before fitting.
# ================================================================

import hashlib
import time


# ================================================================
# 0. Candidate order and true support are NOT used by inference
# ================================================================

assert len(P2_CANDIDATES) == 84
assert len(P2_PAIR_CANDIDATES) == 28
assert len(P2_TRIAD_CANDIDATES) == 56


# ================================================================
# 1. Build physical interaction design for arbitrary states
#
# Each candidate produces a vector field:
#
#       Psi_alpha(X) : (n_states, N)
#
# We vectorize state × output:
#
#       Psi : (n_states * N, 84)
#
# so that:
#
#       vec(F) = Psi beta
#
# One beta_alpha corresponds to one complete physical
# edge/triad interaction.
# ================================================================

def build_P2_physical_design(X):

    X = np.asarray(
        X,
        dtype=float,
    )

    columns = []

    for interaction_type, support in P2_CANDIDATES:

        if interaction_type == "pair":

            field = (
                unit_pair_interaction_field(
                    X,
                    support,
                )
            )

        elif interaction_type == "triad":

            field = (
                unit_triad_interaction_field(
                    X,
                    support,
                )
            )

        else:

            raise RuntimeError(
                interaction_type
            )

        columns.append(
            field.reshape(-1)
        )

    return np.column_stack(
        columns
    )


# ================================================================
# 2. Strict temporal blocks
# ================================================================

P2_PSI_FIT_RAW = (
    build_P2_physical_design(
        X_G1_FIT
    )
)

P2_PSI_VAL_RAW = (
    build_P2_physical_design(
        X_G1_VAL
    )
)

P2_PSI_TEST_RAW = (
    build_P2_physical_design(
        X_G1_TEST
    )
)


assert P2_PSI_FIT_RAW.shape == (
    len(X_G1_FIT) * N,
    84,
)

assert P2_PSI_VAL_RAW.shape == (
    len(X_G1_VAL) * N,
    84,
)

assert P2_PSI_TEST_RAW.shape == (
    len(X_G1_TEST) * N,
    84,
)


# Cell 6 consistency check
assert np.allclose(
    P2_PSI_FIT_RAW,
    P2_PSI_RAW,
    rtol=0.0,
    atol=0.0,
)


# ================================================================
# 3. Feature scaling learned STRICTLY from cycles 0-3
#
# Same rule as Cell 5:
# validation/test do not affect scaling.
# ================================================================

P2_SCALE_CELL7 = np.sqrt(
    np.mean(
        P2_PSI_FIT_RAW ** 2,
        axis=0,
    )
)

P2_SCALE_CELL7 = np.where(
    P2_SCALE_CELL7 > 0.0,
    P2_SCALE_CELL7,
    1.0,
)


assert np.allclose(
    P2_SCALE_CELL7,
    P2_SCALE,
    rtol=1e-14,
    atol=1e-14,
)


P2_PSI_FIT = (
    P2_PSI_FIT_RAW
    /
    P2_SCALE_CELL7[
        None,
        :
    ]
)

P2_PSI_VAL = (
    P2_PSI_VAL_RAW
    /
    P2_SCALE_CELL7[
        None,
        :
    ]
)

P2_PSI_TEST = (
    P2_PSI_TEST_RAW
    /
    P2_SCALE_CELL7[
        None,
        :
    ]
)


# ================================================================
# 4. Targets
#
# IMPORTANT:
#
# inference target = A_SLICE_EST
#
# NOT exact G1.
#
# G1_TRUE_* enters only after fitting for synthetic diagnostics.
# ================================================================

P2_Y_FIT = (
    Y_G1_FIT
    .reshape(
        -1,
        1,
    )
)

P2_Y_VAL = (
    Y_G1_VAL
    .reshape(
        -1,
        1,
    )
)

P2_Y_TEST = (
    Y_G1_TEST
    .reshape(
        -1,
        1,
    )
)


assert P2_Y_FIT.shape == (
    len(X_G1_FIT) * N,
    1,
)

assert P2_Y_VAL.shape == (
    len(X_G1_VAL) * N,
    1,
)

assert P2_Y_TEST.shape == (
    len(X_G1_TEST) * N,
    1,
)


# ================================================================
# 5. Physical-candidate library adapter
#
# _AdaptiveGroupLasso internally expects:
#
#   Theta : (samples, features)
#   Y     : (samples, outputs)
#
# Here:
#
#   "sample" = one state-output component
#   output   = one scalar
#
# Each physical candidate therefore becomes ONE structural group
# containing ONE scalar coefficient.
#
# This does NOT change the solver.
# It only supplies a different physically constrained dictionary.
# ================================================================

class PhysicalCandidateLibrary:

    def __init__(
        self,
        candidates,
        feature_scale,
    ):

        self.candidates = tuple(
            candidates
        )

        self.n_features = len(
            self.candidates
        )

        # Scalarized regression has one output column.
        self.n_nodes = 1

        self.feature_scale = np.asarray(
            feature_scale,
            dtype=float,
        ).copy()

        if self.feature_scale.shape != (
            self.n_features,
        ):
            raise ValueError(
                "feature_scale shape mismatch"
            )

        # Every candidate can contribute to the scalarized field.
        self.valid_mask = np.ones(
            (
                self.n_features,
                1,
            ),
            dtype=bool,
        )

        # One complete physical interaction = one sparse group.
        #
        # Support labels retain the ORIGINAL N=8 node labels.
        self.structural_groups = {}

        for feature_index, (
            interaction_type,
            support,
        ) in enumerate(
            self.candidates
        ):

            support = tuple(
                sorted(
                    int(v)
                    for v in support
                )
            )

            if support in self.structural_groups:

                raise RuntimeError(
                    f"Duplicate physical support: {support}"
                )

            # With one scalar output:
            #
            # flat index = feature_index * 1 + 0
            self.structural_groups[
                support
            ] = np.asarray(
                [
                    feature_index
                ],
                dtype=int,
            )


    def to_raw_coefficients(
        self,
        B_scaled,
    ):

        B_scaled = np.asarray(
            B_scaled,
            dtype=float,
        )

        if B_scaled.shape != (
            self.n_features,
            1,
        ):

            raise ValueError(
                "Unexpected coefficient shape: "
                f"{B_scaled.shape}"
            )

        return (
            B_scaled
            /
            self.feature_scale[
                :,
                None
            ]
        )


P2_LIBRARY_CELL7 = (
    PhysicalCandidateLibrary(
        candidates=
            P2_CANDIDATES,

        feature_scale=
            P2_SCALE_CELL7,
    )
)


assert len(
    P2_LIBRARY_CELL7.structural_groups
) == 84


# ================================================================
# 6. Concatenate fit + validation for v3.6 interface
#
# Temporal block ordering:
#
#   rows [0 : 124*N)
#       -> cycles 0-3
#
#   rows [124*N : 155*N)
#       -> cycle 4
#
# Whole states remain together because reshape is state-major.
# ================================================================

P2_THETA_TRAIN = np.vstack(
    [
        P2_PSI_FIT,
        P2_PSI_VAL,
    ]
)

P2_TARGET_TRAIN = np.vstack(
    [
        P2_Y_FIT,
        P2_Y_VAL,
    ]
)


P2_N_FIT_ROWS = (
    len(
        X_G1_FIT
    )
    *
    N
)

P2_N_VAL_ROWS = (
    len(
        X_G1_VAL
    )
    *
    N
)


assert P2_N_FIT_ROWS == 992
assert P2_N_VAL_ROWS == 248

assert P2_THETA_TRAIN.shape == (
    1240,
    84,
)

assert P2_TARGET_TRAIN.shape == (
    1240,
    1,
)


# ================================================================
# 7. Temporal-block AGLASSO
#
# Reuse the Cell-5 subclass:
#
#       FixedTemporalBlockAGLASSO
#
# Only _fit_validation_split is overridden.
# Core v3.6 solver remains unchanged.
#
# Note:
# validation rows are scalarized state-output components,
# BUT entire physical states remain wholly in one temporal block.
# There is therefore no temporal leakage.
# ================================================================

P2_ESTIMATOR = (
    FixedTemporalBlockAGLASSO(

        library=
            P2_LIBRARY_CELL7,

        fixed_fit_count=
            P2_N_FIT_ROWS,

        fixed_validation_count=
            P2_N_VAL_ROWS,

        n_lambdas=
            STAGE6_CFG._F0_N_LAMBDAS,

        lambda_min_ratio=
            STAGE6_CFG._F0_LAMBDA_MIN_RATIO,

        coarse_n_lambdas=
            STAGE6_CFG._F0_COARSE_N_LAMBDAS,

        refine_n_lambdas=
            STAGE6_CFG._F0_REFINE_N_LAMBDAS,

        gamma=
            STAGE6_CFG._ADAPTIVE_GAMMA,

        delta_ratio=
            STAGE6_CFG._ADAPTIVE_DELTA_RATIO,

        max_iter=
            STAGE6_CFG._MAX_ITER,

        tol=
            STAGE6_CFG._SOLVER_TOL,

        active_group_tol=
            STAGE6_CFG._ACTIVE_GROUP_TOL,

        ridge_condition_target=
            STAGE6_CFG._RIDGE_CONDITION_TARGET,

        ridge_alpha_floor_ratio=
            STAGE6_CFG._RIDGE_ALPHA_FLOOR_RATIO,

        kkt_tol=
            STAGE6_CFG._KKT_TOL,

        max_kkt_expansions=
            STAGE6_CFG._MAX_KKT_EXPANSIONS,

        validation_fraction=
            STAGE6_CFG._VALIDATION_FRACTION,

        validation_seed=
            STAGE6_CFG._VALIDATION_SEED,

        early_stop_min_evals=
            STAGE6_CFG._EARLY_STOP_MIN_EVALS,

        early_stop_patience=
            STAGE6_CFG._EARLY_STOP_PATIENCE,

        early_stop_relative_degradation=
            STAGE6_CFG._EARLY_STOP_RELATIVE_DEGRADATION,

        early_stop_support_growth_ratio=
            STAGE6_CFG._EARLY_STOP_SUPPORT_GROWTH_RATIO,

        exact_fit_plateau_patience=
            STAGE6_CFG._EXACT_FIT_PLATEAU_PATIENCE,

        exact_fit_relative_error=
            STAGE6_CFG._EXACT_FIT_RELATIVE_ERROR,

        adaptive_geometric_ratio=
            STAGE6_CFG._ADAPTIVE_GEOMETRIC_RATIO,

        adaptive_entry_fraction=
            STAGE6_CFG._ADAPTIVE_ENTRY_FRACTION,

        adaptive_max_jump_decades=
            STAGE6_CFG._ADAPTIVE_MAX_JUMP_DECADES,

        adaptive_max_evals=
            STAGE6_CFG._F0_ADAPTIVE_MAX_EVALS,

        enable_pruning=False,

        verbose=True,

        label=
            "Stage6 P2 known-law G1 topology",
    )
)


# ================================================================
# 8. Reproducibility fingerprint
# ================================================================

def stage6_P2_fingerprint():

    h = hashlib.sha256()

    h.update(
        b"STAGE6_P2_G1_KNOWN_FUNCTION_UNKNOWN_TOPOLOGY_v1"
    )

    for name, arr in (

        (
            "Psi_fit_raw",
            P2_PSI_FIT_RAW,
        ),

        (
            "Psi_val_raw",
            P2_PSI_VAL_RAW,
        ),

        (
            "Y_fit",
            P2_Y_FIT,
        ),

        (
            "Y_val",
            P2_Y_VAL,
        ),

        (
            "scale",
            P2_SCALE_CELL7,
        ),
    ):

        arr = np.ascontiguousarray(
            np.asarray(
                arr
            )
        )

        h.update(
            name.encode(
                "utf-8"
            )
        )

        h.update(
            str(
                arr.dtype
            ).encode(
                "utf-8"
            )
        )

        h.update(
            np.asarray(
                arr.shape,
                dtype=np.int64,
            ).tobytes()
        )

        h.update(
            arr.view(
                np.uint8
            ).tobytes()
        )


    h.update(
        repr(
            P2_CANDIDATES
        ).encode(
            "utf-8"
        )
    )

    return h.hexdigest()


P2_FINGERPRINT = (
    stage6_P2_fingerprint()
)


P2_CHECKPOINT = (
    OUTPUT_DIR
    /
    "stage6_G1_P2_known_function_aglasso_partial.pkl.gz"
)


# ================================================================
# 9. Pre-fit report
# ================================================================

print("=" * 78)
print("CELL 7 — P2 BLIND TOPOLOGY INFERENCE")
print("=" * 78)

print(
    f"fit states                   : "
    f"{len(X_G1_FIT)}"
)

print(
    f"validation states            : "
    f"{len(X_G1_VAL)}"
)

print(
    f"external test states         : "
    f"{len(X_G1_TEST)}"
)

print()

print(
    f"scalarized fit rows          : "
    f"{P2_N_FIT_ROWS}"
)

print(
    f"scalarized validation rows   : "
    f"{P2_N_VAL_ROWS}"
)

print(
    f"scalarized external-test rows: "
    f"{P2_Y_TEST.shape[0]}"
)

print()

print(
    f"pair candidates              : "
    f"{len(P2_PAIR_CANDIDATES)}"
)

print(
    f"triad candidates             : "
    f"{len(P2_TRIAD_CANDIDATES)}"
)

print(
    f"total physical candidates    : "
    f"{len(P2_CANDIDATES)}"
)

print()

print(
    "Topology oracle has NOT been supplied to estimator."
)

print(
    "Inference target = multi-epsilon reconstructed A_SLICE_EST."
)


# ================================================================
# 10. Blind topology inference
# ================================================================

print()
print("=" * 78)
print("STARTING v3.6 P2 AGLASSO")
print("=" * 78)


tic = time.perf_counter()


P2_RESULT = (
    P2_ESTIMATOR.fit(

        P2_THETA_TRAIN,
        P2_TARGET_TRAIN,

        Theta_test=
            P2_PSI_TEST,

        Y_test=
            P2_Y_TEST,

        checkpoint_path=
            P2_CHECKPOINT,

        checkpoint_fingerprint=
            P2_FINGERPRINT,

        resume_from_checkpoint=True,
    )
)


P2_RUNTIME = (
    time.perf_counter()
    -
    tic
)


# ================================================================
# 11. Synthetic oracle enters ONLY NOW
# ================================================================

TRUE_P2_SUPPORTS = (
    set(
        TRUE_G1_ACTIVE_PAIRS
    )
    |
    set(
        TRUE_G1_ACTIVE_TRIADS
    )
)


PRED_P2_SUPPORTS = set(
    P2_RESULT.selected_supports
)


assert len(
    TRUE_P2_SUPPORTS
) == 6


P2_SUPPORT_METRICS = (
    support_metrics(
        PRED_P2_SUPPORTS,
        TRUE_P2_SUPPORTS,
    )
)


# ================================================================
# 12. Pair / triad structural metrics
# ================================================================

p2_order_rows = []


for order, label in (
    (
        2,
        "pair",
    ),
    (
        3,
        "triad",
    ),
):

    truth = {
        s
        for s
        in TRUE_P2_SUPPORTS
        if len(s) == order
    }

    predicted = {
        s
        for s
        in PRED_P2_SUPPORTS
        if len(s) == order
    }

    metrics = (
        support_metrics(
            predicted,
            truth,
        )
    )

    p2_order_rows.append({
        "interaction_type":
            label,

        "order":
            order,

        "true":
            len(
                truth
            ),

        "selected":
            len(
                predicted
            ),

        **metrics,
    })


P2_ORDER_METRICS = (
    pd.DataFrame(
        p2_order_rows
    )
)


# ================================================================
# 13. Physical coefficients
# ================================================================

P2_BETA_EST_RAW = (
    P2_RESULT
    .coefficients_raw[
        :,
        0
    ]
)


P2_BETA_EST_SCALED = (
    P2_RESULT
    .coefficients_scaled[
        :,
        0
    ]
)


P2_COEFF_REL_ERROR = (
    relative_vector_error(
        P2_BETA_EST_RAW,
        P2_BETA_TRUE_RAW,
    )
)


# ================================================================
# 14. Candidate-level coefficient table
# ================================================================

candidate_rows = []


for candidate_index, (
    interaction_type,
    support,
) in enumerate(
    P2_CANDIDATES
):

    beta_true = float(
        P2_BETA_TRUE_RAW[
            candidate_index
        ]
    )

    beta_est = float(
        P2_BETA_EST_RAW[
            candidate_index
        ]
    )

    is_true = (
        support
        in
        TRUE_P2_SUPPORTS
    )

    is_selected = (
        support
        in
        PRED_P2_SUPPORTS
    )

    candidate_rows.append({

        "candidate_index":
            candidate_index,

        "interaction_type":
            interaction_type,

        "support":
            support,

        "true_active":
            is_true,

        "selected":
            is_selected,

        "beta_true":
            beta_true,

        "beta_est":
            beta_est,

        "abs_error":
            abs(
                beta_est
                -
                beta_true
            ),
    })


P2_COEFFICIENT_TABLE = (
    pd.DataFrame(
        candidate_rows
    )
)


# ================================================================
# 15. Reconstruct vector fields
# ================================================================

P2_FIELD_FIT_PRED = (
    (
        P2_PSI_FIT
        @
        P2_BETA_EST_SCALED
    )
    .reshape(
        len(
            X_G1_FIT
        ),
        N,
    )
)


P2_FIELD_VAL_PRED = (
    (
        P2_PSI_VAL
        @
        P2_BETA_EST_SCALED
    )
    .reshape(
        len(
            X_G1_VAL
        ),
        N,
    )
)


P2_FIELD_TEST_PRED = (
    (
        P2_PSI_TEST
        @
        P2_BETA_EST_SCALED
    )
    .reshape(
        len(
            X_G1_TEST
        ),
        N,
    )
)


# ================================================================
# 16. Field errors
#
# Distinguish:
#
# A. model-selection validation error
#    -> P2_RESULT.validation_relative_error
#       evaluated before final fit+validation refit
#
# B. post-selection cycle-4 reconstruction below
#    -> NOT held out anymore
#
# C. cycle-5 external test
#    -> completely untouched
# ================================================================

P2_FIT_ERR_AEST = (
    relative_fro_error(
        P2_FIELD_FIT_PRED,
        Y_G1_FIT,
    )
)

P2_FIT_ERR_TRUE = (
    relative_fro_error(
        P2_FIELD_FIT_PRED,
        G1_TRUE_FIT,
    )
)


P2_VAL_POSTREFIT_ERR_AEST = (
    relative_fro_error(
        P2_FIELD_VAL_PRED,
        Y_G1_VAL,
    )
)

P2_VAL_POSTREFIT_ERR_TRUE = (
    relative_fro_error(
        P2_FIELD_VAL_PRED,
        G1_TRUE_VAL,
    )
)


P2_TEST_ERR_AEST = (
    relative_fro_error(
        P2_FIELD_TEST_PRED,
        Y_G1_TEST,
    )
)

P2_TEST_ERR_TRUE = (
    relative_fro_error(
        P2_FIELD_TEST_PRED,
        G1_TRUE_TEST,
    )
)


# ================================================================
# 17. False positives / negatives
# ================================================================

P2_FALSE_POSITIVES = tuple(
    sorted(
        PRED_P2_SUPPORTS
        -
        TRUE_P2_SUPPORTS,

        key=lambda s: (
            len(s),
            s,
        ),
    )
)


P2_FALSE_NEGATIVES = tuple(
    sorted(
        TRUE_P2_SUPPORTS
        -
        PRED_P2_SUPPORTS,

        key=lambda s: (
            len(s),
            s,
        ),
    )
)


# ================================================================
# 18. Main report
# ================================================================

print()
print("=" * 78)
print("P2 G1 MICROSCOPIC-TOPOLOGY RECOVERY")
print("=" * 78)

print(
    f"Runtime                      : "
    f"{P2_RUNTIME:.2f} s "
    f"= {P2_RUNTIME / 60:.2f} min"
)

print()

print(
    f"True physical interactions   : "
    f"{len(TRUE_P2_SUPPORTS)}"
)

print(
    f"Selected interactions        : "
    f"{len(PRED_P2_SUPPORTS)}"
)

print(
    f"TP / FP / FN                 : "
    f"{P2_SUPPORT_METRICS['TP']} / "
    f"{P2_SUPPORT_METRICS['FP']} / "
    f"{P2_SUPPORT_METRICS['FN']}"
)

print(
    f"Precision                    : "
    f"{P2_SUPPORT_METRICS['precision']:.6f}"
)

print(
    f"Recall                       : "
    f"{P2_SUPPORT_METRICS['recall']:.6f}"
)

print(
    f"F1                           : "
    f"{P2_SUPPORT_METRICS['F1']:.6f}"
)


print()
print("Recovery by physical interaction type")
print("-" * 78)

display(
    P2_ORDER_METRICS.style.format({
        "precision":
            "{:.6f}",

        "recall":
            "{:.6f}",

        "F1":
            "{:.6f}",
    })
)


print()
print("Model-selection diagnostics")
print("-" * 78)

print(
    f"Selected lambda              : "
    f"{P2_RESULT.lambda_selected:.6e}"
)

print(
    f"Selection method             : "
    f"{P2_RESULT.selection_method}"
)

print(
    f"True held-out validation err : "
    f"{P2_RESULT.validation_relative_error:.6e}"
)

print(
    f"External test error          : "
    f"{P2_RESULT.test_relative_error:.6e}"
)

print(
    f"Final global KKT             : "
    f"{P2_RESULT.final_kkt_satisfied}"
)

print(
    f"Selected-lambda KKT cert.    : "
    f"{P2_RESULT.kkt_certified}"
)

print(
    f"All-path KKT                 : "
    f"{P2_RESULT.all_path_kkt_certified}"
)

print(
    f"Path boundary hit            : "
    f"{P2_RESULT.path_boundary_hit}"
)

print(
    f"Path stop reason             : "
    f"{P2_RESULT.early_stop_reason}"
)


print()
print("Physical coefficient recovery")
print("-" * 78)

print(
    f"Coefficient relative error   : "
    f"{P2_COEFF_REL_ERROR:.6e}"
)


print()
print("Field reconstruction")
print("-" * 78)

print(
    f"Fit vs reconstructed A_est   : "
    f"{P2_FIT_ERR_AEST:.6e}"
)

print(
    f"Fit vs exact G1              : "
    f"{P2_FIT_ERR_TRUE:.6e}"
)

print()

print(
    f"Cycle-4 post-refit vs A_est  : "
    f"{P2_VAL_POSTREFIT_ERR_AEST:.6e}"
)

print(
    f"Cycle-4 post-refit vs exact  : "
    f"{P2_VAL_POSTREFIT_ERR_TRUE:.6e}"
)

print()

print(
    f"Cycle-5 test vs A_est        : "
    f"{P2_TEST_ERR_AEST:.6e}"
)

print(
    f"Cycle-5 test vs exact G1     : "
    f"{P2_TEST_ERR_TRUE:.6e}"
)


print()
print("Selected physical interactions")
print("-" * 78)

for support in sorted(
    PRED_P2_SUPPORTS,
    key=lambda s: (
        len(s),
        s,
    ),
):

    idx = next(
        i
        for i, (_, s)
        in enumerate(
            P2_CANDIDATES
        )
        if s == support
    )

    print(
        f"{support}: "
        f"beta_hat={P2_BETA_EST_RAW[idx]: .8f} "
        f"| beta_true={P2_BETA_TRUE_RAW[idx]: .8f}"
    )


print()
print("False positives")
print("-" * 78)

print(
    P2_FALSE_POSITIVES
    if P2_FALSE_POSITIVES
    else "None"
)


print()
print("False negatives")
print("-" * 78)

print(
    P2_FALSE_NEGATIVES
    if P2_FALSE_NEGATIVES
    else "None"
)


# ================================================================
# 19. Show only active-or-selected candidate coefficients
# ================================================================

P2_RELEVANT_COEFFICIENTS = (
    P2_COEFFICIENT_TABLE[
        P2_COEFFICIENT_TABLE[
            "true_active"
        ]
        |
        P2_COEFFICIENT_TABLE[
            "selected"
        ]
    ]
    .copy()
    .sort_values(
        by=[
            "interaction_type",
            "support",
        ]
    )
)


print()
print("=" * 78)
print("TRUE / SELECTED COEFFICIENT TABLE")
print("=" * 78)

display(
    P2_RELEVANT_COEFFICIENTS.style.format({
        "beta_true":
            "{:.8f}",

        "beta_est":
            "{:.8f}",

        "abs_error":
            "{:.6e}",
    })
)


# ================================================================
# 20. Save
# ================================================================

P2_ORDER_METRICS.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_order_metrics.csv",
    index=False,
)


P2_COEFFICIENT_TABLE.to_pickle(
    OUTPUT_DIR
    /
    "stage6_G1_P2_candidate_coefficients.pkl"
)


P2_RESULT.path_ledger.to_pickle(
    OUTPUT_DIR
    /
    "stage6_G1_P2_path_ledger.pkl"
)


np.savez_compressed(
    OUTPUT_DIR
    /
    "stage6_G1_P2_complete_inference.npz",

    beta_est_raw=
        P2_BETA_EST_RAW,

    beta_true_raw=
        P2_BETA_TRUE_RAW,

    beta_est_scaled=
        P2_BETA_EST_SCALED,

    feature_scale=
        P2_SCALE_CELL7,

    fit_indices=
        IDX_G1_FIT,

    validation_indices=
        IDX_G1_VAL,

    test_indices=
        IDX_G1_TEST,

    fit_field_pred=
        P2_FIELD_FIT_PRED,

    validation_field_pred=
        P2_FIELD_VAL_PRED,

    test_field_pred=
        P2_FIELD_TEST_PRED,
)


print()
print("=" * 78)
print("Cell 7 PASSED.")
print("=" * 78)

CELL 7 — P2 BLIND TOPOLOGY INFERENCE
fit states                   : 124
validation states            : 31
external test states         : 31

scalarized fit rows          : 992
scalarized validation rows   : 248
scalarized external-test rows: 248

pair candidates              : 28
triad candidates             : 56
total physical candidates    : 84

Topology oracle has NOT been supplied to estimator.
Inference target = multi-epsilon reconstructed A_SLICE_EST.

STARTING v3.6 P2 AGLASSO
Starting KKT working-set Adaptive Group LASSO for Stage6 P2 known-law G1 topology (KKT-event-driven validation path)...
  internal split = 992 fit / 248 validation states
  adaptive path <= 40 lambda evaluations | geometric ratio=0.750 | entry fraction=0.980
  maximum KKT jump = 2.00 decades | emergency floor = 1e-10 * lambda_max
  local refinement <= 4 lambdas around an interior minimum validation-loss adaptive point
  selector = validation one-standard-error rule (largest admissible lambda)
  pilot = adap

,interaction_type,order,true,selected,TP,FP,FN,precision,recall,F1
0,pair,2,4,4,3,1,1,0.750000,0.750000,0.750000
1,triad,3,2,0,0,0,2,0.000000,0.000000,0.000000



Model-selection diagnostics
------------------------------------------------------------------------------
Selected lambda              : 2.309100e-03
Selection method             : internal-validation-1se-largest-lambda
True held-out validation err : 3.461820e-02
External test error          : 5.196969e-02
Final global KKT             : True
Selected-lambda KKT cert.    : True
All-path KKT                 : False
Path boundary hit            : False
Path stop reason             : validation degraded while support kept growing for 2 consecutive lambdas

Physical coefficient recovery
------------------------------------------------------------------------------
Coefficient relative error   : 4.765601e-01

Field reconstruction
------------------------------------------------------------------------------
Fit vs reconstructed A_est   : 1.364634e-01
Fit vs exact G1              : 1.364634e-01

Cycle-4 post-refit vs A_est  : 3.461166e-02
Cycle-4 post-refit vs exact  : 3.461166e-02

Cycle-5

,candidate_index,interaction_type,support,true_active,selected,beta_true,beta_est,abs_error
0,0,pair,"(1, 2)",True,False,1.00600000,0.00000000,1.006000e+00
13,13,pair,"(3, 4)",True,True,1.10500000,1.10225602,2.743975e-03
15,15,pair,"(3, 6)",False,True,0.00000000,0.00481329,4.813290e-03
22,22,pair,"(5, 6)",True,True,1.17700000,1.16989708,7.102924e-03
27,27,pair,"(7, 8)",True,True,0.91700000,0.91393286,3.067144e-03
28,28,triad,"(1, 2, 3)",True,False,0.02000000,0.00000000,2.000000e-02
60,60,triad,"(2, 5, 8)",True,False,0.02000000,0.00000000,2.000000e-02



Cell 7 PASSED.


In [10]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 7b: diagnose P2 support-selection failure
#
# NO sparse inference is run here.
#
# Cell 6 established:
#
#       P2 physical design rank = 84 / 84
#
# so the physical coefficient vector is algebraically unique.
#
# Cell 7 nevertheless recovered only:
#
#       TP / FP / FN = 3 / 1 / 3
#
# This cell asks WHY.
#
# Diagnostics
# ---------------------------------------------------------------
# A. Observational strength of every TRUE interaction
#
# B. How well each TRUE interaction column can be reproduced
#    by the other 83 physical candidates
#
# C. Oracle-six coefficient fit:
#       give ONLY the 6 true physical candidates
#       fit coefficients using A_SLICE_EST
#
# D. Leave-one-true-interaction-out:
#       how much does field reconstruction degrade when each
#       true physical interaction is removed?
#
# No topology-selection hyperparameter is changed.
# ================================================================


# ================================================================
# 0. Candidate lookup
# ================================================================

P2_SUPPORT_TO_INDEX = {
    tuple(support): idx

    for idx, (_, support)
    in enumerate(
        P2_CANDIDATES
    )
}


TRUE_P2_SUPPORTS_SORTED = tuple(
    sorted(
        TRUE_P2_SUPPORTS,
        key=lambda s: (
            len(s),
            s,
        ),
    )
)


TRUE_P2_INDICES = np.asarray(
    [
        P2_SUPPORT_TO_INDEX[
            support
        ]

        for support
        in TRUE_P2_SUPPORTS_SORTED
    ],
    dtype=int,
)


assert len(
    TRUE_P2_INDICES
) == 6


# ================================================================
# Helper: interaction type
# ================================================================

def p2_interaction_type(
    support,
):

    if len(support) == 2:
        return "pair"

    if len(support) == 3:
        return "triad"

    return f"order-{len(support)}"


# ================================================================
# A. TRUE INTERACTION OBSERVATIONAL STRENGTH
#
# For candidate alpha:
#
#       F_alpha(X)
#           =
#       beta_alpha * Psi_alpha(X)
#
# Measure RMS contribution separately in:
#
#       fit / validation / test
#
# Also report fraction relative to total true field RMS.
#
# This distinguishes:
#
#       large physical coefficient
#
# from:
#
#       large observable dynamical contribution.
# ================================================================

TRUE_STRENGTH_ROWS = []


for support in TRUE_P2_SUPPORTS_SORTED:

    idx = (
        P2_SUPPORT_TO_INDEX[
            support
        ]
    )

    beta = float(
        P2_BETA_TRUE_RAW[
            idx
        ]
    )


    contribution_fit = (
        beta
        *
        P2_PSI_FIT_RAW[
            :,
            idx
        ]
    ).reshape(
        len(
            X_G1_FIT
        ),
        N,
    )


    contribution_val = (
        beta
        *
        P2_PSI_VAL_RAW[
            :,
            idx
        ]
    ).reshape(
        len(
            X_G1_VAL
        ),
        N,
    )


    contribution_test = (
        beta
        *
        P2_PSI_TEST_RAW[
            :,
            idx
        ]
    ).reshape(
        len(
            X_G1_TEST
        ),
        N,
    )


    rms_fit = float(
        rms_field(
            contribution_fit
        )
    )

    rms_val = float(
        rms_field(
            contribution_val
        )
    )

    rms_test = float(
        rms_field(
            contribution_test
        )
    )


    total_fit_rms = float(
        rms_field(
            G1_TRUE_FIT
        )
    )

    total_val_rms = float(
        rms_field(
            G1_TRUE_VAL
        )
    )

    total_test_rms = float(
        rms_field(
            G1_TRUE_TEST
        )
    )


    TRUE_STRENGTH_ROWS.append({

        "interaction_type":
            p2_interaction_type(
                support
            ),

        "support":
            support,

        "beta_true":
            beta,

        "selected_Cell7":
            (
                support
                in
                PRED_P2_SUPPORTS
            ),

        "fit_contribution_rms":
            rms_fit,

        "fit_fraction_of_field":
            (
                rms_fit
                /
                total_fit_rms
            ),

        "validation_contribution_rms":
            rms_val,

        "validation_fraction_of_field":
            (
                rms_val
                /
                total_val_rms
            ),

        "test_contribution_rms":
            rms_test,

        "test_fraction_of_field":
            (
                rms_test
                /
                total_test_rms
            ),

        "test_to_fit_strength_ratio":
            (
                rms_test
                /
                max(
                    rms_fit,
                    np.finfo(float).tiny,
                )
            ),
    })


P2_TRUE_STRENGTH_AUDIT = (
    pd.DataFrame(
        TRUE_STRENGTH_ROWS
    )
)


# ================================================================
# B. TRUE-CANDIDATE REPLACEABILITY
#
# For each true physical candidate alpha:
#
# fit on cycles 0-3:
#
#       Psi_alpha
#           ~=
#       Psi_{-alpha} c
#
# using ALL other 83 physical candidates.
#
# Then evaluate the SAME coefficients on:
#
#       validation cycle 4
#       test cycle 5
#
# Define:
#
#       residual_ratio
#         =
#       ||Psi_alpha - Psi_-alpha c||
#       ----------------------------
#              ||Psi_alpha||
#
# Small residual means the candidate is nearly observationally
# replaceable by other physical interactions.
# ================================================================

REPLACEABILITY_ROWS = []


ALL_P2_INDICES = np.arange(
    len(
        P2_CANDIDATES
    ),
    dtype=int,
)


for support in TRUE_P2_SUPPORTS_SORTED:

    target_idx = (
        P2_SUPPORT_TO_INDEX[
            support
        ]
    )

    other_idx = (
        ALL_P2_INDICES[
            ALL_P2_INDICES
            !=
            target_idx
        ]
    )


    target_fit = (
        P2_PSI_FIT_RAW[
            :,
            target_idx
        ]
    )

    target_val = (
        P2_PSI_VAL_RAW[
            :,
            target_idx
        ]
    )

    target_test = (
        P2_PSI_TEST_RAW[
            :,
            target_idx
        ]
    )


    design_fit = (
        P2_PSI_FIT_RAW[
            :,
            other_idx
        ]
    )

    design_val = (
        P2_PSI_VAL_RAW[
            :,
            other_idx
        ]
    )

    design_test = (
        P2_PSI_TEST_RAW[
            :,
            other_idx
        ]
    )


    alias_coef, _, alias_rank, _ = (
        np.linalg.lstsq(
            design_fit,
            target_fit,
            rcond=None,
        )
    )


    pred_fit = (
        design_fit
        @
        alias_coef
    )

    pred_val = (
        design_val
        @
        alias_coef
    )

    pred_test = (
        design_test
        @
        alias_coef
    )


    fit_residual = (
        relative_vector_error(
            pred_fit,
            target_fit,
        )
    )

    val_residual = (
        relative_vector_error(
            pred_val,
            target_val,
        )
    )

    test_residual = (
        relative_vector_error(
            pred_test,
            target_test,
        )
    )


    # ------------------------------------------------------------
    # Also inspect strongest pairwise cosine similarity
    # ------------------------------------------------------------

    target_norm = np.linalg.norm(
        target_fit
    )

    other_norms = np.linalg.norm(
        design_fit,
        axis=0,
    )

    cosine = np.abs(
        (
            design_fit.T
            @
            target_fit
        )
        /
        np.maximum(
            other_norms
            *
            target_norm,
            np.finfo(float).tiny,
        )
    )


    top_local = int(
        np.argmax(
            cosine
        )
    )

    top_idx = int(
        other_idx[
            top_local
        ]
    )

    top_type, top_support = (
        P2_CANDIDATES[
            top_idx
        ]
    )


    REPLACEABILITY_ROWS.append({

        "interaction_type":
            p2_interaction_type(
                support
            ),

        "support":
            support,

        "selected_Cell7":
            (
                support
                in
                PRED_P2_SUPPORTS
            ),

        "fit_alias_rank":
            int(
                alias_rank
            ),

        "fit_replaceability_residual":
            fit_residual,

        "validation_replaceability_residual":
            val_residual,

        "test_replaceability_residual":
            test_residual,

        "strongest_single_competitor":
            top_support,

        "competitor_type":
            top_type,

        "max_abs_cosine":
            float(
                cosine[
                    top_local
                ]
            ),
    })


P2_TRUE_REPLACEABILITY_AUDIT = (
    pd.DataFrame(
        REPLACEABILITY_ROWS
    )
)


# ================================================================
# C. ORACLE-SIX FIT
#
# IMPORTANT:
#
# This does NOT test topology inference.
#
# It asks:
#
#   If support selection were magically correct,
#   is A_SLICE_EST accurate enough to estimate the six physical
#   coefficients?
#
# Fit ONLY on cycles 0-3.
#
# Validation and test are untouched.
# ================================================================

P2_ORACLE6_FIT_RAW = (
    P2_PSI_FIT_RAW[
        :,
        TRUE_P2_INDICES
    ]
)

P2_ORACLE6_VAL_RAW = (
    P2_PSI_VAL_RAW[
        :,
        TRUE_P2_INDICES
    ]
)

P2_ORACLE6_TEST_RAW = (
    P2_PSI_TEST_RAW[
        :,
        TRUE_P2_INDICES
    ]
)


P2_ORACLE6_BETA_EST, _, P2_ORACLE6_RANK, _ = (
    np.linalg.lstsq(
        P2_ORACLE6_FIT_RAW,
        P2_Y_FIT[
            :,
            0
        ],
        rcond=None,
    )
)


assert (
    P2_ORACLE6_RANK
    ==
    6
)


P2_ORACLE6_BETA_TRUE = (
    P2_BETA_TRUE_RAW[
        TRUE_P2_INDICES
    ]
)


P2_ORACLE6_COEFF_ERROR = (
    relative_vector_error(
        P2_ORACLE6_BETA_EST,
        P2_ORACLE6_BETA_TRUE,
    )
)


P2_ORACLE6_FIT_PRED = (
    P2_ORACLE6_FIT_RAW
    @
    P2_ORACLE6_BETA_EST
).reshape(
    len(
        X_G1_FIT
    ),
    N,
)


P2_ORACLE6_VAL_PRED = (
    P2_ORACLE6_VAL_RAW
    @
    P2_ORACLE6_BETA_EST
).reshape(
    len(
        X_G1_VAL
    ),
    N,
)


P2_ORACLE6_TEST_PRED = (
    P2_ORACLE6_TEST_RAW
    @
    P2_ORACLE6_BETA_EST
).reshape(
    len(
        X_G1_TEST
    ),
    N,
)


P2_ORACLE6_ERRORS = {
    "fit_vs_Aest":
        relative_fro_error(
            P2_ORACLE6_FIT_PRED,
            Y_G1_FIT,
        ),

    "fit_vs_exact":
        relative_fro_error(
            P2_ORACLE6_FIT_PRED,
            G1_TRUE_FIT,
        ),

    "validation_vs_Aest":
        relative_fro_error(
            P2_ORACLE6_VAL_PRED,
            Y_G1_VAL,
        ),

    "validation_vs_exact":
        relative_fro_error(
            P2_ORACLE6_VAL_PRED,
            G1_TRUE_VAL,
        ),

    "test_vs_Aest":
        relative_fro_error(
            P2_ORACLE6_TEST_PRED,
            Y_G1_TEST,
        ),

    "test_vs_exact":
        relative_fro_error(
            P2_ORACLE6_TEST_PRED,
            G1_TRUE_TEST,
        ),
}


ORACLE6_COEFF_ROWS = []


for local_idx, support in enumerate(
    TRUE_P2_SUPPORTS_SORTED
):

    ORACLE6_COEFF_ROWS.append({

        "interaction_type":
            p2_interaction_type(
                support
            ),

        "support":
            support,

        "beta_true":
            float(
                P2_ORACLE6_BETA_TRUE[
                    local_idx
                ]
            ),

        "beta_oracle6":
            float(
                P2_ORACLE6_BETA_EST[
                    local_idx
                ]
            ),

        "abs_error":
            float(
                abs(
                    P2_ORACLE6_BETA_EST[
                        local_idx
                    ]
                    -
                    P2_ORACLE6_BETA_TRUE[
                        local_idx
                    ]
                )
            ),
    })


P2_ORACLE6_COEFFICIENTS = (
    pd.DataFrame(
        ORACLE6_COEFF_ROWS
    )
)


# ================================================================
# D. LEAVE-ONE-TRUE-INTERACTION-OUT
#
# For each true interaction alpha:
#
#   remove alpha
#   allow the remaining FIVE true physical interactions to refit
#
# This measures how much unique dynamical information alpha
# contributes INSIDE the true physical model.
#
# Compare against the full oracle-six model.
# ================================================================

LOTO_ROWS = []


for omitted_local_idx, omitted_support in enumerate(
    TRUE_P2_SUPPORTS_SORTED
):

    keep_local_idx = np.asarray(
        [
            i

            for i in range(6)

            if i
            !=
            omitted_local_idx
        ],
        dtype=int,
    )


    design_fit = (
        P2_ORACLE6_FIT_RAW[
            :,
            keep_local_idx
        ]
    )

    design_val = (
        P2_ORACLE6_VAL_RAW[
            :,
            keep_local_idx
        ]
    )

    design_test = (
        P2_ORACLE6_TEST_RAW[
            :,
            keep_local_idx
        ]
    )


    beta5, _, rank5, _ = (
        np.linalg.lstsq(
            design_fit,
            P2_Y_FIT[
                :,
                0
            ],
            rcond=None,
        )
    )


    pred_fit = (
        design_fit
        @
        beta5
    ).reshape(
        len(
            X_G1_FIT
        ),
        N,
    )

    pred_val = (
        design_val
        @
        beta5
    ).reshape(
        len(
            X_G1_VAL
        ),
        N,
    )

    pred_test = (
        design_test
        @
        beta5
    ).reshape(
        len(
            X_G1_TEST
        ),
        N,
    )


    fit_err = (
        relative_fro_error(
            pred_fit,
            Y_G1_FIT,
        )
    )

    val_err = (
        relative_fro_error(
            pred_val,
            Y_G1_VAL,
        )
    )

    test_err = (
        relative_fro_error(
            pred_test,
            Y_G1_TEST,
        )
    )


    LOTO_ROWS.append({

        "omitted_interaction":
            omitted_support,

        "interaction_type":
            p2_interaction_type(
                omitted_support
            ),

        "selected_Cell7":
            (
                omitted_support
                in
                PRED_P2_SUPPORTS
            ),

        "remaining_true_rank":
            int(
                rank5
            ),

        "fit_error":
            fit_err,

        "validation_error":
            val_err,

        "test_error":
            test_err,

        "fit_error_increase_vs_oracle6":
            (
                fit_err
                -
                P2_ORACLE6_ERRORS[
                    "fit_vs_Aest"
                ]
            ),

        "validation_error_increase_vs_oracle6":
            (
                val_err
                -
                P2_ORACLE6_ERRORS[
                    "validation_vs_Aest"
                ]
            ),

        "test_error_increase_vs_oracle6":
            (
                test_err
                -
                P2_ORACLE6_ERRORS[
                    "test_vs_Aest"
                ]
            ),
    })


P2_LEAVE_ONE_TRUE_OUT = (
    pd.DataFrame(
        LOTO_ROWS
    )
)


# ================================================================
# E. Compare Cell-7 selected model vs oracle-six
# ================================================================

P2_MODEL_COMPARISON = pd.DataFrame(
    [
        {
            "model":
                "Cell7 selected sparse model",

            "n_interactions":
                len(
                    PRED_P2_SUPPORTS
                ),

            "fit_error":
                P2_FIT_ERR_AEST,

            # Post-refit; not truly held-out anymore
            "cycle4_postrefit_error":
                P2_VAL_POSTREFIT_ERR_AEST,

            "cycle5_test_error":
                P2_TEST_ERR_AEST,

            "coefficient_error":
                P2_COEFF_REL_ERROR,
        },

        {
            "model":
                "oracle true-six fit",

            "n_interactions":
                6,

            "fit_error":
                P2_ORACLE6_ERRORS[
                    "fit_vs_Aest"
                ],

            "cycle4_postrefit_error":
                P2_ORACLE6_ERRORS[
                    "validation_vs_Aest"
                ],

            "cycle5_test_error":
                P2_ORACLE6_ERRORS[
                    "test_vs_Aest"
                ],

            "coefficient_error":
                P2_ORACLE6_COEFF_ERROR,
        },
    ]
)


# ================================================================
# REPORT
# ================================================================

print("=" * 80)
print("CELL 7b — P2 SUPPORT-SELECTION FAILURE DIAGNOSIS")
print("=" * 80)


print()
print("A. TRUE INTERACTION OBSERVATIONAL STRENGTH")
print("-" * 80)

display(
    P2_TRUE_STRENGTH_AUDIT.style.format({

        "beta_true":
            "{:.8f}",

        "fit_contribution_rms":
            "{:.6e}",

        "fit_fraction_of_field":
            "{:.6e}",

        "validation_contribution_rms":
            "{:.6e}",

        "validation_fraction_of_field":
            "{:.6e}",

        "test_contribution_rms":
            "{:.6e}",

        "test_fraction_of_field":
            "{:.6e}",

        "test_to_fit_strength_ratio":
            "{:.6f}",
    })
)


print()
print("B. TRUE INTERACTION REPLACEABILITY BY OTHER 83 CANDIDATES")
print("-" * 80)

display(
    P2_TRUE_REPLACEABILITY_AUDIT.style.format({

        "fit_replaceability_residual":
            "{:.6e}",

        "validation_replaceability_residual":
            "{:.6e}",

        "test_replaceability_residual":
            "{:.6e}",

        "max_abs_cosine":
            "{:.8f}",
    })
)


print()
print("C. ORACLE TRUE-SIX COEFFICIENT FIT")
print("-" * 80)

print(
    f"Oracle-six rank              : "
    f"{P2_ORACLE6_RANK} / 6"
)

print(
    f"Oracle-six coefficient error : "
    f"{P2_ORACLE6_COEFF_ERROR:.6e}"
)

print()

print(
    f"Fit vs A_est                 : "
    f"{P2_ORACLE6_ERRORS['fit_vs_Aest']:.6e}"
)

print(
    f"Fit vs exact G1              : "
    f"{P2_ORACLE6_ERRORS['fit_vs_exact']:.6e}"
)

print()

print(
    f"Validation vs A_est          : "
    f"{P2_ORACLE6_ERRORS['validation_vs_Aest']:.6e}"
)

print(
    f"Validation vs exact G1       : "
    f"{P2_ORACLE6_ERRORS['validation_vs_exact']:.6e}"
)

print()

print(
    f"Test vs A_est                : "
    f"{P2_ORACLE6_ERRORS['test_vs_Aest']:.6e}"
)

print(
    f"Test vs exact G1             : "
    f"{P2_ORACLE6_ERRORS['test_vs_exact']:.6e}"
)


print()
print("Oracle-six physical coefficients")
print("-" * 80)

display(
    P2_ORACLE6_COEFFICIENTS.style.format({

        "beta_true":
            "{:.8f}",

        "beta_oracle6":
            "{:.8f}",

        "abs_error":
            "{:.6e}",
    })
)


print()
print("D. LEAVE ONE TRUE INTERACTION OUT")
print("-" * 80)

display(
    P2_LEAVE_ONE_TRUE_OUT.style.format({

        "fit_error":
            "{:.6e}",

        "validation_error":
            "{:.6e}",

        "test_error":
            "{:.6e}",

        "fit_error_increase_vs_oracle6":
            "{:.6e}",

        "validation_error_increase_vs_oracle6":
            "{:.6e}",

        "test_error_increase_vs_oracle6":
            "{:.6e}",
    })
)


print()
print("E. CELL-7 MODEL VS ORACLE-SIX")
print("-" * 80)

display(
    P2_MODEL_COMPARISON.style.format({

        "fit_error":
            "{:.6e}",

        "cycle4_postrefit_error":
            "{:.6e}",

        "cycle5_test_error":
            "{:.6e}",

        "coefficient_error":
            "{:.6e}",
    })
)


# ================================================================
# SAVE
# ================================================================

P2_TRUE_STRENGTH_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_true_interaction_strength.csv",
    index=False,
)


P2_TRUE_REPLACEABILITY_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_true_interaction_replaceability.csv",
    index=False,
)


P2_ORACLE6_COEFFICIENTS.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_oracle6_coefficients.csv",
    index=False,
)


P2_LEAVE_ONE_TRUE_OUT.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_leave_one_true_out.csv",
    index=False,
)


P2_MODEL_COMPARISON.to_csv(
    OUTPUT_DIR
    /
    "stage6_G1_P2_model_comparison.csv",
    index=False,
)


print()
print("=" * 80)
print("Cell 7b PASSED.")
print("=" * 80)

CELL 7b — P2 SUPPORT-SELECTION FAILURE DIAGNOSIS

A. TRUE INTERACTION OBSERVATIONAL STRENGTH
--------------------------------------------------------------------------------


,interaction_type,support,beta_true,selected_Cell7,fit_contribution_rms,fit_fraction_of_field,validation_contribution_rms,validation_fraction_of_field,test_contribution_rms,test_fraction_of_field,test_to_fit_strength_ratio
0,pair,"(1, 2)",1.00600000,False,3.107863e-02,1.378292e-01,2.299471e-03,3.538301e-02,2.651387e-03,5.121978e-02,0.085312
1,pair,"(3, 4)",1.10500000,True,1.680526e-01,7.452892e-01,3.339962e-02,5.139353e-01,1.866865e-02,3.606430e-01,0.111088
2,pair,"(5, 6)",1.17700000,True,1.347488e-01,5.975914e-01,1.439009e-02,2.214269e-01,5.775627e-03,1.115742e-01,0.042862
3,pair,"(7, 8)",0.91700000,True,6.137194e-02,2.721757e-01,5.401574e-02,8.311648e-01,4.799052e-02,9.270860e-01,0.781962
4,triad,"(1, 2, 3)",0.02000000,False,3.986965e-04,1.768162e-03,1.142387e-04,1.757843e-03,8.330450e-05,1.609285e-03,0.208942
5,triad,"(2, 5, 8)",0.02000000,False,1.091483e-03,4.840570e-03,3.150499e-04,4.847817e-03,2.355801e-04,4.550961e-03,0.215835



B. TRUE INTERACTION REPLACEABILITY BY OTHER 83 CANDIDATES
--------------------------------------------------------------------------------


,interaction_type,support,selected_Cell7,fit_alias_rank,fit_replaceability_residual,validation_replaceability_residual,test_replaceability_residual,strongest_single_competitor,competitor_type,max_abs_cosine
0,pair,"(1, 2)",False,83,3.634025e-07,5.739620e-02,1.381442e-01,"(1, 2, 7)",triad,0.93426424
1,pair,"(3, 4)",True,83,2.103172e-07,1.021680e-02,5.114193e-02,"(3, 4, 8)",triad,0.87521803
2,pair,"(5, 6)",True,83,7.972178e-07,2.894630e-02,2.043770e-01,"(5, 6, 8)",triad,0.99310419
3,pair,"(7, 8)",True,83,2.544859e-06,7.643809e-03,2.516530e-02,"(2, 7, 8)",triad,0.95551359
4,triad,"(1, 2, 3)",False,83,2.465348e-07,5.945271e-03,2.239560e-02,"(1, 3)",pair,0.93285355
5,triad,"(2, 5, 8)",False,83,2.158936e-07,5.918614e-03,2.219481e-02,"(2, 5)",pair,0.92702841



C. ORACLE TRUE-SIX COEFFICIENT FIT
--------------------------------------------------------------------------------
Oracle-six rank              : 6 / 6
Oracle-six coefficient error : 1.079278e-09

Fit vs A_est                 : 1.170821e-10
Fit vs exact G1              : 1.172544e-09

Validation vs A_est          : 1.493235e-10
Validation vs exact G1       : 5.396781e-10

Test vs A_est                : 9.263528e-11
Test vs exact G1             : 3.471601e-10

Oracle-six physical coefficients
--------------------------------------------------------------------------------


...


D. LEAVE ONE TRUE INTERACTION OUT
--------------------------------------------------------------------------------


,omitted_interaction,interaction_type,selected_Cell7,remaining_true_rank,fit_error,validation_error,test_error,fit_error_increase_vs_oracle6,validation_error_increase_vs_oracle6,test_error_increase_vs_oracle6
0,"(1, 2)",pair,False,5,1.160268e-01,8.072947e-02,1.038549e-01,1.160268e-01,8.072947e-02,1.038549e-01
1,"(3, 4)",pair,True,5,5.973764e-01,4.565848e-01,3.957783e-01,5.973764e-01,4.565848e-01,3.957783e-01
2,"(5, 6)",pair,True,5,4.581630e-01,3.294940e-01,3.674992e-01,4.581630e-01,3.294940e-01,3.674992e-01
3,"(7, 8)",pair,True,5,2.607725e-01,7.956352e-01,8.891797e-01,2.607725e-01,7.956352e-01,8.891797e-01
4,"(1, 2, 3)",triad,False,5,1.340299e-03,1.485002e-03,1.490381e-03,1.340299e-03,1.485002e-03,1.490381e-03
5,"(2, 5, 8)",triad,False,5,3.260485e-03,4.038636e-03,4.359396e-03,3.260485e-03,4.038636e-03,4.359396e-03



E. CELL-7 MODEL VS ORACLE-SIX
--------------------------------------------------------------------------------


,model,n_interactions,fit_error,cycle4_postrefit_error,cycle5_test_error,coefficient_error
0,Cell7 selected sparse model,4,1.364634e-01,3.461166e-02,5.196969e-02,4.765601e-01
1,oracle true-six fit,6,1.170821e-10,1.493235e-10,9.263528e-11,1.079278e-09



Cell 7b PASSED.


Time Resolution Scanning Inference Algorithm

In [11]:
# ================================================================
# RS-TSC prototype
# Cell 1: construct three canonical (s, DeltaT) window ensembles
#
# NO inference in this cell.
# NO reintegration in this cell.
#
# Everything is sliced directly from the ONE stored continuous
# N=8 trajectory.
#
# Canonical cases
# ---------------------------------------------------------------
# Case 1:
#   DeltaT < tau, fully inside G1
#
#       s = 0.015
#       DeltaT = 0.030
#
#   center window:
#       [0.015, 0.045]
#
#   oracle persistent composition:
#       1.0 G1
#
#
# Case 2:
#   DeltaT < tau, crosses G1 -> G2
#
#       s = 0.045
#       DeltaT = 0.030
#
#   center window:
#       [0.045, 0.075]
#
#   oracle persistent composition:
#       0.5 G1 + 0.5 G2
#
#
# Case 3:
#   tau < DeltaT < 2 tau, crosses G1 -> G2 -> G3
#
#       s = 0.045
#       DeltaT = 0.090
#
#   center window:
#       [0.045, 0.135]
#
#   oracle persistent composition:
#       (1/6) G1 + (2/3) G2 + (1/6) G3
#
#
# For each canonical (s, DeltaT):
#
#   center ensemble:
#       exact same phase in every repeated cycle
#
#   local ensemble:
#       nearby starts around s
#
# This prepares the data for later local-stationarity inference.
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ================================================================
# 0. Locate the stored continuous trajectory
# ================================================================

def _find_existing_global(names):
    for name in names:
        if name in globals():
            return np.asarray(globals()[name]), name
    return None, None


# First try variables already present in the notebook.
RS_TIME, RS_TIME_SOURCE = _find_existing_global([
    "T_TRAJ",
    "t_traj",
    "T_OBS_GRID",
    "t_obs",
    "times",
    "time_grid",
])

RS_X, RS_X_SOURCE = _find_existing_global([
    "X_TRAJ",
    "x_traj",
    "X_OBS",
    "trajectory",
    "X_trajectory",
])


# Otherwise load the trajectory saved by the earlier notebook.
if RS_TIME is None or RS_X is None:

    if "OUTPUT_DIR" in globals():
        RS_OUTPUT_DIR = Path(OUTPUT_DIR)
    else:
        RS_OUTPUT_DIR = Path(
            "stage6_n8_timeseries_slicing"
        )

    trajectory_path = (
        RS_OUTPUT_DIR
        /
        "n8_single_continuous_trajectory.npz"
    )

    if not trajectory_path.exists():
        raise FileNotFoundError(
            f"Could not find stored trajectory:\n"
            f"{trajectory_path}"
        )

    _traj_npz = np.load(
        trajectory_path,
        allow_pickle=True,
    )

    # ------------------------------------------------------------
    # Robustly identify:
    #   time = 1D array
    #   state = 2D array with N=8
    # ------------------------------------------------------------

    one_d = []
    two_d = []

    for key in _traj_npz.files:

        arr = np.asarray(
            _traj_npz[key]
        )

        if arr.ndim == 1:
            one_d.append(
                (key, arr)
            )

        elif (
            arr.ndim == 2
            and
            arr.shape[1] == 8
        ):
            two_d.append(
                (key, arr)
            )

    found = False

    for t_key, t_arr in one_d:

        for x_key, x_arr in two_d:

            if len(t_arr) == len(x_arr):

                if np.all(
                    np.diff(t_arr) > 0
                ):
                    RS_TIME = np.asarray(
                        t_arr,
                        dtype=float,
                    )

                    RS_X = np.asarray(
                        x_arr,
                        dtype=float,
                    )

                    RS_TIME_SOURCE = (
                        f"{trajectory_path.name}:{t_key}"
                    )

                    RS_X_SOURCE = (
                        f"{trajectory_path.name}:{x_key}"
                    )

                    found = True
                    break

        if found:
            break

    if not found:
        raise RuntimeError(
            "Could not automatically identify "
            "time/state arrays in the stored trajectory."
        )

else:

    if "OUTPUT_DIR" in globals():
        RS_OUTPUT_DIR = Path(
            OUTPUT_DIR
        )
    else:
        RS_OUTPUT_DIR = Path(
            "stage6_n8_timeseries_slicing"
        )


RS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# 1. Frozen physical timing
# ================================================================

RS_N = int(
    RS_X.shape[1]
)

assert RS_N == 8

assert (
    RS_TIME.ndim == 1
)

assert (
    RS_X.ndim == 2
)

assert (
    len(RS_TIME)
    ==
    len(RS_X)
)


RS_DT_OBS = float(
    np.median(
        np.diff(
            RS_TIME
        )
    )
)

RS_TAU = 0.060

RS_SNAPSHOT_LABELS = (
    "G1",
    "G2",
    "G3",
    "G4",
    "G5",
    "G6",
)

RS_N_SNAPSHOTS_PER_CYCLE = len(
    RS_SNAPSHOT_LABELS
)

RS_CYCLE_DURATION = (
    RS_N_SNAPSHOTS_PER_CYCLE
    *
    RS_TAU
)


assert np.isclose(
    RS_DT_OBS,
    0.0005,
    atol=1e-14,
)

assert np.isclose(
    RS_CYCLE_DURATION,
    0.360,
    atol=1e-14,
)


# Current trajectory contains six cycles.
RS_N_CYCLES_AVAILABLE = int(
    round(
        (
            RS_TIME[-1]
            -
            RS_TIME[0]
        )
        /
        RS_CYCLE_DURATION
    )
)

assert (
    RS_N_CYCLES_AVAILABLE
    >= 6
)

# Keep the exact same temporal split used earlier.
RS_FIT_CYCLES = (
    0,
    1,
    2,
    3,
)

RS_VALIDATION_CYCLES = (
    4,
)

RS_TEST_CYCLES = (
    5,
)

RS_USED_CYCLES = (
    RS_FIT_CYCLES
    +
    RS_VALIDATION_CYCLES
    +
    RS_TEST_CYCLES
)


# ================================================================
# 2. Local ensemble geometry
#
# Each canonical center s gets nearby starts:
#
#       s - 0.003, ..., s, ..., s + 0.003
#
# sampled at the native observation resolution 0.0005.
#
# => 13 phase positions per cycle
# => 78 windows total per case
#
# This bandwidth is NOT frozen.
# It is only the first RS-TSC prototype locality scale.
# ================================================================

RS_LOCAL_HALF_WIDTH = 0.003

RS_LOCAL_OFFSETS = np.arange(
    -RS_LOCAL_HALF_WIDTH,
    RS_LOCAL_HALF_WIDTH
        + 0.5 * RS_DT_OBS,
    RS_DT_OBS,
)

assert len(
    RS_LOCAL_OFFSETS
) == 13

assert np.any(
    np.isclose(
        RS_LOCAL_OFFSETS,
        0.0,
        atol=1e-15,
    )
)


# ================================================================
# 3. Canonical test cases
# ================================================================

RS_CASE_SPECS = {

    "case1_static_G1": {
        "description":
            "DeltaT < tau; window fully inside G1",

        "center_phase":
            0.015,

        "delta_T":
            0.030,

        "expected_sequence":
            ("G1",),

        "expected_center_fractions":
            np.asarray(
                [
                    1.0,
                    0.0,
                    0.0,
                    0.0,
                    0.0,
                    0.0,
                ]
            ),
    },


    "case2_G1_to_G2": {
        "description":
            "DeltaT < tau; crosses one boundary G1 -> G2",

        "center_phase":
            0.045,

        "delta_T":
            0.030,

        "expected_sequence":
            (
                "G1",
                "G2",
            ),

        "expected_center_fractions":
            np.asarray(
                [
                    0.5,
                    0.5,
                    0.0,
                    0.0,
                    0.0,
                    0.0,
                ]
            ),
    },


    "case3_G1_G2_G3": {
        "description":
            "tau < DeltaT < 2 tau; crosses G1 -> G2 -> G3",

        "center_phase":
            0.045,

        "delta_T":
            0.090,

        "expected_sequence":
            (
                "G1",
                "G2",
                "G3",
            ),

        "expected_center_fractions":
            np.asarray(
                [
                    1.0 / 6.0,
                    2.0 / 3.0,
                    1.0 / 6.0,
                    0.0,
                    0.0,
                    0.0,
                ]
            ),
    },
}


# ================================================================
# 4. Exact grid indexing
# ================================================================

def rs_time_to_index(t):

    idx = int(
        round(
            (
                float(t)
                -
                float(RS_TIME[0])
            )
            /
            RS_DT_OBS
        )
    )

    if (
        idx < 0
        or
        idx >= len(RS_TIME)
    ):
        raise IndexError(
            f"time {t:.9f} outside stored trajectory"
        )

    if not np.isclose(
        RS_TIME[idx],
        t,
        atol=1e-12,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Requested RS window boundary does not "
            "lie exactly on the observation grid:\n"
            f"requested={t:.12f}, "
            f"grid={RS_TIME[idx]:.12f}"
        )

    return idx


# ================================================================
# 5. Snapshot identity of every trajectory integration interval
#
# Interval k means:
#
#       [RS_TIME[k], RS_TIME[k+1]]
#
# Use interval midpoint to avoid ambiguity exactly at switching
# boundaries.
# ================================================================

RS_INTERVAL_MIDPOINTS = (
    0.5
    *
    (
        RS_TIME[:-1]
        +
        RS_TIME[1:]
    )
)


RS_INTERVAL_PHASE = np.mod(
    RS_INTERVAL_MIDPOINTS,
    RS_CYCLE_DURATION,
)


RS_INTERVAL_SNAPSHOT_INDEX = np.floor(
    RS_INTERVAL_PHASE
    /
    RS_TAU
).astype(int)


RS_INTERVAL_SNAPSHOT_INDEX = np.clip(
    RS_INTERVAL_SNAPSHOT_INDEX,
    0,
    RS_N_SNAPSHOTS_PER_CYCLE - 1,
)


# ================================================================
# 6. Occupation bookkeeping for one exact observed window
# ================================================================

def rs_window_occupation(
    start_idx,
    end_idx,
):

    if end_idx <= start_idx:
        raise ValueError(
            "end_idx must exceed start_idx"
        )

    snapshot_idx = (
        RS_INTERVAL_SNAPSHOT_INDEX[
            start_idx:end_idx
        ]
    )

    counts = np.bincount(
        snapshot_idx,
        minlength=
            RS_N_SNAPSHOTS_PER_CYCLE,
    )

    durations = (
        counts.astype(float)
        *
        RS_DT_OBS
    )

    total_duration = (
        RS_TIME[end_idx]
        -
        RS_TIME[start_idx]
    )

    fractions = (
        durations
        /
        total_duration
    )


    # Ordered sequence of touched snapshots.
    touched = []

    for idx in snapshot_idx:

        label = (
            RS_SNAPSHOT_LABELS[
                int(idx)
            ]
        )

        if (
            len(touched) == 0
            or
            touched[-1] != label
        ):
            touched.append(
                label
            )


    return (
        durations,
        fractions,
        tuple(touched),
    )


# ================================================================
# 7. Construct local ensembles
# ================================================================

RS_CASES = {}

RS_METADATA_ROWS = []


for case_name, spec in RS_CASE_SPECS.items():

    center_phase = float(
        spec[
            "center_phase"
        ]
    )

    delta_T = float(
        spec[
            "delta_T"
        ]
    )


    X_start_list = []
    X_end_list = []

    start_idx_list = []
    end_idx_list = []

    cycle_list = []
    offset_list = []

    start_time_list = []
    end_time_list = []

    occupation_duration_list = []
    occupation_fraction_list = []

    sequence_list = []
    split_list = []


    for cycle in RS_USED_CYCLES:

        cycle_start = (
            cycle
            *
            RS_CYCLE_DURATION
        )


        for local_offset in RS_LOCAL_OFFSETS:

            phase = (
                center_phase
                +
                float(
                    local_offset
                )
            )

            t_start = (
                cycle_start
                +
                phase
            )

            t_end = (
                t_start
                +
                delta_T
            )


            start_idx = (
                rs_time_to_index(
                    t_start
                )
            )

            end_idx = (
                rs_time_to_index(
                    t_end
                )
            )


            durations, fractions, sequence = (
                rs_window_occupation(
                    start_idx,
                    end_idx,
                )
            )


            # ----------------------------------------------------
            # All local starts must remain in the same qualitative
            # protocol sector as the canonical center.
            #
            # Case 1: G1
            # Case 2: G1 -> G2
            # Case 3: G1 -> G2 -> G3
            # ----------------------------------------------------

            if (
                sequence
                !=
                spec[
                    "expected_sequence"
                ]
            ):
                raise RuntimeError(
                    f"{case_name}: local window changed "
                    f"protocol sector.\n"
                    f"cycle={cycle}, "
                    f"offset={local_offset:.6f}, "
                    f"sequence={sequence}"
                )


            if cycle in RS_FIT_CYCLES:
                split = "fit"

            elif cycle in RS_VALIDATION_CYCLES:
                split = "validation"

            elif cycle in RS_TEST_CYCLES:
                split = "test"

            else:
                raise RuntimeError(
                    cycle
                )


            X_start_list.append(
                RS_X[
                    start_idx
                ].copy()
            )

            X_end_list.append(
                RS_X[
                    end_idx
                ].copy()
            )

            start_idx_list.append(
                start_idx
            )

            end_idx_list.append(
                end_idx
            )

            cycle_list.append(
                cycle
            )

            offset_list.append(
                float(
                    local_offset
                )
            )

            start_time_list.append(
                float(
                    t_start
                )
            )

            end_time_list.append(
                float(
                    t_end
                )
            )

            occupation_duration_list.append(
                durations
            )

            occupation_fraction_list.append(
                fractions
            )

            sequence_list.append(
                sequence
            )

            split_list.append(
                split
            )


            metadata_row = {

                "case":
                    case_name,

                "cycle":
                    cycle,

                "split":
                    split,

                "center_phase":
                    center_phase,

                "local_offset":
                    float(
                        local_offset
                    ),

                "phase_start":
                    phase,

                "delta_T":
                    delta_T,

                "t_start":
                    float(
                        t_start
                    ),

                "t_end":
                    float(
                        t_end
                    ),

                "start_idx":
                    start_idx,

                "end_idx":
                    end_idx,

                "is_center":
                    bool(
                        np.isclose(
                            local_offset,
                            0.0,
                            atol=1e-15,
                        )
                    ),

                "sequence":
                    "->".join(
                        sequence
                    ),
            }


            for m, label in enumerate(
                RS_SNAPSHOT_LABELS
            ):

                metadata_row[
                    f"duration_{label}"
                ] = float(
                    durations[m]
                )

                metadata_row[
                    f"fraction_{label}"
                ] = float(
                    fractions[m]
                )


            RS_METADATA_ROWS.append(
                metadata_row
            )


    X_start = np.asarray(
        X_start_list
    )

    X_end = np.asarray(
        X_end_list
    )

    occupation_duration = np.asarray(
        occupation_duration_list
    )

    occupation_fraction = np.asarray(
        occupation_fraction_list
    )

    cycle_array = np.asarray(
        cycle_list,
        dtype=int,
    )

    offset_array = np.asarray(
        offset_list,
        dtype=float,
    )

    split_array = np.asarray(
        split_list,
        dtype=object,
    )


    center_mask = np.isclose(
        offset_array,
        0.0,
        atol=1e-15,
    )


    fit_mask = (
        split_array
        ==
        "fit"
    )

    validation_mask = (
        split_array
        ==
        "validation"
    )

    test_mask = (
        split_array
        ==
        "test"
    )


    # ------------------------------------------------------------
    # Canonical-center oracle fractions must be exact.
    # ------------------------------------------------------------

    center_fractions = (
        occupation_fraction[
            center_mask
        ]
    )

    expected_center = np.asarray(
        spec[
            "expected_center_fractions"
        ],
        dtype=float,
    )


    assert np.allclose(
        center_fractions,
        expected_center[
            None,
            :
        ],
        atol=1e-13,
        rtol=0.0,
    )


    RS_CASES[
        case_name
    ] = {

        "description":
            spec[
                "description"
            ],

        "center_phase":
            center_phase,

        "delta_T":
            delta_T,

        "expected_sequence":
            spec[
                "expected_sequence"
            ],

        "X_start":
            X_start,

        "X_end":
            X_end,

        "start_idx":
            np.asarray(
                start_idx_list,
                dtype=int,
            ),

        "end_idx":
            np.asarray(
                end_idx_list,
                dtype=int,
            ),

        "start_time":
            np.asarray(
                start_time_list,
                dtype=float,
            ),

        "end_time":
            np.asarray(
                end_time_list,
                dtype=float,
            ),

        "cycle":
            cycle_array,

        "local_offset":
            offset_array,

        "occupation_duration":
            occupation_duration,

        "occupation_fraction":
            occupation_fraction,

        "center_mask":
            center_mask,

        "fit_mask":
            fit_mask,

        "validation_mask":
            validation_mask,

        "test_mask":
            test_mask,
    }


# ================================================================
# 8. Global metadata table
# ================================================================

RS_WINDOW_METADATA = pd.DataFrame(
    RS_METADATA_ROWS
)


# ================================================================
# 9. Source-data audit
#
# Verify every extracted endpoint is EXACTLY a stored trajectory
# state; there has been no reintegration/interpolation.
# ================================================================

for case_name, case in RS_CASES.items():

    assert np.array_equal(
        case[
            "X_start"
        ],
        RS_X[
            case[
                "start_idx"
            ]
        ],
    )

    assert np.array_equal(
        case[
            "X_end"
        ],
        RS_X[
            case[
                "end_idx"
            ]
        ],
    )


# ================================================================
# 10. Report
# ================================================================

print("=" * 82)
print("RS-TSC PROTOTYPE — CELL 1")
print("THREE CANONICAL WINDOW ENSEMBLES")
print("=" * 82)

print(
    f"time source                  : "
    f"{RS_TIME_SOURCE}"
)

print(
    f"state source                 : "
    f"{RS_X_SOURCE}"
)

print(
    f"trajectory shape             : "
    f"{RS_X.shape}"
)

print(
    f"native dt                    : "
    f"{RS_DT_OBS:.6f}"
)

print(
    f"snapshot duration tau        : "
    f"{RS_TAU:.6f}"
)

print(
    f"cycle duration               : "
    f"{RS_CYCLE_DURATION:.6f}"
)

print(
    f"local half-width             : "
    f"{RS_LOCAL_HALF_WIDTH:.6f}"
)

print(
    f"local phase positions/cycle  : "
    f"{len(RS_LOCAL_OFFSETS)}"
)

print()


for case_name, case in RS_CASES.items():

    frac = (
        case[
            "occupation_fraction"
        ]
    )

    center_frac = (
        frac[
            case[
                "center_mask"
            ]
        ][0]
    )


    print("-" * 82)
    print(case_name)
    print("-" * 82)

    print(
        case[
            "description"
        ]
    )

    print(
        f"center phase s               : "
        f"{case['center_phase']:.6f}"
    )

    print(
        f"DeltaT                       : "
        f"{case['delta_T']:.6f}"
    )

    print(
        f"protocol sequence            : "
        f"{' -> '.join(case['expected_sequence'])}"
    )

    print(
        f"total local windows          : "
        f"{len(case['X_start'])}"
    )

    print(
        f"fit / validation / test      : "
        f"{np.sum(case['fit_mask'])} / "
        f"{np.sum(case['validation_mask'])} / "
        f"{np.sum(case['test_mask'])}"
    )

    print(
        f"exact-center windows         : "
        f"{np.sum(case['center_mask'])}"
    )

    print()

    print(
        "center occupation fractions : "
        +
        ", ".join(
            f"{label}={center_frac[m]:.6f}"
            for m, label
            in enumerate(
                RS_SNAPSHOT_LABELS
            )
            if center_frac[m] > 0
        )
    )

    print(
        "local fraction ranges       : "
        +
        ", ".join(
            (
                f"{label}="
                f"[{np.min(frac[:, m]):.6f}, "
                f"{np.max(frac[:, m]):.6f}]"
            )
            for m, label
            in enumerate(
                RS_SNAPSHOT_LABELS
            )
            if np.max(frac[:, m]) > 0
        )
    )

    print()


# ================================================================
# 11. Compact canonical-center summary
# ================================================================

center_summary_rows = []

for case_name, case in RS_CASES.items():

    center_frac = (
        case[
            "occupation_fraction"
        ][
            case[
                "center_mask"
            ]
        ][0]
    )

    row = {
        "case":
            case_name,

        "s":
            case[
                "center_phase"
            ],

        "DeltaT":
            case[
                "delta_T"
            ],

        "sequence":
            "->".join(
                case[
                    "expected_sequence"
                ]
            ),

        "local_windows":
            len(
                case[
                    "X_start"
                ]
            ),

        "center_windows":
            int(
                np.sum(
                    case[
                        "center_mask"
                    ]
                )
            ),
    }

    for m, label in enumerate(
        RS_SNAPSHOT_LABELS
    ):
        row[
            f"fraction_{label}"
        ] = center_frac[m]

    center_summary_rows.append(
        row
    )


RS_CANONICAL_SUMMARY = pd.DataFrame(
    center_summary_rows
)


print("=" * 82)
print("CANONICAL CENTER WINDOWS")
print("=" * 82)

display(
    RS_CANONICAL_SUMMARY.style.format({
        "s":
            "{:.6f}",

        "DeltaT":
            "{:.6f}",

        **{
            f"fraction_{label}":
                "{:.6f}"

            for label
            in RS_SNAPSHOT_LABELS
        },
    })
)


# ================================================================
# 12. Save the slices
# ================================================================

RS_WINDOW_METADATA.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell1_canonical_window_metadata.csv",
    index=False,
)


save_payload = {}

for case_name, case in RS_CASES.items():

    prefix = (
        case_name
        +
        "__"
    )

    for key in (
        "X_start",
        "X_end",
        "start_idx",
        "end_idx",
        "start_time",
        "end_time",
        "cycle",
        "local_offset",
        "occupation_duration",
        "occupation_fraction",
        "center_mask",
        "fit_mask",
        "validation_mask",
        "test_mask",
    ):

        save_payload[
            prefix
            +
            key
        ] = case[
            key
        ]


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell1_canonical_window_slices.npz",
    **save_payload,
)


print()
print("=" * 82)
print("Cell 1 PASSED.")
print("=" * 82)

RS-TSC PROTOTYPE — CELL 1
THREE CANONICAL WINDOW ENSEMBLES
time source                  : n8_single_continuous_trajectory.npz:times
state source                 : n8_single_continuous_trajectory.npz:states
trajectory shape             : (4321, 8)
native dt                    : 0.000500
snapshot duration tau        : 0.060000
cycle duration               : 0.360000
local half-width             : 0.003000
local phase positions/cycle  : 13

----------------------------------------------------------------------------------
case1_static_G1
----------------------------------------------------------------------------------
DeltaT < tau; window fully inside G1
center phase s               : 0.015000
DeltaT                       : 0.030000
protocol sequence            : G1
total local windows          : 78
fit / validation / test      : 52 / 13 / 13
exact-center windows         : 6

center occupation fractions : G1=1.000000
local fraction ranges       : G1=[1.000000, 1.000000]

----------------

,case,s,DeltaT,sequence,local_windows,center_windows,fraction_G1,fraction_G2,fraction_G3,fraction_G4,fraction_G5,fraction_G6
0,case1_static_G1,0.015000,0.030000,G1,78,6,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,case2_G1_to_G2,0.045000,0.030000,G1->G2,78,6,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000
2,case3_G1_G2_G3,0.045000,0.090000,G1->G2->G3,78,6,0.166667,0.666667,0.166667,0.000000,0.000000,0.000000



Cell 1 PASSED.


In [12]:
# ================================================================
# RS-TSC prototype
# Cell 2: Case 1 — static-window identity test
#
# Case 1:
#
#       s      = 0.015
#       DeltaT = 0.030
#
# Every local window lies completely inside G1.
#
# Goal
# ---------------------------------------------------------------
# 1. Extract four SHORT endpoint resolutions from the stored
#    trajectory:
#
#       eps = [0.005, 0.0075, 0.010, 0.015]
#
# 2. Recover the local instantaneous generator by cubic
#    eps -> 0 extrapolation.
#
# 3. Verify against the exact synthetic G1 field.
#
# 4. Audit the P2 known-function physical design locally:
#       28 pairs + 56 triads = 84 candidates.
#
# 5. For this static window:
#
#       F_bar       = G1
#       F_eff       = G1
#       DeltaF_temp = 0
#
# NO AGLASSO.
# NO topology oracle enters the dense 84-candidate fit.
# Oracle-six fit is diagnostic only.
# ================================================================

from itertools import combinations


# ================================================================
# 0. Case and short-resolution protocol
# ================================================================

RS_CASE1 = RS_CASES[
    "case1_static_G1"
]


RS_MICRO_EPS = np.asarray(
    [
        0.0050,
        0.0075,
        0.0100,
        0.0150,
    ],
    dtype=float,
)


RS_MICRO_EPS_STEPS = np.rint(
    RS_MICRO_EPS
    /
    RS_DT_OBS
).astype(int)


assert np.allclose(
    RS_MICRO_EPS_STEPS
    *
    RS_DT_OBS,
    RS_MICRO_EPS,
    atol=1e-14,
    rtol=0.0,
)


# ================================================================
# 1. Extract nested short-epsilon endpoints
#
# These use ONLY the stored trajectory.
# No reintegration / interpolation.
# ================================================================

C1_START_IDX = np.asarray(
    RS_CASE1[
        "start_idx"
    ],
    dtype=int,
)


C1_X0 = RS_X[
    C1_START_IDX
].copy()


C1_XF = np.stack(
    [
        RS_X[
            C1_START_IDX
            +
            step
        ]

        for step
        in RS_MICRO_EPS_STEPS
    ],
    axis=0,
)


assert C1_XF.shape == (
    len(RS_MICRO_EPS),
    len(C1_X0),
    RS_N,
)


# ---------------------------------------------------------------
# Protocol audit:
# every short epsilon trajectory segment must remain entirely G1.
# ---------------------------------------------------------------

for start_idx in C1_START_IDX:

    for step in RS_MICRO_EPS_STEPS:

        interval_labels = (
            RS_INTERVAL_SNAPSHOT_INDEX[
                start_idx:
                start_idx + step
            ]
        )

        assert np.all(
            interval_labels == 0
        )


# ================================================================
# 2. Four-point cubic resolution extrapolation
#
#       Y(eps) = [X(t+eps)-X(t)] / eps
#
# Fit exactly:
#
#       Y(eps)
#         =
#       A + c1 eps + c2 eps^2 + c3 eps^3
#
# A is the eps -> 0 generator estimate.
# ================================================================

C1_SECANT = (
    (
        C1_XF
        -
        C1_X0[
            None,
            :,
            :
        ]
    )
    /
    RS_MICRO_EPS[
        :,
        None,
        None,
    ]
)


C1_VANDERMONDE = np.column_stack(
    [
        np.ones_like(
            RS_MICRO_EPS
        ),

        RS_MICRO_EPS,

        RS_MICRO_EPS ** 2,

        RS_MICRO_EPS ** 3,
    ]
)


C1_POLY_COEFF = np.linalg.solve(
    C1_VANDERMONDE,
    C1_SECANT.reshape(
        len(
            RS_MICRO_EPS
        ),
        -1,
    ),
)


C1_A_EST = (
    C1_POLY_COEFF[
        0
    ]
    .reshape(
        len(
            C1_X0
        ),
        RS_N,
    )
)


# ================================================================
# 3. Frozen P2 physical dictionary
#
# Known microscopic functional forms:
#
# pair:
#
#   phi(x) = x + 0.5 x^2
#
# triad:
#
#   T_i = x_j x_k
#         - 0.5 x_i x_j
#         - 0.5 x_i x_k
#
# Unknown topology = all 28 pairs + all 56 triads.
# ================================================================

def rs_phi(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    return (
        x
        +
        0.5 * x ** 2
    )


def rs_unit_pair_field(
    X,
    support,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    i, j = (
        int(support[0]) - 1,
        int(support[1]) - 1,
    )

    F = np.zeros_like(
        X
    )

    flux = (
        rs_phi(
            X[:, j]
        )
        -
        rs_phi(
            X[:, i]
        )
    )

    F[:, i] += flux
    F[:, j] -= flux

    return F


def rs_unit_triad_field(
    X,
    support,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    i, j, k = [
        int(v) - 1
        for v in support
    ]

    xi = X[:, i]
    xj = X[:, j]
    xk = X[:, k]

    F = np.zeros_like(
        X
    )

    F[:, i] += (
        xj * xk
        -
        0.5 * xi * xj
        -
        0.5 * xi * xk
    )

    F[:, j] += (
        xi * xk
        -
        0.5 * xj * xi
        -
        0.5 * xj * xk
    )

    F[:, k] += (
        xi * xj
        -
        0.5 * xk * xi
        -
        0.5 * xk * xj
    )

    return F


RS_P2_PAIRS = tuple(
    combinations(
        range(
            1,
            RS_N + 1
        ),
        2,
    )
)


RS_P2_TRIADS = tuple(
    combinations(
        range(
            1,
            RS_N + 1
        ),
        3,
    )
)


RS_P2_CANDIDATES = (
    tuple(
        (
            "pair",
            support,
        )
        for support
        in RS_P2_PAIRS
    )
    +
    tuple(
        (
            "triad",
            support,
        )
        for support
        in RS_P2_TRIADS
    )
)


assert len(
    RS_P2_CANDIDATES
) == 84


def rs_build_physical_design(
    X,
):

    columns = []

    for interaction_type, support in RS_P2_CANDIDATES:

        if interaction_type == "pair":

            field = (
                rs_unit_pair_field(
                    X,
                    support,
                )
            )

        elif interaction_type == "triad":

            field = (
                rs_unit_triad_field(
                    X,
                    support,
                )
            )

        else:

            raise RuntimeError(
                interaction_type
            )

        columns.append(
            field.reshape(-1)
        )

    return np.column_stack(
        columns
    )


# ================================================================
# 4. Exact synthetic G1 oracle
#
# Used ONLY for diagnostic comparison.
# ================================================================

RS_EDGE_WEIGHT = {

    (1, 2): 1.006,
    (2, 3): 0.911,
    (3, 4): 1.105,
    (4, 5): 0.917,
    (5, 6): 1.177,
    (6, 7): 1.119,
    (7, 8): 0.917,
    (1, 8): 0.944,

    (1, 3): 1.026,
    (2, 4): 1.002,
    (3, 5): 1.067,
    (5, 7): 1.031,
}


RS_G1_TRUE_PAIRS = {
    (1, 2),
    (3, 4),
    (5, 6),
    (7, 8),
}


RS_G1_TRUE_TRIADS = {
    (1, 2, 3),
    (2, 5, 8),
}


RS_NATIVE_G = 0.020


RS_BETA_G1_TRUE = np.zeros(
    len(
        RS_P2_CANDIDATES
    ),
    dtype=float,
)


for idx, (
    interaction_type,
    support,
) in enumerate(
    RS_P2_CANDIDATES
):

    if (
        interaction_type == "pair"
        and
        support in RS_G1_TRUE_PAIRS
    ):

        canonical_edge = tuple(
            sorted(
                support
            )
        )

        RS_BETA_G1_TRUE[
            idx
        ] = (
            RS_EDGE_WEIGHT[
                canonical_edge
            ]
        )

    elif (
        interaction_type == "triad"
        and
        support in RS_G1_TRUE_TRIADS
    ):

        RS_BETA_G1_TRUE[
            idx
        ] = RS_NATIVE_G


assert np.count_nonzero(
    RS_BETA_G1_TRUE
) == 6


C1_PSI_RAW = (
    rs_build_physical_design(
        C1_X0
    )
)


C1_G1_TRUE = (
    C1_PSI_RAW
    @
    RS_BETA_G1_TRUE
).reshape(
    len(
        C1_X0
    ),
    RS_N,
)


# ================================================================
# 5. Basic relative-error helper
# ================================================================

def rs_relative_error(
    estimate,
    truth,
):

    estimate = np.asarray(
        estimate,
        dtype=float,
    )

    truth = np.asarray(
        truth,
        dtype=float,
    )

    denominator = np.linalg.norm(
        truth.ravel()
    )

    return float(
        np.linalg.norm(
            (
                estimate
                -
                truth
            ).ravel()
        )
        /
        max(
            denominator,
            np.finfo(float).tiny,
        )
    )


C1_GENERATOR_RECOVERY_ERROR = (
    rs_relative_error(
        C1_A_EST,
        C1_G1_TRUE,
    )
)


# ================================================================
# 6. Split exactly as before
# ================================================================

C1_FIT_MASK = np.asarray(
    RS_CASE1[
        "fit_mask"
    ],
    dtype=bool,
)

C1_VAL_MASK = np.asarray(
    RS_CASE1[
        "validation_mask"
    ],
    dtype=bool,
)

C1_TEST_MASK = np.asarray(
    RS_CASE1[
        "test_mask"
    ],
    dtype=bool,
)


def rs_scalar_rows_from_state_mask(
    mask,
):

    return np.repeat(
        mask,
        RS_N,
    )


C1_FIT_ROWS = (
    rs_scalar_rows_from_state_mask(
        C1_FIT_MASK
    )
)

C1_VAL_ROWS = (
    rs_scalar_rows_from_state_mask(
        C1_VAL_MASK
    )
)

C1_TEST_ROWS = (
    rs_scalar_rows_from_state_mask(
        C1_TEST_MASK
    )
)


C1_Y = C1_A_EST.reshape(
    -1
)


# ================================================================
# 7. Blind dense 84-candidate physical fit
#
# This is NOT the final RS sparse algorithm.
#
# Purpose:
#   measure local design rank / conditioning and determine whether
#   the known physical dictionary itself is usable on this slice.
#
# Scaling learned only from fit cycles.
# ================================================================

C1_PSI_FIT_RAW = (
    C1_PSI_RAW[
        C1_FIT_ROWS
    ]
)

C1_PSI_VAL_RAW = (
    C1_PSI_RAW[
        C1_VAL_ROWS
    ]
)

C1_PSI_TEST_RAW = (
    C1_PSI_RAW[
        C1_TEST_ROWS
    ]
)


C1_Y_FIT = C1_Y[
    C1_FIT_ROWS
]

C1_Y_VAL = C1_Y[
    C1_VAL_ROWS
]

C1_Y_TEST = C1_Y[
    C1_TEST_ROWS
]


C1_SCALE = np.sqrt(
    np.mean(
        C1_PSI_FIT_RAW ** 2,
        axis=0,
    )
)


C1_SCALE = np.where(
    C1_SCALE > 0.0,
    C1_SCALE,
    1.0,
)


C1_PSI_FIT = (
    C1_PSI_FIT_RAW
    /
    C1_SCALE[
        None,
        :
    ]
)

C1_PSI_VAL = (
    C1_PSI_VAL_RAW
    /
    C1_SCALE[
        None,
        :
    ]
)

C1_PSI_TEST = (
    C1_PSI_TEST_RAW
    /
    C1_SCALE[
        None,
        :
    ]
)


C1_DENSE_BETA_SCALED, _, C1_DESIGN_RANK, C1_SINGULAR_VALUES = (
    np.linalg.lstsq(
        C1_PSI_FIT,
        C1_Y_FIT,
        rcond=None,
    )
)


C1_DENSE_BETA_RAW = (
    C1_DENSE_BETA_SCALED
    /
    C1_SCALE
)


C1_DESIGN_CONDITION = float(
    C1_SINGULAR_VALUES[0]
    /
    C1_SINGULAR_VALUES[-1]
)


# ================================================================
# 8. Dense-fit field reconstruction
# ================================================================

C1_DENSE_FIT_PRED = (
    C1_PSI_FIT_RAW
    @
    C1_DENSE_BETA_RAW
)

C1_DENSE_VAL_PRED = (
    C1_PSI_VAL_RAW
    @
    C1_DENSE_BETA_RAW
)

C1_DENSE_TEST_PRED = (
    C1_PSI_TEST_RAW
    @
    C1_DENSE_BETA_RAW
)


C1_DENSE_FIT_ERROR = (
    rs_relative_error(
        C1_DENSE_FIT_PRED,
        C1_Y_FIT,
    )
)

C1_DENSE_VAL_ERROR = (
    rs_relative_error(
        C1_DENSE_VAL_PRED,
        C1_Y_VAL,
    )
)

C1_DENSE_TEST_ERROR = (
    rs_relative_error(
        C1_DENSE_TEST_PRED,
        C1_Y_TEST,
    )
)


C1_DENSE_COEFF_ERROR = (
    rs_relative_error(
        C1_DENSE_BETA_RAW,
        RS_BETA_G1_TRUE,
    )
)


# ================================================================
# 9. Oracle-six fit
#
# Diagnostic ONLY:
# if correct support is supplied, do local data recover the
# coefficients accurately?
# ================================================================

C1_TRUE_INDICES = np.flatnonzero(
    RS_BETA_G1_TRUE != 0.0
)


C1_ORACLE6_DESIGN = (
    C1_PSI_FIT_RAW[
        :,
        C1_TRUE_INDICES
    ]
)


C1_ORACLE6_BETA, _, C1_ORACLE6_RANK, C1_ORACLE6_SVALS = (
    np.linalg.lstsq(
        C1_ORACLE6_DESIGN,
        C1_Y_FIT,
        rcond=None,
    )
)


C1_ORACLE6_TRUE = (
    RS_BETA_G1_TRUE[
        C1_TRUE_INDICES
    ]
)


C1_ORACLE6_COEFF_ERROR = (
    rs_relative_error(
        C1_ORACLE6_BETA,
        C1_ORACLE6_TRUE,
    )
)


C1_ORACLE6_CONDITION = float(
    C1_ORACLE6_SVALS[0]
    /
    C1_ORACLE6_SVALS[-1]
)


# ================================================================
# 10. Coefficient-separation diagnostic for dense 84 fit
#
# This uses the synthetic oracle only AFTER fitting.
# It does NOT define an inference threshold.
# ================================================================

C1_TRUE_ABS = np.abs(
    C1_DENSE_BETA_RAW[
        C1_TRUE_INDICES
    ]
)


C1_FALSE_INDICES = np.asarray(
    [
        i
        for i
        in range(
            len(
                RS_P2_CANDIDATES
            )
        )
        if i not in set(
            C1_TRUE_INDICES.tolist()
        )
    ],
    dtype=int,
)


C1_FALSE_ABS = np.abs(
    C1_DENSE_BETA_RAW[
        C1_FALSE_INDICES
    ]
)


C1_MIN_TRUE_COEFF = float(
    np.min(
        C1_TRUE_ABS
    )
)

C1_MAX_FALSE_COEFF = float(
    np.max(
        C1_FALSE_ABS
    )
)


C1_COEFF_SEPARATION_RATIO = (
    C1_MIN_TRUE_COEFF
    /
    max(
        C1_MAX_FALSE_COEFF,
        np.finfo(float).tiny,
    )
)


# ================================================================
# 11. Case-1 RS identities
#
# Since the entire window is static G1:
#
#       F_bar       = G1
#       F_eff       = G1
#       DeltaF_temp = 0
#
# At the reconstructed field-sample level:
# ================================================================

C1_F_BAR_EST = (
    C1_A_EST.copy()
)

C1_F_EFF_EST = (
    C1_A_EST.copy()
)

C1_DELTA_F_TEMP_EST = (
    C1_F_EFF_EST
    -
    C1_F_BAR_EST
)


C1_TEMPORAL_RESIDUAL_REL = (
    np.linalg.norm(
        C1_DELTA_F_TEMP_EST
    )
    /
    max(
        np.linalg.norm(
            C1_F_EFF_EST
        ),
        np.finfo(float).tiny,
    )
)


# ================================================================
# 12. Candidate coefficient table
# ================================================================

C1_COEFF_ROWS = []


for idx, (
    interaction_type,
    support,
) in enumerate(
    RS_P2_CANDIDATES
):

    C1_COEFF_ROWS.append({

        "candidate_index":
            idx,

        "interaction_type":
            interaction_type,

        "support":
            support,

        "true_active":
            bool(
                RS_BETA_G1_TRUE[
                    idx
                ]
                !=
                0.0
            ),

        "beta_true":
            float(
                RS_BETA_G1_TRUE[
                    idx
                ]
            ),

        "beta_dense84":
            float(
                C1_DENSE_BETA_RAW[
                    idx
                ]
            ),

        "abs_beta_dense84":
            float(
                abs(
                    C1_DENSE_BETA_RAW[
                        idx
                    ]
                )
            ),
    })


C1_COEFF_TABLE = (
    pd.DataFrame(
        C1_COEFF_ROWS
    )
)


C1_TOP_COEFFS = (
    C1_COEFF_TABLE
    .sort_values(
        "abs_beta_dense84",
        ascending=False,
    )
    .head(
        15
    )
    .copy()
)


# ================================================================
# 13. Report
# ================================================================

print("=" * 82)
print("RS-TSC PROTOTYPE — CELL 2")
print("CASE 1: STATIC-WINDOW IDENTITY")
print("=" * 82)

print(
    f"long-window s                : "
    f"{RS_CASE1['center_phase']:.6f}"
)

print(
    f"long-window DeltaT           : "
    f"{RS_CASE1['delta_T']:.6f}"
)

print(
    f"short epsilon values         : "
    f"{RS_MICRO_EPS.tolist()}"
)

print(
    f"local states                 : "
    f"{len(C1_X0)}"
)

print(
    f"fit / validation / test      : "
    f"{np.sum(C1_FIT_MASK)} / "
    f"{np.sum(C1_VAL_MASK)} / "
    f"{np.sum(C1_TEST_MASK)}"
)


print()
print("-" * 82)
print("A. MULTI-EPSILON GENERATOR RECOVERY")
print("-" * 82)

print(
    f"A_est vs exact G1            : "
    f"{C1_GENERATOR_RECOVERY_ERROR:.6e}"
)


print()
print("-" * 82)
print("B. LOCAL P2 DESIGN")
print("-" * 82)

print(
    f"scaled design shape          : "
    f"{C1_PSI_FIT.shape}"
)

print(
    f"rank                         : "
    f"{C1_DESIGN_RANK} / 84"
)

print(
    f"nullity                      : "
    f"{84 - C1_DESIGN_RANK}"
)

print(
    f"condition                    : "
    f"{C1_DESIGN_CONDITION:.6e}"
)


print()
print("-" * 82)
print("C. BLIND DENSE 84-CANDIDATE FIT")
print("-" * 82)

print(
    f"fit field error              : "
    f"{C1_DENSE_FIT_ERROR:.6e}"
)

print(
    f"validation field error       : "
    f"{C1_DENSE_VAL_ERROR:.6e}"
)

print(
    f"test field error             : "
    f"{C1_DENSE_TEST_ERROR:.6e}"
)

print(
    f"coefficient relative error   : "
    f"{C1_DENSE_COEFF_ERROR:.6e}"
)

print(
    f"min |true beta_hat|          : "
    f"{C1_MIN_TRUE_COEFF:.6e}"
)

print(
    f"max |false beta_hat|         : "
    f"{C1_MAX_FALSE_COEFF:.6e}"
)

print(
    f"true/false separation ratio  : "
    f"{C1_COEFF_SEPARATION_RATIO:.6e}"
)


print()
print("15 largest dense-fit coefficients")
print("-" * 82)

display(
    C1_TOP_COEFFS.style.format({

        "beta_true":
            "{:.8f}",

        "beta_dense84":
            "{:.8f}",

        "abs_beta_dense84":
            "{:.6e}",
    })
)


print()
print("-" * 82)
print("D. ORACLE-SIX DIAGNOSTIC")
print("-" * 82)

print(
    f"oracle-six rank              : "
    f"{C1_ORACLE6_RANK} / 6"
)

print(
    f"oracle-six condition         : "
    f"{C1_ORACLE6_CONDITION:.6e}"
)

print(
    f"oracle-six coeff error       : "
    f"{C1_ORACLE6_COEFF_ERROR:.6e}"
)


print()
print("-" * 82)
print("E. RS STATIC IDENTITY")
print("-" * 82)

print(
    "F_bar                       : reconstructed G1"
)

print(
    "F_eff                       : reconstructed G1"
)

print(
    f"||DeltaF_temp|| / ||F_eff|| : "
    f"{C1_TEMPORAL_RESIDUAL_REL:.6e}"
)


# ================================================================
# 14. Pass flags
#
# Do NOT require blind coefficient recovery here.
# That is a separate model-selection problem.
#
# Cell 2 tests:
#   - short-epsilon generator reconstruction
#   - static-window closure identity
# ================================================================

C1_GENERATOR_PASS = bool(
    C1_GENERATOR_RECOVERY_ERROR
    <
    1e-6
)


C1_STATIC_IDENTITY_PASS = bool(
    C1_TEMPORAL_RESIDUAL_REL
    <
    1e-14
)


print()
print("=" * 82)
print("DECISION")
print("=" * 82)

print(
    f"generator reconstruction     : "
    f"{'PASS' if C1_GENERATOR_PASS else 'FAIL'}"
)

print(
    f"static F_eff = F_bar         : "
    f"{'PASS' if C1_STATIC_IDENTITY_PASS else 'FAIL'}"
)

print(
    f"CASE 1 overall               : "
    f"{'PASS' if (C1_GENERATOR_PASS and C1_STATIC_IDENTITY_PASS) else 'FAIL'}"
)


# ================================================================
# 15. Save
# ================================================================

C1_COEFF_TABLE.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell2_case1_dense84_coefficients.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell2_case1_static_identity.npz",

    epsilon=
        RS_MICRO_EPS,

    X0=
        C1_X0,

    XF=
        C1_XF,

    A_est=
        C1_A_EST,

    G1_true=
        C1_G1_TRUE,

    beta_dense84=
        C1_DENSE_BETA_RAW,

    beta_true=
        RS_BETA_G1_TRUE,

    F_bar_est=
        C1_F_BAR_EST,

    F_eff_est=
        C1_F_EFF_EST,

    DeltaF_temp_est=
        C1_DELTA_F_TEMP_EST,
)


print()
print("=" * 82)
print("Cell 2 PASSED.")
print("=" * 82)

RS-TSC PROTOTYPE — CELL 2
CASE 1: STATIC-WINDOW IDENTITY
long-window s                : 0.015000
long-window DeltaT           : 0.030000
short epsilon values         : [0.005, 0.0075, 0.01, 0.015]
local states                 : 78
fit / validation / test      : 52 / 13 / 13

----------------------------------------------------------------------------------
A. MULTI-EPSILON GENERATOR RECOVERY
----------------------------------------------------------------------------------
A_est vs exact G1            : 1.163452e-09

----------------------------------------------------------------------------------
B. LOCAL P2 DESIGN
----------------------------------------------------------------------------------
scaled design shape          : (416, 84)
rank                         : 84 / 84
nullity                      : 0
condition                    : 2.681933e+10

----------------------------------------------------------------------------------
C. BLIND DENSE 84-CANDIDATE FIT
-------------------

,candidate_index,interaction_type,support,true_active,beta_true,beta_dense84,abs_beta_dense84
22,22,pair,"(5, 6)",True,1.17700000,1.17699863,1.176999e+00
13,13,pair,"(3, 4)",True,1.10500000,1.10499095,1.104991e+00
0,0,pair,"(1, 2)",True,1.00600000,1.00601536,1.006015e+00
27,27,pair,"(7, 8)",True,0.91700000,0.91699822,9.169982e-01
28,28,triad,"(1, 2, 3)",True,0.02000000,0.02001557,2.001557e-02
60,60,triad,"(2, 5, 8)",True,0.02000000,0.02000887,2.000887e-02
32,32,triad,"(1, 2, 7)",False,0.00000000,0.00008703,8.702911e-05
30,30,triad,"(1, 2, 5)",False,0.00000000,0.00007159,7.158705e-05
41,41,triad,"(1, 4, 7)",False,0.00000000,-0.00006144,6.143514e-05
29,29,triad,"(1, 2, 4)",False,0.00000000,0.00005787,5.786529e-05



----------------------------------------------------------------------------------
D. ORACLE-SIX DIAGNOSTIC
----------------------------------------------------------------------------------
oracle-six rank              : 6 / 6
oracle-six condition         : 1.061890e+01
oracle-six coeff error       : 1.079333e-09

----------------------------------------------------------------------------------
E. RS STATIC IDENTITY
----------------------------------------------------------------------------------
F_bar                       : reconstructed G1
F_eff                       : reconstructed G1
||DeltaF_temp|| / ||F_eff|| : 0.000000e+00

DECISION
generator reconstruction     : PASS
static F_eff = F_bar         : PASS
CASE 1 overall               : PASS

Cell 2 PASSED.


In [13]:
# ================================================================
# RS-TSC prototype
# Cell 3: freeze the Case-1 microscopic reconstruction
#
# Goal
# ---------------------------------------------------------------
# Convert the successful Cell-2 feasibility result into an actual
# reusable reconstructed microscopic generator:
#
#       G1_hat(x)
#         =
#       sum_alpha beta1_hat[alpha] Psi_alpha(x)
#
# Known:
#   physical interaction functional forms
#
# Unknown:
#   all 84 physical coefficients / topology
#
# Strategy:
#   - use Case-1 local multi-epsilon generator samples
#   - fit ALL 84 physical candidates
#   - no oracle support
#   - no hard threshold
#   - no AGLASSO
#
# The full continuous coefficient vector is retained.
#
# Fit:
#   cycles 0-3 only
#
# Validation:
#   cycle 4
#
# External test:
#   cycle 5
#
# Synthetic truth enters ONLY for diagnostics after reconstruction.
# ================================================================


# ================================================================
# 0. Freeze the Case-1 training objects from Cell 2
# ================================================================

assert C1_PSI_FIT_RAW.shape == (
    52 * RS_N,
    84,
)

assert C1_PSI_VAL_RAW.shape == (
    13 * RS_N,
    84,
)

assert C1_PSI_TEST_RAW.shape == (
    13 * RS_N,
    84,
)

assert C1_DESIGN_RANK == 84


# ================================================================
# 1. Final microscopic coefficient estimate
#
# Use the fit-block scaling learned in Cell 2.
#
# This is deliberately the same full-rank dense physical inversion
# that succeeded in Cell 2.
#
# No validation information enters coefficient estimation.
# ================================================================

RS_G1_BETA_SCALED, _, RS_G1_RANK, RS_G1_SVALS = (
    np.linalg.lstsq(
        C1_PSI_FIT,
        C1_Y_FIT,
        rcond=None,
    )
)


RS_G1_BETA = (
    RS_G1_BETA_SCALED
    /
    C1_SCALE
)


assert RS_G1_RANK == 84


RS_G1_CONDITION = float(
    RS_G1_SVALS[0]
    /
    RS_G1_SVALS[-1]
)


# ================================================================
# 2. Reusable reconstructed microscopic vector field
#
# IMPORTANT:
#
# This is the object later cells will use.
#
# It does NOT know the true G1 topology.
# ================================================================

def RS_G1_HAT(X):

    X = np.asarray(
        X,
        dtype=float,
    )

    single_state = (
        X.ndim == 1
    )

    if single_state:
        X = X[
            None,
            :
        ]

    if (
        X.ndim != 2
        or
        X.shape[1] != RS_N
    ):
        raise ValueError(
            f"Expected X with shape (M,{RS_N}) "
            f"or ({RS_N},), got {X.shape}"
        )


    Psi = (
        rs_build_physical_design(
            X
        )
    )


    F = (
        Psi
        @
        RS_G1_BETA
    ).reshape(
        len(X),
        RS_N,
    )


    if single_state:
        return F[0]

    return F


# ================================================================
# 3. Reconstruction errors against generator samples
# ================================================================

RS_G1_FIT_PRED = (
    RS_G1_HAT(
        C1_X0[
            C1_FIT_MASK
        ]
    )
)


RS_G1_VAL_PRED = (
    RS_G1_HAT(
        C1_X0[
            C1_VAL_MASK
        ]
    )
)


RS_G1_TEST_PRED = (
    RS_G1_HAT(
        C1_X0[
            C1_TEST_MASK
        ]
    )
)


RS_G1_FIT_TARGET = (
    C1_A_EST[
        C1_FIT_MASK
    ]
)

RS_G1_VAL_TARGET = (
    C1_A_EST[
        C1_VAL_MASK
    ]
)

RS_G1_TEST_TARGET = (
    C1_A_EST[
        C1_TEST_MASK
    ]
)


RS_G1_FIT_ERROR = (
    rs_relative_error(
        RS_G1_FIT_PRED,
        RS_G1_FIT_TARGET,
    )
)


RS_G1_VAL_ERROR = (
    rs_relative_error(
        RS_G1_VAL_PRED,
        RS_G1_VAL_TARGET,
    )
)


RS_G1_TEST_ERROR = (
    rs_relative_error(
        RS_G1_TEST_PRED,
        RS_G1_TEST_TARGET,
    )
)


# ================================================================
# 4. Synthetic-oracle diagnostics
#
# Truth enters only here.
# ================================================================

RS_G1_COEFF_ERROR_ORACLE = (
    rs_relative_error(
        RS_G1_BETA,
        RS_BETA_G1_TRUE,
    )
)


RS_G1_FIT_TRUE_ERROR = (
    rs_relative_error(
        RS_G1_FIT_PRED,
        C1_G1_TRUE[
            C1_FIT_MASK
        ],
    )
)


RS_G1_VAL_TRUE_ERROR = (
    rs_relative_error(
        RS_G1_VAL_PRED,
        C1_G1_TRUE[
            C1_VAL_MASK
        ],
    )
)


RS_G1_TEST_TRUE_ERROR = (
    rs_relative_error(
        RS_G1_TEST_PRED,
        C1_G1_TRUE[
            C1_TEST_MASK
        ],
    )
)


# ================================================================
# 5. Blind coefficient-spectrum diagnostics
#
# Still NO support threshold is imposed.
#
# We report:
#   - coefficient magnitude
#   - rank by magnitude
#
# and only afterwards mark synthetic true_active for evaluation.
# ================================================================

RS_G1_COEFF_ROWS = []


for idx, (
    interaction_type,
    support,
) in enumerate(
    RS_P2_CANDIDATES
):

    RS_G1_COEFF_ROWS.append({

        "candidate_index":
            idx,

        "interaction_type":
            interaction_type,

        "support":
            support,

        "beta_hat":
            float(
                RS_G1_BETA[
                    idx
                ]
            ),

        "abs_beta_hat":
            float(
                abs(
                    RS_G1_BETA[
                        idx
                    ]
                )
            ),

        # synthetic diagnostic only
        "true_active":
            bool(
                RS_BETA_G1_TRUE[
                    idx
                ]
                !=
                0.0
            ),

        "beta_true":
            float(
                RS_BETA_G1_TRUE[
                    idx
                ]
            ),
    })


RS_G1_COEFF_TABLE = (
    pd.DataFrame(
        RS_G1_COEFF_ROWS
    )
    .sort_values(
        "abs_beta_hat",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


RS_G1_COEFF_TABLE[
    "magnitude_rank"
] = np.arange(
    1,
    len(
        RS_G1_COEFF_TABLE
    ) + 1,
)


# ================================================================
# 6. Oracle-only true/false gap
#
# This is NOT used to construct G1_hat.
# It merely measures how cleanly the blind coefficient spectrum
# separates the known synthetic truth.
# ================================================================

RS_G1_TRUE_MAG = (
    RS_G1_COEFF_TABLE.loc[
        RS_G1_COEFF_TABLE[
            "true_active"
        ],
        "abs_beta_hat",
    ].to_numpy()
)


RS_G1_FALSE_MAG = (
    RS_G1_COEFF_TABLE.loc[
        ~RS_G1_COEFF_TABLE[
            "true_active"
        ],
        "abs_beta_hat",
    ].to_numpy()
)


RS_G1_MIN_TRUE_MAG = float(
    np.min(
        RS_G1_TRUE_MAG
    )
)


RS_G1_MAX_FALSE_MAG = float(
    np.max(
        RS_G1_FALSE_MAG
    )
)


RS_G1_TRUE_FALSE_GAP = (
    RS_G1_MIN_TRUE_MAG
    /
    max(
        RS_G1_MAX_FALSE_MAG,
        np.finfo(float).tiny,
    )
)


# ================================================================
# 7. Check whether the six true interactions are exactly the six
#    largest coefficients.
#
# Again: evaluation only, not part of inference.
# ================================================================

RS_G1_TOP6 = (
    RS_G1_COEFF_TABLE
    .head(6)
    .copy()
)


RS_G1_TOP6_SUPPORTS = set(
    RS_G1_TOP6[
        "support"
    ].tolist()
)


RS_G1_TRUE_SUPPORTS = (
    RS_G1_TRUE_PAIRS
    |
    RS_G1_TRUE_TRIADS
)


RS_G1_TOP6_EXACT = bool(
    RS_G1_TOP6_SUPPORTS
    ==
    RS_G1_TRUE_SUPPORTS
)


# ================================================================
# 8. Locality / extrapolation diagnostic
#
# Evaluate reconstructed G1_hat not only on the Case-1 starts,
# but on EVERY stored trajectory state whose phase lies inside G1.
#
# This checks whether the reconstructed vector field generalizes
# across the entire observed G1 sector rather than only the
# +/-0.003 local neighborhood.
# ================================================================

RS_ALL_STATE_PHASE = np.mod(
    RS_TIME,
    RS_CYCLE_DURATION,
)


# Avoid the exact right switching boundary.
RS_ALL_G1_MASK = (
    (RS_ALL_STATE_PHASE >= 0.0)
    &
    (
        RS_ALL_STATE_PHASE
        <
        RS_TAU - 0.5 * RS_DT_OBS
    )
)


RS_ALL_G1_X = (
    RS_X[
        RS_ALL_G1_MASK
    ]
)


RS_ALL_G1_PSI = (
    rs_build_physical_design(
        RS_ALL_G1_X
    )
)


RS_ALL_G1_TRUE = (
    RS_ALL_G1_PSI
    @
    RS_BETA_G1_TRUE
).reshape(
    len(
        RS_ALL_G1_X
    ),
    RS_N,
)


RS_ALL_G1_PRED = (
    RS_G1_HAT(
        RS_ALL_G1_X
    )
)


RS_G1_FULL_SECTOR_ERROR = (
    rs_relative_error(
        RS_ALL_G1_PRED,
        RS_ALL_G1_TRUE,
    )
)


# ================================================================
# 9. Report
# ================================================================

print("=" * 82)
print("RS-TSC PROTOTYPE — CELL 3")
print("CASE 1 MICROSCOPIC RECONSTRUCTION")
print("=" * 82)


print()
print("-" * 82)
print("A. RECONSTRUCTION SYSTEM")
print("-" * 82)

print(
    f"physical candidates          : "
    f"{len(RS_P2_CANDIDATES)}"
)

print(
    f"fit states                   : "
    f"{np.sum(C1_FIT_MASK)}"
)

print(
    f"fit scalar rows              : "
    f"{C1_PSI_FIT.shape[0]}"
)

print(
    f"rank                         : "
    f"{RS_G1_RANK} / 84"
)

print(
    f"condition                    : "
    f"{RS_G1_CONDITION:.6e}"
)


print()
print("-" * 82)
print("B. BLIND GENERATOR RECONSTRUCTION")
print("-" * 82)

print(
    f"fit error                    : "
    f"{RS_G1_FIT_ERROR:.6e}"
)

print(
    f"validation error             : "
    f"{RS_G1_VAL_ERROR:.6e}"
)

print(
    f"external test error          : "
    f"{RS_G1_TEST_ERROR:.6e}"
)


print()
print("-" * 82)
print("C. SYNTHETIC ORACLE DIAGNOSTICS")
print("-" * 82)

print(
    f"coefficient relative error   : "
    f"{RS_G1_COEFF_ERROR_ORACLE:.6e}"
)

print(
    f"fit vs exact G1              : "
    f"{RS_G1_FIT_TRUE_ERROR:.6e}"
)

print(
    f"validation vs exact G1       : "
    f"{RS_G1_VAL_TRUE_ERROR:.6e}"
)

print(
    f"test vs exact G1             : "
    f"{RS_G1_TEST_TRUE_ERROR:.6e}"
)

print(
    f"whole observed G1 sector err : "
    f"{RS_G1_FULL_SECTOR_ERROR:.6e}"
)

print(
    f"min true |beta_hat|          : "
    f"{RS_G1_MIN_TRUE_MAG:.6e}"
)

print(
    f"max false |beta_hat|         : "
    f"{RS_G1_MAX_FALSE_MAG:.6e}"
)

print(
    f"true/false magnitude gap     : "
    f"{RS_G1_TRUE_FALSE_GAP:.6e}"
)

print(
    f"top-6 exactly true support   : "
    f"{RS_G1_TOP6_EXACT}"
)


print()
print("12 largest reconstructed physical coefficients")
print("-" * 82)

display(
    RS_G1_COEFF_TABLE
    .head(12)
    .style
    .format({

        "beta_hat":
            "{:.8f}",

        "abs_beta_hat":
            "{:.6e}",

        "beta_true":
            "{:.8f}",
    })
)


# ================================================================
# 10. Pass flags
#
# Important:
#
# The scientific reconstruction pass criterion is based on
# held-out FIELD accuracy, not oracle topology.
#
# Top-6 exact support is reported but NOT required.
# ================================================================

RS_G1_RECONSTRUCTION_PASS = bool(
    RS_G1_TEST_ERROR
    <
    1e-5
)


RS_G1_GENERALIZATION_PASS = bool(
    RS_G1_FULL_SECTOR_ERROR
    <
    1e-4
)


print()
print("=" * 82)
print("DECISION")
print("=" * 82)

print(
    f"held-out reconstruction      : "
    f"{'PASS' if RS_G1_RECONSTRUCTION_PASS else 'FAIL'}"
)

print(
    f"whole-G1-sector generalize   : "
    f"{'PASS' if RS_G1_GENERALIZATION_PASS else 'FAIL'}"
)

print(
    f"CASE-1 reconstruction        : "
    f"{'PASS' if (RS_G1_RECONSTRUCTION_PASS and RS_G1_GENERALIZATION_PASS) else 'FAIL'}"
)


# ================================================================
# 11. Save frozen reconstructed G1
# ================================================================

RS_G1_COEFF_TABLE.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell3_G1_reconstructed_coefficients.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell3_G1_microscopic_reconstruction.npz",

    beta_hat=
        RS_G1_BETA,

    beta_true=
        RS_BETA_G1_TRUE,

    feature_scale=
        C1_SCALE,

    singular_values=
        RS_G1_SVALS,

    fit_error=
        RS_G1_FIT_ERROR,

    validation_error=
        RS_G1_VAL_ERROR,

    test_error=
        RS_G1_TEST_ERROR,

    full_G1_sector_error=
        RS_G1_FULL_SECTOR_ERROR,
)


print()
print("=" * 82)
print("Cell 3 PASSED.")
print("=" * 82)

RS-TSC PROTOTYPE — CELL 3
CASE 1 MICROSCOPIC RECONSTRUCTION

----------------------------------------------------------------------------------
A. RECONSTRUCTION SYSTEM
----------------------------------------------------------------------------------
physical candidates          : 84
fit states                   : 52
fit scalar rows              : 416
rank                         : 84 / 84
condition                    : 2.681933e+10

----------------------------------------------------------------------------------
B. BLIND GENERATOR RECONSTRUCTION
----------------------------------------------------------------------------------
fit error                    : 1.040769e-13
validation error             : 3.432112e-08
external test error          : 1.199268e-07

----------------------------------------------------------------------------------
C. SYNTHETIC ORACLE DIAGNOSTICS
----------------------------------------------------------------------------------
coefficient relative error   :

,candidate_index,interaction_type,support,beta_hat,abs_beta_hat,true_active,beta_true,magnitude_rank
0,22,pair,"(5, 6)",1.17699863,1.176999e+00,True,1.17700000,1
1,13,pair,"(3, 4)",1.10499095,1.104991e+00,True,1.10500000,2
2,0,pair,"(1, 2)",1.00601536,1.006015e+00,True,1.00600000,3
3,27,pair,"(7, 8)",0.91699822,9.169982e-01,True,0.91700000,4
4,28,triad,"(1, 2, 3)",0.02001557,2.001557e-02,True,0.02000000,5
5,60,triad,"(2, 5, 8)",0.02000887,2.000887e-02,True,0.02000000,6
6,32,triad,"(1, 2, 7)",0.00008703,8.702911e-05,False,0.00000000,7
7,30,triad,"(1, 2, 5)",0.00007159,7.158705e-05,False,0.00000000,8
8,41,triad,"(1, 4, 7)",-0.00006144,6.143514e-05,False,0.00000000,9
9,29,triad,"(1, 2, 4)",0.00005787,5.786529e-05,False,0.00000000,10



DECISION
held-out reconstruction      : PASS
whole-G1-sector generalize   : PASS
CASE-1 reconstruction        : PASS

Cell 3 PASSED.


In [14]:
# ================================================================
# RS-TSC prototype
# Cell 4: reconstruct the second microscopic generator G2
#
# G2 static sector:
#
#       phase in [0.060, 0.120)
#
# Reconstruction center:
#
#       phase = 0.075
#
# Local starts:
#
#       0.075 +/- 0.003
#
# Short endpoint horizons:
#
#       eps = [0.005, 0.0075, 0.010, 0.015]
#
# Therefore every short slice remains strictly inside G2.
#
# Goal:
#
#       local time series
#           ->
#       A_est
#           ->
#       blind 84-candidate physical reconstruction
#           ->
#       reusable G2_hat(x)
#
# NO AGLASSO.
# NO oracle topology enters reconstruction.
# ================================================================


# ================================================================
# 0. G2 microscopic local ensemble
# ================================================================

RS_G2_MICRO_CENTER_PHASE = 0.075


G2_START_IDX_LIST = []
G2_CYCLE_LIST = []
G2_OFFSET_LIST = []
G2_SPLIT_LIST = []


for cycle in RS_USED_CYCLES:

    cycle_start = (
        cycle
        *
        RS_CYCLE_DURATION
    )

    for local_offset in RS_LOCAL_OFFSETS:

        phase = (
            RS_G2_MICRO_CENTER_PHASE
            +
            float(local_offset)
        )

        t_start = (
            cycle_start
            +
            phase
        )

        start_idx = (
            rs_time_to_index(
                t_start
            )
        )


        # --------------------------------------------------------
        # Audit all four short endpoint horizons:
        # every integration interval must lie inside G2.
        # --------------------------------------------------------

        for step in RS_MICRO_EPS_STEPS:

            labels = (
                RS_INTERVAL_SNAPSHOT_INDEX[
                    start_idx:
                    start_idx + step
                ]
            )

            assert np.all(
                labels == 1
            )


        if cycle in RS_FIT_CYCLES:
            split = "fit"

        elif cycle in RS_VALIDATION_CYCLES:
            split = "validation"

        elif cycle in RS_TEST_CYCLES:
            split = "test"

        else:
            raise RuntimeError(
                cycle
            )


        G2_START_IDX_LIST.append(
            start_idx
        )

        G2_CYCLE_LIST.append(
            cycle
        )

        G2_OFFSET_LIST.append(
            float(local_offset)
        )

        G2_SPLIT_LIST.append(
            split
        )


G2_START_IDX = np.asarray(
    G2_START_IDX_LIST,
    dtype=int,
)

G2_CYCLE = np.asarray(
    G2_CYCLE_LIST,
    dtype=int,
)

G2_LOCAL_OFFSET = np.asarray(
    G2_OFFSET_LIST,
    dtype=float,
)

G2_SPLIT = np.asarray(
    G2_SPLIT_LIST,
    dtype=object,
)


assert len(
    G2_START_IDX
) == 78


G2_FIT_MASK = (
    G2_SPLIT
    ==
    "fit"
)

G2_VAL_MASK = (
    G2_SPLIT
    ==
    "validation"
)

G2_TEST_MASK = (
    G2_SPLIT
    ==
    "test"
)


assert np.sum(
    G2_FIT_MASK
) == 52

assert np.sum(
    G2_VAL_MASK
) == 13

assert np.sum(
    G2_TEST_MASK
) == 13


# ================================================================
# 1. Extract X0 and four nested endpoints directly from trajectory
# ================================================================

G2_X0 = (
    RS_X[
        G2_START_IDX
    ]
    .copy()
)


G2_XF = np.stack(
    [
        RS_X[
            G2_START_IDX
            +
            step
        ]

        for step
        in RS_MICRO_EPS_STEPS
    ],
    axis=0,
)


assert G2_XF.shape == (
    len(RS_MICRO_EPS),
    78,
    RS_N,
)


# Exact source audit
assert np.array_equal(
    G2_X0,
    RS_X[
        G2_START_IDX
    ],
)


# ================================================================
# 2. Four-point cubic eps -> 0 extrapolation
# ================================================================

G2_SECANT = (
    (
        G2_XF
        -
        G2_X0[
            None,
            :,
            :
        ]
    )
    /
    RS_MICRO_EPS[
        :,
        None,
        None,
    ]
)


G2_POLY_COEFF = np.linalg.solve(
    C1_VANDERMONDE,
    G2_SECANT.reshape(
        len(
            RS_MICRO_EPS
        ),
        -1,
    ),
)


G2_A_EST = (
    G2_POLY_COEFF[
        0
    ]
    .reshape(
        len(
            G2_X0
        ),
        RS_N,
    )
)


# ================================================================
# 3. Blind P2 physical design
# ================================================================

G2_PSI_RAW = (
    rs_build_physical_design(
        G2_X0
    )
)


G2_FIT_ROWS = (
    rs_scalar_rows_from_state_mask(
        G2_FIT_MASK
    )
)

G2_VAL_ROWS = (
    rs_scalar_rows_from_state_mask(
        G2_VAL_MASK
    )
)

G2_TEST_ROWS = (
    rs_scalar_rows_from_state_mask(
        G2_TEST_MASK
    )
)


G2_Y = (
    G2_A_EST
    .reshape(-1)
)


G2_PSI_FIT_RAW = (
    G2_PSI_RAW[
        G2_FIT_ROWS
    ]
)

G2_PSI_VAL_RAW = (
    G2_PSI_RAW[
        G2_VAL_ROWS
    ]
)

G2_PSI_TEST_RAW = (
    G2_PSI_RAW[
        G2_TEST_ROWS
    ]
)


G2_Y_FIT = (
    G2_Y[
        G2_FIT_ROWS
    ]
)

G2_Y_VAL = (
    G2_Y[
        G2_VAL_ROWS
    ]
)

G2_Y_TEST = (
    G2_Y[
        G2_TEST_ROWS
    ]
)


# ================================================================
# 4. Fit-only RMS scaling
# ================================================================

G2_SCALE = np.sqrt(
    np.mean(
        G2_PSI_FIT_RAW ** 2,
        axis=0,
    )
)


G2_SCALE = np.where(
    G2_SCALE > 0.0,
    G2_SCALE,
    1.0,
)


G2_PSI_FIT = (
    G2_PSI_FIT_RAW
    /
    G2_SCALE[
        None,
        :
    ]
)


# ================================================================
# 5. Blind dense 84-candidate microscopic reconstruction
# ================================================================

RS_G2_BETA_SCALED, _, RS_G2_RANK, RS_G2_SVALS = (
    np.linalg.lstsq(
        G2_PSI_FIT,
        G2_Y_FIT,
        rcond=None,
    )
)


RS_G2_BETA = (
    RS_G2_BETA_SCALED
    /
    G2_SCALE
)


RS_G2_CONDITION = float(
    RS_G2_SVALS[0]
    /
    RS_G2_SVALS[-1]
)


# ================================================================
# 6. Reusable reconstructed G2 vector field
# ================================================================

def RS_G2_HAT(X):

    X = np.asarray(
        X,
        dtype=float,
    )

    single_state = (
        X.ndim == 1
    )

    if single_state:
        X = X[
            None,
            :
        ]


    if (
        X.ndim != 2
        or
        X.shape[1] != RS_N
    ):

        raise ValueError(
            f"Expected X with shape "
            f"(M,{RS_N}) or ({RS_N},), "
            f"got {X.shape}"
        )


    Psi = (
        rs_build_physical_design(
            X
        )
    )


    F = (
        Psi
        @
        RS_G2_BETA
    ).reshape(
        len(X),
        RS_N,
    )


    if single_state:
        return F[0]

    return F


# ================================================================
# 7. Held-out generator reconstruction
# ================================================================

RS_G2_FIT_PRED = (
    RS_G2_HAT(
        G2_X0[
            G2_FIT_MASK
        ]
    )
)

RS_G2_VAL_PRED = (
    RS_G2_HAT(
        G2_X0[
            G2_VAL_MASK
        ]
    )
)

RS_G2_TEST_PRED = (
    RS_G2_HAT(
        G2_X0[
            G2_TEST_MASK
        ]
    )
)


RS_G2_FIT_ERROR = (
    rs_relative_error(
        RS_G2_FIT_PRED,
        G2_A_EST[
            G2_FIT_MASK
        ],
    )
)

RS_G2_VAL_ERROR = (
    rs_relative_error(
        RS_G2_VAL_PRED,
        G2_A_EST[
            G2_VAL_MASK
        ],
    )
)

RS_G2_TEST_ERROR = (
    rs_relative_error(
        RS_G2_TEST_PRED,
        G2_A_EST[
            G2_TEST_MASK
        ],
    )
)


# ================================================================
# 8. Synthetic G2 oracle
#
# Truth enters ONLY from here onward.
#
# G2 pair edges:
#
#   (2,3)
#   (4,5)
#   (6,7)
#   (8,1) -> canonical (1,8)
#
# Native triads remain:
#
#   (1,2,3)
#   (2,5,8)
# ================================================================

RS_G2_TRUE_PAIRS = {
    (2, 3),
    (4, 5),
    (6, 7),
    (1, 8),
}


RS_G2_TRUE_TRIADS = {
    (1, 2, 3),
    (2, 5, 8),
}


RS_BETA_G2_TRUE = np.zeros(
    len(
        RS_P2_CANDIDATES
    ),
    dtype=float,
)


for idx, (
    interaction_type,
    support,
) in enumerate(
    RS_P2_CANDIDATES
):

    if (
        interaction_type == "pair"
        and
        support in RS_G2_TRUE_PAIRS
    ):

        RS_BETA_G2_TRUE[
            idx
        ] = (
            RS_EDGE_WEIGHT[
                tuple(
                    sorted(
                        support
                    )
                )
            ]
        )

    elif (
        interaction_type == "triad"
        and
        support in RS_G2_TRUE_TRIADS
    ):

        RS_BETA_G2_TRUE[
            idx
        ] = (
            RS_NATIVE_G
        )


assert np.count_nonzero(
    RS_BETA_G2_TRUE
) == 6


G2_TRUE_FIELD = (
    G2_PSI_RAW
    @
    RS_BETA_G2_TRUE
).reshape(
    len(
        G2_X0
    ),
    RS_N,
)


# ================================================================
# 9. Generator / coefficient oracle errors
# ================================================================

RS_G2_AEST_TRUE_ERROR = (
    rs_relative_error(
        G2_A_EST,
        G2_TRUE_FIELD,
    )
)


RS_G2_COEFF_ERROR_ORACLE = (
    rs_relative_error(
        RS_G2_BETA,
        RS_BETA_G2_TRUE,
    )
)


RS_G2_FIT_TRUE_ERROR = (
    rs_relative_error(
        RS_G2_FIT_PRED,
        G2_TRUE_FIELD[
            G2_FIT_MASK
        ],
    )
)

RS_G2_VAL_TRUE_ERROR = (
    rs_relative_error(
        RS_G2_VAL_PRED,
        G2_TRUE_FIELD[
            G2_VAL_MASK
        ],
    )
)

RS_G2_TEST_TRUE_ERROR = (
    rs_relative_error(
        RS_G2_TEST_PRED,
        G2_TRUE_FIELD[
            G2_TEST_MASK
        ],
    )
)


# ================================================================
# 10. Whole observed G2-sector generalization
# ================================================================

RS_ALL_G2_MASK = (
    (
        RS_ALL_STATE_PHASE
        >=
        RS_TAU
    )
    &
    (
        RS_ALL_STATE_PHASE
        <
        2.0 * RS_TAU
        -
        0.5 * RS_DT_OBS
    )
)


RS_ALL_G2_X = (
    RS_X[
        RS_ALL_G2_MASK
    ]
)


RS_ALL_G2_PSI = (
    rs_build_physical_design(
        RS_ALL_G2_X
    )
)


RS_ALL_G2_TRUE = (
    RS_ALL_G2_PSI
    @
    RS_BETA_G2_TRUE
).reshape(
    len(
        RS_ALL_G2_X
    ),
    RS_N,
)


RS_ALL_G2_PRED = (
    RS_G2_HAT(
        RS_ALL_G2_X
    )
)


RS_G2_FULL_SECTOR_ERROR = (
    rs_relative_error(
        RS_ALL_G2_PRED,
        RS_ALL_G2_TRUE,
    )
)


# ================================================================
# 11. Coefficient spectrum diagnostics
# ================================================================

RS_G2_COEFF_ROWS = []


for idx, (
    interaction_type,
    support,
) in enumerate(
    RS_P2_CANDIDATES
):

    RS_G2_COEFF_ROWS.append({

        "candidate_index":
            idx,

        "interaction_type":
            interaction_type,

        "support":
            support,

        "beta_hat":
            float(
                RS_G2_BETA[
                    idx
                ]
            ),

        "abs_beta_hat":
            float(
                abs(
                    RS_G2_BETA[
                        idx
                    ]
                )
            ),

        "true_active":
            bool(
                RS_BETA_G2_TRUE[
                    idx
                ]
                !=
                0.0
            ),

        "beta_true":
            float(
                RS_BETA_G2_TRUE[
                    idx
                ]
            ),
    })


RS_G2_COEFF_TABLE = (
    pd.DataFrame(
        RS_G2_COEFF_ROWS
    )
    .sort_values(
        "abs_beta_hat",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


RS_G2_COEFF_TABLE[
    "magnitude_rank"
] = np.arange(
    1,
    85,
)


RS_G2_TRUE_MAG = (
    RS_G2_COEFF_TABLE.loc[
        RS_G2_COEFF_TABLE[
            "true_active"
        ],
        "abs_beta_hat",
    ]
    .to_numpy()
)


RS_G2_FALSE_MAG = (
    RS_G2_COEFF_TABLE.loc[
        ~RS_G2_COEFF_TABLE[
            "true_active"
        ],
        "abs_beta_hat",
    ]
    .to_numpy()
)


RS_G2_MIN_TRUE_MAG = float(
    np.min(
        RS_G2_TRUE_MAG
    )
)

RS_G2_MAX_FALSE_MAG = float(
    np.max(
        RS_G2_FALSE_MAG
    )
)


RS_G2_TRUE_FALSE_GAP = (
    RS_G2_MIN_TRUE_MAG
    /
    max(
        RS_G2_MAX_FALSE_MAG,
        np.finfo(float).tiny,
    )
)


RS_G2_TOP6_SUPPORTS = set(
    RS_G2_COEFF_TABLE
    .head(6)[
        "support"
    ]
    .tolist()
)


RS_G2_TRUE_SUPPORTS = (
    RS_G2_TRUE_PAIRS
    |
    RS_G2_TRUE_TRIADS
)


RS_G2_TOP6_EXACT = bool(
    RS_G2_TOP6_SUPPORTS
    ==
    RS_G2_TRUE_SUPPORTS
)


# ================================================================
# 12. Oracle-six diagnostic
# ================================================================

RS_G2_TRUE_INDICES = np.flatnonzero(
    RS_BETA_G2_TRUE
    !=
    0.0
)


G2_ORACLE6_DESIGN = (
    G2_PSI_FIT_RAW[
        :,
        RS_G2_TRUE_INDICES
    ]
)


G2_ORACLE6_BETA, _, G2_ORACLE6_RANK, G2_ORACLE6_SVALS = (
    np.linalg.lstsq(
        G2_ORACLE6_DESIGN,
        G2_Y_FIT,
        rcond=None,
    )
)


G2_ORACLE6_TRUE = (
    RS_BETA_G2_TRUE[
        RS_G2_TRUE_INDICES
    ]
)


G2_ORACLE6_COEFF_ERROR = (
    rs_relative_error(
        G2_ORACLE6_BETA,
        G2_ORACLE6_TRUE,
    )
)


G2_ORACLE6_CONDITION = float(
    G2_ORACLE6_SVALS[0]
    /
    G2_ORACLE6_SVALS[-1]
)


# ================================================================
# 13. Report
# ================================================================

print("=" * 82)
print("RS-TSC PROTOTYPE — CELL 4")
print("G2 MICROSCOPIC RECONSTRUCTION")
print("=" * 82)


print()
print("-" * 82)
print("A. MULTI-EPSILON GENERATOR RECOVERY")
print("-" * 82)

print(
    f"A_est vs exact G2            : "
    f"{RS_G2_AEST_TRUE_ERROR:.6e}"
)


print()
print("-" * 82)
print("B. RECONSTRUCTION SYSTEM")
print("-" * 82)

print(
    f"center phase                 : "
    f"{RS_G2_MICRO_CENTER_PHASE:.6f}"
)

print(
    f"fit / validation / test      : "
    f"{np.sum(G2_FIT_MASK)} / "
    f"{np.sum(G2_VAL_MASK)} / "
    f"{np.sum(G2_TEST_MASK)}"
)

print(
    f"physical candidates          : "
    f"{len(RS_P2_CANDIDATES)}"
)

print(
    f"rank                         : "
    f"{RS_G2_RANK} / 84"
)

print(
    f"condition                    : "
    f"{RS_G2_CONDITION:.6e}"
)


print()
print("-" * 82)
print("C. BLIND GENERATOR RECONSTRUCTION")
print("-" * 82)

print(
    f"fit error                    : "
    f"{RS_G2_FIT_ERROR:.6e}"
)

print(
    f"validation error             : "
    f"{RS_G2_VAL_ERROR:.6e}"
)

print(
    f"external test error          : "
    f"{RS_G2_TEST_ERROR:.6e}"
)


print()
print("-" * 82)
print("D. SYNTHETIC ORACLE DIAGNOSTICS")
print("-" * 82)

print(
    f"coefficient relative error   : "
    f"{RS_G2_COEFF_ERROR_ORACLE:.6e}"
)

print(
    f"fit vs exact G2              : "
    f"{RS_G2_FIT_TRUE_ERROR:.6e}"
)

print(
    f"validation vs exact G2       : "
    f"{RS_G2_VAL_TRUE_ERROR:.6e}"
)

print(
    f"test vs exact G2             : "
    f"{RS_G2_TEST_TRUE_ERROR:.6e}"
)

print(
    f"whole observed G2 sector err : "
    f"{RS_G2_FULL_SECTOR_ERROR:.6e}"
)

print(
    f"min true |beta_hat|          : "
    f"{RS_G2_MIN_TRUE_MAG:.6e}"
)

print(
    f"max false |beta_hat|         : "
    f"{RS_G2_MAX_FALSE_MAG:.6e}"
)

print(
    f"true/false magnitude gap     : "
    f"{RS_G2_TRUE_FALSE_GAP:.6e}"
)

print(
    f"top-6 exactly true support   : "
    f"{RS_G2_TOP6_EXACT}"
)


print()
print("12 largest reconstructed physical coefficients")
print("-" * 82)

display(
    RS_G2_COEFF_TABLE
    .head(12)
    .style
    .format({

        "beta_hat":
            "{:.8f}",

        "abs_beta_hat":
            "{:.6e}",

        "beta_true":
            "{:.8f}",
    })
)


print()
print("-" * 82)
print("E. ORACLE-SIX DIAGNOSTIC")
print("-" * 82)

print(
    f"oracle-six rank              : "
    f"{G2_ORACLE6_RANK} / 6"
)

print(
    f"oracle-six condition         : "
    f"{G2_ORACLE6_CONDITION:.6e}"
)

print(
    f"oracle-six coeff error       : "
    f"{G2_ORACLE6_COEFF_ERROR:.6e}"
)


# ================================================================
# 14. Pass flags
# ================================================================

RS_G2_RECONSTRUCTION_PASS = bool(
    RS_G2_TEST_ERROR
    <
    1e-5
)


RS_G2_GENERALIZATION_PASS = bool(
    RS_G2_FULL_SECTOR_ERROR
    <
    1e-4
)


print()
print("=" * 82)
print("DECISION")
print("=" * 82)

print(
    f"held-out reconstruction      : "
    f"{'PASS' if RS_G2_RECONSTRUCTION_PASS else 'FAIL'}"
)

print(
    f"whole-G2-sector generalize   : "
    f"{'PASS' if RS_G2_GENERALIZATION_PASS else 'FAIL'}"
)

print(
    f"G2 reconstruction            : "
    f"{'PASS' if (RS_G2_RECONSTRUCTION_PASS and RS_G2_GENERALIZATION_PASS) else 'FAIL'}"
)


# ================================================================
# 15. Save reconstructed G2
# ================================================================

RS_G2_COEFF_TABLE.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell4_G2_reconstructed_coefficients.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell4_G2_microscopic_reconstruction.npz",

    beta_hat=
        RS_G2_BETA,

    beta_true=
        RS_BETA_G2_TRUE,

    feature_scale=
        G2_SCALE,

    singular_values=
        RS_G2_SVALS,

    A_est=
        G2_A_EST,

    fit_error=
        RS_G2_FIT_ERROR,

    validation_error=
        RS_G2_VAL_ERROR,

    test_error=
        RS_G2_TEST_ERROR,

    full_G2_sector_error=
        RS_G2_FULL_SECTOR_ERROR,
)


print()
print("=" * 82)
print("Cell 4 PASSED.")
print("=" * 82)

RS-TSC PROTOTYPE — CELL 4
G2 MICROSCOPIC RECONSTRUCTION

----------------------------------------------------------------------------------
A. MULTI-EPSILON GENERATOR RECOVERY
----------------------------------------------------------------------------------
A_est vs exact G2            : 3.856325e-10

----------------------------------------------------------------------------------
B. RECONSTRUCTION SYSTEM
----------------------------------------------------------------------------------
center phase                 : 0.075000
fit / validation / test      : 52 / 13 / 13
physical candidates          : 84
rank                         : 84 / 84
condition                    : 3.677523e+10

----------------------------------------------------------------------------------
C. BLIND GENERATOR RECONSTRUCTION
----------------------------------------------------------------------------------
fit error                    : 1.072004e-13
validation error             : 1.462304e-07
external test e

,candidate_index,interaction_type,support,beta_hat,abs_beta_hat,true_active,beta_true,magnitude_rank
0,25,pair,"(6, 7)",1.11899769,1.118998e+00,True,1.11900000,1
1,6,pair,"(1, 8)",0.94400004,9.440000e-01,True,0.94400000,2
2,18,pair,"(4, 5)",0.91708704,9.170870e-01,True,0.91700000,3
3,7,pair,"(2, 3)",0.91103755,9.110375e-01,True,0.91100000,4
4,28,triad,"(1, 2, 3)",0.02004247,2.004247e-02,True,0.02000000,5
5,60,triad,"(2, 5, 8)",0.01990132,1.990132e-02,True,0.02000000,6
6,52,triad,"(2, 3, 7)",-0.00048415,4.841488e-04,False,0.00000000,7
7,51,triad,"(2, 3, 6)",-0.00042689,4.268860e-04,False,0.00000000,8
8,59,triad,"(2, 5, 7)",0.00034269,3.426908e-04,False,0.00000000,9
9,66,triad,"(3, 4, 7)",0.00033338,3.333779e-04,False,0.00000000,10



----------------------------------------------------------------------------------
E. ORACLE-SIX DIAGNOSTIC
----------------------------------------------------------------------------------
oracle-six rank              : 6 / 6
oracle-six condition         : 1.839423e+01
oracle-six coeff error       : 4.661894e-10

DECISION
held-out reconstruction      : PASS
whole-G2-sector generalize   : PASS
G2 reconstruction            : PASS

Cell 4 PASSED.


In [15]:
# ================================================================
# RS-TSC prototype
# Cell 5: Case 2 — G1 -> G2 temporal-composition test
#
# Window:
#
#       DeltaT = 0.030 < tau
#
# but every local window crosses exactly one switching boundary:
#
#       G1 -> G2
#
# For each window:
#
#       a = time spent in G1
#       b = time spent in G2
#
#       a + b = DeltaT
#
# Persistent generator:
#
#       F_bar
#         =
#       (a/DeltaT) G1_hat
#       +
#       (b/DeltaT) G2_hat
#
# First BCH temporal correction for chronological flow
#
#       G1 for a
#           ->
#       G2 for b
#
# is defined here as
#
#       B_1to2
#         =
#       DG2 G1 - DG1 G2
#
# so
#
#       F_eff_BCH1
#         =
#       F_bar
#       +
#       [ab/(2 DeltaT)] B_1to2.
#
#
# We compare THREE forward predictions:
#
#   1. persistent:
#          autonomous flow of F_bar
#
#   2. BCH1 effective:
#          autonomous flow of F_eff_BCH1
#
#   3. reconstructed microscopic composition:
#          flow G1_hat for a
#          then flow G2_hat for b
#
# against the ACTUAL stored time-series endpoint.
#
# No sparse inference.
# No topology oracle in the main calculation.
# ================================================================


# ================================================================
# 0. Case-2 data
# ================================================================

RS_CASE2 = RS_CASES[
    "case2_G1_to_G2"
]


C2_X0 = np.asarray(
    RS_CASE2[
        "X_start"
    ],
    dtype=float,
)


C2_X_TRUE = np.asarray(
    RS_CASE2[
        "X_end"
    ],
    dtype=float,
)


C2_A = np.asarray(
    RS_CASE2[
        "occupation_duration"
    ][:, 0],
    dtype=float,
)


C2_B = np.asarray(
    RS_CASE2[
        "occupation_duration"
    ][:, 1],
    dtype=float,
)


C2_DELTA_T = float(
    RS_CASE2[
        "delta_T"
    ]
)


C2_ALPHA = (
    C2_A
    /
    C2_DELTA_T
)


C2_FIT_MASK = np.asarray(
    RS_CASE2[
        "fit_mask"
    ],
    dtype=bool,
)

C2_VAL_MASK = np.asarray(
    RS_CASE2[
        "validation_mask"
    ],
    dtype=bool,
)

C2_TEST_MASK = np.asarray(
    RS_CASE2[
        "test_mask"
    ],
    dtype=bool,
)

C2_CENTER_MASK = np.asarray(
    RS_CASE2[
        "center_mask"
    ],
    dtype=bool,
)


assert np.allclose(
    C2_A + C2_B,
    C2_DELTA_T,
    atol=1e-14,
    rtol=0.0,
)

assert np.all(
    C2_A > 0.0
)

assert np.all(
    C2_B > 0.0
)


# ================================================================
# 1. Generic physical field from an 84-vector beta
# ================================================================

def rs_field_from_beta(
    X,
    beta,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    beta = np.asarray(
        beta,
        dtype=float,
    )

    single_state = (
        X.ndim == 1
    )

    if single_state:
        X = X[
            None,
            :
        ]

    Psi = (
        rs_build_physical_design(
            X
        )
    )

    F = (
        Psi
        @
        beta
    ).reshape(
        len(X),
        RS_N,
    )

    if single_state:
        return F[0]

    return F


# ================================================================
# 2. Analytic Jacobian of a reconstructed physical field
#
# Needed for the BCH / Lie-bracket correction.
# ================================================================

def rs_physical_jacobian(
    x,
    beta,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    beta = np.asarray(
        beta,
        dtype=float,
    )

    assert x.shape == (
        RS_N,
    )

    J = np.zeros(
        (
            RS_N,
            RS_N,
        ),
        dtype=float,
    )


    for coeff, (
        interaction_type,
        support,
    ) in zip(
        beta,
        RS_P2_CANDIDATES,
    ):

        if coeff == 0.0:
            continue


        if interaction_type == "pair":

            i, j = [
                int(v) - 1
                for v in support
            ]

            dpi = (
                1.0
                +
                x[i]
            )

            dpj = (
                1.0
                +
                x[j]
            )

            J[i, i] += (
                coeff
                *
                (-dpi)
            )

            J[i, j] += (
                coeff
                *
                dpj
            )

            J[j, i] += (
                coeff
                *
                dpi
            )

            J[j, j] += (
                coeff
                *
                (-dpj)
            )


        elif interaction_type == "triad":

            i, j, k = [
                int(v) - 1
                for v in support
            ]

            xi = x[i]
            xj = x[j]
            xk = x[k]


            # Fi
            J[i, i] += coeff * (
                -0.5 * (xj + xk)
            )

            J[i, j] += coeff * (
                xk - 0.5 * xi
            )

            J[i, k] += coeff * (
                xj - 0.5 * xi
            )


            # Fj
            J[j, j] += coeff * (
                -0.5 * (xi + xk)
            )

            J[j, i] += coeff * (
                xk - 0.5 * xj
            )

            J[j, k] += coeff * (
                xi - 0.5 * xj
            )


            # Fk
            J[k, k] += coeff * (
                -0.5 * (xi + xj)
            )

            J[k, i] += coeff * (
                xj - 0.5 * xk
            )

            J[k, j] += coeff * (
                xi - 0.5 * xk
            )


        else:

            raise RuntimeError(
                interaction_type
            )


    return J


# ================================================================
# 3. Chronological bracket
#
# For:
#
#       G1 first
#       G2 second
#
# define
#
#       B_1to2
#         =
#       DG2 G1 - DG1 G2.
#
# This convention is chosen directly from the state-flow expansion,
# avoiding ambiguity in [G1,G2] sign conventions across literature.
# ================================================================

def rs_chronological_bracket_1to2(
    x,
    beta1,
    beta2,
):

    G1 = rs_field_from_beta(
        x,
        beta1,
    )

    G2 = rs_field_from_beta(
        x,
        beta2,
    )

    J1 = rs_physical_jacobian(
        x,
        beta1,
    )

    J2 = rs_physical_jacobian(
        x,
        beta2,
    )

    return (
        J2 @ G1
        -
        J1 @ G2
    )


# ================================================================
# 4. Generic fixed-step RK4 for autonomous reconstructed fields
# ================================================================

def rs_rk4_autonomous(
    x0,
    duration,
    field,
    dt=RS_DT_OBS,
):

    x = np.asarray(
        x0,
        dtype=float,
    ).copy()

    duration = float(
        duration
    )


    if duration == 0.0:
        return x


    n_steps = int(
        round(
            duration
            /
            dt
        )
    )


    if not np.isclose(
        n_steps * dt,
        duration,
        atol=1e-12,
        rtol=0.0,
    ):

        raise ValueError(
            "duration must lie on the native "
            "observation grid"
        )


    for _ in range(
        n_steps
    ):

        k1 = field(
            x
        )

        k2 = field(
            x
            +
            0.5 * dt * k1
        )

        k3 = field(
            x
            +
            0.5 * dt * k2
        )

        k4 = field(
            x
            +
            dt * k3
        )

        x = (
            x
            +
            (dt / 6.0)
            *
            (
                k1
                +
                2.0 * k2
                +
                2.0 * k3
                +
                k4
            )
        )


    return x


# ================================================================
# 5. Endpoint-error metrics
#
# State-relative error can be misleading because the state itself
# contains a large conserved background.
#
# Main metric:
#
#   error relative to the TRUE displacement over the window.
# ================================================================

def rs_displacement_relative_error(
    X_pred,
    X0,
    X_true,
):

    pred_disp = (
        np.asarray(
            X_pred
        )
        -
        np.asarray(
            X0
        )
    )

    true_disp = (
        np.asarray(
            X_true
        )
        -
        np.asarray(
            X0
        )
    )

    return float(
        np.linalg.norm(
            (
                pred_disp
                -
                true_disp
            ).ravel()
        )
        /
        max(
            np.linalg.norm(
                true_disp.ravel()
            ),
            np.finfo(float).tiny,
        )
    )


# ================================================================
# 6. Forward predictions for all 78 Case-2 windows
# ================================================================

C2_X_PERSISTENT = []
C2_X_BCH1 = []
C2_X_MICRO_COMPOSED = []

C2_TEMPORAL_RATIO_AT_START = []
C2_BRACKET_NORM_AT_START = []


for n in range(
    len(C2_X0)
):

    x0 = C2_X0[n]

    a = float(
        C2_A[n]
    )

    b = float(
        C2_B[n]
    )

    alpha = float(
        C2_ALPHA[n]
    )


    # ------------------------------------------------------------
    # Persistent coefficient vector
    #
    # Because G1_hat and G2_hat share exactly the same physical
    # interaction dictionary:
    #
    #   beta_bar = alpha beta1 + (1-alpha) beta2.
    # ------------------------------------------------------------

    beta_bar = (
        alpha
        *
        RS_G1_BETA
        +
        (
            1.0
            -
            alpha
        )
        *
        RS_G2_BETA
    )


    def F_bar(x):

        return rs_field_from_beta(
            x,
            beta_bar,
        )


    # ------------------------------------------------------------
    # BCH1 temporal correction
    # ------------------------------------------------------------

    temporal_prefactor = (
        a
        *
        b
        /
        (
            2.0
            *
            C2_DELTA_T
        )
    )


    def F_temp(x):

        return (
            temporal_prefactor
            *
            rs_chronological_bracket_1to2(
                x,
                RS_G1_BETA,
                RS_G2_BETA,
            )
        )


    def F_eff_BCH1(x):

        return (
            F_bar(x)
            +
            F_temp(x)
        )


    # ------------------------------------------------------------
    # 1. Persistent prediction
    # ------------------------------------------------------------

    x_persistent = (
        rs_rk4_autonomous(
            x0,
            C2_DELTA_T,
            F_bar,
        )
    )


    # ------------------------------------------------------------
    # 2. BCH1 effective prediction
    # ------------------------------------------------------------

    x_bch1 = (
        rs_rk4_autonomous(
            x0,
            C2_DELTA_T,
            F_eff_BCH1,
        )
    )


    # ------------------------------------------------------------
    # 3. Reconstructed microscopic composition
    #
    # G1_hat for a, then G2_hat for b.
    #
    # This is the control for microscopic reconstruction quality.
    # ------------------------------------------------------------

    x_mid = (
        rs_rk4_autonomous(
            x0,
            a,
            RS_G1_HAT,
        )
    )

    x_micro = (
        rs_rk4_autonomous(
            x_mid,
            b,
            RS_G2_HAT,
        )
    )


    C2_X_PERSISTENT.append(
        x_persistent
    )

    C2_X_BCH1.append(
        x_bch1
    )

    C2_X_MICRO_COMPOSED.append(
        x_micro
    )


    # ------------------------------------------------------------
    # Temporal correction strength at window start
    # ------------------------------------------------------------

    fbar0 = F_bar(
        x0
    )

    ftemp0 = F_temp(
        x0
    )

    C2_BRACKET_NORM_AT_START.append(
        float(
            np.linalg.norm(
                rs_chronological_bracket_1to2(
                    x0,
                    RS_G1_BETA,
                    RS_G2_BETA,
                )
            )
        )
    )

    C2_TEMPORAL_RATIO_AT_START.append(
        float(
            np.linalg.norm(
                ftemp0
            )
            /
            max(
                np.linalg.norm(
                    fbar0
                ),
                np.finfo(float).tiny,
            )
        )
    )


C2_X_PERSISTENT = np.asarray(
    C2_X_PERSISTENT
)

C2_X_BCH1 = np.asarray(
    C2_X_BCH1
)

C2_X_MICRO_COMPOSED = np.asarray(
    C2_X_MICRO_COMPOSED
)

C2_TEMPORAL_RATIO_AT_START = np.asarray(
    C2_TEMPORAL_RATIO_AT_START
)

C2_BRACKET_NORM_AT_START = np.asarray(
    C2_BRACKET_NORM_AT_START
)


# ================================================================
# 7. Global / split errors
# ================================================================

def c2_error_for_mask(
    prediction,
    mask,
):

    return (
        rs_displacement_relative_error(
            prediction[
                mask
            ],
            C2_X0[
                mask
            ],
            C2_X_TRUE[
                mask
            ],
        )
    )


C2_ERROR_ROWS = []


for split_name, mask in (

    (
        "fit-cycles",
        C2_FIT_MASK,
    ),

    (
        "validation-cycle",
        C2_VAL_MASK,
    ),

    (
        "external-test-cycle",
        C2_TEST_MASK,
    ),

    (
        "all",
        np.ones(
            len(C2_X0),
            dtype=bool,
        ),
    ),

    (
        "center-only",
        C2_CENTER_MASK,
    ),
):

    persistent_err = (
        c2_error_for_mask(
            C2_X_PERSISTENT,
            mask,
        )
    )

    bch1_err = (
        c2_error_for_mask(
            C2_X_BCH1,
            mask,
        )
    )

    micro_err = (
        c2_error_for_mask(
            C2_X_MICRO_COMPOSED,
            mask,
        )
    )


    C2_ERROR_ROWS.append({

        "split":
            split_name,

        "n_windows":
            int(
                np.sum(
                    mask
                )
            ),

        "persistent_error":
            persistent_err,

        "BCH1_error":
            bch1_err,

        "micro_composed_error":
            micro_err,

        "BCH1_improvement_factor":
            (
                persistent_err
                /
                max(
                    bch1_err,
                    np.finfo(float).tiny,
                )
            ),

        "micro_improvement_factor":
            (
                persistent_err
                /
                max(
                    micro_err,
                    np.finfo(float).tiny,
                )
            ),
    })


C2_ERROR_TABLE = pd.DataFrame(
    C2_ERROR_ROWS
)


# ================================================================
# 8. Mini position scan over the 13 local offsets
#
# Pool the six repeated cycles at each phase offset.
# ================================================================

C2_OFFSET_ROWS = []


for offset in np.unique(
    RS_CASE2[
        "local_offset"
    ]
):

    offset_mask = np.isclose(
        RS_CASE2[
            "local_offset"
        ],
        offset,
        atol=1e-15,
        rtol=0.0,
    )


    alpha = float(
        np.mean(
            C2_ALPHA[
                offset_mask
            ]
        )
    )


    a = float(
        np.mean(
            C2_A[
                offset_mask
            ]
        )
    )

    b = float(
        np.mean(
            C2_B[
                offset_mask
            ]
        )
    )


    persistent_err = (
        c2_error_for_mask(
            C2_X_PERSISTENT,
            offset_mask,
        )
    )

    bch1_err = (
        c2_error_for_mask(
            C2_X_BCH1,
            offset_mask,
        )
    )

    micro_err = (
        c2_error_for_mask(
            C2_X_MICRO_COMPOSED,
            offset_mask,
        )
    )


    C2_OFFSET_ROWS.append({

        "local_offset":
            float(
                offset
            ),

        "phase_start":
            float(
                RS_CASE2[
                    "center_phase"
                ]
                +
                offset
            ),

        "a_G1":
            a,

        "b_G2":
            b,

        "alpha_G1":
            alpha,

        "alpha_G2":
            1.0 - alpha,

        "alpha_product":
            alpha
            *
            (
                1.0
                -
                alpha
            ),

        "temporal_prefactor":
            (
                a
                *
                b
                /
                (
                    2.0
                    *
                    C2_DELTA_T
                )
            ),

        "mean_temporal_ratio_at_start":
            float(
                np.mean(
                    C2_TEMPORAL_RATIO_AT_START[
                        offset_mask
                    ]
                )
            ),

        "persistent_error":
            persistent_err,

        "BCH1_error":
            bch1_err,

        "micro_composed_error":
            micro_err,

        "BCH1_improvement_factor":
            (
                persistent_err
                /
                max(
                    bch1_err,
                    np.finfo(float).tiny,
                )
            ),
    })


C2_POSITION_SCAN = pd.DataFrame(
    C2_OFFSET_ROWS
)


# ================================================================
# 9. Oracle BCH diagnostic
#
# This is NOT used above.
#
# It tells us whether any remaining BCH1 error is caused primarily
# by microscopic reconstruction error or by BCH truncation.
# ================================================================

C2_X_BCH1_ORACLE = []


for n in range(
    len(C2_X0)
):

    x0 = C2_X0[n]

    a = float(
        C2_A[n]
    )

    b = float(
        C2_B[n]
    )

    alpha = float(
        C2_ALPHA[n]
    )


    beta_bar_true = (
        alpha
        *
        RS_BETA_G1_TRUE
        +
        (
            1.0
            -
            alpha
        )
        *
        RS_BETA_G2_TRUE
    )


    temporal_prefactor = (
        a
        *
        b
        /
        (
            2.0
            *
            C2_DELTA_T
        )
    )


    def F_eff_oracle_BCH1(x):

        return (
            rs_field_from_beta(
                x,
                beta_bar_true,
            )
            +
            temporal_prefactor
            *
            rs_chronological_bracket_1to2(
                x,
                RS_BETA_G1_TRUE,
                RS_BETA_G2_TRUE,
            )
        )


    C2_X_BCH1_ORACLE.append(
        rs_rk4_autonomous(
            x0,
            C2_DELTA_T,
            F_eff_oracle_BCH1,
        )
    )


C2_X_BCH1_ORACLE = np.asarray(
    C2_X_BCH1_ORACLE
)


C2_BCH1_ORACLE_TEST_ERROR = (
    c2_error_for_mask(
        C2_X_BCH1_ORACLE,
        C2_TEST_MASK,
    )
)


# ================================================================
# 10. Report
# ================================================================

print("=" * 84)
print("RS-TSC PROTOTYPE — CELL 5")
print("CASE 2: G1 -> G2 TEMPORAL COMPOSITION")
print("=" * 84)

print(
    f"DeltaT                       : "
    f"{C2_DELTA_T:.6f}"
)

print(
    f"number of windows            : "
    f"{len(C2_X0)}"
)

print(
    f"alpha_G1 range               : "
    f"[{np.min(C2_ALPHA):.6f}, "
    f"{np.max(C2_ALPHA):.6f}]"
)

print(
    f"center composition           : "
    f"G1=0.500000, G2=0.500000"
)


print()
print("-" * 84)
print("A. ENDPOINT PREDICTION")
print("-" * 84)

display(
    C2_ERROR_TABLE.style.format({

        "persistent_error":
            "{:.6e}",

        "BCH1_error":
            "{:.6e}",

        "micro_composed_error":
            "{:.6e}",

        "BCH1_improvement_factor":
            "{:.6f}",

        "micro_improvement_factor":
            "{:.6f}",
    })
)


print()
print("-" * 84)
print("B. MINI POSITION SCAN")
print("-" * 84)

display(
    C2_POSITION_SCAN.style.format({

        "local_offset":
            "{:.6f}",

        "phase_start":
            "{:.6f}",

        "a_G1":
            "{:.6f}",

        "b_G2":
            "{:.6f}",

        "alpha_G1":
            "{:.6f}",

        "alpha_G2":
            "{:.6f}",

        "alpha_product":
            "{:.6f}",

        "temporal_prefactor":
            "{:.6e}",

        "mean_temporal_ratio_at_start":
            "{:.6e}",

        "persistent_error":
            "{:.6e}",

        "BCH1_error":
            "{:.6e}",

        "micro_composed_error":
            "{:.6e}",

        "BCH1_improvement_factor":
            "{:.6f}",
    })
)


print()
print("-" * 84)
print("C. ORACLE BCH1 TRUNCATION DIAGNOSTIC")
print("-" * 84)

print(
    f"oracle BCH1 external-test err: "
    f"{C2_BCH1_ORACLE_TEST_ERROR:.6e}"
)


# ================================================================
# 11. Decision
# ================================================================

C2_TEST_ROW = (
    C2_ERROR_TABLE[
        C2_ERROR_TABLE[
            "split"
        ]
        ==
        "external-test-cycle"
    ]
    .iloc[0]
)


C2_MICRO_COMPOSITION_PASS = bool(
    C2_TEST_ROW[
        "micro_composed_error"
    ]
    <
    1e-4
)


C2_BCH_IMPROVES_PASS = bool(
    C2_TEST_ROW[
        "BCH1_error"
    ]
    <
    C2_TEST_ROW[
        "persistent_error"
    ]
)


print()
print("=" * 84)
print("DECISION")
print("=" * 84)

print(
    f"reconstructed micro composition: "
    f"{'PASS' if C2_MICRO_COMPOSITION_PASS else 'FAIL'}"
)

print(
    f"BCH1 improves persistent model : "
    f"{'PASS' if C2_BCH_IMPROVES_PASS else 'FAIL'}"
)

print(
    f"CASE-2 temporal composition    : "
    f"{'PASS' if (C2_MICRO_COMPOSITION_PASS and C2_BCH_IMPROVES_PASS) else 'FAIL'}"
)


# ================================================================
# 12. Save
# ================================================================

C2_ERROR_TABLE.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell5_case2_endpoint_errors.csv",
    index=False,
)


C2_POSITION_SCAN.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell5_case2_position_scan.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell5_case2_temporal_composition.npz",

    X0=
        C2_X0,

    X_true=
        C2_X_TRUE,

    X_persistent=
        C2_X_PERSISTENT,

    X_BCH1=
        C2_X_BCH1,

    X_micro_composed=
        C2_X_MICRO_COMPOSED,

    X_BCH1_oracle=
        C2_X_BCH1_ORACLE,

    a=
        C2_A,

    b=
        C2_B,

    alpha=
        C2_ALPHA,

    temporal_ratio_at_start=
        C2_TEMPORAL_RATIO_AT_START,
)


print()
print("=" * 84)
print("Cell 5 PASSED.")
print("=" * 84)

RS-TSC PROTOTYPE — CELL 5
CASE 2: G1 -> G2 TEMPORAL COMPOSITION
DeltaT                       : 0.030000
number of windows            : 78
alpha_G1 range               : [0.400000, 0.600000]
center composition           : G1=0.500000, G2=0.500000

------------------------------------------------------------------------------------
A. ENDPOINT PREDICTION
------------------------------------------------------------------------------------


,split,n_windows,persistent_error,BCH1_error,micro_composed_error,BCH1_improvement_factor,micro_improvement_factor
0,fit-cycles,52,3.170319e-03,2.026783e-05,7.426474e-10,156.421231,4268942.390436
1,validation-cycle,13,5.424695e-03,3.623598e-05,9.550956e-08,149.704641,56797.400953
2,external-test-cycle,13,6.205257e-03,4.148645e-05,3.365158e-07,149.573097,18439.721391
3,all,78,3.280202e-03,2.106247e-05,3.802392e-08,155.736850,86266.805418
4,center-only,6,3.336883e-03,2.106821e-05,3.773260e-08,158.384777,88435.009454



------------------------------------------------------------------------------------
B. MINI POSITION SCAN
------------------------------------------------------------------------------------


,local_offset,phase_start,a_G1,b_G2,alpha_G1,alpha_G2,alpha_product,temporal_prefactor,mean_temporal_ratio_at_start,persistent_error,BCH1_error,micro_composed_error,BCH1_improvement_factor
0,-0.003000,0.042000,0.018000,0.012000,0.600000,0.400000,0.240000,3.600000e-03,4.330557e-03,3.196361e-03,2.238598e-05,3.011813e-08,142.784051
1,-0.002500,0.042500,0.017500,0.012500,0.583333,0.416667,0.243056,3.645833e-03,4.383291e-03,3.240044e-03,2.221701e-05,3.137351e-08,145.836206
2,-0.002000,0.043000,0.017000,0.013000,0.566667,0.433333,0.245556,3.683333e-03,4.423821e-03,3.275666e-03,2.201958e-05,3.263709e-08,148.761518
3,-0.001500,0.043500,0.016500,0.013500,0.550000,0.450000,0.247500,3.712500e-03,4.452139e-03,3.303186e-03,2.180023e-05,3.390669e-08,151.520645
4,-0.001000,0.044000,0.016000,0.014000,0.533333,0.466667,0.248889,3.733333e-03,4.468277e-03,3.322572e-03,2.156503e-05,3.518036e-08,154.072216
5,-0.000500,0.044500,0.015500,0.014500,0.516667,0.483333,0.249722,3.745833e-03,4.472307e-03,3.333807e-03,2.131944e-05,3.645623e-08,156.374068
6,0.000000,0.045000,0.015000,0.015000,0.500000,0.500000,0.250000,3.750000e-03,4.464339e-03,3.336883e-03,2.106821e-05,3.773260e-08,158.384777
7,0.000500,0.045500,0.014500,0.015500,0.483333,0.516667,0.249722,3.745833e-03,4.444518e-03,3.331803e-03,2.081525e-05,3.900788e-08,160.065441
8,0.001000,0.046000,0.014000,0.016000,0.466667,0.533333,0.248889,3.733333e-03,4.413024e-03,3.318582e-03,2.056357e-05,4.028056e-08,161.381566
9,0.001500,0.046500,0.013500,0.016500,0.450000,0.550000,0.247500,3.712500e-03,4.370065e-03,3.297246e-03,2.031513e-05,4.154924e-08,162.304922



------------------------------------------------------------------------------------
C. ORACLE BCH1 TRUNCATION DIAGNOSTIC
------------------------------------------------------------------------------------
oracle BCH1 external-test err: 4.150178e-05

DECISION
reconstructed micro composition: PASS
BCH1 improves persistent model : PASS
CASE-2 temporal composition    : PASS

Cell 5 PASSED.


In [16]:
# ================================================================
# RS-TSC prototype
# Cell 6: local-perturbation identifiability audit
#
# QUESTION
# ---------------------------------------------------------------
# Around the Case-2 inference point
#
#       s0      = 0.045
#       DeltaT  = 0.030
#
# do slightly shifted starts
#
#       s = s0 + delta
#
# provide genuinely useful independent information?
#
#
# We DO NOT infer F_eff in this cell.
#
# Instead, as the local bandwidth h increases, we audit:
#
#   A. state-cloud singular spectrum
#
#   B. P2 physical-design singular spectrum
#      (84 known microscopic interaction templates)
#
#   C. broad polynomial feature singular spectrum
#      degree <= 3, N=8 -> 164 scalar monomials
#
#   D. protocol drift away from the center 50/50
#      G1 -> G2 composition.
#
#
# Three pooling modes:
#
#   one_cycle  : cycle 0 only
#   fit_cycles : cycles 0-3
#   all_cycles : cycles 0-5
#
#
# This separates:
#
#   dense local sampling
#
# from
#
#   repeated-protocol state diversity across cycles.
# ================================================================

from itertools import product


# ================================================================
# 0. Bandwidth grid
# ================================================================

C6_H_VALUES = np.asarray(
    [
        0.0000,
        0.0005,
        0.0010,
        0.0015,
        0.0020,
        0.0025,
        0.0030,
    ],
    dtype=float,
)


C6_POOLING_MODES = {

    "one_cycle": (
        0,
    ),

    "fit_cycles": (
        0,
        1,
        2,
        3,
    ),

    "all_cycles": (
        0,
        1,
        2,
        3,
        4,
        5,
    ),
}


# ================================================================
# 1. Broad degree <= 3 scalar polynomial library
#
# Number of non-constant monomials:
#
#   degree 1 : 8
#   degree 2 : 36
#   degree 3 : 120
#
# total:
#
#   164
#
# This matches the broad library used in the earlier audit.
# ================================================================

def c6_generate_exponents(
    n_variables,
    max_degree,
):

    exponents = []

    def recurse(
        remaining_degree,
        variable_index,
        current,
    ):

        if variable_index == n_variables - 1:

            exp = (
                current
                +
                [
                    remaining_degree
                ]
            )

            exponents.append(
                tuple(exp)
            )

            return


        for power in range(
            remaining_degree + 1
        ):

            recurse(
                remaining_degree - power,
                variable_index + 1,
                current + [power],
            )


    for degree in range(
        1,
        max_degree + 1
    ):

        recurse(
            degree,
            0,
            [],
        )


    return tuple(
        exponents
    )


C6_POLY_EXPONENTS = (
    c6_generate_exponents(
        RS_N,
        3,
    )
)


assert len(
    C6_POLY_EXPONENTS
) == 164


def c6_polynomial_design(
    X,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    Phi = np.empty(
        (
            len(X),
            len(
                C6_POLY_EXPONENTS
            ),
        ),
        dtype=float,
    )


    for q, exponent in enumerate(
        C6_POLY_EXPONENTS
    ):

        exponent = np.asarray(
            exponent,
            dtype=int,
        )

        Phi[
            :,
            q
        ] = np.prod(
            X ** exponent[
                None,
                :
            ],
            axis=1,
        )


    return Phi


# ================================================================
# 2. Scaled-SVD helper
#
# Column RMS scaling avoids declaring a direction unimportant only
# because one feature naturally has a much smaller physical scale.
#
# We report several effective ranks:
#
#       sigma_i / sigma_max > tolerance
#
# because exact algebraic rank alone can be misleading for a
# nearly singular local orbit.
# ================================================================

def c6_scaled_svd_metrics(
    A,
    n_parameters=None,
):

    A = np.asarray(
        A,
        dtype=float,
    )


    if A.ndim != 2:
        raise ValueError(
            "A must be a matrix."
        )


    if A.shape[0] == 0:
        raise ValueError(
            "A has no rows."
        )


    scale = np.sqrt(
        np.mean(
            A ** 2,
            axis=0,
        )
    )


    valid = (
        scale
        >
        1e-30
    )


    A_scaled = (
        A[
            :,
            valid
        ]
        /
        scale[
            valid
        ][
            None,
            :
        ]
    )


    singular_values = np.linalg.svd(
        A_scaled,
        compute_uv=False,
    )


    if len(
        singular_values
    ) == 0:

        return {
            "algebraic_rank": 0,
            "effective_rank_1e-6": 0,
            "effective_rank_1e-8": 0,
            "effective_rank_1e-10": 0,
            "effective_rank_1e-12": 0,
            "condition_nonzero": np.inf,
            "sigma_min_rel": 0.0,
            "n_valid_columns": int(
                np.sum(valid)
            ),
            "singular_values":
                singular_values,
        }


    sigma_max = float(
        singular_values[0]
    )


    rel = (
        singular_values
        /
        max(
            sigma_max,
            np.finfo(float).tiny,
        )
    )


    algebraic_rank = int(
        np.linalg.matrix_rank(
            A_scaled
        )
    )


    positive = (
        singular_values
        >
        sigma_max
        *
        np.finfo(float).eps
        *
        max(
            A_scaled.shape
        )
    )


    if np.any(
        positive
    ):

        sigma_min_nonzero = float(
            singular_values[
                np.flatnonzero(
                    positive
                )[-1]
            ]
        )

        condition_nonzero = (
            sigma_max
            /
            sigma_min_nonzero
        )

    else:

        condition_nonzero = np.inf


    if n_parameters is None:
        n_parameters = (
            A.shape[1]
        )


    return {

        "algebraic_rank":
            algebraic_rank,

        "effective_rank_1e-6":
            int(
                np.sum(
                    rel > 1e-6
                )
            ),

        "effective_rank_1e-8":
            int(
                np.sum(
                    rel > 1e-8
                )
            ),

        "effective_rank_1e-10":
            int(
                np.sum(
                    rel > 1e-10
                )
            ),

        "effective_rank_1e-12":
            int(
                np.sum(
                    rel > 1e-12
                )
            ),

        "condition_nonzero":
            float(
                condition_nonzero
            ),

        "sigma_min_rel":
            float(
                rel[-1]
            ),

        "n_valid_columns":
            int(
                np.sum(
                    valid
                )
            ),

        "parameter_nullity_lower_bound":
            int(
                max(
                    0,
                    n_parameters
                    -
                    algebraic_rank
                )
            ),

        "singular_values":
            singular_values,
    }


# ================================================================
# 3. State-cloud geometry
#
# For states themselves we center the cloud first.
#
# Because total mass is conserved, the state cloud can occupy at
# most a 7-dimensional affine subspace in R^8.
#
# More importantly, a very local piece of one trajectory should
# show a strongly dominant tangent direction.
# ================================================================

def c6_state_cloud_metrics(
    X,
):

    X = np.asarray(
        X,
        dtype=float,
    )


    X_centered = (
        X
        -
        np.mean(
            X,
            axis=0,
            keepdims=True,
        )
    )


    singular_values = np.linalg.svd(
        X_centered,
        compute_uv=False,
    )


    if (
        len(singular_values) == 0
        or
        singular_values[0] == 0.0
    ):

        rel = np.zeros_like(
            singular_values
        )

    else:

        rel = (
            singular_values
            /
            singular_values[0]
        )


    return {

        "state_rank":
            int(
                np.linalg.matrix_rank(
                    X_centered
                )
            ),

        "state_effrank_1e-6":
            int(
                np.sum(
                    rel > 1e-6
                )
            ),

        "state_effrank_1e-8":
            int(
                np.sum(
                    rel > 1e-8
                )
            ),

        "state_effrank_1e-10":
            int(
                np.sum(
                    rel > 1e-10
                )
            ),

        "state_sigma2_over_sigma1":
            float(
                rel[1]
                if len(rel) > 1
                else 0.0
            ),

        "state_sigma3_over_sigma1":
            float(
                rel[2]
                if len(rel) > 2
                else 0.0
            ),
    }


# ================================================================
# 4. Center-protocol BCH1 field
#
# Used ONLY to quantify protocol drift.
#
# We compare:
#
#       F_eff_BCH1(x ; alpha)
#
# with
#
#       F_eff_BCH1(x ; alpha=0.5)
#
# at THE SAME STATE x.
#
# Therefore this diagnostic isolates protocol variation from
# ordinary state variation along the trajectory.
# ================================================================

def c6_bch1_effective_field(
    x,
    alpha,
):

    alpha = float(
        alpha
    )


    beta_bar = (
        alpha
        *
        RS_G1_BETA
        +
        (
            1.0
            -
            alpha
        )
        *
        RS_G2_BETA
    )


    temporal_prefactor = (
        0.5
        *
        C2_DELTA_T
        *
        alpha
        *
        (
            1.0
            -
            alpha
        )
    )


    return (
        rs_field_from_beta(
            x,
            beta_bar,
        )
        +
        temporal_prefactor
        *
        rs_chronological_bracket_1to2(
            x,
            RS_G1_BETA,
            RS_G2_BETA,
        )
    )


def c6_protocol_drift(
    x,
    alpha,
):

    F_center = (
        c6_bch1_effective_field(
            x,
            0.5,
        )
    )


    F_shifted = (
        c6_bch1_effective_field(
            x,
            alpha,
        )
    )


    return float(
        np.linalg.norm(
            F_shifted
            -
            F_center
        )
        /
        max(
            np.linalg.norm(
                F_center
            ),
            np.finfo(float).tiny,
        )
    )


# ================================================================
# 5. Run bandwidth audit
# ================================================================

C6_ROWS = []


C6_CASE2_OFFSETS = np.asarray(
    RS_CASE2[
        "local_offset"
    ],
    dtype=float,
)


C6_CASE2_CYCLES = np.asarray(
    RS_CASE2[
        "cycle"
    ],
    dtype=int,
)


for pooling_name, cycles_used in (
    C6_POOLING_MODES.items()
):

    cycle_mask = np.isin(
        C6_CASE2_CYCLES,
        np.asarray(
            cycles_used,
            dtype=int,
        ),
    )


    for h in C6_H_VALUES:

        bandwidth_mask = (
            np.abs(
                C6_CASE2_OFFSETS
            )
            <=
            h + 1e-15
        )


        mask = (
            cycle_mask
            &
            bandwidth_mask
        )


        X_local = (
            C2_X0[
                mask
            ]
        )


        alpha_local = (
            C2_ALPHA[
                mask
            ]
        )


        offsets_local = (
            C6_CASE2_OFFSETS[
                mask
            ]
        )


        # --------------------------------------------------------
        # A. State-cloud geometry
        # --------------------------------------------------------

        state_metrics = (
            c6_state_cloud_metrics(
                X_local
            )
        )


        # --------------------------------------------------------
        # B. P2 physical design
        #
        # Vector-valued design:
        #
        #       (M * 8) x 84
        #
        # --------------------------------------------------------

        P2_design = (
            rs_build_physical_design(
                X_local
            )
        )


        P2_metrics = (
            c6_scaled_svd_metrics(
                P2_design,
                n_parameters=84,
            )
        )


        # --------------------------------------------------------
        # C. Broad degree<=3 scalar polynomial design
        #
        #       M x 164
        #
        # Each output would have its own 164 coefficients in the
        # completely unrestricted vector-field representation.
        #
        # --------------------------------------------------------

        poly_design = (
            c6_polynomial_design(
                X_local
            )
        )


        poly_metrics = (
            c6_scaled_svd_metrics(
                poly_design,
                n_parameters=164,
            )
        )


        # --------------------------------------------------------
        # D. Protocol variation relative to alpha=0.5
        # --------------------------------------------------------

        protocol_drifts = np.asarray(
            [
                c6_protocol_drift(
                    x,
                    alpha,
                )

                for x, alpha
                in zip(
                    X_local,
                    alpha_local,
                )
            ],
            dtype=float,
        )


        max_alpha_deviation = float(
            np.max(
                np.abs(
                    alpha_local
                    -
                    0.5
                )
            )
        )


        C6_ROWS.append({

            "pooling":
                pooling_name,

            "h":
                float(h),

            "n_cycles":
                len(
                    cycles_used
                ),

            "n_states":
                len(
                    X_local
                ),

            "n_offsets":
                len(
                    np.unique(
                        offsets_local
                    )
                ),

            "alpha_min":
                float(
                    np.min(
                        alpha_local
                    )
                ),

            "alpha_max":
                float(
                    np.max(
                        alpha_local
                    )
                ),

            "max_alpha_deviation":
                max_alpha_deviation,

            "mean_protocol_drift":
                float(
                    np.mean(
                        protocol_drifts
                    )
                ),

            "max_protocol_drift":
                float(
                    np.max(
                        protocol_drifts
                    )
                ),


            # state cloud
            "state_rank":
                state_metrics[
                    "state_rank"
                ],

            "state_erank_1e-6":
                state_metrics[
                    "state_effrank_1e-6"
                ],

            "state_erank_1e-8":
                state_metrics[
                    "state_effrank_1e-8"
                ],

            "state_erank_1e-10":
                state_metrics[
                    "state_effrank_1e-10"
                ],

            "state_sigma2_sigma1":
                state_metrics[
                    "state_sigma2_over_sigma1"
                ],

            "state_sigma3_sigma1":
                state_metrics[
                    "state_sigma3_over_sigma1"
                ],


            # P2
            "P2_rank":
                P2_metrics[
                    "algebraic_rank"
                ],

            "P2_erank_1e-6":
                P2_metrics[
                    "effective_rank_1e-6"
                ],

            "P2_erank_1e-8":
                P2_metrics[
                    "effective_rank_1e-8"
                ],

            "P2_erank_1e-10":
                P2_metrics[
                    "effective_rank_1e-10"
                ],

            "P2_erank_1e-12":
                P2_metrics[
                    "effective_rank_1e-12"
                ],

            "P2_condition":
                P2_metrics[
                    "condition_nonzero"
                ],

            "P2_nullity_lb":
                P2_metrics[
                    "parameter_nullity_lower_bound"
                ],


            # broad polynomial
            "P0_rank":
                poly_metrics[
                    "algebraic_rank"
                ],

            "P0_erank_1e-6":
                poly_metrics[
                    "effective_rank_1e-6"
                ],

            "P0_erank_1e-8":
                poly_metrics[
                    "effective_rank_1e-8"
                ],

            "P0_erank_1e-10":
                poly_metrics[
                    "effective_rank_1e-10"
                ],

            "P0_erank_1e-12":
                poly_metrics[
                    "effective_rank_1e-12"
                ],

            "P0_condition":
                poly_metrics[
                    "condition_nonzero"
                ],

            "P0_nullity_lb":
                poly_metrics[
                    "parameter_nullity_lower_bound"
                ],
        })


C6_AUDIT = pd.DataFrame(
    C6_ROWS
)


# ================================================================
# 6. Main table: all six cycles
# ================================================================

C6_ALL_CYCLES = (
    C6_AUDIT[
        C6_AUDIT[
            "pooling"
        ]
        ==
        "all_cycles"
    ]
    .copy()
)


print("=" * 94)
print("RS-TSC PROTOTYPE — CELL 6")
print("LOCAL-PERTURBATION / BANDWIDTH IDENTIFIABILITY AUDIT")
print("=" * 94)


print()
print("-" * 94)
print("A. ALL SIX CYCLES: BANDWIDTH TRADE-OFF")
print("-" * 94)


display(
    C6_ALL_CYCLES[
        [
            "h",
            "n_states",
            "alpha_min",
            "alpha_max",
            "max_protocol_drift",

            "state_rank",
            "state_erank_1e-8",
            "state_sigma2_sigma1",

            "P2_rank",
            "P2_erank_1e-8",
            "P2_condition",

            "P0_rank",
            "P0_erank_1e-8",
            "P0_nullity_lb",
        ]
    ]
    .style
    .format({

        "h":
            "{:.6f}",

        "alpha_min":
            "{:.6f}",

        "alpha_max":
            "{:.6f}",

        "max_protocol_drift":
            "{:.6e}",

        "state_sigma2_sigma1":
            "{:.6e}",

        "P2_condition":
            "{:.6e}",
    })
)


# ================================================================
# 7. Compare local sampling vs repeated cycles
#
# Use the maximum tested bandwidth h=0.003.
# ================================================================

C6_MAX_H = float(
    np.max(
        C6_H_VALUES
    )
)


C6_POOL_COMPARISON = (
    C6_AUDIT[
        np.isclose(
            C6_AUDIT[
                "h"
            ],
            C6_MAX_H,
        )
    ]
    .copy()
)


print()
print("-" * 94)
print("B. DENSE LOCAL SAMPLING VS REPEATED-CYCLE STATE DIVERSITY")
print(f"   evaluated at h = {C6_MAX_H:.6f}")
print("-" * 94)


display(
    C6_POOL_COMPARISON[
        [
            "pooling",
            "n_cycles",
            "n_states",

            "state_rank",
            "state_erank_1e-8",

            "P2_rank",
            "P2_erank_1e-8",
            "P2_condition",
            "P2_nullity_lb",

            "P0_rank",
            "P0_erank_1e-8",
            "P0_nullity_lb",
        ]
    ]
    .style
    .format({

        "P2_condition":
            "{:.6e}",
    })
)


# ================================================================
# 8. Smallest nonzero bandwidth comparison
#
# This is the regime closest to the Method-C assumption:
#
#       "small start perturbation,
#        approximately unchanged protocol"
#
# Current native dt means the smallest nonzero half-bandwidth is
# 0.0005.
# ================================================================

C6_MIN_NONZERO_H = float(
    C6_H_VALUES[
        C6_H_VALUES > 0.0
    ][0]
)


C6_SMALL_H = (
    C6_AUDIT[
        np.isclose(
            C6_AUDIT[
                "h"
            ],
            C6_MIN_NONZERO_H,
        )
    ]
    .copy()
)


print()
print("-" * 94)
print("C. SMALLEST NONZERO LOCAL BANDWIDTH")
print(f"   h = {C6_MIN_NONZERO_H:.6f}")
print("-" * 94)


display(
    C6_SMALL_H[
        [
            "pooling",
            "n_states",
            "alpha_min",
            "alpha_max",
            "max_protocol_drift",

            "state_erank_1e-8",

            "P2_rank",
            "P2_erank_1e-8",
            "P2_condition",

            "P0_rank",
            "P0_erank_1e-8",
            "P0_nullity_lb",
        ]
    ]
    .style
    .format({

        "alpha_min":
            "{:.6f}",

        "alpha_max":
            "{:.6f}",

        "max_protocol_drift":
            "{:.6e}",

        "P2_condition":
            "{:.6e}",
    })
)


# ================================================================
# 9. Information-gain summary
#
# We are particularly interested in whether increasing h gives
# effective-rank gain before protocol drift becomes appreciable.
# ================================================================

print()
print("-" * 94)
print("D. ALL-CYCLE INFORMATION GAIN")
print("-" * 94)


for _, row in C6_ALL_CYCLES.iterrows():

    print(
        f"h={row['h']:.4f} | "
        f"M={int(row['n_states']):2d} | "
        f"protocol drift max={row['max_protocol_drift']:.3e} | "
        f"P2 rank={int(row['P2_rank']):2d}/84 "
        f"(erank1e-8={int(row['P2_erank_1e-8']):2d}) | "
        f"P0 rank={int(row['P0_rank']):2d}/164 "
        f"(erank1e-8={int(row['P0_erank_1e-8']):2d})"
    )


# ================================================================
# 10. Save
# ================================================================

C6_AUDIT.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell6_local_bandwidth_identifiability_audit.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell6_local_bandwidth_identifiability_audit.npz",

    h_values=
        C6_H_VALUES,

    broad_polynomial_exponents=
        np.asarray(
            C6_POLY_EXPONENTS,
            dtype=int,
        ),
)


print()
print("=" * 94)
print("Cell 6 audit completed.")
print("=" * 94)

RS-TSC PROTOTYPE — CELL 6
LOCAL-PERTURBATION / BANDWIDTH IDENTIFIABILITY AUDIT

----------------------------------------------------------------------------------------------
A. ALL SIX CYCLES: BANDWIDTH TRADE-OFF
----------------------------------------------------------------------------------------------


,h,n_states,alpha_min,alpha_max,max_protocol_drift,state_rank,state_erank_1e-8,state_sigma2_sigma1,P2_rank,P2_erank_1e-8,P2_condition,P0_rank,P0_erank_1e-8,P0_nullity_lb
14,0.000000,6,0.500000,0.500000,0.000000e+00,5,5,8.502771e-02,42,42,2.405572e+05,6,6,158
15,0.000500,18,0.483333,0.516667,2.746290e-02,7,7,8.502807e-02,84,79,5.621136e+08,18,14,146
16,0.001000,30,0.466667,0.533333,5.493842e-02,7,7,8.502879e-02,84,82,2.200595e+08,23,15,141
17,0.001500,42,0.450000,0.550000,8.242656e-02,7,7,8.502988e-02,84,83,1.248340e+08,25,15,139
18,0.002000,54,0.433333,0.566667,1.099273e-01,7,7,8.503132e-02,84,84,8.185424e+07,26,16,138
19,0.002500,66,0.416667,0.583333,1.374406e-01,7,7,8.503313e-02,84,84,5.827374e+07,26,16,138
20,0.003000,78,0.400000,0.600000,1.649664e-01,7,7,8.503529e-02,84,84,4.387773e+07,27,17,137



----------------------------------------------------------------------------------------------
B. DENSE LOCAL SAMPLING VS REPEATED-CYCLE STATE DIVERSITY
   evaluated at h = 0.003000
----------------------------------------------------------------------------------------------


,pooling,n_cycles,n_states,state_rank,state_erank_1e-8,P2_rank,P2_erank_1e-8,P2_condition,P2_nullity_lb,P0_rank,P0_erank_1e-8,P0_nullity_lb
6,one_cycle,1,13,8,3,35,21,4.820174e+12,49,5,4,159
13,fit_cycles,4,52,7,7,84,73,2.795768e+10,0,19,13,145
20,all_cycles,6,78,7,7,84,84,4.387773e+07,0,27,17,137



----------------------------------------------------------------------------------------------
C. SMALLEST NONZERO LOCAL BANDWIDTH
   h = 0.000500
----------------------------------------------------------------------------------------------


,pooling,n_states,alpha_min,alpha_max,max_protocol_drift,state_erank_1e-8,P2_rank,P2_erank_1e-8,P2_condition,P0_rank,P0_erank_1e-8,P0_nullity_lb
1,one_cycle,3,0.483333,0.516667,1.305170e-02,2,21,21,2.011897e+07,3,3,161
8,fit_cycles,12,0.483333,0.516667,1.995571e-02,7,84,62,1.932671e+12,12,10,152
15,all_cycles,18,0.483333,0.516667,2.746290e-02,7,84,79,5.621136e+08,18,14,146



----------------------------------------------------------------------------------------------
D. ALL-CYCLE INFORMATION GAIN
----------------------------------------------------------------------------------------------
h=0.0000 | M= 6 | protocol drift max=0.000e+00 | P2 rank=42/84 (erank1e-8=42) | P0 rank= 6/164 (erank1e-8= 6)
h=0.0005 | M=18 | protocol drift max=2.746e-02 | P2 rank=84/84 (erank1e-8=79) | P0 rank=18/164 (erank1e-8=14)
h=0.0010 | M=30 | protocol drift max=5.494e-02 | P2 rank=84/84 (erank1e-8=82) | P0 rank=23/164 (erank1e-8=15)
h=0.0015 | M=42 | protocol drift max=8.243e-02 | P2 rank=84/84 (erank1e-8=83) | P0 rank=25/164 (erank1e-8=15)
h=0.0020 | M=54 | protocol drift max=1.099e-01 | P2 rank=84/84 (erank1e-8=84) | P0 rank=26/164 (erank1e-8=16)
h=0.0025 | M=66 | protocol drift max=1.374e-01 | P2 rank=84/84 (erank1e-8=84) | P0 rank=26/164 (erank1e-8=16)
h=0.0030 | M=78 | protocol drift max=1.650e-01 | P2 rank=84/84 (erank1e-8=84) | P0 rank=27/164 (erank1e-8=17)

Cell 6 a

In [17]:
# ================================================================
# RS-TSC prototype
# Cell 7: topology-agnostic first-closure dictionary audit
#
# PURPOSE
# ---------------------------------------------------------------
# Construct the first effective closure
#
#     D_eff^(1) = D_micro U [D_micro, D_micro]
#
# WITHOUT using the true microscopic topology.
#
# Assumptions allowed:
#
#   - microscopic interaction FUNCTIONS / TYPES are known
#   - all possible pair / triad supports are allowed
#   - actual nonzero topology is unknown
#
#
# We:
#
#   1. Treat all 84 P2 physical candidates as admissible
#      microscopic interaction channels.
#
#   2. Generate every potentially nonzero chronological bracket
#
#          [Psi_b, Psi_a]
#          = D Psi_b Psi_a - D Psi_a Psi_b
#
#      for overlapping supports.
#
#   3. Remove numerically zero brackets.
#
#   4. Compress algebraically redundant bracket directions on
#      generic state-space probes.
#
#   5. Preserve all 84 microscopic channels, then add genuinely
#      new bracket directions in increasing support order.
#
#   6. Export a reusable closure basis for later direct
#      finite-window inference.
#
#
# NOTE
# ---------------------------------------------------------------
# Generic probes are NOT trajectory observations.
# They are used only to determine algebraic independence of the
# candidate vector fields themselves.
#
# No F_eff inference is performed in this cell.
# ================================================================

import numpy as np
import pandas as pd
from scipy.linalg import qr


# ================================================================
# 0. Configuration
# ================================================================

C7_SEED = 20260825

# Degree <= 3 scalar polynomial space has 164 monomials.
# 192 generic states -> 1536 vector-valued rows, safely above
# the full conservative cubic ambient dimension 7*164 = 1148.
C7_N_PROBE = 192

C7_ZERO_REL_TOL = 1e-12
C7_RANK_REL_TOL = 1e-10

C7_Q_MICRO = len(RS_G1_BETA)

assert C7_Q_MICRO == 84
assert RS_N == 8


# ================================================================
# 1. Generic state-space probes
#
# These are deliberately NOT restricted to the observed orbit or
# the conserved-mass manifold.
#
# The goal here is intrinsic functional independence.
# ================================================================

_rng_c7 = np.random.default_rng(C7_SEED)

C7_X_PROBE = _rng_c7.uniform(
    low=-0.8,
    high=1.2,
    size=(C7_N_PROBE, RS_N),
)


# ================================================================
# 2. Evaluate every microscopic basis field and Jacobian
#
# F_basis[m, i, q]     = Psi_q(x_m)_i
# J_basis[m, i, j, q]  = d Psi_q,i / d x_j
# ================================================================

C7_F_BASIS = np.empty(
    (
        C7_N_PROBE,
        RS_N,
        C7_Q_MICRO,
    ),
    dtype=float,
)

C7_J_BASIS = np.empty(
    (
        C7_N_PROBE,
        RS_N,
        RS_N,
        C7_Q_MICRO,
    ),
    dtype=float,
)


for q in range(C7_Q_MICRO):

    beta_q = np.zeros(
        C7_Q_MICRO,
        dtype=float,
    )

    beta_q[q] = 1.0

    for m, x in enumerate(C7_X_PROBE):

        C7_F_BASIS[m, :, q] = (
            rs_field_from_beta(
                x,
                beta_q,
            )
        )

        C7_J_BASIS[m, :, :, q] = (
            rs_physical_jacobian(
                x,
                beta_q,
            )
        )


# Flatten vector-valued evaluations:
#
#     (n_probe * N) x 84
#
C7_A_MICRO = C7_F_BASIS.reshape(
    C7_N_PROBE * RS_N,
    C7_Q_MICRO,
)


# ================================================================
# 3. Infer structural support of each microscopic basis function
#
# No assumption about ordering of the 84 candidates is required.
#
# A node is included if it appears either:
#
#   - as an active output component, or
#   - as an input variable through the Jacobian.
# ================================================================

C7_MICRO_SUPPORTS = []


for q in range(C7_Q_MICRO):

    Fq = C7_F_BASIS[:, :, q]
    Jq = C7_J_BASIS[:, :, :, q]

    scale_q = max(
        float(np.max(np.abs(Fq))),
        float(np.max(np.abs(Jq))),
        1.0,
    )

    tol_q = 1e-11 * scale_q

    active_output = (
        np.max(
            np.abs(Fq),
            axis=0,
        )
        >
        tol_q
    )

    active_input = (
        np.max(
            np.abs(Jq),
            axis=(0, 1),
        )
        >
        tol_q
    )

    support = tuple(
        np.flatnonzero(
            active_output
            |
            active_input
        )
        .astype(int)
        .tolist()
    )

    C7_MICRO_SUPPORTS.append(
        support
    )


C7_MICRO_SUPPORT_SIZES = np.asarray(
    [
        len(s)
        for s in C7_MICRO_SUPPORTS
    ],
    dtype=int,
)


# ================================================================
# 4. Check microscopic basis structure
# ================================================================

C7_MICRO_SUPPORT_COUNTS = (
    pd.Series(
        C7_MICRO_SUPPORT_SIZES
    )
    .value_counts()
    .sort_index()
)


# ================================================================
# 5. Intrinsic microscopic rank
#
# Column normalization before rank tests.
# ================================================================

def c7_normalize_columns(A):

    A = np.asarray(
        A,
        dtype=float,
    )

    norms = np.linalg.norm(
        A,
        axis=0,
    )

    valid = (
        norms
        >
        np.finfo(float).tiny
    )

    A_norm = np.zeros_like(
        A,
        dtype=float,
    )

    A_norm[:, valid] = (
        A[:, valid]
        /
        norms[valid][None, :]
    )

    return (
        A_norm,
        norms,
        valid,
    )


C7_A_MICRO_NORM, _, _ = (
    c7_normalize_columns(
        C7_A_MICRO
    )
)


U_micro, S_micro, _ = np.linalg.svd(
    C7_A_MICRO_NORM,
    full_matrices=False,
)


C7_MICRO_RANK = int(
    np.sum(
        S_micro
        >
        S_micro[0]
        *
        C7_RANK_REL_TOL
    )
)


C7_Q_CURRENT = (
    U_micro[
        :,
        :C7_MICRO_RANK
    ]
)


# ================================================================
# 6. Enumerate all potentially nonzero first brackets
#
# If two structural supports are disjoint, their Lie bracket is
# identically zero and does not need to be evaluated.
#
# IMPORTANT:
# This uses ALL admissible microscopic candidates, not true edges.
# ================================================================

C7_BRACKET_PARENT_A = []
C7_BRACKET_PARENT_B = []
C7_BRACKET_SUPPORTS = []
C7_BRACKET_PARENT_TYPES = []


for a in range(C7_Q_MICRO):

    support_a = set(
        C7_MICRO_SUPPORTS[a]
    )

    for b in range(
        a + 1,
        C7_Q_MICRO,
    ):

        support_b = set(
            C7_MICRO_SUPPORTS[b]
        )

        if support_a.isdisjoint(
            support_b
        ):
            continue

        union_support = tuple(
            sorted(
                support_a
                |
                support_b
            )
        )

        size_a = len(
            support_a
        )

        size_b = len(
            support_b
        )

        if (
            size_a == 2
            and
            size_b == 2
        ):
            parent_type = "pair-pair"

        elif (
            {size_a, size_b}
            ==
            {2, 3}
        ):
            parent_type = "pair-triad"

        elif (
            size_a == 3
            and
            size_b == 3
        ):
            parent_type = "triad-triad"

        else:
            parent_type = (
                f"{size_a}-{size_b}"
            )

        C7_BRACKET_PARENT_A.append(
            a
        )

        C7_BRACKET_PARENT_B.append(
            b
        )

        C7_BRACKET_SUPPORTS.append(
            union_support
        )

        C7_BRACKET_PARENT_TYPES.append(
            parent_type
        )


C7_BRACKET_PARENT_A = np.asarray(
    C7_BRACKET_PARENT_A,
    dtype=int,
)

C7_BRACKET_PARENT_B = np.asarray(
    C7_BRACKET_PARENT_B,
    dtype=int,
)

C7_BRACKET_SUPPORT_SIZES = np.asarray(
    [
        len(s)
        for s in C7_BRACKET_SUPPORTS
    ],
    dtype=int,
)

C7_N_BRACKET_RAW = len(
    C7_BRACKET_PARENT_A
)


# ================================================================
# 7. Evaluate all chronological brackets on generic probes
#
#     B_{a->b}
#       = D Psi_b Psi_a - D Psi_a Psi_b
#
# Chunking avoids building a very large Jacobian tensor at once.
# ================================================================

C7_A_BRACKET = np.empty(
    (
        C7_N_PROBE * RS_N,
        C7_N_BRACKET_RAW,
    ),
    dtype=float,
)


C7_CHUNK = 128


for lo in range(
    0,
    C7_N_BRACKET_RAW,
    C7_CHUNK,
):

    hi = min(
        lo + C7_CHUNK,
        C7_N_BRACKET_RAW,
    )

    a_idx = (
        C7_BRACKET_PARENT_A[
            lo:hi
        ]
    )

    b_idx = (
        C7_BRACKET_PARENT_B[
            lo:hi
        ]
    )

    # Shapes:
    #
    # J_a, J_b : (M, N, N, K)
    # F_a, F_b : (M, N, K)

    J_a = (
        C7_J_BASIS[
            :,
            :,
            :,
            a_idx
        ]
    )

    J_b = (
        C7_J_BASIS[
            :,
            :,
            :,
            b_idx
        ]
    )

    F_a = (
        C7_F_BASIS[
            :,
            :,
            a_idx
        ]
    )

    F_b = (
        C7_F_BASIS[
            :,
            :,
            b_idx
        ]
    )

    term_ba = np.einsum(
        "mijk,mjk->mik",
        J_b,
        F_a,
        optimize=True,
    )

    term_ab = np.einsum(
        "mijk,mjk->mik",
        J_a,
        F_b,
        optimize=True,
    )

    bracket_chunk = (
        term_ba
        -
        term_ab
    )

    C7_A_BRACKET[
        :,
        lo:hi
    ] = bracket_chunk.reshape(
        C7_N_PROBE * RS_N,
        hi - lo,
    )


# ================================================================
# 8. Remove numerically zero brackets
# ================================================================

C7_BRACKET_NORMS = np.linalg.norm(
    C7_A_BRACKET,
    axis=0,
)


C7_MAX_BRACKET_NORM = float(
    np.max(
        C7_BRACKET_NORMS
    )
)


C7_NONZERO_MASK = (
    C7_BRACKET_NORMS
    >
    C7_ZERO_REL_TOL
    *
    C7_MAX_BRACKET_NORM
)


C7_N_BRACKET_NONZERO = int(
    np.sum(
        C7_NONZERO_MASK
    )
)


C7_A_BRACKET_NZ = (
    C7_A_BRACKET[
        :,
        C7_NONZERO_MASK
    ]
)


C7_BRACKET_PARENT_A_NZ = (
    C7_BRACKET_PARENT_A[
        C7_NONZERO_MASK
    ]
)

C7_BRACKET_PARENT_B_NZ = (
    C7_BRACKET_PARENT_B[
        C7_NONZERO_MASK
    ]
)

C7_BRACKET_SUPPORT_SIZES_NZ = (
    C7_BRACKET_SUPPORT_SIZES[
        C7_NONZERO_MASK
    ]
)

C7_BRACKET_SUPPORTS_NZ = [
    C7_BRACKET_SUPPORTS[i]

    for i in np.flatnonzero(
        C7_NONZERO_MASK
    )
]

C7_BRACKET_PARENT_TYPES_NZ = [
    C7_BRACKET_PARENT_TYPES[i]

    for i in np.flatnonzero(
        C7_NONZERO_MASK
    )
]


# ================================================================
# 9. Hierarchical algebraic compression
#
# Preserve all microscopic directions first.
#
# Then, for support order k = 2,3,4,5,...:
#
#   - project bracket candidates onto orthogonal complement of
#     the already accepted closure space
#
#   - pivoted QR finds genuinely new independent directions
#
# This gives a topology-agnostic but structurally interpretable
# first-closure basis.
# ================================================================

C7_SELECTED_NZ_INDICES = []
C7_ADDED_BY_ORDER = {}

C7_Q_SPAN = C7_Q_CURRENT.copy()


for support_order in sorted(
    np.unique(
        C7_BRACKET_SUPPORT_SIZES_NZ
    )
):

    local_idx = np.flatnonzero(
        C7_BRACKET_SUPPORT_SIZES_NZ
        ==
        support_order
    )

    B = (
        C7_A_BRACKET_NZ[
            :,
            local_idx
        ]
    )

    B_norm, _, valid = (
        c7_normalize_columns(
            B
        )
    )

    local_idx = (
        local_idx[
            valid
        ]
    )

    B_norm = (
        B_norm[
            :,
            valid
        ]
    )


    # Remove everything already generated by lower-order/current
    # closure directions.

    residual = (
        B_norm
        -
        C7_Q_SPAN
        @
        (
            C7_Q_SPAN.T
            @
            B_norm
        )
    )


    residual_norms = np.linalg.norm(
        residual,
        axis=0,
    )


    independent_candidate_mask = (
        residual_norms
        >
        C7_RANK_REL_TOL
    )


    if not np.any(
        independent_candidate_mask
    ):

        C7_ADDED_BY_ORDER[
            int(support_order)
        ] = 0

        continue


    residual = (
        residual[
            :,
            independent_candidate_mask
        ]
    )

    local_idx = (
        local_idx[
            independent_candidate_mask
        ]
    )


    # Normalize projected residual columns before rank-revealing QR.

    residual = (
        residual
        /
        np.linalg.norm(
            residual,
            axis=0,
        )[None, :]
    )


    Qk, Rk, pivk = qr(
        residual,
        mode="economic",
        pivoting=True,
    )


    diag_Rk = np.abs(
        np.diag(
            Rk
        )
    )


    if len(
        diag_Rk
    ) == 0:

        rank_k = 0

    else:

        rank_k = int(
            np.sum(
                diag_Rk
                >
                diag_Rk[0]
                *
                C7_RANK_REL_TOL
            )
        )


    selected_local = (
        local_idx[
            pivk[
                :rank_k
            ]
        ]
    )


    C7_SELECTED_NZ_INDICES.extend(
        selected_local.tolist()
    )

    C7_ADDED_BY_ORDER[
        int(support_order)
    ] = int(
        rank_k
    )


    if rank_k > 0:

        C7_Q_SPAN = np.column_stack(
            [
                C7_Q_SPAN,
                Qk[
                    :,
                    :rank_k
                ],
            ]
        )

        # Small numerical re-orthogonalization.
        C7_Q_SPAN, _ = np.linalg.qr(
            C7_Q_SPAN,
            mode="reduced",
        )


C7_SELECTED_NZ_INDICES = np.asarray(
    C7_SELECTED_NZ_INDICES,
    dtype=int,
)


C7_N_BRACKET_INDEPENDENT = len(
    C7_SELECTED_NZ_INDICES
)


C7_CLOSURE_DIM = (
    C7_MICRO_RANK
    +
    C7_N_BRACKET_INDEPENDENT
)


# ================================================================
# 10. Selected bracket metadata
# ================================================================

C7_SELECTED_PARENT_A = (
    C7_BRACKET_PARENT_A_NZ[
        C7_SELECTED_NZ_INDICES
    ]
)

C7_SELECTED_PARENT_B = (
    C7_BRACKET_PARENT_B_NZ[
        C7_SELECTED_NZ_INDICES
    ]
)

C7_SELECTED_SUPPORT_SIZES = (
    C7_BRACKET_SUPPORT_SIZES_NZ[
        C7_SELECTED_NZ_INDICES
    ]
)

C7_SELECTED_SUPPORTS = [
    C7_BRACKET_SUPPORTS_NZ[i]

    for i in C7_SELECTED_NZ_INDICES
]

C7_SELECTED_PARENT_TYPES = [
    C7_BRACKET_PARENT_TYPES_NZ[i]

    for i in C7_SELECTED_NZ_INDICES
]


C7_SELECTED_BRACKET_META = pd.DataFrame({

    "closure_index":
        np.arange(
            C7_Q_MICRO,
            C7_Q_MICRO
            +
            C7_N_BRACKET_INDEPENDENT,
        ),

    "parent_a":
        C7_SELECTED_PARENT_A,

    "parent_b":
        C7_SELECTED_PARENT_B,

    "parent_type":
        C7_SELECTED_PARENT_TYPES,

    "support_order":
        C7_SELECTED_SUPPORT_SIZES,

    "support":
        [
            tuple(
                node + 1
                for node in support
            )

            for support in
            C7_SELECTED_SUPPORTS
        ],
})


# ================================================================
# 11. Full reusable basis metadata
# ================================================================

C7_MICRO_META = pd.DataFrame({

    "closure_index":
        np.arange(
            C7_Q_MICRO
        ),

    "kind":
        "microscopic",

    "parent_a":
        -1,

    "parent_b":
        -1,

    "parent_type":
        [
            (
                "pair"
                if len(s) == 2
                else
                "triad"
                if len(s) == 3
                else
                f"order-{len(s)}"
            )

            for s in
            C7_MICRO_SUPPORTS
        ],

    "support_order":
        C7_MICRO_SUPPORT_SIZES,

    "support":
        [
            tuple(
                node + 1
                for node in support
            )

            for support in
            C7_MICRO_SUPPORTS
        ],
})


C7_BRACKET_META_FOR_FULL = (
    C7_SELECTED_BRACKET_META.copy()
)

C7_BRACKET_META_FOR_FULL[
    "kind"
] = "first-bracket"


RS_CLOSURE1_BASIS_META = pd.concat(
    [
        C7_MICRO_META,
        C7_BRACKET_META_FOR_FULL[
            C7_MICRO_META.columns
        ],
    ],
    ignore_index=True,
)


# ================================================================
# 12. Reusable closure-basis evaluator
#
# Returns:
#
#       B(x) : N x Q_closure
#
# so that
#
#       F_eff(x) = B(x) @ theta
#
# in later direct finite-window inference.
# ================================================================

RS_CLOSURE1_SELECTED_BRACKETS = tuple(
    zip(
        C7_SELECTED_PARENT_A.tolist(),
        C7_SELECTED_PARENT_B.tolist(),
    )
)


def rs_closure1_basis_at_x(
    x,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    F_micro = np.empty(
        (
            RS_N,
            C7_Q_MICRO,
        ),
        dtype=float,
    )

    J_micro = np.empty(
        (
            RS_N,
            RS_N,
            C7_Q_MICRO,
        ),
        dtype=float,
    )


    for q in range(
        C7_Q_MICRO
    ):

        beta_q = np.zeros(
            C7_Q_MICRO,
            dtype=float,
        )

        beta_q[q] = 1.0

        F_micro[
            :,
            q
        ] = rs_field_from_beta(
            x,
            beta_q,
        )

        J_micro[
            :,
            :,
            q
        ] = rs_physical_jacobian(
            x,
            beta_q,
        )


    B = np.empty(
        (
            RS_N,
            C7_CLOSURE_DIM,
        ),
        dtype=float,
    )

    B[
        :,
        :C7_Q_MICRO
    ] = F_micro


    for k, (
        a,
        b,
    ) in enumerate(
        RS_CLOSURE1_SELECTED_BRACKETS
    ):

        B[
            :,
            C7_Q_MICRO + k
        ] = (
            J_micro[
                :,
                :,
                b
            ]
            @
            F_micro[
                :,
                a
            ]
            -
            J_micro[
                :,
                :,
                a
            ]
            @
            F_micro[
                :,
                b
            ]
        )


    return B


RS_CLOSURE1_DIM = int(
    C7_CLOSURE_DIM
)


# ================================================================
# 13. Diagnostics
# ================================================================

print("=" * 96)
print("RS-TSC PROTOTYPE — CELL 7")
print("TOPOLOGY-AGNOSTIC FIRST-CLOSURE DICTIONARY")
print("=" * 96)


print()
print("-" * 96)
print("A. MICROSCOPIC ADMISSIBLE DICTIONARY")
print("-" * 96)

print(
    f"microscopic candidates          : "
    f"{C7_Q_MICRO}"
)

print(
    f"intrinsic microscopic rank      : "
    f"{C7_MICRO_RANK}"
)

print(
    "support-order counts:"
)

for order, count in (
    C7_MICRO_SUPPORT_COUNTS.items()
):

    print(
        f"  order {int(order)} : "
        f"{int(count)}"
    )


print()
print("-" * 96)
print("B. RAW FIRST-BRACKET GENERATION")
print("-" * 96)

print(
    f"overlapping parent pairs tested : "
    f"{C7_N_BRACKET_RAW}"
)

print(
    f"nonzero brackets                : "
    f"{C7_N_BRACKET_NONZERO}"
)

print(
    f"numerically zero brackets       : "
    f"{C7_N_BRACKET_RAW - C7_N_BRACKET_NONZERO}"
)


C7_RAW_BRACKET_SUMMARY = (
    pd.DataFrame({

        "parent_type":
            C7_BRACKET_PARENT_TYPES_NZ,

        "support_order":
            C7_BRACKET_SUPPORT_SIZES_NZ,

    })
    .value_counts()
    .rename(
        "n_nonzero_candidates"
    )
    .reset_index()
    .sort_values(
        [
            "support_order",
            "parent_type",
        ]
    )
)


display(
    C7_RAW_BRACKET_SUMMARY
)


print()
print("-" * 96)
print("C. ALGEBRAIC CLOSURE COMPRESSION")
print("-" * 96)

print(
    f"microscopic dimensions retained : "
    f"{C7_MICRO_RANK}"
)

for order in sorted(
    C7_ADDED_BY_ORDER
):

    print(
        f"new bracket dimensions "
        f"(support <=/at order {order}) : "
        f"{C7_ADDED_BY_ORDER[order]}"
    )

print()

print(
    f"independent bracket additions   : "
    f"{C7_N_BRACKET_INDEPENDENT}"
)

print(
    f"TOTAL first-closure dimension   : "
    f"{C7_CLOSURE_DIM}"
)


print()
print(
    f"generic degree<=3 vector field  : "
    f"{RS_N * 164} coefficients"
)

print(
    f"generic conservative ambient    : "
    f"{(RS_N - 1) * 164} dimensions"
)


print()
print("-" * 96)
print("D. SELECTED CLOSURE BASIS BY STRUCTURAL ORDER")
print("-" * 96)


C7_SELECTED_SUMMARY = (
    RS_CLOSURE1_BASIS_META
    .groupby(
        [
            "kind",
            "support_order",
        ]
    )
    .size()
    .rename(
        "n_basis_channels"
    )
    .reset_index()
)


display(
    C7_SELECTED_SUMMARY
)


print()
print("-" * 96)
print("E. FIRST 30 SELECTED BRACKET CHANNELS")
print("-" * 96)


display(
    C7_SELECTED_BRACKET_META.head(
        30
    )
)


# ================================================================
# 14. Sanity check:
#     evaluator dimension + generic closure rank
# ================================================================

C7_TEST_BASIS = (
    rs_closure1_basis_at_x(
        C7_X_PROBE[0]
    )
)

assert C7_TEST_BASIS.shape == (
    RS_N,
    C7_CLOSURE_DIM,
)


print()
print("-" * 96)
print("F. SANITY CHECK")
print("-" * 96)

print(
    "closure basis at one state     : "
    f"{C7_TEST_BASIS.shape}"
)

print(
    "topology information used      : "
    "NONE"
)

print(
    "interaction-class information  : "
    "YES"
)


# ================================================================
# 15. Save
# ================================================================

RS_CLOSURE1_BASIS_META.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell7_closure1_basis_metadata.csv",
    index=False,
)


C7_SELECTED_BRACKET_META.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell7_selected_first_brackets.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell7_closure1_dictionary.npz",

    closure_dimension=
        np.asarray(
            [C7_CLOSURE_DIM],
            dtype=int,
        ),

    microscopic_dimension=
        np.asarray(
            [C7_MICRO_RANK],
            dtype=int,
        ),

    selected_parent_a=
        C7_SELECTED_PARENT_A,

    selected_parent_b=
        C7_SELECTED_PARENT_B,

    selected_support_order=
        C7_SELECTED_SUPPORT_SIZES,
)


print()
print("=" * 96)
print("Cell 7 closure audit completed.")
print("=" * 96)

RS-TSC PROTOTYPE — CELL 7
TOPOLOGY-AGNOSTIC FIRST-CLOSURE DICTIONARY

------------------------------------------------------------------------------------------------
A. MICROSCOPIC ADMISSIBLE DICTIONARY
------------------------------------------------------------------------------------------------
microscopic candidates          : 84
intrinsic microscopic rank      : 84
support-order counts:
  order 2 : 28
  order 3 : 56

------------------------------------------------------------------------------------------------
B. RAW FIRST-BRACKET GENERATION
------------------------------------------------------------------------------------------------
overlapping parent pairs tested : 2436
nonzero brackets                : 2436
numerically zero brackets       : 0


,parent_type,support_order,n_nonzero_candidates
3,pair-pair,3,168
4,pair-triad,3,168
0,pair-triad,4,840
2,triad-triad,4,420
1,triad-triad,5,840



------------------------------------------------------------------------------------------------
C. ALGEBRAIC CLOSURE COMPRESSION
------------------------------------------------------------------------------------------------
microscopic dimensions retained : 84
new bracket dimensions (support <=/at order 3) : 336
new bracket dimensions (support <=/at order 4) : 566
new bracket dimensions (support <=/at order 5) : 21

independent bracket additions   : 923
TOTAL first-closure dimension   : 1007

generic degree<=3 vector field  : 1312 coefficients
generic conservative ambient    : 1148 dimensions

------------------------------------------------------------------------------------------------
D. SELECTED CLOSURE BASIS BY STRUCTURAL ORDER
------------------------------------------------------------------------------------------------


,kind,support_order,n_basis_channels
0,first-bracket,3,336
1,first-bracket,4,566
2,first-bracket,5,21
3,microscopic,2,28
4,microscopic,3,56



------------------------------------------------------------------------------------------------
E. FIRST 30 SELECTED BRACKET CHANNELS
------------------------------------------------------------------------------------------------


,closure_index,parent_a,parent_b,parent_type,support_order,support
0,84,0,30,pair-triad,3,"(1, 2, 5)"
1,85,13,16,pair-pair,3,"(3, 4, 7)"
2,86,2,8,pair-pair,3,"(1, 2, 4)"
3,87,25,27,pair-pair,3,"(6, 7, 8)"
4,88,26,72,pair-triad,3,"(3, 6, 8)"
5,89,9,23,pair-pair,3,"(2, 5, 7)"
6,90,18,24,pair-pair,3,"(4, 5, 8)"
7,91,7,12,pair-pair,3,"(2, 3, 8)"
8,92,1,15,pair-pair,3,"(1, 3, 6)"
9,93,11,56,pair-triad,3,"(2, 4, 7)"



------------------------------------------------------------------------------------------------
F. SANITY CHECK
------------------------------------------------------------------------------------------------
closure basis at one state     : (8, 1007)
topology information used      : NONE
interaction-class information  : YES

Cell 7 closure audit completed.


In [18]:
# ================================================================
# RS-TSC prototype
# Cell 8: Case-2 low-rank Lie-sector / support-reachability audit
#
# PURPOSE
# ---------------------------------------------------------------
# For the two-stage Case-2 protocol
#
#       G1 -> G2
#
# test the TSC structural statement
#
#   [G2,G1]
#     = sum_{a<b} A_ab [Psi_a,Psi_b]
#
# with
#
#   A_ab = beta1_a beta2_b - beta2_a beta1_b
#
# so that the antisymmetric coefficient matrix
#
#   A = beta1 beta2^T - beta2 beta1^T
#
# must satisfy
#
#   rank(A) <= 2.
#
#
# This cell also audits:
#
#   - how sparse the actually activated raw Lie channels are;
#   - which structural orders are reachable at first Lie depth;
#   - exact reconstruction of [G2,G1] from the low-rank tensor.
#
#
# IMPORTANT:
# This is a mechanism / oracle diagnostic.
# It is NOT yet the blind finite-window inference.
# ================================================================

import numpy as np
import pandas as pd


# ================================================================
# 0. Coefficient vectors
#
# Use the already reconstructed microscopic generators.
# Their reconstruction errors are negligible compared with the
# BCH1 truncation error, so they are appropriate for this audit.
# ================================================================

C8_BETA1 = np.asarray(
    RS_G1_BETA,
    dtype=float,
)

C8_BETA2 = np.asarray(
    RS_G2_BETA,
    dtype=float,
)

C8_Q = len(C8_BETA1)

assert C8_Q == 84


# ================================================================
# 1. Antisymmetric temporal coefficient tensor
#
# Our raw basis convention from Cell 7 is
#
#   B_ab = D Psi_b Psi_a - D Psi_a Psi_b
#          for a < b.
#
# Therefore
#
#   [G2,G1]
#     = sum_{a<b}
#         (beta1_a beta2_b - beta2_a beta1_b)
#         B_ab.
# ================================================================

C8_A = (
    np.outer(
        C8_BETA1,
        C8_BETA2,
    )
    -
    np.outer(
        C8_BETA2,
        C8_BETA1,
    )
)


# Exact antisymmetry check
C8_ANTISYM_ERROR = (
    np.linalg.norm(
        C8_A + C8_A.T
    )
    /
    max(
        np.linalg.norm(C8_A),
        np.finfo(float).tiny,
    )
)


# ================================================================
# 2. Singular spectrum / numerical rank
# ================================================================

C8_SINGULAR_VALUES = np.linalg.svd(
    C8_A,
    compute_uv=False,
)

C8_SINGULAR_REL = (
    C8_SINGULAR_VALUES
    /
    max(
        C8_SINGULAR_VALUES[0],
        np.finfo(float).tiny,
    )
)

C8_RANK_1E8 = int(
    np.sum(
        C8_SINGULAR_REL > 1e-8
    )
)

C8_RANK_1E10 = int(
    np.sum(
        C8_SINGULAR_REL > 1e-10
    )
)

C8_RANK_1E12 = int(
    np.sum(
        C8_SINGULAR_REL > 1e-12
    )
)


# ================================================================
# 3. BCH temporal prefactor at the Case-2 center
#
# a = b = 0.015
# DeltaT = 0.030
#
# c = ab / (2 DeltaT) = 0.00375
# ================================================================

C8_ALPHA = 0.5

C8_TEMPORAL_PREFACTOR = (
    0.5
    *
    C2_DELTA_T
    *
    C8_ALPHA
    *
    (1.0 - C8_ALPHA)
)

assert np.isclose(
    C8_TEMPORAL_PREFACTOR,
    0.00375,
)


C8_A_BCH = (
    C8_TEMPORAL_PREFACTOR
    *
    C8_A
)


# ================================================================
# 4. Map Cell-7 raw bracket candidates to actual coefficients
#
# Cell 7 contains every overlapping pair a<b.
# ================================================================

C8_RAW_COEFF = np.empty(
    C7_N_BRACKET_RAW,
    dtype=float,
)


for k, (a, b) in enumerate(
    zip(
        C7_BRACKET_PARENT_A,
        C7_BRACKET_PARENT_B,
    )
):

    C8_RAW_COEFF[k] = (
        C8_A[
            a,
            b
        ]
    )


C8_RAW_BCH_COEFF = (
    C8_TEMPORAL_PREFACTOR
    *
    C8_RAW_COEFF
)


# ================================================================
# 5. Define active temporal channels
#
# Relative threshold only for reporting structural sparsity.
# No threshold is used in the field-equivalence test below.
# ================================================================

C8_COEFF_SCALE = max(
    float(
        np.max(
            np.abs(
                C8_RAW_COEFF
            )
        )
    ),
    np.finfo(float).tiny,
)

C8_ACTIVE_TOL = (
    1e-8
    *
    C8_COEFF_SCALE
)

C8_ACTIVE_MASK = (
    np.abs(
        C8_RAW_COEFF
    )
    >
    C8_ACTIVE_TOL
)

C8_N_ACTIVE_RAW = int(
    np.sum(
        C8_ACTIVE_MASK
    )
)


# ================================================================
# 6. Metadata for activated raw Lie channels
# ================================================================

C8_ACTIVE_META = pd.DataFrame({

    "parent_a":
        C7_BRACKET_PARENT_A[
            C8_ACTIVE_MASK
        ],

    "parent_b":
        C7_BRACKET_PARENT_B[
            C8_ACTIVE_MASK
        ],

    "parent_type":
        np.asarray(
            C7_BRACKET_PARENT_TYPES,
            dtype=object,
        )[
            C8_ACTIVE_MASK
        ],

    "support_order":
        C7_BRACKET_SUPPORT_SIZES[
            C8_ACTIVE_MASK
        ],

    "support":
        [
            tuple(
                node + 1
                for node in
                C7_BRACKET_SUPPORTS[i]
            )

            for i in np.flatnonzero(
                C8_ACTIVE_MASK
            )
        ],

    "lie_coefficient":
        C8_RAW_COEFF[
            C8_ACTIVE_MASK
        ],

    "bch1_coefficient":
        C8_RAW_BCH_COEFF[
            C8_ACTIVE_MASK
        ],
})


C8_ACTIVE_SUMMARY = (
    C8_ACTIVE_META
    .groupby(
        [
            "parent_type",
            "support_order",
        ]
    )
    .size()
    .rename(
        "n_active_channels"
    )
    .reset_index()
    .sort_values(
        [
            "support_order",
            "parent_type",
        ]
    )
)


# ================================================================
# 7. Count active microscopic channels in G1 and G2
#
# Reporting threshold is relative to each generator.
# ================================================================

def c8_active_support(beta, rel_tol=1e-6):

    beta = np.asarray(
        beta,
        dtype=float,
    )

    scale = max(
        np.max(
            np.abs(beta)
        ),
        np.finfo(float).tiny,
    )

    return np.flatnonzero(
        np.abs(beta)
        >
        rel_tol * scale
    )


C8_ACTIVE_G1 = c8_active_support(
    C8_BETA1
)

C8_ACTIVE_G2 = c8_active_support(
    C8_BETA2
)


# ================================================================
# 8. Verify the low-rank tensor reproduces the bracket FIELD
#
# We deliberately use generic probes, not only the observed orbit.
#
# Direct:
#
#       DG2 G1 - DG1 G2
#
# Tensor reconstruction:
#
#       sum_{a<b} A_ab B_ab
# ================================================================

C8_X_TEST = C7_X_PROBE[
    :64
]


C8_DIRECT_FIELDS = []
C8_TENSOR_FIELDS = []


for x in C8_X_TEST:

    F1 = rs_field_from_beta(
        x,
        C8_BETA1,
    )

    F2 = rs_field_from_beta(
        x,
        C8_BETA2,
    )

    J1 = rs_physical_jacobian(
        x,
        C8_BETA1,
    )

    J2 = rs_physical_jacobian(
        x,
        C8_BETA2,
    )


    direct = (
        J2 @ F1
        -
        J1 @ F2
    )


    # Build all 84 elementary fields/Jacobians once at x.
    F_basis = np.empty(
        (
            RS_N,
            C8_Q,
        ),
        dtype=float,
    )

    J_basis = np.empty(
        (
            RS_N,
            RS_N,
            C8_Q,
        ),
        dtype=float,
    )


    for q in range(C8_Q):

        beta_q = np.zeros(
            C8_Q,
            dtype=float,
        )

        beta_q[q] = 1.0

        F_basis[:, q] = (
            rs_field_from_beta(
                x,
                beta_q,
            )
        )

        J_basis[:, :, q] = (
            rs_physical_jacobian(
                x,
                beta_q,
            )
        )


    reconstructed = np.zeros(
        RS_N,
        dtype=float,
    )


    for k, (a, b) in enumerate(
        zip(
            C7_BRACKET_PARENT_A,
            C7_BRACKET_PARENT_B,
        )
    ):

        coeff = C8_RAW_COEFF[k]

        # Keep exact coefficients here.
        if coeff == 0.0:
            continue

        B_ab = (
            J_basis[:, :, b]
            @
            F_basis[:, a]
            -
            J_basis[:, :, a]
            @
            F_basis[:, b]
        )

        reconstructed += (
            coeff
            *
            B_ab
        )


    C8_DIRECT_FIELDS.append(
        direct
    )

    C8_TENSOR_FIELDS.append(
        reconstructed
    )


C8_DIRECT_FIELDS = np.asarray(
    C8_DIRECT_FIELDS
)

C8_TENSOR_FIELDS = np.asarray(
    C8_TENSOR_FIELDS
)


C8_FIELD_RECON_ERROR = (
    np.linalg.norm(
        C8_TENSOR_FIELDS
        -
        C8_DIRECT_FIELDS
    )
    /
    max(
        np.linalg.norm(
            C8_DIRECT_FIELDS
        ),
        np.finfo(float).tiny,
    )
)


# ================================================================
# 9. Structural reachability audit
#
# General first-bracket rule:
#
#     support([F_S,F_T]) subseteq S union T
#
# and nonzero requires overlap.
#
# Thus:
#
#   pair-pair   : max order 3
#   pair-triad  : max order 4
#   triad-triad : max order 5
#
# For a pairwise-only microscopic class, first-depth order >=4
# is therefore impossible.
# ================================================================

C8_PAIR_PAIR_4PLUS = int(
    np.sum(
        (
            np.asarray(
                C7_BRACKET_PARENT_TYPES,
                dtype=object,
            )
            ==
            "pair-pair"
        )
        &
        (
            C7_BRACKET_SUPPORT_SIZES
            >=
            4
        )
    )
)


# ================================================================
# 10. Diagnostics
# ================================================================

print("=" * 96)
print("RS-TSC PROTOTYPE — CELL 8")
print("CASE-2 LOW-RANK LIE-SECTOR / SUPPORT-REACHABILITY AUDIT")
print("=" * 96)


print()
print("-" * 96)
print("A. TEMPORAL COEFFICIENT MATRIX")
print("-" * 96)

print(
    f"A shape                         : "
    f"{C8_A.shape}"
)

print(
    f"antisymmetry relative error     : "
    f"{C8_ANTISYM_ERROR:.6e}"
)

print(
    f"rank at rel tol 1e-8            : "
    f"{C8_RANK_1E8}"
)

print(
    f"rank at rel tol 1e-10           : "
    f"{C8_RANK_1E10}"
)

print(
    f"rank at rel tol 1e-12           : "
    f"{C8_RANK_1E12}"
)

print(
    "leading relative singular values:"
)

print(
    C8_SINGULAR_REL[:10]
)


print()
print("-" * 96)
print("B. PARAMETERIZATION COMPRESSION")
print("-" * 96)

print(
    f"raw overlapping bracket channels : "
    f"{C7_N_BRACKET_RAW}"
)

print(
    f"linear first-closure dimension    : "
    f"{C7_CLOSURE_DIM}"
)

print(
    f"antisymmetric matrix entries      : "
    f"{C8_Q * (C8_Q - 1) // 2}"
)

print(
    f"rank-2 factorized representation  : "
    f"2 x {C8_Q} = {2 * C8_Q} raw factor parameters"
)

print(
    "NOTE: factor representation has gauge redundancy,"
)
print(
    "      so its intrinsic dimension is smaller than 168."
)


print()
print("-" * 96)
print("C. ACTUAL CASE-2 TEMPORAL SPARSITY")
print("-" * 96)

print(
    f"active microscopic channels G1  : "
    f"{len(C8_ACTIVE_G1)}"
)

print(
    f"active microscopic channels G2  : "
    f"{len(C8_ACTIVE_G2)}"
)

print(
    f"active raw Lie channels         : "
    f"{C8_N_ACTIVE_RAW} / {C7_N_BRACKET_RAW}"
)

print()

display(
    C8_ACTIVE_SUMMARY
)


print()
print("-" * 96)
print("D. LOW-RANK TENSOR -> BRACKET FIELD CHECK")
print("-" * 96)

print(
    f"generic probe states            : "
    f"{len(C8_X_TEST)}"
)

print(
    f"relative field reconstruction   : "
    f"{C8_FIELD_RECON_ERROR:.6e}"
)


print()
print("-" * 96)
print("E. SUPPORT-REACHABILITY RULE")
print("-" * 96)

print(
    "first Lie depth:"
)

print(
    "  pair-pair   -> support order <= 3"
)

print(
    "  pair-triad  -> support order <= 4"
)

print(
    "  triad-triad -> support order <= 5"
)

print()

print(
    f"pair-pair candidates with order >=4 : "
    f"{C8_PAIR_PAIR_4PLUS}"
)

print()

print(
    "Therefore, for a pairwise-only microscopic class:"
)

print(
    "  first-depth 4-body support is FORBIDDEN;"
)

print(
    "  4-body support requires at least a nested"
)

print(
    "  second-depth Lie bracket."
)


# ================================================================
# 11. Decision
# ================================================================

C8_PASS_RANK = (
    C8_RANK_1E10
    ==
    2
)

C8_PASS_FIELD = (
    C8_FIELD_RECON_ERROR
    <
    1e-10
)

C8_PASS_SUPPORT = (
    C8_PAIR_PAIR_4PLUS
    ==
    0
)


print()
print("=" * 96)
print("DECISION")
print("=" * 96)

print(
    "rank-2 temporal Lie sector      : "
    + (
        "PASS"
        if C8_PASS_RANK
        else
        "CHECK"
    )
)

print(
    "factorized bracket equivalence  : "
    + (
        "PASS"
        if C8_PASS_FIELD
        else
        "CHECK"
    )
)

print(
    "pairwise support-depth rule     : "
    + (
        "PASS"
        if C8_PASS_SUPPORT
        else
        "CHECK"
    )
)


# ================================================================
# 12. Save
# ================================================================

C8_ACTIVE_META.to_csv(
    RS_OUTPUT_DIR
    /
    "rs_cell8_case2_active_lie_channels.csv",
    index=False,
)


np.savez_compressed(
    RS_OUTPUT_DIR
    /
    "rs_cell8_case2_lowrank_lie_sector.npz",

    A=
        C8_A,

    A_BCH=
        C8_A_BCH,

    singular_values=
        C8_SINGULAR_VALUES,

    singular_values_relative=
        C8_SINGULAR_REL,

    active_G1=
        C8_ACTIVE_G1,

    active_G2=
        C8_ACTIVE_G2,

    raw_lie_coefficients=
        C8_RAW_COEFF,

    raw_bch_coefficients=
        C8_RAW_BCH_COEFF,
)


print()
print("=" * 96)
print("Cell 8 audit completed.")
print("=" * 96)

RS-TSC PROTOTYPE — CELL 8
CASE-2 LOW-RANK LIE-SECTOR / SUPPORT-REACHABILITY AUDIT

------------------------------------------------------------------------------------------------
A. TEMPORAL COEFFICIENT MATRIX
------------------------------------------------------------------------------------------------
A shape                         : (84, 84)
antisymmetry relative error     : 0.000000e+00
rank at rel tol 1e-8            : 2
rank at rel tol 1e-10           : 2
rank at rel tol 1e-12           : 2
leading relative singular values:
[1.00000000e+00 1.00000000e+00 7.89510839e-16 7.58644936e-16
 7.43034019e-16 6.30891406e-16 5.27334616e-16 5.24635340e-16
 4.86284372e-16 4.24275202e-16]

------------------------------------------------------------------------------------------------
B. PARAMETERIZATION COMPRESSION
------------------------------------------------------------------------------------------------
raw overlapping bracket channels : 2436
linear first-closure dimension    : 100

,parent_type,support_order,n_active_channels
0,pair-pair,3,88
1,pair-triad,3,52
2,pair-triad,4,262
3,triad-triad,4,39
4,triad-triad,5,86



------------------------------------------------------------------------------------------------
D. LOW-RANK TENSOR -> BRACKET FIELD CHECK
------------------------------------------------------------------------------------------------
generic probe states            : 64
relative field reconstruction   : 1.560466e-15

------------------------------------------------------------------------------------------------
E. SUPPORT-REACHABILITY RULE
------------------------------------------------------------------------------------------------
first Lie depth:
  pair-pair   -> support order <= 3
  pair-triad  -> support order <= 4
  triad-triad -> support order <= 5

pair-pair candidates with order >=4 : 0

Therefore, for a pairwise-only microscopic class:
  first-depth 4-body support is FORBIDDEN;
  4-body support requires at least a nested
  second-depth Lie bracket.

DECISION
rank-2 temporal Lie sector      : PASS
factorized bracket equivalence  : PASS
pairwise support-depth rule     : P

In [19]:
# ================================================================
# RS-TSC prototype
# Cell 9: theory-guided direct finite-flow inference solver
#
# MODEL
# ---------------------------------------------------------------
#
#   F_eff(x)
#     = G_mu(x)
#       + B_chr(G_mu, G_nu)(x)
#
# where
#
#   G_mu = sum_q mu_q Psi_q
#   G_nu = sum_q nu_q Psi_q
#
# and
#
#   B_chr(F,G)
#     = D G F - D F G.
#
#
# For a two-stage BCH1 protocol this is equivalent to
#
#   F_eff
#     = persistent sector
#       + rank-2 temporal Lie sector.
#
#
# IMPORTANT
# ---------------------------------------------------------------
# The solver NEVER uses:
#
#   RS_G1_BETA
#   RS_G2_BETA
#   true microscopic support
#
# Those may be used only AFTER inference for oracle diagnostics.
#
#
# FITTING STRATEGY
# ---------------------------------------------------------------
#
# Stage 0:
#   cheap secant/Ridge pilot ONLY as initialization.
#
# Stage 1:
#   direct nonlinear finite-flow fit of mu:
#
#       X_F ~= Phi_{G_mu}^{DeltaT}(X_0)
#
#   with analytic parameter sensitivities.
#
# Stage 2:
#   infer temporal coordinate nu from endpoint residuals using
#   finite-flow variational sensitivities:
#
#       residual ~= S_nu nu
#
#   adaptive LASSO + validation selection.
#
# Stage 3:
#   optional small active-set nonlinear joint refit of (mu,nu).
#
# ================================================================

import numpy as np

from scipy.optimize import least_squares
from sklearn.linear_model import Lasso


# ================================================================
# 0. Small helpers
# ================================================================

def rs9_rk4_step_augmented(
    rhs,
    y,
    dt,
):

    k1 = rhs(y)

    k2 = rhs(
        y
        +
        0.5 * dt * k1
    )

    k3 = rhs(
        y
        +
        0.5 * dt * k2
    )

    k4 = rhs(
        y
        +
        dt * k3
    )

    return (
        y
        +
        (dt / 6.0)
        *
        (
            k1
            +
            2.0 * k2
            +
            2.0 * k3
            +
            k4
        )
    )


def rs9_n_steps(
    DeltaT,
    dt,
):

    n_steps = int(
        round(
            DeltaT / dt
        )
    )

    if not np.isclose(
        n_steps * dt,
        DeltaT,
        atol=1e-12,
        rtol=0.0,
    ):

        raise ValueError(
            "DeltaT must be an integer multiple of dt."
        )

    return n_steps


def rs9_project_temporal_gauge(
    mu,
    nu,
):

    """
    Remove the gauge component nu -> nu + c mu.

    Since

        B_chr(G_mu, G_{nu+c mu})
        =
        B_chr(G_mu, G_nu),

    the component of nu parallel to mu is dynamically invisible.
    """

    mu = np.asarray(
        mu,
        dtype=float,
    )

    nu = np.asarray(
        nu,
        dtype=float,
    )

    denom = float(
        mu @ mu
    )

    if denom <= np.finfo(float).tiny:
        return nu.copy()

    return (
        nu
        -
        mu
        *
        float(
            mu @ nu
        )
        /
        denom
    )


# ================================================================
# 1. Main solver
# ================================================================

class RSTSCRank2FiniteFlowSolver:

    def __init__(
        self,
        DeltaT,
        dt,
        n_nodes=RS_N,
        n_channels=84,
    ):

        self.DeltaT = float(
            DeltaT
        )

        self.dt = float(
            dt
        )

        self.N = int(
            n_nodes
        )

        self.Q = int(
            n_channels
        )

        self.n_steps = rs9_n_steps(
            self.DeltaT,
            self.dt,
        )

        # one-hot microscopic coordinate vectors
        self._eye_beta = np.eye(
            self.Q,
            dtype=float,
        )


    # ============================================================
    # 2. Admissible microscopic interaction basis
    # ============================================================

    def basis_matrix(
        self,
        x,
    ):

        """
        Return

            Psi(x) : N x Q

        where column q is the complete vector-valued interaction
        candidate Psi_q(x).
        """

        x = np.asarray(
            x,
            dtype=float,
        )

        B = rs_build_physical_design(
            x[
                None,
                :
            ]
        )

        return np.asarray(
            B,
            dtype=float,
        ).reshape(
            self.N,
            self.Q,
        )


    def basis_jacobians(
        self,
        x,
    ):

        """
        Return

            J_basis[:,:,q]
                = D Psi_q(x)

        Shape:

            N x N x Q
        """

        x = np.asarray(
            x,
            dtype=float,
        )

        J = np.empty(
            (
                self.N,
                self.N,
                self.Q,
            ),
            dtype=float,
        )

        for q in range(
            self.Q
        ):

            J[
                :,
                :,
                q
            ] = rs_physical_jacobian(
                x,
                self._eye_beta[q],
            )

        return J


    # ============================================================
    # 3. Aggregate fields
    # ============================================================

    def G(
        self,
        x,
        beta,
    ):

        return rs_field_from_beta(
            x,
            beta,
        )


    def DG(
        self,
        x,
        beta,
    ):

        return rs_physical_jacobian(
            x,
            beta,
        )


    def temporal_bracket(
        self,
        x,
        mu,
        nu,
    ):

        """
        Chronological Lie field

            B_chr(G_mu,G_nu)
              = D G_nu G_mu
                - D G_mu G_nu.
        """

        G_mu = self.G(
            x,
            mu,
        )

        G_nu = self.G(
            x,
            nu,
        )

        J_mu = self.DG(
            x,
            mu,
        )

        J_nu = self.DG(
            x,
            nu,
        )

        return (
            J_nu @ G_mu
            -
            J_mu @ G_nu
        )


    def F_eff(
        self,
        x,
        mu,
        nu,
    ):

        return (
            self.G(
                x,
                mu,
            )
            +
            self.temporal_bracket(
                x,
                mu,
                nu,
            )
        )


    # ============================================================
    # 4. Forward integration
    # ============================================================

    def integrate_m0(
        self,
        x0,
        mu,
    ):

        x = np.asarray(
            x0,
            dtype=float,
        ).copy()

        for _ in range(
            self.n_steps
        ):

            def rhs(z):

                return self.G(
                    z,
                    mu,
                )

            x = rs9_rk4_step_augmented(
                rhs,
                x,
                self.dt,
            )

        return x


    def integrate_m1(
        self,
        x0,
        mu,
        nu,
    ):

        x = np.asarray(
            x0,
            dtype=float,
        ).copy()

        for _ in range(
            self.n_steps
        ):

            def rhs(z):

                return self.F_eff(
                    z,
                    mu,
                    nu,
                )

            x = rs9_rk4_step_augmented(
                rhs,
                x,
                self.dt,
            )

        return x


    # ============================================================
    # 5. M0 state + parameter sensitivity integration
    #
    # dS_mu/dt
    #   = D G_mu S_mu + Psi(x)
    #
    # so endpoint sensitivity is exact for the M0 finite flow.
    # ============================================================

    def integrate_m0_with_sensitivity(
        self,
        x0,
        mu,
    ):

        x0 = np.asarray(
            x0,
            dtype=float,
        )

        S0 = np.zeros(
            (
                self.N,
                self.Q,
            ),
            dtype=float,
        )

        y = np.concatenate(
            [
                x0,
                S0.ravel(),
            ]
        )


        def rhs(y_aug):

            x = y_aug[
                :self.N
            ]

            S = y_aug[
                self.N:
            ].reshape(
                self.N,
                self.Q,
            )

            F = self.G(
                x,
                mu,
            )

            J = self.DG(
                x,
                mu,
            )

            Psi = self.basis_matrix(
                x
            )

            dS = (
                J @ S
                +
                Psi
            )

            return np.concatenate(
                [
                    F,
                    dS.ravel(),
                ]
            )


        for _ in range(
            self.n_steps
        ):

            y = rs9_rk4_step_augmented(
                rhs,
                y,
                self.dt,
            )


        xF = y[
            :self.N
        ]

        SF = y[
            self.N:
        ].reshape(
            self.N,
            self.Q,
        )


        return (
            xF,
            SF,
        )


    # ============================================================
    # 6. Temporal endpoint sensitivity around M0
    #
    # Model:
    #
    #   F_eff
    #     = G_mu
    #       + sum_q nu_q H_q
    #
    # where
    #
    #   H_q
    #     = B_chr(G_mu, Psi_q)
    #
    #     = D Psi_q G_mu
    #       - D G_mu Psi_q.
    #
    #
    # Along the M0 trajectory:
    #
    #   dS_nu/dt
    #     = D G_mu S_nu + H(x).
    # ============================================================

    def temporal_forcing_matrix(
        self,
        x,
        mu,
    ):

        G_mu = self.G(
            x,
            mu,
        )

        J_mu = self.DG(
            x,
            mu,
        )

        Psi = self.basis_matrix(
            x
        )

        J_basis = self.basis_jacobians(
            x
        )


        # First term:
        #
        #   D Psi_q @ G_mu
        #
        first = np.einsum(
            "ijq,j->iq",
            J_basis,
            G_mu,
            optimize=True,
        )


        # Second term:
        #
        #   D G_mu @ Psi_q
        #
        second = (
            J_mu
            @
            Psi
        )


        return (
            first
            -
            second
        )


    def integrate_temporal_sensitivity(
        self,
        x0,
        mu,
    ):

        x0 = np.asarray(
            x0,
            dtype=float,
        )

        S0 = np.zeros(
            (
                self.N,
                self.Q,
            ),
            dtype=float,
        )

        y = np.concatenate(
            [
                x0,
                S0.ravel(),
            ]
        )


        def rhs(y_aug):

            x = y_aug[
                :self.N
            ]

            S = y_aug[
                self.N:
            ].reshape(
                self.N,
                self.Q,
            )

            F0 = self.G(
                x,
                mu,
            )

            J0 = self.DG(
                x,
                mu,
            )

            H = self.temporal_forcing_matrix(
                x,
                mu,
            )

            dS = (
                J0 @ S
                +
                H
            )

            return np.concatenate(
                [
                    F0,
                    dS.ravel(),
                ]
            )


        for _ in range(
            self.n_steps
        ):

            y = rs9_rk4_step_augmented(
                rhs,
                y,
                self.dt,
            )


        xF0 = y[
            :self.N
        ]

        Snu = y[
            self.N:
        ].reshape(
            self.N,
            self.Q,
        )


        return (
            xF0,
            Snu,
        )


    # ============================================================
    # 7. Window normalization
    #
    # Each window is normalized by its observed displacement.
    # This prevents later, more-relaxed cycles from being ignored.
    # ============================================================

    def window_scales(
        self,
        X0,
        XF,
        floor=1e-10,
    ):

        disp = (
            np.asarray(
                XF
            )
            -
            np.asarray(
                X0
            )
        )

        scales = np.linalg.norm(
            disp,
            axis=1,
        )

        return np.maximum(
            scales,
            floor,
        )


    # ============================================================
    # 8. Cheap initialization ONLY
    #
    # Secant approximation is never used as the final estimator.
    # It simply provides an initial point for nonlinear finite-flow
    # optimization.
    # ============================================================

    def secant_ridge_initialization(
        self,
        X0,
        XF,
        weights=None,
        ridge=1e-8,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )

        M = len(
            X0
        )

        if weights is None:

            weights = np.ones(
                M,
                dtype=float,
            )

        weights = np.asarray(
            weights,
            dtype=float,
        )

        scales = self.window_scales(
            X0,
            XF,
        )


        A_blocks = []
        y_blocks = []


        for r in range(M):

            Psi = self.basis_matrix(
                X0[r]
            )

            velocity = (
                XF[r]
                -
                X0[r]
            ) / self.DeltaT


            fac = (
                np.sqrt(
                    weights[r]
                )
                /
                scales[r]
            )


            A_blocks.append(
                fac
                *
                Psi
            )

            y_blocks.append(
                fac
                *
                velocity
            )


        A = np.vstack(
            A_blocks
        )

        y = np.concatenate(
            y_blocks
        )


        ATA = (
            A.T @ A
            +
            ridge
            *
            np.eye(
                self.Q
            )
        )

        ATy = (
            A.T @ y
        )


        mu0 = np.linalg.solve(
            ATA,
            ATy,
        )

        return mu0


    # ============================================================
    # 9. Direct finite-flow M0 fit
    # ============================================================

    def fit_m0(
        self,
        X0,
        XF,
        weights=None,
        mu0=None,
        ridge=1e-8,
        max_nfev=80,
        verbose=0,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )

        M = len(
            X0
        )

        if weights is None:

            weights = np.ones(
                M,
                dtype=float,
            )

        weights = np.asarray(
            weights,
            dtype=float,
        )

        scales = self.window_scales(
            X0,
            XF,
        )


        if mu0 is None:

            mu0 = (
                self.secant_ridge_initialization(
                    X0,
                    XF,
                    weights=weights,
                    ridge=max(
                        ridge,
                        1e-10,
                    ),
                )
            )


        cache = {
            "mu": None,
            "residual": None,
            "jacobian": None,
        }


        def evaluate(mu):

            if (
                cache["mu"]
                is not None
                and
                np.array_equal(
                    mu,
                    cache["mu"],
                )
            ):

                return (
                    cache["residual"],
                    cache["jacobian"],
                )


            residual_blocks = []
            jacobian_blocks = []


            for r in range(M):

                x_pred, S_mu = (
                    self.integrate_m0_with_sensitivity(
                        X0[r],
                        mu,
                    )
                )


                fac = (
                    np.sqrt(
                        weights[r]
                    )
                    /
                    scales[r]
                )


                residual_blocks.append(
                    fac
                    *
                    (
                        x_pred
                        -
                        XF[r]
                    )
                )

                jacobian_blocks.append(
                    fac
                    *
                    S_mu
                )


            residual = np.concatenate(
                residual_blocks
            )

            jacobian = np.vstack(
                jacobian_blocks
            )


            if ridge > 0.0:

                residual = np.concatenate(
                    [
                        residual,
                        np.sqrt(
                            ridge
                        )
                        *
                        mu,
                    ]
                )

                jacobian = np.vstack(
                    [
                        jacobian,
                        np.sqrt(
                            ridge
                        )
                        *
                        np.eye(
                            self.Q
                        ),
                    ]
                )


            cache["mu"] = mu.copy()
            cache["residual"] = residual
            cache["jacobian"] = jacobian


            return (
                residual,
                jacobian,
            )


        result = least_squares(

            fun=lambda mu:
                evaluate(mu)[0],

            jac=lambda mu:
                evaluate(mu)[1],

            x0=np.asarray(
                mu0,
                dtype=float,
            ),

            method="trf",

            x_scale="jac",

            max_nfev=int(
                max_nfev
            ),

            ftol=1e-10,
            xtol=1e-10,
            gtol=1e-10,

            verbose=int(
                verbose
            ),
        )


        return {
            "mu":
                result.x.copy(),

            "result":
                result,

            "initial_mu":
                np.asarray(
                    mu0
                ).copy(),
        }


    # ============================================================
    # 10. Build temporal linear inverse problem
    #
    #     R ~= S_nu nu
    # ============================================================

    def build_temporal_problem(
        self,
        X0,
        XF,
        mu,
        weights=None,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )

        M = len(
            X0
        )

        if weights is None:

            weights = np.ones(
                M,
                dtype=float,
            )

        weights = np.asarray(
            weights,
            dtype=float,
        )

        scales = self.window_scales(
            X0,
            XF,
        )


        residual_blocks = []
        sensitivity_blocks = []
        m0_predictions = []


        for r in range(M):

            xF0, Snu = (
                self.integrate_temporal_sensitivity(
                    X0[r],
                    mu,
                )
            )


            fac = (
                np.sqrt(
                    weights[r]
                )
                /
                scales[r]
            )


            # Observed minus M0 prediction.
            residual_blocks.append(
                fac
                *
                (
                    XF[r]
                    -
                    xF0
                )
            )

            sensitivity_blocks.append(
                fac
                *
                Snu
            )

            m0_predictions.append(
                xF0
            )


        return {

            "residual":
                np.concatenate(
                    residual_blocks
                ),

            "sensitivity":
                np.vstack(
                    sensitivity_blocks
                ),

            "m0_predictions":
                np.asarray(
                    m0_predictions
                ),
        }


    # ============================================================
    # 11. Adaptive sparse temporal solve
    #
    # Validation is performed using the FULL nonlinear M1 flow,
    # not the linearized residual approximation.
    # ============================================================

    def fit_temporal_adaptive_lasso(
        self,
        X0_fit,
        XF_fit,
        X0_val,
        XF_val,
        mu,
        weights_fit=None,
        weights_val=None,
        ridge_pilot=1e-8,
        adaptive_gamma=1.0,
        n_lambda=30,
        lambda_ratio_min=1e-5,
        lasso_max_iter=50000,
    ):

        problem_fit = (
            self.build_temporal_problem(
                X0_fit,
                XF_fit,
                mu,
                weights=weights_fit,
            )
        )


        S = problem_fit[
            "sensitivity"
        ]

        r = problem_fit[
            "residual"
        ]


        # --------------------------------------------------------
        # Column scaling
        # --------------------------------------------------------

        col_scale = np.sqrt(
            np.mean(
                S ** 2,
                axis=0,
            )
        )

        col_scale = np.maximum(
            col_scale,
            1e-14,
        )

        S_std = (
            S
            /
            col_scale[
                None,
                :
            ]
        )


        # --------------------------------------------------------
        # Ridge pilot
        # --------------------------------------------------------

        ridge_matrix = (
            S_std.T
            @
            S_std
            +
            ridge_pilot
            *
            np.eye(
                self.Q
            )
        )

        ridge_rhs = (
            S_std.T
            @
            r
        )

        z_ridge = np.linalg.solve(
            ridge_matrix,
            ridge_rhs,
        )

        nu_ridge = (
            z_ridge
            /
            col_scale
        )


        # --------------------------------------------------------
        # Adaptive weights in ORIGINAL coefficient coordinates
        # --------------------------------------------------------

        pilot_scale = max(
            float(
                np.max(
                    np.abs(
                        nu_ridge
                    )
                )
            ),
            1e-12,
        )

        adaptive_eps = (
            1e-4
            *
            pilot_scale
        )

        adaptive_weight = (
            1.0
            /
            (
                np.abs(
                    nu_ridge
                )
                +
                adaptive_eps
            )
            **
            adaptive_gamma
        )


        # Transform
        #
        #   theta_q
        #      = adaptive_weight_q * nu_q
        #
        # so standard LASSO acts on theta.
        #
        X_lasso = (
            S
            /
            adaptive_weight[
                None,
                :
            ]
        )


        n_obs = len(
            r
        )

        lambda_max = (
            np.max(
                np.abs(
                    X_lasso.T
                    @
                    r
                )
            )
            /
            max(
                n_obs,
                1,
            )
        )


        lambda_max = max(
            float(
                lambda_max
            ),
            1e-14,
        )


        lambda_grid = np.geomspace(
            lambda_max,
            lambda_max
            *
            lambda_ratio_min,
            int(
                n_lambda
            ),
        )


        records = []
        candidates = []


        for lam in lambda_grid:

            model = Lasso(

                alpha=float(
                    lam
                ),

                fit_intercept=False,

                max_iter=int(
                    lasso_max_iter
                ),

                tol=1e-10,

                selection="cyclic",
            )


            model.fit(
                X_lasso,
                r,
            )


            theta = model.coef_

            nu = (
                theta
                /
                adaptive_weight
            )


            # Remove dynamically invisible gauge direction.
            nu = rs9_project_temporal_gauge(
                mu,
                nu,
            )


            # Full nonlinear validation prediction.
            X_val_pred = np.asarray(
                [
                    self.integrate_m1(
                        x0,
                        mu,
                        nu,
                    )

                    for x0 in X0_val
                ]
            )


            val_error = (
                self.endpoint_relative_error(
                    X0_val,
                    XF_val,
                    X_val_pred,
                    weights=weights_val,
                )
            )


            fit_linear_error = (
                np.linalg.norm(
                    S @ nu
                    -
                    r
                )
                /
                max(
                    np.linalg.norm(
                        r
                    ),
                    np.finfo(float).tiny,
                )
            )


            n_active = int(
                np.sum(
                    np.abs(
                        nu
                    )
                    >
                    1e-10
                    *
                    max(
                        np.max(
                            np.abs(
                                nu
                            )
                        ),
                        1.0,
                    )
                )
            )


            records.append({

                "lambda":
                    float(lam),

                "n_active":
                    n_active,

                "fit_linear_relative_error":
                    float(
                        fit_linear_error
                    ),

                "validation_fullflow_error":
                    float(
                        val_error
                    ),
            })


            candidates.append(
                nu
            )


        validation_errors = np.asarray(
            [
                row[
                    "validation_fullflow_error"
                ]

                for row in records
            ]
        )


        best_index = int(
            np.argmin(
                validation_errors
            )
        )


        nu_best = (
            candidates[
                best_index
            ]
            .copy()
        )


        return {

            "nu":
                nu_best,

            "best_index":
                best_index,

            "best_lambda":
                float(
                    lambda_grid[
                        best_index
                    ]
                ),

            "records":
                records,

            "ridge_pilot":
                nu_ridge,

            "adaptive_weight":
                adaptive_weight,

            "fit_problem":
                problem_fit,
        }


    # ============================================================
    # 12. Endpoint prediction / error
    # ============================================================

    def predict(
        self,
        X0,
        mu,
        nu=None,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )


        if nu is None:

            return np.asarray(
                [
                    self.integrate_m0(
                        x0,
                        mu,
                    )

                    for x0 in X0
                ]
            )


        return np.asarray(
            [
                self.integrate_m1(
                    x0,
                    mu,
                    nu,
                )

                for x0 in X0
            ]
        )


    def endpoint_relative_error(
        self,
        X0,
        XF,
        X_pred,
        weights=None,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )

        X_pred = np.asarray(
            X_pred,
            dtype=float,
        )

        M = len(
            X0
        )


        if weights is None:

            weights = np.ones(
                M,
                dtype=float,
            )

        weights = np.asarray(
            weights,
            dtype=float,
        )


        sqrt_w = np.sqrt(
            weights
        )[
            :,
            None
        ]


        numerator = np.linalg.norm(
            sqrt_w
            *
            (
                X_pred
                -
                XF
            )
        )

        denominator = np.linalg.norm(
            sqrt_w
            *
            (
                XF
                -
                X0
            )
        )


        return float(
            numerator
            /
            max(
                denominator,
                np.finfo(float).tiny,
            )
        )


    # ============================================================
    # 13. Small active-set joint nonlinear refit
    #
    # Numerical Jacobian is acceptable here because the working
    # set should already be small.
    # ============================================================

    def joint_active_refit(
        self,
        X0,
        XF,
        mu,
        nu,
        weights=None,
        mu_rel_threshold=5e-3,
        nu_rel_threshold=1e-4,
        max_nfev=100,
    ):

        mu = np.asarray(
            mu,
            dtype=float,
        )

        nu = np.asarray(
            nu,
            dtype=float,
        )


        mu_scale = max(
            np.max(
                np.abs(
                    mu
                )
            ),
            1e-14,
        )

        nu_scale = max(
            np.max(
                np.abs(
                    nu
                )
            ),
            1e-14,
        )


        active_mu = np.flatnonzero(
            np.abs(
                mu
            )
            >=
            mu_rel_threshold
            *
            mu_scale
        )

        active_nu = np.flatnonzero(
            np.abs(
                nu
            )
            >=
            nu_rel_threshold
            *
            nu_scale
        )


        p0 = np.concatenate(
            [
                mu[
                    active_mu
                ],
                nu[
                    active_nu
                ],
            ]
        )


        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )


        if weights is None:

            weights = np.ones(
                len(X0),
                dtype=float,
            )

        weights = np.asarray(
            weights,
            dtype=float,
        )


        scales = self.window_scales(
            X0,
            XF,
        )


        def unpack(p):

            mu_full = np.zeros(
                self.Q,
                dtype=float,
            )

            nu_full = np.zeros(
                self.Q,
                dtype=float,
            )


            n_mu = len(
                active_mu
            )


            mu_full[
                active_mu
            ] = p[
                :n_mu
            ]

            nu_full[
                active_nu
            ] = p[
                n_mu:
            ]


            return (
                mu_full,
                nu_full,
            )


        def residual(p):

            mu_full, nu_full = unpack(
                p
            )


            blocks = []


            for r in range(
                len(X0)
            ):

                pred = self.integrate_m1(
                    X0[r],
                    mu_full,
                    nu_full,
                )


                fac = (
                    np.sqrt(
                        weights[r]
                    )
                    /
                    scales[r]
                )


                blocks.append(
                    fac
                    *
                    (
                        pred
                        -
                        XF[r]
                    )
                )


            return np.concatenate(
                blocks
            )


        result = least_squares(

            residual,

            p0,

            jac="2-point",

            method="trf",

            x_scale="jac",

            max_nfev=int(
                max_nfev
            ),

            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
        )


        mu_refit, nu_refit = unpack(
            result.x
        )


        return {

            "mu":
                mu_refit,

            "nu":
                nu_refit,

            "active_mu":
                active_mu,

            "active_nu":
                active_nu,

            "result":
                result,
        }


# ================================================================
# 14. Instantiate solver for current Case 2
# ================================================================

RS_TSC_RANK2_SOLVER = (
    RSTSCRank2FiniteFlowSolver(

        DeltaT=C2_DELTA_T,

        dt=RS_DT_OBS,

        n_nodes=RS_N,

        n_channels=84,
    )
)


print("=" * 86)
print("RS-TSC PROTOTYPE — CELL 9")
print("TSC RANK-2 DIRECT FINITE-FLOW SOLVER")
print("=" * 86)

print(
    f"N                     : "
    f"{RS_TSC_RANK2_SOLVER.N}"
)

print(
    f"admissible channels   : "
    f"{RS_TSC_RANK2_SOLVER.Q}"
)

print(
    f"DeltaT                : "
    f"{RS_TSC_RANK2_SOLVER.DeltaT:.6f}"
)

print(
    f"integration dt        : "
    f"{RS_TSC_RANK2_SOLVER.dt:.6f}"
)

print(
    f"RK4 steps / window    : "
    f"{RS_TSC_RANK2_SOLVER.n_steps}"
)

print()

print(
    "model:"
)

print(
    "  F_eff = G_mu + B_chr(G_mu, G_nu)"
)

print()

print(
    "nominal coordinates  : 84 mu + 84 nu"
)

print(
    "temporal gauge        : nu -> nu + c mu"
)

print(
    "effective intrinsic   : <= 167 coordinates"
)

print()

print(
    "solver definition     : READY"
)

print(
    "oracle topology used  : NO"
)

print("=" * 86)

RS-TSC PROTOTYPE — CELL 9
TSC RANK-2 DIRECT FINITE-FLOW SOLVER
N                     : 8
admissible channels   : 84
DeltaT                : 0.030000
integration dt        : 0.000500
RK4 steps / window    : 60

model:
  F_eff = G_mu + B_chr(G_mu, G_nu)

nominal coordinates  : 84 mu + 84 nu
temporal gauge        : nu -> nu + c mu
effective intrinsic   : <= 167 coordinates

solver definition     : READY
oracle topology used  : NO
